# Week 2: adapting pricing to the emerging market

I am Manuel Rodríguez, an actuarial student and individual participant in the IAA AI Hackathon 2026. In this notebook I explain and reconstruct my Week 2 adaptation, developed with AI assistance.

I estimate the 2025 ultimate cost for each emerging-market policy period. The developed market provides a reference; I keep joins within each market. The separate commercial premium exercise applies expense assumptions to twenty reference policies and does not change the risk-premium submission.


## Notebook guide

| Purpose | What I reconstruct |
| --- | --- |
| Selected recipe | F3c, my Week 2 submission |
| Required inputs | 11 developed-market and 13 emerging-market Parquet files from Part 2 |
| Final exports | `submission_week2_F3c.csv` and `commercial_reference_premiums.csv` |
| External model or repository download | None; source and frozen configuration are embedded |

For a first reading, I follow the [method](#modeling-and-evidence), [execution plan](#inspect-the-execution-plan) and [output checks](#check-and-save-the-risk-premium-csv). The [embedded source package](#embedded-source-package) preserves the individual programs for inspection. Its source cells are marked as hidden in JupyterLab's presentation metadata; I can expand them. GitHub may display them in full.

For execution, I attach the data, check [dependencies and storage](#dependencies-and-run-workspace) and run cells in order. The later [recipe execution](#run-the-frozen-stages) trains models and can take hours. Reading the notebook does not require starting that execution.

A fresh local execution of the preceding frozen edition completed all 91 code cells and 48 stages and recovered the historical F3c CSV exactly. Its twenty commercial premiums and fixed-expense sensitivities also matched the report source. This English edition uses neutral runtime names and revised comments/docstrings. Its 123 embedded sources across both notebooks passed normalized executable-structure checks; the data and path checks are documented in the [source review](../registro/documentacion_codigo_actual.json). Those checks are separate from another full Run All of this edition. Hosted Kaggle execution and independent professional review remain unverified.


## Scope and reproducibility

I rebuild the historical F3c risk-premium chain from the official Parquet inputs, including its developed-market T2b models, A1 transfer, public-probe calibration and F3c adjustment. I then calculate the twenty commercial reference premiums. The embedded code and frozen aggregate constants are part of this notebook; no repository checkout, previous model, prediction file or team script is required.

This is a notebook adaptation of my historical reconstruction, not an independent professional validation. Its full execution on Kaggle has not yet been verified. Numerical differences can arise from library versions and hardware; a hash mismatch must be investigated rather than presented as an exact reproduction.

Attach Part 2 and run the cells in order. The input files remain read-only. I keep intermediate data, models and logs in a fresh temporary directory outside `/kaggle/working`; only final CSVs go into `/kaggle/working`. The temporary directory does not persist with the notebook output. Keep the row-level final outputs private under the competition rules.


## Attach the competition data

I use the eleven developed-market files and thirteen emerging-market files from Part 2. I resolve the folders in this order: explicit market variables (`IAA_W2_DEVELOPED` and `IAA_W2_EMERGING`), an optional `IAA_DATA_ROOT`, the local workspace's `RUTAS_PROYECTO.json`, then attached data under `/kaggle/input`.

In my local project, I can open this notebook from the workspace without entering machine-specific paths. `IAA_PROJECT_ROOT` identifies that workspace when the kernel starts elsewhere. For another local checkout I set the market variables to the actual Parquet directories, or point `IAA_DATA_ROOT` to their common parent. In Kaggle I attach the corresponding competition; folder discovery stops if multiple candidates are possible.

Before training, I check every required file against the frozen sizes and SHA-256 fingerprints of my archived official downloads, then check the template's identifiers and order against the test files. Week 2 validates both markets. An incorrect path, missing file, mixed market or different fingerprint stops execution. These hashes identify the downloaded bytes; they are not represented as hashes published by Kaggle.

Only after these checks pass do I create the run folder. Its `input_manifest.json` records the resolved paths, input fingerprints and template counts. I keep inputs read-only and prevent output directories inside them.


## Optional dependency installation

Kaggle usually includes these libraries. If the environment does not match the recorded reference, set the switch to `True` and run this cell with Internet enabled. Restart the kernel after installation, set the switch back to `False`, and run from the top. Keep the same Kaggle session so the installed packages remain available. The reference versions may not support every Kaggle Python image.


In [ ]:
INSTALL_REFERENCE_PACKAGES = False
if INSTALL_REFERENCE_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==2.5.3", "pandas==3.0.5", "scipy==1.18.1", "scikit-learn==1.9.1", "lightgbm==4.7.0", "pyarrow==26.0.0"])
    raise RuntimeError("Packages installed. Restart the kernel, turn installation off, and run all cells again.")


In [ ]:
import os, sys, hashlib, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

for package in ("numpy", "pandas", "pyarrow"):
    print(f"{package}: {importlib.metadata.version(package)}")
print("Python:", sys.version)

DEVELOPED_DIR = os.environ.get("IAA_W2_DEVELOPED", "")  # e.g. /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/developed
EMERGING_DIR = os.environ.get("IAA_W2_EMERGING", "")  # e.g. /kaggle/input/competitions/iaa-ai-assisted-hackathon-2026-part-2/emerging


In [ ]:
"""Resolve official notebook inputs and validate them without training models."""
from pathlib import Path
import hashlib
import json
import os

LINES = ('motor', 'buildings', 'contents')
BASE_FILES = ['addresses.parquet', 'claims.parquet', 'customers.parquet',
              'weather.parquet', 'sample_submission.parquet'] + [
    f'{prefix}_{line}.parquet' for prefix in ('policies', 'test') for line in LINES]
INPUT_CONTRACT = {}
NOTEBOOK_WEEK = None
_RESOLVED_INPUTS = {}
_VERIFIED_INPUTS = {}
_VERIFIED_TEMPLATES = {}
_VARIABLES = {'public': 'IAA_W1_DATA', 'developed': 'IAA_W2_DEVELOPED',
              'emerging': 'IAA_W2_EMERGING'}
_ROUTE_KEYS = {'public': 'week_1_data', 'developed': 'week_2_developed_reference',
               'emerging': 'week_2_emerging_data'}


def _directory(value):
    return Path(value).expanduser().resolve()


def _missing(folder, required):
    return [name for name in required if not (folder / name).is_file()]


def _discover(root, required, market):
    """Search only an explicitly chosen data tree or the Kaggle input mount."""
    candidates = set()
    if not root.is_dir():
        return []
    for current, directories, filenames in os.walk(root, followlinks=False):
        folder = Path(current)
        if len(folder.relative_to(root).parts) >= 6:
            directories[:] = []
        else:
            directories[:] = [d for d in directories if not d.startswith('.')]
        if 'sample_submission.parquet' not in filenames or _missing(folder, required):
            continue
        # The emerging folder also contains the eleven base filenames.
        if market != 'emerging' and (folder / 'reference_policies.parquet').is_file():
            continue
        candidates.add(folder.resolve())
    return sorted(candidates, key=str)


def _project_folder(market):
    explicit = os.environ.get('IAA_PROJECT_ROOT', '')
    if explicit:
        configs = [_directory(explicit) / 'RUTAS_PROYECTO.json']
        if not configs[0].is_file():
            raise RuntimeError(f'IAA_PROJECT_ROOT has no RUTAS_PROYECTO.json: {configs[0]}')
    else:
        start = Path.cwd().resolve()
        configs = [p / 'RUTAS_PROYECTO.json' for p in (start, *start.parents)
                   if (p / 'RUTAS_PROYECTO.json').is_file()][:1]
    if not configs:
        return None
    config = configs[0]
    routes = json.loads(config.read_text(encoding='utf-8-sig'))
    value = routes.get(_ROUTE_KEYS[market])
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f'{config.name} lacks the data route {_ROUTE_KEYS[market]}')
    return (config.parent / value).resolve()


def find_market(market, required, override):
    """Resolve one market, never silently select between multiple candidates."""
    if market not in INPUT_CONTRACT or market not in _VARIABLES:
        raise RuntimeError(f'No frozen input contract for {market!r}')
    origin = _VARIABLES[market]
    if override:
        candidates = [_directory(override)]
    elif os.environ.get('IAA_DATA_ROOT'):
        origin = 'IAA_DATA_ROOT'
        candidates = _discover(_directory(os.environ['IAA_DATA_ROOT']), required, market)
    else:
        project = _project_folder(market)
        origin = 'RUTAS_PROYECTO.json' if project is not None else '/kaggle/input'
        candidates = [project] if project is not None else _discover(Path('/kaggle/input'), required, market)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one {market} directory via {origin}; found '
                           f'{[str(p) for p in candidates]}. Set {_VARIABLES[market]} '
                           'to the directory containing the required Parquet files.')
    folder = candidates[0]
    missing = _missing(folder, required)
    if missing:
        raise RuntimeError(f'Missing {market} files in {folder}: {", ".join(missing)}. '
                           f'Check {_VARIABLES[market]}. No fallback was selected.')
    if market != 'emerging' and (folder / 'reference_policies.parquet').is_file():
        raise RuntimeError(f'{folder} contains emerging-market reference policies; '
                           f'check {_VARIABLES[market]} instead of mixing markets.')
    _RESOLVED_INPUTS[str(folder)] = {'market': market, 'origin': origin}
    _VERIFIED_INPUTS.pop(market, None)
    _VERIFIED_TEMPLATES.pop(market, None)
    return folder


def inventory(folder, filenames):
    import pandas as pd
    import pyarrow.parquet as pq
    folder = _directory(folder)
    binding = _RESOLVED_INPUTS.get(str(folder))
    if binding is None:
        raise RuntimeError('Resolve this input with find_market before inventory.')
    market = binding['market']
    _VERIFIED_INPUTS.pop(market, None)
    _VERIFIED_TEMPLATES.pop(market, None)
    expected = INPUT_CONTRACT[market]
    if set(filenames) != set(expected):
        raise RuntimeError(f'File list does not match the frozen {market} contract')
    rows = []
    for name in filenames:
        path = folder / name
        before = path.stat()
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(chunk)
        actual = digest.hexdigest()
        if before.st_size != expected[name]['bytes'] or actual != expected[name]['sha256']:
            raise RuntimeError(f'Official-download fingerprint mismatch: {market}/{name}. '
                               'Check the input version; training has not started.')
        parquet = pq.ParquetFile(path)
        after = path.stat()
        if (before.st_size, before.st_mtime_ns) != (after.st_size, after.st_mtime_ns):
            raise RuntimeError(f'Input changed while being checked: {path}')
        rows.append({'file': name, 'rows': parquet.metadata.num_rows,
                     'columns': len(parquet.schema_arrow), 'bytes': after.st_size,
                     'sha256': actual, 'matches_download': True,
                     'mtime_ns': after.st_mtime_ns})
    _VERIFIED_INPUTS[market] = {'directory': str(folder), 'origin': binding['origin'], 'files': rows}
    return pd.DataFrame([{k: v for k, v in row.items() if k != 'mtime_ns'} for row in rows])


def check_template(folder):
    import pandas as pd
    folder = _directory(folder)
    market = _RESOLVED_INPUTS[str(folder)]['market']
    _VERIFIED_TEMPLATES.pop(market, None)
    if market not in _VERIFIED_INPUTS:
        raise RuntimeError('Verify input fingerprints before checking the template.')
    sample = pd.read_parquet(folder / 'sample_submission.parquet')
    test = pd.concat([pd.read_parquet(folder / f'test_{line}.parquet', columns=['row_id'])
                      for line in LINES], ignore_index=True)
    if ('row_id' not in sample or not sample.row_id.notna().all()
            or not test.row_id.notna().all() or not test.row_id.is_unique
            or not sample.row_id.is_unique or not sample.row_id.equals(test.row_id)):
        raise RuntimeError(f'{market}: template and test identifiers/order do not agree')
    _VERIFIED_TEMPLATES[market] = len(sample)
    print(f'{market}: template matches test identifiers in order ({len(sample):,} rows)')
    return sample


def _safe_output_root(root):
    root = _directory(root)
    for binding in _RESOLVED_INPUTS:
        source = Path(binding)
        if root == source or source in root.parents:
            raise RuntimeError(f'Output directory is inside read-only input data: {root}')
    return root


def prepare_workspace(week):
    """Create a run only after all input contracts and templates have passed."""
    import tempfile
    import uuid
    from datetime import datetime, timezone
    if week not in (1, 2) or week != NOTEBOOK_WEEK:
        raise RuntimeError('Workspace week does not match the notebook input contract')
    required = {'public'} if week == 1 else {'developed', 'emerging'}
    if not required.issubset(_VERIFIED_INPUTS) or not required.issubset(_VERIFIED_TEMPLATES):
        raise RuntimeError('Run the input inventory and template cells before creating the workspace.')
    for market in required:
        for row in _VERIFIED_INPUTS[market]['files']:
            path = Path(_VERIFIED_INPUTS[market]['directory']) / row['file']
            stat = path.stat()
            if (stat.st_size, stat.st_mtime_ns) != (row['bytes'], row['mtime_ns']):
                raise RuntimeError(f'Input changed after validation: {market}/{row["file"]}')
    work_root = _safe_output_root(os.environ.get('IAA_NOTEBOOK_WORK') or tempfile.gettempdir())
    export_value = os.environ.get('IAA_NOTEBOOK_EXPORT', '')
    export_root = _safe_output_root(export_value) if export_value else (
        Path('/kaggle/working') if Path('/kaggle/working').is_dir() else None)
    run = work_root / ('iaa_week' + str(week) + '_' + uuid.uuid4().hex[:12])
    run.mkdir(parents=True, exist_ok=False)
    export = (export_root if export_root is not None else run) / (run.name + '_exports')
    export.mkdir(parents=True, exist_ok=False)
    receipt = {'utc': datetime.now(timezone.utc).isoformat(), 'week': week,
               'inputs': {m: _VERIFIED_INPUTS[m] for m in sorted(required)},
               'template_rows': {m: _VERIFIED_TEMPLATES[m] for m in sorted(required)},
               'training_started': False}
    (run / 'input_manifest.json').write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
    print('New run:', run)
    print('Final outputs:', export)
    return run, export



NOTEBOOK_WEEK = 2
INPUT_CONTRACT = {'developed': {'addresses.parquet': {'bytes': 8209304, 'sha256': '2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1'}, 'claims.parquet': {'bytes': 6393494, 'sha256': '323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0'}, 'customers.parquet': {'bytes': 9840123, 'sha256': '065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb'}, 'policies_buildings.parquet': {'bytes': 28411001, 'sha256': 'eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe'}, 'policies_contents.parquet': {'bytes': 28203870, 'sha256': '8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd'}, 'policies_motor.parquet': {'bytes': 44218528, 'sha256': '4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e'}, 'sample_submission.parquet': {'bytes': 3271617, 'sha256': '9529f3b6a679d3410350baa7c2ef3fb085d1d8843a838360c5852dfe820d4d95'}, 'test_buildings.parquet': {'bytes': 7917305, 'sha256': '7b2b96507de29241c70617c17df66146ed30b51f826ce5219bbda5da536d4957'}, 'test_contents.parquet': {'bytes': 7795903, 'sha256': 'c2212f62f3b7a21431c0073277549c3db5e63857565a8b88855e79a65fa1b8b6'}, 'test_motor.parquet': {'bytes': 12035693, 'sha256': '8d28fb9fd091caf2a044913a013592226a4c0917d5c6fcc9c5e8cdcec2ab4fc5'}, 'weather.parquet': {'bytes': 41863008, 'sha256': 'b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49'}}, 'emerging': {'addresses.parquet': {'bytes': 2094327, 'sha256': '4e9bf9d2a1777896e84fd4ee471096ff075aaefefd544f91b731324b25425632'}, 'claims.parquet': {'bytes': 373376, 'sha256': '1a2fce8a26cb1e05de1f53fe4e39bde772a1898e845755487625355fec4cc0e9'}, 'customers.parquet': {'bytes': 2319251, 'sha256': 'c00a0095d5fb7b2b65cd9647216add49afb3703d863d5c358b1441f0fab1fb62'}, 'expenses.parquet': {'bytes': 3124, 'sha256': '9033f666a5a45f0b44501d57e40ab1cc571c51c384f23a62942b75db323b1cdf'}, 'policies_buildings.parquet': {'bytes': 1049236, 'sha256': 'ccc948fafc11e623d5cb010c2b494f6813f6048719eae7bcde28794467598319'}, 'policies_contents.parquet': {'bytes': 1272728, 'sha256': '84d7f48d03933b018bec8fdc2f4bbd96df184cca7ae1971c38d3acc955720f8e'}, 'policies_motor.parquet': {'bytes': 1988978, 'sha256': '7c5dbb5e5b46629ebf01a66c991183d20b8585393b589bc2644fcbd6653a3ec1'}, 'reference_policies.parquet': {'bytes': 30934, 'sha256': '6d4c2839e87d092fccd4b3c3ddc16a2a4c4993c35e18481e322462cd382dd8f4'}, 'sample_submission.parquet': {'bytes': 1197460, 'sha256': '160fc59043f1012337605c8d8446c0efbf81458e046cf02df9186d6f909d59ef'}, 'test_buildings.parquet': {'bytes': 2507817, 'sha256': '21f91aaa66a1b9efd216ad0e04ff12a507dbeb721084330d9b9f06c99edd0ca6'}, 'test_contents.parquet': {'bytes': 3038763, 'sha256': 'af5ef2553e35a2d563ca091b173500daac55acc4969d1d3a96fa6b4ec3c6a012'}, 'test_motor.parquet': {'bytes': 4626510, 'sha256': '108d5c68e9bab5ccd90ee5502bc3543aa515a5aa07fda44a3b227cf070a28aa5'}, 'weather.parquet': {'bytes': 6060339, 'sha256': '645067069f68de8c4746d1ba48a454ff00155e1aa6fbbf64cdee48ae7269cbbd'}}}


In [ ]:
emerging_files = BASE_FILES + ["expenses.parquet", "reference_policies.parquet"]
developed = find_market("developed", BASE_FILES, DEVELOPED_DIR)
emerging = find_market("emerging", emerging_files, EMERGING_DIR)
for name, folder, files in (("developed", developed, BASE_FILES), ("emerging", emerging, emerging_files)):
    print(name, folder)
    display(inventory(folder, files))
developed_sample = check_template(developed)
sample = check_template(emerging)


## Modeling and evidence

I use claims valued at 31 December 2024 as an incomplete observation of ultimate losses. T2b learns developed-market risk patterns; A1 transfers a separate developed-market structure. The F3c recipe combines the two and applies historical public-score information. Those public factors are frozen, so this is a reconstruction of the historical submission rather than a new blind evaluation. I do not recalculate leaderboard probes or select an alternative using the 2024 results.

Two parts of the historical source were restored: a global reporting curve reconstructed from the documented formula and the old motor confirmation join before the ID01 correction. They are clearly distinguished from recovered original code. Availability at quotation, case reserves and the stability of new-business results remain open limitations. The reconstruction previously matched F3c in a Windows Python 3.12 environment; it was performed within the same AI-assisted workflow and is not independent professional review.

I directed the source modeling and reconstruction work with AI assistance. The repository history and project records retain the authorship and provenance of those contributions. The source files are embedded as text and written into this run's temporary workspace so that their imports and the historical stage boundaries remain intact.


## What changes in the emerging market

The emerging market has a shorter claims history than the developed market. I use the developed portfolio to learn risk patterns, then adapt their level and other components to emerging-market experience. The markets stay separate when policies, customers and claims are joined: a matching identifier across markets is not evidence of the same entity.

For each market I assign accepted claims to policy periods by line, policy and occurrence date and calculate observed net incurred cost. Claims are valued at 31 December 2024. I estimate reporting completeness to construct the effective exposure used by the models. This does not establish that open-claim reserves are adequate or that the reporting pattern transports perfectly between markets.

## Training and validation timeline

| Stage | Developed data | Emerging data | Purpose |
| --- | --- | --- | --- |
| `dev` | 2021–2023 | 2023, with customer folds | Out-of-fold predictions for local development |
| `hold` | 2021–2023 | Fit on 2023, evaluate 2024 | Temporal contrast |
| `pcv` | 2021–2024 | 2023–2024, with customer folds | Out-of-fold inputs for production calibration |
| `prod` | 2021–2024 | 2023–2024 | Predictions for 2025 |

T2b is a developed-market LightGBM Tweedie model with line-specific settings. It uses an internal customer split to choose the number of rounds, then refits on the permitted source training data. A1 is a separate transfer line; the production reconstruction uses three seeds and its recorded claims-history variant. The intermediate 2024 contrast and the production A1 recipe have different scopes. I do not call the contrast an out-of-fold replay of the entire production file.

Customer folds reduce same-customer leakage across fold boundaries; the year split tests transport across time. They address different problems. Both 2023 and 2024 contributed to development, so the resulting diagnostics are adaptive rather than an independent final validation.

## From components to F3c

The historical chain constructs a 50/50 blend of its designated T2b and A1 production components. It then applies the frozen global public-probe level, F2 adjustments and the credible final level and shape. F3c adds the recorded residual factors for gender in motor and contents, and for working from home and marketing segment in contents, followed by their historical level adjustment. The code recalculates the local factors from the newly generated inputs and uses the saved aggregate public-probe constants; it does not make new submissions.

The public calibration makes this a competition reconstruction. It would need a different source of level information in an actual pricing deployment. Gender is included because it belongs to the historical F3c specification; its use would need a jurisdiction-specific review and an assessment of differential effects before any real pricing application. New-business support, feature timing, influential claims and reserve uncertainty remain material limits. The conditional sensitivity checks do not establish that the method will perform the same way on a new portfolio.

## Reading the output

I expect one positive finite prediction for each emerging-market test row, in the sample template's order. The notebook checks that contract and compares the CSV with the historical SHA-256. An exact match is a reproducibility check for the file, not independent validation of the actuarial assumptions. A mismatch stops the final step and leaves the generated file available for investigation. The reference premium calculation below is separate from the submission and does not reapply exposure to a period risk premium.


## Interpreting the metric

For nonnegative observed cost `y` and positive prediction `mu`, Tweedie deviance at power 1.5 is `4 * (y / sqrt(mu) + sqrt(mu) - 2 * sqrt(y))`. The local D² is one minus the total model deviance divided by the deviance of the constant mean prediction. Higher is better; a negative value means worse than that constant on the evaluated rows. The observed target remains a valuation proxy for ultimate cost. I do not claim this local calculation proves every detail of Kaggle's unpublished evaluator.

When comparing models, I keep the same rows and distinguish a global D² from segment-specific D² values, which have different null predictions. I also inspect the support and uncertainty of each segment instead of treating a small score difference as conclusive.


In [ ]:
from sklearn.metrics import mean_tweedie_deviance, d2_tweedie_score
# A small synthetic calculation, including a zero-cost observation.
y_demo = np.array([0.0, 0.25, 2.0, 100.0])
mu_demo = np.array([0.1, 0.3, 1.5, 25.0])
d_demo = 4 * (y_demo / np.sqrt(mu_demo) + np.sqrt(mu_demo) - 2 * np.sqrt(y_demo))
null_demo = 4 * (y_demo / np.sqrt(y_demo.mean()) + np.sqrt(y_demo.mean()) - 2 * np.sqrt(y_demo))
assert np.isclose(d_demo.mean(), mean_tweedie_deviance(y_demo, mu_demo, power=1.5), rtol=1e-13)
assert np.isclose(1 - d_demo.sum() / null_demo.sum(), d2_tweedie_score(y_demo, mu_demo, power=1.5), rtol=1e-13)
print("The formula agrees with the local scikit-learn calculation.")


## Dependencies and run workspace

The reference reconstruction used Python 3.12.14, NumPy 2.5.3, pandas 3.0.5, PyArrow 26.0.0, SciPy 1.18.1, scikit-learn 1.9.1 and LightGBM 4.7.0. I check the installed versions before running. If the Kaggle image differs, install compatible versions in a separate setup step and restart the kernel; package installation may require enabling Internet. Exact bytes are not promised on a different platform. The historical local 57-stage reconstruction, which also included alternatives beyond F3c, took about 57 minutes excluding installation on its specific hardware. This notebook stops at F3c. [Kaggle's notebook documentation](https://www.kaggle.com/docs/notebooks), consulted on 28 October 2026, describes a 12-hour CPU session and 20 GB of saved working storage. Runtime and storage in a hosted run remain unverified.


In [ ]:
import json, shutil, subprocess, time, uuid
from datetime import datetime, timezone
from importlib import metadata
REFERENCE = {"numpy": "2.5.3", "pandas": "3.0.5", "pyarrow": "26.0.0", "scipy": "1.18.1", "scikit-learn": "1.9.1", "lightgbm": "4.7.0"}
installed = {name: metadata.version(name) for name in REFERENCE}
print("Installed:", installed)
print("Reference:", REFERENCE)
if installed != REFERENCE:
    print("Environment differs from the recorded reference; output identity must be checked.")
RUN, EXPORT = prepare_workspace(2)
SOURCES = {}


## Embedded source package

I keep the source files below as text assignments. Running these cells registers the code in memory; the later execution section writes it into the new workspace and starts the recipe. I retain the original program boundaries, authorship and frozen configuration. No past row-level predictions or fitted models are embedded.


### Embedded source: `ml/codigo/abrir_hold.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/abrir_hold.py'] = '"""Open the registered 2024 HOLD evaluation once for the frozen candidate list.\nRead entrega/lista_congelada.json; use DEV OOF blend/calibration parameters and\nthe recorded hold-phase trend. This historical procedure is not a new selection.\n"""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import metricas, bootstrap_dif\nimport receta\n\nlista = json.load(open(ENTREGA / "lista_congelada.json"))\ntend = json.load(open(TRABAJO / "pred" / "hold" / "tendencia.json"))\npesos = pd.read_parquet(TRABAJO / "pesos_2025_hold.parquet")\nres = {"candidatos": {}, "bootstrap": {}}\nmus = {}\nfor nombre, spec in lista["candidatos"].items():\n    params = receta.parametros("dev", spec["componentes"], spec["calib"])\n    t, mu_sin = receta.aplicar("hold", params, None)\n    _, mu_con = receta.aplicar("hold", params, tend)\n    t = t.merge(pesos, on="row_id", how="left")\n    mus[nombre] = mu_con if lista.get("tendencia", True) else mu_sin\n    res["candidatos"][nombre] = {"con_tendencia": metricas(t, mu_con), "sin_tendencia": metricas(t, mu_sin), "params": params}\n    print(nombre, round(res["candidatos"][nombre]["con_tendencia"]["D2"], 4), round(res["candidatos"][nombre]["sin_tendencia"]["D2"], 4), flush=True)\nref = lista["referencias"]\nfor nombre in lista["candidatos"]:\n    for r in ref:\n        if nombre != r:\n            res["bootstrap"][f"{nombre}-{r}"] = bootstrap_dif(t, mus[nombre], mus[r], B=1000)\nelegido = lista["elegido"]\nres["elegido"] = elegido\nguardar_json(res, ENTREGA / "resultados_hold.json")\nprint(json.dumps({k: v for k, v in res["bootstrap"].items() if k.startswith(elegido)}, indent=1))\n'


### Embedded source: `ml/codigo/adaptacion.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/adaptacion.py'] = '"""T5: ridge-regularized GLM adaptation of transferred T2 relativities (E03).\nlog(mu)=raw_T2+log(e*c)+b0+Z*b, with credibility shrinkage on b.\nUsage: python adaptacion.py <phase>\n"""\nfrom __future__ import annotations\nimport sys, hashlib\nimport numpy as np, pandas as pd\nfrom scipy.optimize import minimize\nfrom comun import *\nfrom variables import cargar\n\nFASE = sys.argv[1] if len(sys.argv) > 1 else "dev"\nCONF = {"dev": ("hasta2023", [2023], None), "hold": ("hasta2023", [2023], 2024), "prod": ("hasta2024", [2023, 2024], 2025),\n        "pcv": ("hasta2024", [2023, 2024], None)}[FASE]\nOUT = TRABAJO / "pred" / FASE\nFACT = ["region", "channel", "age_band", "gender", "occupation"]\nLAMBDAS = [1e2, 1e3, 3e3, 1e4, 3e4, 1e5, 3e5, 1e6, 3e6, 1e7]\n\n\ndef diseno(df, niveles=None):\n    cols = [df.new_business.astype(float).values[:, None], df.self_employed.fillna(False).astype(float).values[:, None]]\n    nombres = ["new_business", "self_employed"]\n    niv = {} if niveles is None else niveles\n    for f in FACT:\n        v = df[f].astype(object).where(df[f].notna(), "NA").astype(str).values\n        if niveles is None:\n            niv[f] = sorted(set(v))\n        for n in niv[f]:\n            cols.append((v == n).astype(float)[:, None]); nombres.append(f"{f}={n}")\n    return np.hstack(cols), nombres, niv\n\n\ndef ajustar(Z, y, off, lam):\n    n, k = Z.shape\n\n    def f(beta):\n        eta = off + beta[0] + Z @ beta[1:]\n        a = np.exp(0.5 * eta); b = np.exp(-0.5 * eta)  # mu^(2-p) and mu^(1-p), with p=1.5.\n        loss = np.sum(2 * (y * 2 * b + 2 * a)) + lam * np.sum(beta[1:] ** 2)  # Omit terms depending only on y.\n        g_eta = 2 * (-y * b + a)\n        grad = np.concatenate([[g_eta.sum()], Z.T @ g_eta + 2 * lam * beta[1:]])\n        return loss, grad\n    b0 = np.log(max(np.sum(y / np.sqrt(np.exp(off))) / np.sum(np.sqrt(np.exp(off))), 1e-9))\n    x0 = np.zeros(k + 1); x0[0] = b0\n    r = minimize(f, x0, jac=True, method="L-BFGS-B", options={"maxiter": 2000})\n    return r.x\n\n\ndef interno(cids):\n    return np.array([int(hashlib.sha256(("mlind_inner|" + c).encode()).hexdigest()[:8], 16) % 5 == 0 for c in cids])\n\n\ncoefs = {}\nfor l in LINEAS:\n    em = cargar("emerging", l, CONF[0])\n    raw = pd.read_parquet(OUT / f"T2raw_{l}.parquet")\n    em = em.merge(raw, on="row_id")\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF[1])].reset_index(drop=True)\n    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF[2] is None else [(tr, em[em.year == CONF[2]].reset_index(drop=True))]\n    preds = []; lams = []\n    for a, b in sets:\n        Za, nom, niv = diseno(a); Zb, _, _ = diseno(b, niv)\n        offa = a.raw.values + np.log(a.exposure.values * a.c_fila.values)\n        offb = b.raw.values + np.log(b.exposure.values)\n        ya = a.y.values\n        v = interno(a.customer_id.values)\n        best = None\n        for lam in LAMBDAS:\n            bt = ajustar(Za[~v], ya[~v], offa[~v], lam)\n            mu = np.exp(offa[v] + bt[0] + Za[v] @ bt[1:])\n            dev = desviacion(ya[v], mu).sum()\n            if best is None or dev < best[0]:\n                best = (dev, lam)\n        lam = best[1]; lams.append(lam)\n        bt = ajustar(Za, ya, offa, lam)\n        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": np.exp(offb + bt[0] + Zb @ bt[1:])}))\n        coefs.setdefault(l, []).append({"lambda": lam, "b0": bt[0], **dict(zip(nom, bt[1:]))})\n    pd.concat(preds).to_parquet(OUT / f"T5_{l}.parquet", index=False)\n    print(l, "lambdas", lams, "b_NB", [round(c["new_business"], 3) for c in coefs[l]])\nguardar_json(coefs, OUT / "T5_coeficientes.json")\n'


### Embedded source: `ml/codigo/anidado.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/anidado.py'] = '"""Evaluate saved raw predictions with fully nested OOF calibration (A05).\n\nFor outer fold j, estimate component levels, blend weights, C1 and C2 only on\nother folds. Internal predictions for fold i use levels k_{-i,-j}, excluding\nboth i and j when estimating calibration parameters.\nUsage: python anidado.py <dev|pcv> <output.json> <recipe> [...]\nRecipe format: A+B+C:shrinkage, with shrinkage in [0,1].\n"""\nimport sys, json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import mezcla_pesos, metricas, bootstrap_dif\nimport receta\n\nFASE, SALIDA, RECETAS = sys.argv[1], sys.argv[2], sys.argv[3:]\nimport os\nFIJOS = {}\nif os.environ.get("MLIND_PESOS_FIJOS"):  # Positional weights from the e26_pesos_des24.py JSON record.\n    _j = json.load(open(os.environ["MLIND_PESOS_FIJOS"]))\n    FIJOS = {l: list(_j["ramos"][l]["pesos"].values()) for l in LINEAS}\nK_NB = 200.0\nt = receta.tabla(FASE).reset_index(drop=True)\nfolds = pliegue(t.customer_id.values)\ny = t.y.values; e = t.exposure.values; cf = t.c_fila.values; nbv = t.new_business.values\n\n\ndef raw_de(c):\n    """Load raw component predictions; dev/hold share fits, as do pcv/production."""\n    sub = {"dev": ["dev", "hold"], "pcv": ["pcv", "prod"]}[FASE]\n    out = []\n    for l in LINEAS:\n        for s in sub:\n            f = TRABAJO / "pred" / s / f"{c}raw_{l}.parquet"\n            if f.exists():\n                out.append(pd.read_parquet(f)); break\n        else:\n            raise FileNotFoundError(f"{c}raw_{l} en {sub}")\n    r = pd.concat(out, ignore_index=True)\n    v = t[["row_id"]].merge(r, on="row_id", how="left").raw.values\n    assert not np.isnan(v).any(), c\n    return v\n\n\ndef k_nivel(rw, m):\n    return factor_tweedie(y[m], np.exp(rw[m]) * e[m] * cf[m])\n\n\ndef predecir(cs, enc, entreno, objetivo, R):\n    """Estimate blend, C1 and C2 on the training mask using inner OOF levels, then apply to the target mask."""\n    pli = np.unique(folds[entreno])\n    U_in = np.zeros((entreno.sum(), len(cs))); idx = np.where(entreno)[0]\n    U_out = np.zeros((objetivo.sum(), len(cs))); ido = np.where(objetivo)[0]\n    for l in LINEAS:  # Estimate k per insurance line, as in transfer2.salidas and gbm.py.\n        L = (t.line.values == l)\n        for i in pli:\n            mi = (folds[idx] == i) & L[idx]\n            fit = entreno & (folds != i) & L\n            for a, c in enumerate(cs):\n                U_in[mi, a] = k_nivel(R[c], fit) * np.exp(R[c][idx[mi]]) * e[idx[mi]]\n        mo = L[ido]\n        for a, c in enumerate(cs):\n            U_out[mo, a] = k_nivel(R[c], entreno & L) * np.exp(R[c][ido[mo]]) * e[ido[mo]]\n    res = {}\n    for l in LINEAS:\n        il_in = (t.line.values[idx] == l); il_out = (t.line.values[objetivo] == l)\n        M = U_in[il_in] * cf[idx][il_in][:, None]\n        if enc < 0:  # E26 uses externally fixed DES24 weights in component order.\n            w = np.array(FIJOS[l])\n        else:\n            w = mezcla_pesos(y[idx][il_in], M) if len(cs) > 1 else np.ones(1)\n            w = (1 - enc) * w + enc / len(cs)\n        m_in = M @ w; yy = y[idx][il_in]\n        k1 = factor_tweedie(yy, m_in)\n        base_out = U_out[il_out] @ w\n        f2 = np.zeros(il_out.sum())\n        for nb in (True, False):\n            s = nbv[idx][il_in] == nb\n            ks = factor_tweedie(yy[s], m_in[s] * k1); n = int((yy[s] > 0).sum()); z = n / (n + K_NB)\n            f2[nbv[objetivo][il_out] == nb] = k1 * (z * ks + 1 - z)\n        res[l] = (il_out, base_out, k1, f2, w)\n    out = {"sin": np.zeros(objetivo.sum()), "C1": np.zeros(objetivo.sum()), "C2": np.zeros(objetivo.sum())}\n    for l, (il, b, k1, f2, w) in res.items():\n        out["sin"][il] = b; out["C1"][il] = b * k1; out["C2"][il] = b * f2\n    return out\n\n\ndef main():\n    recs = []\n    for r in RECETAS:\n        cs, enc = r.split(":") if ":" in r else (r, "0")\n        recs.append((cs.split("+"), -1.0 if enc == "fijo" else float(enc)))\n    todos = sorted({c for cs, _ in recs for c in cs} | {"T2b"})\n    R = {c: raw_de(c) for c in todos}\n    if ("T2b", 0.0) not in [(tuple(cs)[0] if len(cs) == 1 else None, enc) for cs, enc in recs]:\n        recs = [(["T2b"], 0.0)] + recs\n    preds = {}\n    for cs, enc in recs:\n        nom = "+".join(cs) + ("_des24" if enc < 0 else (f"_e{int(enc * 100)}" if len(cs) > 1 else ""))\n        P = {k: np.zeros(len(t)) for k in ("sin", "C1", "C2")}\n        for j in np.unique(folds):\n            ob = folds == j\n            o = predecir(cs, enc, ~ob, ob, R)\n            for k in P:\n                P[k][ob] = o[k]\n        preds[nom] = P\n        print(nom, {k: round(metricas(t, v)["D2"], 4) for k, v in P.items()}, flush=True)\n    res = {"fase": FASE, "metricas": {n: {k: metricas(t, v) for k, v in P.items()} for n, P in preds.items()}, "bootstrap_vs_T2b": {}}\n    for n, P in preds.items():\n        if n == "T2b":\n            continue\n        res["bootstrap_vs_T2b"][n] = {k: bootstrap_dif(t, P[k], preds["T2b"][k]) for k in ("C1", "C2")}\n        if FASE == "pcv":\n            for nb in (True, False):\n                s = nbv == nb\n                res["bootstrap_vs_T2b"][n]["C1_" + ("NB" if nb else "REN")] = bootstrap_dif(\n                    t[s].reset_index(drop=True), P["C1"][s], preds["T2b"]["C1"][s])\n        print(n, {k: (round(v["dif"], 4), [round(x, 4) for x in v["ic95"]]) for k, v in res["bootstrap_vs_T2b"][n].items()}, flush=True)\n    guardar_json(res, ENTREGA / "evolucion" / SALIDA)\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `ml/codigo/audit_checks.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/audit_checks.py'] = '"""Measure two historical audit sensitivities on the fixed emerging-market recipe.\n\nCompare 2023-only peril-cost completeness weights with the exported 2023-2024\nweights, and customer-address connected-component folds with customer-only\nfolds for DEV T2b. Preserve the registered scope and original audit records.\n"""\nimport hashlib, json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom completitud import por_peligro\nfrom evaluar import metricas\nimport receta\n\nres = {"a_c_fila": {}, "b_agrupacion": {}}\nPEL = {"motor": ["bodily_injury", "glass_hail", "own_damage", "theft", "third_party_prop"],\n       "buildings": ["accidental_damage", "escape_of_water", "fire", "flood", "storm", "subsidence"],\n       "contents": ["accidental_damage", "escape_of_water", "flood", "theft"]}\nnuevas = {}\nfor l in LINEAS:\n    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet")\n    tr = b[b.conjunto == "train"]\n    comp = {p: por_peligro(l, p, b.period_start, b.period_end) for p in PEL[l]}\n    def pesos(mask):\n        ult = {p: tr.loc[mask, f"y_{p}"].sum() / np.average(comp[p][(b.conjunto == "train").values][mask.values], weights=tr.exposure[mask]) for p in PEL[l]}\n        s = sum(ult.values()); return {p: v / s for p, v in ult.items()}\n    w23 = pesos(tr.year == 2023)\n    c23 = sum(w23[p] * comp[p] for p in PEL[l])\n    c_old = b.c_fila.values\n    r = np.where(b.conjunto == "test", 1.0, c23 / c_old)\n    res["a_c_fila"][l] = {"pesos_2023": {k: round(v, 4) for k, v in w23.items()},\n                          "cambio_relativo_max_2023": float(np.abs(r[(b.year == 2023).values] - 1).max()),\n                          "cambio_relativo_max_2024": float(np.abs(r[(b.year == 2024).values] - 1).max()),\n                          "cambio_medio_2024": float(np.average(r[(b.year == 2024).values], weights=b.exposure[(b.year == 2024).values]) - 1)}\n    nuevas[l] = pd.DataFrame({"row_id": b.row_id, "c_nueva": np.where(b.conjunto == "test", 1.0, c23)})\n# Effect on the fixed final model\'s evaluation metrics.\ndef con_c(t):\n    c = pd.concat(nuevas.values()); t2 = t.merge(c, on="row_id", how="left"); t2["c_fila"] = t2.c_nueva; return t2\nt, mu, _ = receta.oof_cruzado("dev", ["T2b"], "sin")\nres["a_c_fila"]["DEV_T2b_sin"] = {"exportada": metricas(t, mu)["D2"], "pesos_2023": metricas(con_c(t), mu)["D2"]}\np = receta.parametros("dev", ["T2b"], "ramo")\nth, mh = receta.aplicar("hold", p, None)\nres["a_c_fila"]["HOLD_T2b_C1"] = {"exportada": metricas(th, mh)["D2"], "pesos_2023_solo_en_evaluacion": metricas(con_c(th), mh)["D2"]}\n# Customer-address connected components in emerging-market 2023.\nfilas = []\nfor l in LINEAS:\n    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "customer_id", "address_id", "year", "conjunto", "fold"])\n    filas.append(b[(b.conjunto == "train") & (b.year == 2023)].assign(line=l))\ne = pd.concat(filas, ignore_index=True)\npadre = {}\ndef f(x):\n    while padre.setdefault(x, x) != x:\n        padre[x] = padre[padre[x]]; x = padre[x]\n    return x\nfor c, a in zip("C" + e.customer_id, "A" + e.address_id):\n    rc, ra = f(c), f(a)\n    if rc != ra: padre[ra] = rc\ncomp_id = np.array([f("C" + c) for c in e.customer_id])\ne["fold_ca"] = [int(hashlib.sha256(("mlind_ca|" + x).encode()).hexdigest()[:16], 16) % 5 for x in comp_id]\ncruce = e.groupby("address_id").fold.nunique()\nres["b_agrupacion"]["direcciones_en_mas_de_un_pliegue_por_cliente"] = int((cruce > 1).sum())\nres["b_agrupacion"]["componentes"] = int(len(set(comp_id)))\n# Recompute T2b\'s line-level emerging calibration k2 under the alternate folds.\nraw = pd.concat([pd.read_parquet(TRABAJO / "pred" / "dev" / f"T2braw_{l}.parquet") for l in LINEAS])\nbb = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "exposure", "c_fila", "y"]) for l in LINEAS])\ne = e.merge(raw, on="row_id").merge(bb, on="row_id")\nmu = np.zeros(len(e))\nfor k in range(5):\n    for l in LINEAS:\n        a = ((e.fold_ca != k) & (e.line == l)).values; bm = ((e.fold_ca == k) & (e.line == l)).values\n        k2 = factor_tweedie(e.y.values[a], np.exp(e.raw.values[a]) * e.exposure.values[a] * e.c_fila.values[a])\n        mu[bm] = k2 * np.exp(e.raw.values[bm]) * e.exposure.values[bm]\ntt = receta.tabla("dev")\nmu_al = tt[["row_id"]].merge(pd.DataFrame({"row_id": e.row_id, "mu": mu}), on="row_id").mu.values\nres["b_agrupacion"]["DEV_T2b_sin"] = {"pliegues_cliente": metricas(t, receta.componentes("dev", ["T2b"], t)["T2b"])["D2"],\n                                       "pliegues_cliente_direccion": metricas(tt, mu_al)["D2"]}\nguardar_json(res, ENTREGA / "respuesta_audit_checks.json")\nprint(json.dumps(res, indent=1, default=float))\n'


### Embedded source: `ml/codigo/bootstrap_final.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/bootstrap_final.py'] = '"""Customer-bootstrap intervals for the comparisons cited in the report."""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import bootstrap_dif\nimport receta\n\nres = {}\nt = receta.tabla("dev")\nm = {}\nfor c, cal in [("T2b", "ramo"), ("R0", "ramo"), ("T1wp", "ramo"), ("G1", "ramo"), ("N1", "ramo_nb"), ("N2a", "ramo_nb"), ("T2b", "ramo_nb"), ("N3", "ramo_nb")]:\n    _, mu, _ = receta.oof_cruzado("dev", [c], cal); m[(c, cal)] = mu\nres["DEV_T2b_C1-R0_C1"] = bootstrap_dif(t, m[("T2b", "ramo")], m[("R0", "ramo")])\nres["DEV_T2b_C1-T1wp_C1"] = bootstrap_dif(t, m[("T2b", "ramo")], m[("T1wp", "ramo")])\nres["DEV_T2b_C1-G1_C1"] = bootstrap_dif(t, m[("T2b", "ramo")], m[("G1", "ramo")])\nres["DEV_N2a_C2-N1_C2"] = bootstrap_dif(t, m[("N2a", "ramo_nb")], m[("N1", "ramo_nb")])\nres["DEV_N3_C2-T2b_C2"] = bootstrap_dif(t, m[("N3", "ramo_nb")], m[("T2b", "ramo_nb")])\nlista = json.load(open(ENTREGA / "lista_congelada.json"))\nth = None; hm = {}\nfor nom in ["T2b_C1", "R0_C1", "T1wp_C2", "G1_C2", "N1_C2", "N2a_C2", "N3_C2", "T2b_C2"]:\n    spec = lista["candidatos"][nom]\n    p = receta.parametros("dev", spec["componentes"], spec["calib"])\n    th, hm[nom] = receta.aplicar("hold", p, None)\nres["HOLD_T2b_C1-R0_C1"] = bootstrap_dif(th, hm["T2b_C1"], hm["R0_C1"])\nres["HOLD_T2b_C1-T1wp_C2"] = bootstrap_dif(th, hm["T2b_C1"], hm["T1wp_C2"])\nres["HOLD_N2a_C2-N1_C2"] = bootstrap_dif(th, hm["N2a_C2"], hm["N1_C2"])\nres["HOLD_N3_C2-T2b_C2"] = bootstrap_dif(th, hm["N3_C2"], hm["T2b_C2"])\nres["HOLD_T2b_C2-T2b_C1"] = bootstrap_dif(th, hm["T2b_C2"], hm["T2b_C1"])\nguardar_json(res, ENTREGA / "bootstrap_final.json")\nfor k, v in res.items():\n    print(k, round(v["dif"], 4), [round(x, 4) for x in v["ic95"]], v["p_mayor"])\n'


### Embedded source: `ml/codigo/calibracion.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/calibracion.py'] = '"""Inspect power-1.5 Tweedie calibration with isotonic Murphy decomposition.\nReport observed/expected ratios by decile and segment.\nUsage: python calibracion.py <dev|pcv> <recipe.json>\n"""\nimport sys, json\nimport numpy as np, pandas as pd, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom sklearn.isotonic import IsotonicRegression\nfrom comun import *\nimport receta\n\nplt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})\nC = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}\n\n\ndef murphy(y, m, w):\n    """Decompose mean score as S=UNC-DSC+MCB; fit isotonic observed rates with exposure weights."""\n    e = w\n    r = m / e\n    # For power-1.5 amount deviance, optimal isotonic blocks weight rates by e^(2-p)=sqrt(e).\n    iso = IsotonicRegression(out_of_bounds="clip").fit(r, y / e, sample_weight=np.sqrt(e))\n    mc = np.maximum(iso.predict(r) * e, 1e-9)\n    ybar = np.sum(y / np.sqrt(e)) / np.sum(np.sqrt(e)) * e  # Tweedie-optimal constant rate.\n    S = desviacion(y, m).mean(); Sc = desviacion(y, mc).mean(); Su = desviacion(y, ybar).mean()\n    return {"S": S, "UNC": Su, "DSC": Su - Sc, "MCB": S - Sc, "D2": 1 - S / Su, "D2_recal": 1 - Sc / Su}\n\n\nif __name__ == "__main__":\n    fase = sys.argv[1]; spec = json.load(open(sys.argv[2]))\n    t, mu, _ = receta.oof_cruzado(fase, spec["componentes"], spec["calib"])\n    m = mu * t.c_fila.values; y = t.y.values\n    out = {"murphy": {}, "deciles": {}, "segmentos": {}}\n    fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))\n    for ax, l in zip(axs, LINEAS):\n        i = (t.line == l).values\n        out["murphy"][l] = murphy(y[i], m[i], t.exposure.values[i] * t.c_fila.values[i])\n        r = m[i] / (t.exposure.values[i] * t.c_fila.values[i])\n        dec = pd.qcut(r, 10, labels=False, duplicates="drop")\n        g = pd.DataFrame({"d": dec, "y": y[i], "m": m[i], "e": t.exposure.values[i] * t.c_fila.values[i]}).groupby("d").sum()\n        g["obs"] = g.y / g.e; g["pred"] = g.m / g.e; g["AE"] = g.y / g.m\n        out["deciles"][l] = g[["obs", "pred", "AE"]].round(4).to_dict()\n        ax.plot(g.pred, g.obs, "o-", color=C[l], ms=4, label="observado")\n        lim = [0, max(g.pred.max(), g.obs.max()) * 1.05]\n        ax.plot(lim, lim, color="#999", lw=0.8, ls="--", label="diagonal")\n        ax.set_title(f"{l}  (MCB/S = {out[\'murphy\'][l][\'MCB\'] / out[\'murphy\'][l][\'S\']:.2%})")\n        ax.set_xlabel("tasa predicha (CU por año)"); ax.legend(frameon=False, fontsize=7)\n    axs[0].set_ylabel("tasa observada (CU por año)")\n    fig.suptitle(f"Calibración por deciles de la predicción, OOF {fase}", fontsize=10)\n    fig.tight_layout(); fig.savefig(ENTREGA / "figuras" / f"f05_calibracion_deciles_{fase}.png"); plt.close(fig)\n    # Shape diagnostic: transferred-relativity exponent a; a=1 indicates unchanged shape.\n    out["forma"] = {}\n    for l in LINEAS:\n        i = (t.line == l).values; e = t.exposure.values[i] * t.c_fila.values[i]; r = m[i] / e\n        g = np.exp(np.average(np.log(r), weights=e))\n        mejor = None; curva = {}\n        for a in np.round(np.arange(0.6, 1.41, 0.05), 2):\n            ma = e * g * (r / g) ** a\n            ma *= factor_tweedie(y[i], ma)\n            dv_ = desviacion(y[i], ma).sum(); curva[float(a)] = dv_\n            if mejor is None or dv_ < mejor[1]:\n                mejor = (float(a), dv_)\n        base = curva[1.0]; d0 = desviacion(y[i], np.full(i.sum(), y[i].mean())).sum()\n        out["forma"][l] = {"a_optimo": mejor[0], "ganancia_D2_en_muestra": (base - mejor[1]) / d0}\n    for seg in ["new_business", "year"]:\n        g = pd.DataFrame({"line": t.line, "s": t[seg], "y": y, "m": m}).groupby(["line", "s"]).sum()\n        g["AE"] = g.y / g.m\n        out["segmentos"][seg] = {f"{a}|{b}": round(float(v), 4) for (a, b), v in g.AE.items()}\n    guardar_json(out, ENTREGA / f"calibracion_{fase}.json")\n    print(json.dumps(out["murphy"], indent=1, default=float)); print(out["segmentos"]); print(out["forma"])\n'


### Embedded source: `ml/codigo/comparar_lineas.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/comparar_lineas.py'] = '"""Compare model families on common 2024 rows and production 2025 predictions.\n\nUse the same 59,202 2024 rows and T0 labels, allowing each model\'s documented\ncompleteness on the observed scale. Bootstrap customer-address components.\nFor 2025 inspect totals, rates, row correlations and power-1.5 Tweedie Bregman\ndivergence between predictions, without treating predictions as observed labels.\n"""\nimport json, hashlib\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import tabla_eval\nimport receta\n\nOTROS = RAIZ / "comparacion" / "otros"\nres = {"fuentes": {}, "2024": {}, "2025": {}}\nfor f in sorted(OTROS.iterdir()):\n    res["fuentes"][f.name] = sha256(f)\n\n# 2024 comparison.\no = pd.read_parquet(OTROS / "predicciones_2024.parquet")\nlista = json.load(open(ENTREGA / "lista_congelada.json"))\np = receta.parametros("dev", ["T2b"], "ramo")\nt, mu_c1 = receta.aplicar("hold", p, None)\np2 = receta.parametros("dev", ["T2b"], "ramo_nb"); _, mu_c2 = receta.aplicar("hold", p2, None)\np3 = receta.parametros("dev", ["R0"], "ramo"); _, mu_r0 = receta.aplicar("hold", p3, None)\nd = t.merge(o[["row_id", "address_id", "R0", "A1", "A2", "A3"]], on="row_id", how="inner")\nassert len(d) == len(t) == len(o) == 59202\nd["MLind_T2b_C1"] = mu_c1 * t.c_fila.values\nd["MLind_T2b_C2"] = mu_c2 * t.c_fila.values\nd["MLind_R0"] = mu_r0 * t.c_fila.values\nd = d.rename(columns={"R0": "noct_R0", "A1": "noct_A1", "A2": "noct_A2", "A3": "noct_A3"})\nmods = ["MLind_T2b_C1", "MLind_T2b_C2", "MLind_R0", "noct_R0", "noct_A1", "noct_A2", "noct_A3"]\ny = d.y.values\n# Customer-address connected components.\npadre = {}\ndef f(x):\n    while padre.setdefault(x, x) != x:\n        padre[x] = padre[padre[x]]; x = padre[x]\n    return x\nfor c, a in zip("C" + d.customer_id, "A" + d.address_id):\n    rc, ra = f(c), f(a)\n    if rc != ra: padre[ra] = rc\ncomp = np.array([f("C" + c) for c in d.customer_id])\n_, gi = np.unique(comp, return_inverse=True)\nres["2024"]["filas"] = int(len(d)); res["2024"]["componentes"] = int(gi.max() + 1)\nfor m in mods:\n    r = {"D2": d2(y, d[m].values)}\n    for l in LINEAS:\n        i = (d.line == l).values\n        r[f"D2_{l}"] = d2(y[i], d[m].values[i]); r[f"AE_{l}"] = float(y[i].sum() / d[m].values[i].sum())\n    for nb in (True, False):\n        i = d.new_business.values == nb\n        r[f"D2_{\'NB\' if nb else \'REN\'}"] = d2(y[i], d[m].values[i])\n    res["2024"][m] = r\nrng = np.random.default_rng(20261024)\ndev = {m: desviacion(y, d[m].values) for m in mods}\nsy = np.sqrt(y); B = 2000\npares = [("MLind_T2b_C1", "noct_A1"), ("MLind_T2b_C1", "noct_R0"), ("MLind_T2b_C2", "noct_A1"), ("noct_A1", "noct_R0"), ("MLind_T2b_C1", "noct_A2")]\nboot = {k: [] for k in pares}; boot33 = {k: [] for k in pares}\nG = gi.max() + 1\nfor s in range(0, B, 100):\n    W = rng.poisson(1.0, size=(100, G)).astype(float)[:, gi]\n    # Sample 33% of components as a public-subset-size diagnostic.\n    W33 = (rng.random(size=(100, G)) < 0.33).astype(float)[:, gi]\n    for WW, store in ((W, boot), (W33, boot33)):\n        ybar = (WW @ y) / WW.sum(1)\n        dn = 2 * ((WW @ sy) / (-0.25) - (WW @ y) * ybar ** -0.5 / (-0.5) + WW.sum(1) * ybar ** 0.5 / 0.5)\n        for a, b in pares:\n            store[(a, b)].append((WW @ dev[b] - WW @ dev[a]) / dn)\nres["2024"]["pares"] = {}\nfor k in pares:\n    r = np.concatenate(boot[k]); r33 = np.concatenate(boot33[k])\n    res["2024"]["pares"][f"{k[0]} - {k[1]}"] = {"dif": res["2024"][k[0]]["D2"] - res["2024"][k[1]]["D2"],\n        "ic95": [float(np.quantile(r, .025)), float(np.quantile(r, .975))], "p_mayor": float((r > 0).mean()),\n        "sd_submuestra_33": float(np.std(r33)), "p_mayor_submuestra_33": float((r33 > 0).mean())}\n# Correlation between models on 2024 rows.\nres["2024"]["corr_log_T2b_A1"] = float(np.corrcoef(np.log(d.MLind_T2b_C1), np.log(d.noct_A1))[0, 1])\n\n# 2025 prediction comparison.\nsub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")[["row_id"]]\nbase = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "exposure", "new_business", "conjunto"]).assign(line=l)\n                  for l in LINEAS])\nbase = base[base.conjunto == "test"]\nq = sub.merge(base, on="row_id", how="left")\narch = {"MLind_final": PRIVADO / "prediccion_2025_mlind.csv", "MLind_C2": PRIVADO / "prediccion_2025_mlind_C2.csv",\n        "MLind_C1_tendencia": PRIVADO / "prediccion_2025_mlind_C1_tendencia.csv",\n        "noct_A1fixF1b3n_designada": OTROS / "A1fixF1b3n_2025.csv", "noct_A1": OTROS / "A1_2025.csv", "noct_R0": OTROS / "R0_2025.csv",\n        "noct_A1fixTc": OTROS / "A1fixTc_2025.csv", "noct_A1fixF1nTc": OTROS / "A1fixF1nTc_2025.csv"}\nfor k, fpath in arch.items():\n    x = pd.read_csv(fpath, dtype={"row_id": str})\n    assert (x.row_id.values == sub.row_id.values).all() and x.prediction.notna().all()\n    q[k] = x.prediction.values\nfor k in arch:\n    r = {"total": float(q[k].sum())}\n    for l in LINEAS:\n        i = (q.line == l).values; e = q.exposure.values[i]; nb = q.new_business.values[i]\n        r[l] = {"total": float(q[k].values[i].sum()), "tasa": float(q[k].values[i].sum() / e.sum()),\n                "tasa_NB": float(q[k].values[i][nb].sum() / e[nb].sum()), "tasa_REN": float(q[k].values[i][~nb].sum() / e[~nb].sum())}\n    res["2025"][k] = r\n# Distance between the candidate and designated delivery predictions.\na = q.MLind_final.values; b = q.noct_A1fixF1b3n_designada.values\nres["2025"]["final_vs_designada"] = {\n    "corr_log": float(np.corrcoef(np.log(a), np.log(b))[0, 1]),\n    "spearman": float(pd.Series(a).rank().corr(pd.Series(b).rank())),\n    "cociente_p01_p50_p99": [float(x) for x in np.quantile(a / b, [0.01, 0.5, 0.99])],\n    "por_ramo": {}}\n# Mean Bregman divergence relative to expected null deviance per exposure year.\nnull24 = {}\nfor l in LINEAS:\n    i = (d.line == l).values\n    yl = y[i]; ybar = yl.mean()\n    null24[l] = float(desviacion(yl, np.full(i.sum(), ybar)).sum() / t.exposure.values[i].sum())\nnulo25 = sum(null24[l] * q.exposure.values[(q.line == l).values].sum() for l in LINEAS)\ndef breg(mt, mo):  # E_y~mt[d(y,mo) - d(y,mt)] = d(mt, mo)\n    return desviacion(mt, mo)\nres["2025"]["final_vs_designada"]["gap_D2_si_la_verdad_es_designada"] = float(breg(b, a).sum() / nulo25)\nres["2025"]["final_vs_designada"]["gap_D2_si_la_verdad_es_final"] = float(breg(a, b).sum() / nulo25)\nfor l in LINEAS:\n    i = (q.line == l).values\n    res["2025"]["final_vs_designada"]["por_ramo"][l] = {"corr_log": float(np.corrcoef(np.log(a[i]), np.log(b[i]))[0, 1]),\n                                                       "cociente_total": float(a[i].sum() / b[i].sum())}\n# Portion of the distance attributable solely to line-level differences.\na_niv = a.copy()\nfor l in LINEAS:\n    i = (q.line == l).values; a_niv[i] = a[i] * b[i].sum() / a[i].sum()\nres["2025"]["final_vs_designada"]["gap_D2_misma_forma_nivel_designada"] = float(breg(b, a_niv).sum() / nulo25)\nguardar_json(res, RAIZ / "comparacion" / "comparacion_lineas.json")\nprint(json.dumps({k: v for k, v in res.items() if k != "fuentes"}, indent=1, default=float)[:9000])\n'


### Embedded source: `ml/codigo/comparar_lineas_2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/comparar_lineas_2.py'] = '"""Explore 2024 new-business segments weighted to the 2025 mix and a fixed 50/50 A1 blend."""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nimport receta\n\nOTROS = RAIZ / "comparacion" / "otros"\no = pd.read_parquet(OTROS / "predicciones_2024.parquet")\np = receta.parametros("dev", ["T2b"], "ramo"); t, mu = receta.aplicar("hold", p, None)\nd = t.merge(o[["row_id", "address_id", "A1", "R0"]], on="row_id").merge(pd.read_parquet(TRABAJO / "pesos_2025_hold.parquet"), on="row_id", how="left")\nd["T2b"] = mu * t.c_fila.values\nfor w in (0.25, 0.5, 0.75):\n    d[f"mez{int(w*100)}"] = w * d.T2b + (1 - w) * d.A1\ny = d.y.values\npadre = {}\ndef f(x):\n    while padre.setdefault(x, x) != x:\n        padre[x] = padre[padre[x]]; x = padre[x]\n    return x\nfor c, a in zip("C" + d.customer_id, "A" + d.address_id):\n    rc, ra = f(c), f(a)\n    if rc != ra: padre[ra] = rc\n_, gi = np.unique([f("C" + c) for c in d.customer_id], return_inverse=True)\nG = gi.max() + 1; rng = np.random.default_rng(7)\ndef boot(a, b, mask=None, w=None, B=2000):\n    m = np.ones(len(y), bool) if mask is None else mask\n    ww = np.ones(len(y)) if w is None else w\n    da = desviacion(y, d[a].values); db = desviacion(y, d[b].values); sy = np.sqrt(y); out = []\n    for s in range(0, B, 100):\n        W = rng.poisson(1.0, size=(100, G)).astype(float)[:, gi] * ww * m\n        ybar = (W @ y) / W.sum(1)\n        dn = 2 * ((W @ sy) / (-0.25) - (W @ y) * ybar ** -0.5 / (-0.5) + W.sum(1) * ybar ** 0.5 / 0.5)\n        out.append((W @ db - W @ da) / dn)\n    r = np.concatenate(out)\n    pt = d2(y[m], d[a].values[m], ww[m]) - d2(y[m], d[b].values[m], ww[m])\n    return {"dif": float(pt), "ic95": [float(np.quantile(r, .025)), float(np.quantile(r, .975))], "p_mayor": float((r > 0).mean())}\nres = {"D2": {}, "pares": {}}\nnb = d.new_business.values; w25 = d.w2025.values\nfor m in ["T2b", "A1", "mez25", "mez50", "mez75", "R0"]:\n    res["D2"][m] = {"todas": d2(y, d[m].values), "NB": d2(y[nb], d[m].values[nb]), "REN": d2(y[~nb], d[m].values[~nb]),\n                    "pond_2025": d2(y, d[m].values, w25)}\nres["pares"]["T2b-A1_NB"] = boot("T2b", "A1", mask=nb)\nres["pares"]["T2b-A1_REN"] = boot("T2b", "A1", mask=~nb)\nres["pares"]["T2b-A1_pond2025"] = boot("T2b", "A1", w=w25)\nres["pares"]["mez50-A1"] = boot("mez50", "A1")\nres["pares"]["mez50-T2b"] = boot("mez50", "T2b")\nres["pares"]["mez50-A1_pond2025"] = boot("mez50", "A1", w=w25)\nres["NB_filas_2024"] = int(nb.sum()); res["NB_siniestros_2024"] = int((y[nb] > 0).sum())\n# New-business 2024 line level: observed/predicted ratio.\nres["AE_NB"] = {m: {l: float(y[nb & (d.line == l).values].sum() / d[m].values[nb & (d.line == l).values].sum()) for l in LINEAS} for m in ["T2b", "A1"]}\nguardar_json(res, RAIZ / "comparacion" / "comparacion_lineas_2.json")\nprint(json.dumps(res, indent=1))\n'


### Embedded source: `ml/codigo/completitud.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/completitud.py'] = '"""Compute cost-weighted reporting completeness by line/peril, matching preparar_datos.py."""\nimport numpy as np, pandas as pd\nfrom comun import *\n\nD = 1095\n_G = None\n\n\ndef _estimar():\n    c = pd.read_parquet(DATOS / "developed" / "claims.parquet")\n    cols = [x for x in c.columns if x != "claim_id"]\n    c = c[~(c.duplicated(cols, keep=False) & (c.claim_id.str.len() == 11))]\n    c = c[c.status != "repudiated"].copy()\n    c["loss_date"] = pd.to_datetime(c.loss_date); c["report_date"] = pd.to_datetime(c.report_date)\n    c["neto"] = c.incurred_to_date - c.recoveries\n    G = {}\n    for (l, p), g in c.groupby(["line", "peril"]):\n        g = g[(V - g.loss_date.values.astype("datetime64[D]")).astype(int) >= D]\n        dly = (g.report_date - g.loss_date).dt.days.values; w = g.neto.values; ok = dly <= D\n        cum = np.clip(np.bincount(dly[ok], weights=w[ok], minlength=D + 1).cumsum() / w[ok].sum(), 0, 1)\n        G[(l, p)] = np.concatenate([[0.0], np.cumsum(np.concatenate([cum, np.ones(4000 - len(cum))]))])\n    return G\n\n\ndef por_peligro(linea, peligro, inicio, fin):\n    global _G\n    if _G is None:\n        _G = _estimar()\n    g = _G[(linea, peligro)]\n    s = pd.to_datetime(inicio).values.astype("datetime64[D]"); e = pd.to_datetime(fin).values.astype("datetime64[D]")\n    a = np.clip((V - e).astype(int), 0, None); b = np.clip((V - s).astype(int), 0, None)\n    return (g[b + 1] - g[a]) / (b - a + 1)\n'


### Embedded source: `ml/codigo/comun.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/comun.py'] = '"""Shared Week 2 transfer-model utilities.\nResolve paths, read fingerprint-verified data, compute Tweedie metrics and assign folds.\n"""\nfrom __future__ import annotations\nimport hashlib, json, os\nfrom pathlib import Path\nimport numpy as np\n\nRAIZ = Path(os.environ.get("MLIND_RAIZ", "/home/participant/ml"))\nDATOS = RAIZ / "datos"\nTRABAJO = RAIZ / "trabajo"\nENTREGA = RAIZ / "entrega"\nPRIVADO = RAIZ / "privado"\nV = np.datetime64("2024-12-31")\nLINEAS = ["motor", "buildings", "contents"]\nP = 1.5\n\nHUELLAS = {\n    "emerging/addresses.parquet": "4e9bf9d2a1777896e84fd4ee471096ff075aaefefd544f91b731324b25425632",\n    "emerging/claims.parquet": "1a2fce8a26cb1e05de1f53fe4e39bde772a1898e845755487625355fec4cc0e9",\n    "emerging/customers.parquet": "c00a0095d5fb7b2b65cd9647216add49afb3703d863d5c358b1441f0fab1fb62",\n    "emerging/expenses.parquet": "9033f666a5a45f0b44501d57e40ab1cc571c51c384f23a62942b75db323b1cdf",\n    "emerging/policies_buildings.parquet": "ccc948fafc11e623d5cb010c2b494f6813f6048719eae7bcde28794467598319",\n    "emerging/policies_contents.parquet": "84d7f48d03933b018bec8fdc2f4bbd96df184cca7ae1971c38d3acc955720f8e",\n    "emerging/policies_motor.parquet": "7c5dbb5e5b46629ebf01a66c991183d20b8585393b589bc2644fcbd6653a3ec1",\n    "emerging/reference_policies.parquet": "6d4c2839e87d092fccd4b3c3ddc16a2a4c4993c35e18481e322462cd382dd8f4",\n    "emerging/sample_submission.parquet": "160fc59043f1012337605c8d8446c0efbf81458e046cf02df9186d6f909d59ef",\n    "emerging/test_buildings.parquet": "21f91aaa66a1b9efd216ad0e04ff12a507dbeb721084330d9b9f06c99edd0ca6",\n    "emerging/test_contents.parquet": "af5ef2553e35a2d563ca091b173500daac55acc4969d1d3a96fa6b4ec3c6a012",\n    "emerging/test_motor.parquet": "108d5c68e9bab5ccd90ee5502bc3543aa515a5aa07fda44a3b227cf070a28aa5",\n    "emerging/weather.parquet": "645067069f68de8c4746d1ba48a454ff00155e1aa6fbbf64cdee48ae7269cbbd",\n    "developed/addresses.parquet": "2f1a9a157649a73ed1b0ee9b2caa3660b4c2a11766af3c64ab5131389079c0b1",\n    "developed/claims.parquet": "323adf2e0f6c5975a026479d725d4cd04f51fce3aa069f897a99f664c6e392c0",\n    "developed/customers.parquet": "065458a0b3aa9d3499e80b3b25b76baf0ef6245146477b6861f6ea35edf581eb",\n    "developed/policies_buildings.parquet": "eb84372c23c16ed0619d89c8eb1396ab6916e5ec46a5907ce42c3dfce4a91cbe",\n    "developed/policies_contents.parquet": "8c067cc80baed583100df999e1c3d5e389295702f7bac21d5331bbd639b83fcd",\n    "developed/policies_motor.parquet": "4d2d7ae03971d5d6df3a71bbaf0ebc168250a6807b27d1561d38b638434dc95e",\n    "developed/weather.parquet": "b8407f0f3c7279a49297c969a4ab2905641b232910e316fb806730065987cd49",\n}\n\n\ndef sha256(path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef comprobar_huellas() -> dict:\n    out = {}\n    for rel, esperado in HUELLAS.items():\n        obs = sha256(DATOS / rel)\n        if obs != esperado:\n            raise SystemExit(f"Huella distinta en {rel}: {obs}")\n        out[rel] = obs\n    return out\n\n\ndef pliegue(customer_ids) -> np.ndarray:\n    return np.array([int(hashlib.sha256(("mlind_v1|" + c).encode()).hexdigest()[:16], 16) % 5\n                     for c in customer_ids], dtype=np.int8)\n\n\ndef desviacion(y, mu, w=None, p=P):\n    y = np.asarray(y, float); mu = np.asarray(mu, float)\n    d = 2 * (np.power(np.maximum(y, 0), 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p)\n             + np.power(mu, 2 - p) / (2 - p))\n    return d if w is None else d * w\n\n\ndef d2(y, mu, w=None, p=P) -> float:\n    y = np.asarray(y, float); mu = np.asarray(mu, float)\n    w = np.ones_like(y) if w is None else np.asarray(w, float)\n    ybar = np.sum(w * y) / np.sum(w)\n    return 1 - np.sum(desviacion(y, mu, w, p)) / np.sum(desviacion(y, np.full_like(y, ybar), w, p))\n\n\ndef factor_tweedie(y, mu, w=None):\n    """Return k minimizing power-1.5 Tweedie deviance of k*mu."""\n    w = np.ones_like(np.asarray(y, float)) if w is None else w\n    return float(np.sum(w * y / np.sqrt(mu)) / np.sum(w * np.sqrt(mu)))\n\n\ndef guardar_json(obj, path):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8") as f:\n        json.dump(obj, f, ensure_ascii=False, indent=1, default=float)\n'


### Embedded source: `ml/codigo/conteos_peligro.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/conteos_peligro.py'] = '"""Count accepted claims by peril and policy row using the prepared-data assignment.\nWrite row_id and n_<peril> to trabajo/conteos_<market>_<line>.parquet. Check\nthat peril counts reproduce n_sin and peril costs reproduce y_<peril>.\n"""\nimport numpy as np, pandas as pd\nfrom comun import *\n\nfor m in ["developed", "emerging"]:\n    c = pd.read_parquet(DATOS / m / "claims.parquet")\n    if m == "developed":\n        cols = [x for x in c.columns if x != "claim_id"]\n        c = c[~(c.duplicated(cols, keep=False) & (c.claim_id.str.len() == 11))]\n    c = c[c.status != "repudiated"].copy()\n    c["loss_date"] = pd.to_datetime(c.loss_date)\n    c["neto"] = c.incurred_to_date - c.recoveries\n    for l in LINEAS:\n        b = pd.read_parquet(TRABAJO / f"base_{m}_{l}.parquet")\n        mm = c[c.line == l].merge(b[["row_id", "policy_id", "period_start", "period_end"]], on="policy_id")\n        mm = mm[(mm.loss_date >= pd.to_datetime(mm.period_start)) & (mm.loss_date <= pd.to_datetime(mm.period_end))]\n        out = pd.DataFrame({"row_id": b.row_id.values})\n        perils = sorted(c[c.line == l].peril.unique())\n        for p in perils:\n            g = mm[mm.peril == p]\n            out[f"n_{p}"] = out.row_id.map(g.groupby("row_id").size()).fillna(0).astype(int)\n            cost = out.row_id.map(g.groupby("row_id").neto.sum()).fillna(0.0)\n            tr = (b.conjunto == "train").values\n            assert np.allclose(cost.values[tr], b[f"y_{p}"].values[tr]), (m, l, p)\n        tr = (b.conjunto == "train").values\n        assert (out[[f"n_{p}" for p in perils]].sum(1).values[tr] == b.n_sin.values[tr]).all(), (m, l)\n        out.to_parquet(TRABAJO / f"conteos_{m}_{l}.parquet", index=False)\n        print(m, l, {p: int(out[f"n_{p}"].values[tr].sum()) for p in perils}, flush=True)\n'


### Embedded source: `ml/codigo/credibilidad_entre_anios.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/credibilidad_entre_anios.py'] = '"""Cross-year check: estimate the new-business factor from one PCV OOF year and apply it to the other."""\nimport numpy as np, pandas as pd\nfrom comun import *\nimport receta\nt, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")\nm = mu * t.c_fila.values; y = t.y.values; res = {}\nfor l in LINEAS:\n    il = (t.line == l).values; curva = {}\n    for z in np.round(np.arange(0, 1.01, 0.1), 1):\n        dv = 0.0\n        for a, b in [(2023, 2024), (2024, 2023)]:\n            ia = il & (t.year.values == a); ib = il & (t.year.values == b)\n            nb_a = ia & t.new_business.values; nb_b = ib & t.new_business.values\n            ka = factor_tweedie(y[ia], m[ia]); ks = factor_tweedie(y[nb_a], m[nb_a] * ka)\n            mb = m.copy(); mb[nb_b] = m[nb_b] * (1 + z * (ks - 1))\n            kb = factor_tweedie(y[ib], mb[ib])\n            dv += desviacion(y[ib], mb[ib] * kb).sum()\n        curva[float(z)] = dv\n    d0 = curva[0.0]\n    res[l] = {"z_optimo": min(curva, key=curva.get), "cambio_relativo_desviacion_por_10mil": {k: round((v - d0) / d0 * 1e4, 2) for k, v in curva.items()}}\n    print(l, res[l]["z_optimo"])\nguardar_json(res, ENTREGA / "credibilidad_entre_anios.json")\n'


### Embedded source: `ml/codigo/e09_e12_tabla.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e09_e12_tabla.py'] = '"""Compare E10-E12 with references in DEV (uncalibrated/C1) and HOLD (DEV-derived C1), using customer resampling."""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import metricas, bootstrap_dif\nimport receta\nres = {}\ntdev = receta.tabla("dev")\nfor fase in ["dev", "hold"]:\n    res[fase] = {}\n    for c in ["T2", "T2cap", "T2b", "T2bs4", "T2bw"]:\n        if fase == "dev":\n            t, mu, _ = receta.oof_cruzado("dev", [c], "ramo"); _, mu0, _ = receta.oof_cruzado("dev", [c], "sin")\n            res[fase][c] = {"C1": metricas(t, mu), "sin": metricas(t, mu0)}\n        else:\n            t, mu = receta.aplicar("hold", receta.parametros("dev", [c], "ramo"), None)\n            res[fase][c] = {"C1": metricas(t, mu)}\n        res[fase][c]["_mu"] = mu\n    for a, b in [("T2cap", "T2"), ("T2bs4", "T2b"), ("T2bw", "T2b")]:\n        res[fase][f"{a}-{b}"] = bootstrap_dif(t, res[fase][a]["_mu"], res[fase][b]["_mu"], B=1000)\n    for c in ["T2", "T2cap", "T2b", "T2bs4", "T2bw"]:\n        res[fase][c].pop("_mu")\nguardar_json(res, ENTREGA / "e10_e12_resultados.json")\nfor fase in res:\n    for k, v in res[fase].items():\n        if "-" in k: print(fase, k, round(v["dif"], 4), [round(x, 4) for x in v["ic95"]], v["p_mayor"])\n        else: print(fase, k, {kk: round(vv["D2"], 4) for kk, vv in v.items()}, {l: round(v["C1"][f"D2_{l}"], 4) for l in LINEAS})\n'


### Embedded source: `ml/codigo/e09_nivel.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e09_nivel.py'] = '"""E09: sensitivity to line and new-business levels for T2b+C1 in DEV 2023 and HOLD 2024."""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nimport receta\n\nres = {}\nfs = [0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.3]\nfor fase in ["dev", "hold"]:\n    if fase == "dev":\n        t, mu, _ = receta.oof_cruzado("dev", ["T2b"], "ramo")\n    else:\n        t, mu = receta.aplicar("hold", receta.parametros("dev", ["T2b"], "ramo"), None)\n    m = mu * t.c_fila.values; y = t.y.values; base = d2(y, m)\n    r = {"base": base, "ramo": {}, "nuevo_negocio": {}, "optimo": {}}\n    for l in LINEAS:\n        i = (t.line == l).values\n        r["ramo"][l] = {str(f): d2(y, np.where(i, m * f, m)) - base for f in fs}\n        r["nuevo_negocio"][l] = {str(f): d2(y, np.where(i & t.new_business.values, m * f, m)) - base for f in fs}\n        r["optimo"][l] = {"factor_tweedie": factor_tweedie(y[i], m[i]),\n                          "factor_tweedie_NB": factor_tweedie(y[i & t.new_business.values], m[i & t.new_business.values])}\n    res[fase] = r\nguardar_json(res, ENTREGA / "e09_nivel.json")\nfor fase in res:\n    print(fase, round(res[fase]["base"], 4))\n    for l in LINEAS:\n        print(" ", l, "ramo", {k: round(v, 4) for k, v in res[fase]["ramo"][l].items()})\n        print(" ", l, "NB  ", {k: round(v, 4) for k, v in res[fase]["nuevo_negocio"][l].items()}, {k: round(v, 3) for k, v in res[fase]["optimo"][l].items()})\n'


### Embedded source: `ml/codigo/e11_e12.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e11_e12.py'] = '"""E11 averages four T2b seeds; E12 reweights developed data toward emerging 2025.\nKeep T2b\'s DEV-fixed rounds. Save T2bs4/T2bw predictions in pred/dev and pred/hold.\n"""\nimport json\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom gbm import P_DEVB\n\nRONDAS = {"motor": 249, "buildings": 194, "contents": 169}\nout = {}\nfor l in LINEAS:\n    em = cargar("emerging", l, "hasta2023"); dv = cargar("developed", l, "hasta2023")\n    dv = dv[dv.year.isin([2021, 2022, 2023])].reset_index(drop=True)\n    cols = comunes(l)\n    cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)\n    Xd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats)\n    off = np.log(dv.exposure.values * dv.c_fila.values)\n    # E11: original seed 20261024 plus three additional seeds.\n    raws = []\n    for s in [20261024, 11, 12, 13]:\n        b = lgb.train(dict(P_DEVB[l], seed=s), lgb.Dataset(Xd, dv.y.values, init_score=off), RONDAS[l])\n        raws.append(b.predict(Xe, raw_score=True))\n    r_s4 = np.log(np.mean(np.exp(np.column_stack(raws)), axis=1))\n    # E12: developed-to-emerging-test density ratio using common non-history features.\n    cw = [c for c in cols if c not in ("hist_pol_12m", "hist_cli_12m")]\n    te = em[em.conjunto == "test"]\n    Xc = pd.concat([matriz(dv, cw, cats), matriz(te, cw, cats)], ignore_index=True)\n    z = np.r_[np.zeros(len(dv)), np.ones(len(te))]\n    pc = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8,\n              bagging_fraction=0.8, bagging_freq=1, verbose=-1, num_threads=2, seed=5)\n    fo = dv.fold.values\n    p = np.zeros(len(dv))\n    clf = lgb.train(pc, lgb.Dataset(Xc, z), 200)\n    p = clf.predict(matriz(dv, cw, cats))  # In-sample predictions are used for weighting, not evaluation.\n    w = p / (1 - p) * (len(dv) / len(te)); w = np.clip(w, 0.2, 5); w = w / w.mean()\n    ess = w.sum() ** 2 / (w ** 2).sum()\n    b = lgb.train(P_DEVB[l], lgb.Dataset(Xd, dv.y.values, weight=w, init_score=off), RONDAS[l])\n    r_w = b.predict(Xe, raw_score=True)\n    out[l] = {"ess_frac": float(ess / len(dv))}\n    for nom, r in [("T2bs4", r_s4), ("T2bw", r_w)]:\n        em["raw"] = r\n        tr = em[(em.conjunto == "train") & (em.year == 2023)].reset_index(drop=True)\n        for fase, sets in [("dev", [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)]),\n                           ("hold", [(tr, em[em.year == 2024].reset_index(drop=True))])]:\n            preds = []\n            for a, bb in sets:\n                k = factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)\n                preds.append(pd.DataFrame({"row_id": bb.row_id.values, "mu": k * np.exp(bb.raw.values) * bb.exposure.values}))\n            pd.concat(preds).to_parquet(TRABAJO / "pred" / fase / f"{nom}_{l}.parquet", index=False)\n    print(l, out[l], flush=True)\nguardar_json(out, ENTREGA / "e11_e12_info.json")\n'


### Embedded source: `ml/codigo/e13_clima_anual.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e13_clima_anual.py'] = '"""E13 exploratory variant: T2b with weather from the policy\'s own year."""\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom gbm import P_DEVB\nRONDAS = {"motor": 249, "buildings": 194, "contents": 169}\nVARS = ["hail_days", "wind_exposure_index", "freezing_days", "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]\nw = {m: pd.read_parquet(DATOS / m / "weather.parquet").rename(columns={v: "w_" + v for v in VARS}) for m in ["developed", "emerging"]}\nfor l in LINEAS:\n    em = cargar("emerging", l, "hasta2023"); dv = cargar("developed", l, "hasta2023")\n    dv = dv[dv.year.isin([2021, 2022, 2023])].reset_index(drop=True)\n    dv = dv.merge(w["developed"], left_on=["address_id", "year"], right_on=["address_id", "year"], how="left")\n    em = em.merge(w["emerging"], left_on=["address_id", "year"], right_on=["address_id", "year"], how="left")\n    cols = [c for c in comunes(l) if not c.startswith("clima_")] + ["w_" + v for v in VARS]\n    cats = categorias_union([dv, em], cols)\n    off = np.log(dv.exposure.values * dv.c_fila.values)\n    b = lgb.train(P_DEVB[l], lgb.Dataset(matriz(dv, cols, cats), dv.y.values, init_score=off), RONDAS[l])\n    em["raw"] = b.predict(matriz(em, cols, cats), raw_score=True)\n    tr = em[(em.conjunto == "train") & (em.year == 2023)].reset_index(drop=True)\n    for fase, sets in [("dev", [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)]), ("hold", [(tr, em[em.year == 2024].reset_index(drop=True))])]:\n        preds = []\n        for a, bb in sets:\n            k = factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)\n            preds.append(pd.DataFrame({"row_id": bb.row_id.values, "mu": k * np.exp(bb.raw.values) * bb.exposure.values}))\n        pd.concat(preds).to_parquet(TRABAJO / "pred" / fase / f"T2bclima_{l}.parquet", index=False)\n    imp = pd.Series(b.feature_importance("gain"), index=cols); imp = imp / imp.sum()\n    print(l, "ganancia clima del año", round(float(imp[[c for c in cols if c.startswith("w_")]].sum()), 3), flush=True)\n'


### Embedded source: `ml/codigo/e14_eval.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e14_eval.py'] = '"""Inspect E14-E18 individually and in blends in DEV OOF and exploratory HOLD.\nApply C1 by line and C2 by line/new-business with K=200. Fit convex weights\nout of fold in DEV and on all DEV for HOLD. Compare fitted weights (e0), 50%\nshrinkage toward equal weights (e50), and equal weights (e100).\nUsage: python e14_eval.py <output.json> <candidate> [...] [--mezclas A+B+C,...]\n"""\nfrom __future__ import annotations\nimport sys, json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import mezcla_pesos, metricas, bootstrap_dif, calibrar_cruzado\nfrom receta import tabla, componentes, K_NB\n\n\ndef pesos(t, P, comps, il, encoge):\n    M = np.column_stack([P[c][il] * t.c_fila.values[il] for c in comps])\n    w = mezcla_pesos(t.y.values[il], M) if len(comps) > 1 else np.ones(1)\n    return (1 - encoge) * w + encoge / len(comps)\n\n\ndef dev_oof(t, P, comps, encoge):\n    folds = pliegue(t.customer_id.values)\n    if len(comps) == 1:\n        return P[comps[0]], folds\n    U = np.column_stack([P[c] for c in comps]); out = np.zeros(len(t))\n    for k in np.unique(folds):\n        te = folds == k\n        for l in LINEAS:\n            il = (t.line == l).values\n            w = pesos(t[~te].reset_index(drop=True), {c: P[c][~te] for c in comps}, comps, il[~te], encoge)\n            out[te & il] = U[te & il] @ w\n    return out, folds\n\n\ndef hold_pred(td, Pd, th, Ph, comps, calib, encoge):\n    mu = np.zeros(len(th)); info = {}\n    for l in LINEAS:\n        il = (td.line == l).values; ih = (th.line == l).values\n        w = pesos(td, Pd, comps, il, encoge)\n        m = np.column_stack([Pd[c][il] for c in comps]) @ w * td.c_fila.values[il]\n        y = td.y.values[il]; kl = factor_tweedie(y, m)\n        base = np.column_stack([Ph[c][ih] for c in comps]) @ w\n        if calib == "ramo":\n            f = np.full(ih.sum(), kl)\n        else:\n            f = np.zeros(ih.sum())\n            for nb in (True, False):\n                s = td.new_business.values[il] == nb\n                ks = factor_tweedie(y[s], m[s] * kl); n = int((y[s] > 0).sum()); z = n / (n + K_NB)\n                f[th.new_business.values[ih] == nb] = kl * (z * ks + 1 - z)\n        mu[ih] = base * f\n        info[l] = {"pesos": dict(zip(comps, np.round(w, 4).tolist())), "k_ramo": kl}\n    return mu, info\n\n\ndef main():\n    salida = sys.argv[1]\n    args = sys.argv[2:]\n    mezclas = []\n    if "--mezclas" in args:\n        i = args.index("--mezclas"); mezclas = [m.split("+") for m in args[i + 1].split(",")]; args = args[:i]\n    cands = args if "T2b" in args else ["T2b"] + args\n    td = tabla("dev").merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")\n    th = tabla("hold")\n    todos = sorted(set(cands) | {c for m in mezclas for c in m} | {"T2b"})\n    Pd = componentes("dev", todos, td); Ph = componentes("hold", todos, th)\n    recetas = [([c], 0.0) for c in cands] + [(m, e) for m in mezclas for e in (0.0, 0.5, 1.0)]\n    res = {}; preds_dev = {}; preds_hold = {}\n    for comps, enc in recetas:\n        nom = "+".join(comps) + ("" if len(comps) == 1 else f"_e{int(enc * 100)}")\n        base, folds = dev_oof(td, Pd, comps, enc)\n        r = {"sin": metricas(td, base)} if len(comps) == 1 else {}\n        for calib in ("ramo", "ramo_nb"):\n            md = calibrar_cruzado(td, base, folds, por_nb=(calib == "ramo_nb"), K=K_NB)\n            mh, info = hold_pred(td, Pd, th, Ph, comps, calib, enc)\n            preds_dev[(nom, calib)] = md; preds_hold[(nom, calib)] = mh\n            r[calib] = {"dev": metricas(td, md), "hold": metricas(th, mh), "pesos_hold": info}\n        res[nom] = r\n        print(nom, "DEV C1 %.4f C2 %.4f w25 %.4f | HOLD C1 %.4f C2 %.4f" % (\n            r["ramo"]["dev"]["D2"], r["ramo_nb"]["dev"]["D2"], r["ramo"]["dev"]["D2_w2025"],\n            r["ramo"]["hold"]["D2"], r["ramo_nb"]["hold"]["D2"]), flush=True)\n    # Bootstrap against T2b under the same calibration.\n    boot = {}\n    for (nom, calib), md in preds_dev.items():\n        if nom == "T2b":\n            continue\n        boot[f"{nom}|{calib}"] = {\n            "dev": bootstrap_dif(td, md, preds_dev[("T2b", calib)], B=1000),\n            "hold": bootstrap_dif(th, preds_hold[(nom, calib)], preds_hold[("T2b", calib)], B=1000)}\n    # DES24 predictions from the meta-models.\n    des = {}\n    for c in todos:\n        for l in LINEAS:\n            f = TRABAJO / "pred" / "hold" / f"{c}_{l}_meta.json"\n            if f.exists():\n                m = json.load(open(f))\n                if "DES24" in m:\n                    des.setdefault(c, {})[l] = m["DES24"]["D2"]\n    guardar_json({"resultados": res, "bootstrap_vs_T2b": boot, "DES24": des}, ENTREGA / salida)\n    print(json.dumps({k: {kk: [round(v["dif"], 4), [round(x, 4) for x in v["ic95"]]] for kk, v in b.items()} for k, b in boot.items()}, indent=0))\n    print("DES24", json.dumps({c: {l: round(v, 4) for l, v in d.items()} for c, d in des.items()}))\n\n\nif __name__ == "__main__":\n    main()\n'


### Embedded source: `ml/codigo/e25_severidad.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e25_severidad.py'] = '"""E25: select gamma-severity grid settings on developed 2021-2023 inner fold 0.\nWrite validation deviance and the chosen setting per line under trabajo/modelos/dev.\nDo not inspect developed 2024 or emerging data.\nUsage: python e25_severidad.py [<lines>]\n"""\nimport sys, json, itertools, time\nimport numpy as np, lightgbm as lgb\nfrom comun import *\nfrom variables import *\n\nSEMILLA = 20261024\nREJILLA = list(itertools.product([7, 15, 31, 63], [50, 200, 1000], [1.0, 10.0, 100.0]))\n\n\ndef dev_gamma(y, mu, w):\n    return float(np.sum(w * 2 * (-np.log(y / mu) + (y - mu) / mu)) / np.sum(w))\n\n\ndef correr(linea):\n    t0 = time.time()\n    em = cargar("emerging", linea, "hasta2023"); dv = cargar("developed", linea, "hasta2023")\n    dv = dv[dv.year.isin([2021, 2022, 2023])].reset_index(drop=True)\n    cols = comunes(linea); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)\n    s = (dv.n_sin.values > 0) & (dv.y.values > 0)\n    X = matriz(dv, cols, cats)[s].reset_index(drop=True)\n    n = dv.n_sin.values[s].astype(float); ysev = dv.y.values[s] / n\n    val = (dv.fold.values == 0)[s]\n    msev = float(dv.y.values[s].sum() / n.sum())\n    off = np.full(len(ysev), np.log(msev))\n    filas = []\n    for hojas, mind, l2 in REJILLA:\n        p = dict(objective="gamma", learning_rate=0.03, num_leaves=hojas, min_data_in_leaf=mind, lambda_l2=l2,\n                 feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, max_cat_to_onehot=8, cat_smooth=50,\n                 verbose=-1, num_threads=2, seed=SEMILLA)\n        dtr = lgb.Dataset(X[~val], ysev[~val], init_score=off[~val], weight=n[~val], free_raw_data=False)\n        dva = lgb.Dataset(X[val], ysev[val], init_score=off[val], weight=n[val], reference=dtr)\n        b = lgb.train(p, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])\n        mu = np.exp(b.predict(X[val], raw_score=True, num_iteration=b.best_iteration) + off[val])\n        filas.append({"num_leaves": hojas, "min_data_in_leaf": mind, "lambda_l2": l2, "rondas": int(b.best_iteration),\n                      "desv_val": dev_gamma(ysev[val], mu, n[val])})\n    base = dev_gamma(ysev[val], np.full(val.sum(), np.average(ysev[~val], weights=n[~val])), n[val])\n    mejor = min(filas, key=lambda r: r["desv_val"])\n    actual = [r for r in filas if (r["num_leaves"], r["min_data_in_leaf"], r["lambda_l2"]) == (31, 200, 10.0)][0]\n    guardar_json({"linea": linea, "filas_val": int(val.sum()), "siniestros_val": float(n[val].sum()), "desv_constante": base,\n                  "rejilla": filas, "elegida": mejor, "FS_actual": actual, "s": round(time.time() - t0)},\n                 TRABAJO / "modelos" / "dev" / f"e25_rejilla_{linea}.json")\n    print(linea, "constante %.5f" % base, "FS actual", actual, "elegida", mejor, round(time.time() - t0), "s", flush=True)\n\n\nif __name__ == "__main__":\n    for l in (sys.argv[1].split(",") if len(sys.argv) > 1 else LINEAS):\n        correr(l)\n'


### Embedded source: `ml/codigo/e26_hold.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e26_hold.py'] = '"""E26 exploratory HOLD: fixed DES24 weights or DEV e50 weights, with DEV-derived C1."""\nimport sys, json, numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import metricas, bootstrap_dif, mezcla_pesos\nfrom receta import tabla, componentes\nfij = json.load(open(ENTREGA / "evolucion" / "e26_pesos_des24.json"))\ntd = tabla("dev"); th = tabla("hold")\ndef pred(comps, modo):\n    Pd = componentes("dev", comps, td); Ph = componentes("hold", comps, th); mu = np.zeros(len(th))\n    for l in LINEAS:\n        il = (td.line == l).values; ih = (th.line == l).values\n        M = np.column_stack([Pd[c][il] * td.c_fila.values[il] for c in comps])\n        if modo == "des24":\n            w = np.array(list(fij["ramos"][l]["pesos"].values()))\n        else:\n            w = mezcla_pesos(td.y.values[il], M) if len(comps) > 1 else np.ones(1)\n            w = 0.5 * w + 0.5 / len(comps) if len(comps) > 1 else w\n        k = factor_tweedie(td.y.values[il], M @ w)\n        mu[ih] = np.column_stack([Ph[c][ih] for c in comps]) @ w * k\n    return mu\nR = {"T2b": pred(["T2b"], "e50"), "v2": pred(["T2b", "T2h", "FS", "T2o"], "e50"),\n     "v2t": pred(["T2b", "T2h", "FSt", "T2o"], "e50"), "v3": pred(["T2b", "T2h", "FSt", "T2o"], "des24"), "FSt": pred(["FSt"], "e50")}\nout = {}\nfor k, v in R.items():\n    m = metricas(th, v); out[k] = {"D2": m["D2"], **{f"D2_{l}": m[f"D2_{l}"] for l in LINEAS}}\n    if k != "T2b":\n        out[k]["vs_T2b"] = bootstrap_dif(th, v, R["T2b"])\n    if k in ("v2t", "v3"):\n        out[k]["vs_v2"] = bootstrap_dif(th, v, R["v2"])\n    print(k, {a: (round(b, 4) if isinstance(b, float) else (round(b["dif"], 4), [round(x, 4) for x in b["ic95"]])) for a, b in out[k].items()})\nguardar_json(out, ENTREGA / "evolucion" / "e26_hold.json")\n'


### Embedded source: `ml/codigo/e26_pesos_des24.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e26_pesos_des24.py'] = '"""E26: fit convex component weights on developed 2024 (DES24).\nComponents train on 2021-2023. Relevel each by its line Tweedie factor and\nminimize power-1.5 blend deviance. Also report customer-fold-crossed weight\nperformance within 2024 as an overfitting diagnostic.\nUsage: python e26_pesos_des24.py A,B,C,D <output.json>\n"""\nimport sys, json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import mezcla_pesos\n\nCOMPS = sys.argv[1].split(","); SALIDA = sys.argv[2]\nres = {"componentes": COMPS, "ramos": {}}\nfor l in LINEAS:\n    base = None; M = []\n    for c in COMPS:\n        d = pd.read_parquet(TRABAJO / "pred" / "hold" / f"{c}des24_{l}.parquet")\n        if base is None:\n            base = d\n        assert (d.row_id.values == base.row_id.values).all()\n        mu = np.exp(d.raw.values) * d.exposure.values * d.c_fila.values\n        M.append(mu * factor_tweedie(d.y.values, mu))\n    M = np.column_stack(M); y = base.y.values\n    w = mezcla_pesos(y, M)\n    cust = pd.read_parquet(TRABAJO / f"base_developed_{l}.parquet", columns=["row_id", "customer_id"])\n    folds = pliegue(base[["row_id"]].merge(cust, on="row_id", how="left").customer_id.values)\n    cruz = np.zeros(len(y))\n    for f in range(5):\n        te = folds == f\n        cruz[te] = M[te] @ mezcla_pesos(y[~te], M[~te])\n    ind = {c: d2(y, M[:, i]) for i, c in enumerate(COMPS)}\n    eq = M.mean(1)\n    res["ramos"][l] = {"pesos": dict(zip(COMPS, np.round(w, 4).tolist())), "D2_individual": ind,\n                       "D2_mezcla_en_muestra": d2(y, M @ w), "D2_mezcla_cruzada": d2(y, cruz),\n                       "D2_pesos_iguales": d2(y, eq), "filas": int(len(y))}\n    print(l, {c: round(v, 4) for c, v in ind.items()}, "pesos", dict(zip(COMPS, np.round(w, 3))),\n          "mezcla %.4f cruzada %.4f iguales %.4f" % (d2(y, M @ w), d2(y, cruz), d2(y, eq)), flush=True)\nguardar_json(res, ENTREGA / "evolucion" / SALIDA)\n'


### Embedded source: `ml/codigo/e27_backtest.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e27_backtest.py'] = '"""E27 preregistered exploratory backtest: apply the 2023-only rule to HOLD T2b.\nWith one year, between-year variance is unavailable; declare customer-bootstrap variance.\n"""\nimport json, numpy as np, pandas as pd\nfrom comun import *\nfrom receta import tabla, componentes\nTAU, K = 0.07, 200.0\nrows = []\nfor l in LINEAS:\n    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "customer_id", "year", "conjunto", "y", "n_sin", "c_fila", "exposure", "written_premium", "new_business"])\n    rows.append(b.assign(line=l))\nd = pd.concat(rows, ignore_index=True); d["seg"] = d.line + "|" + np.where(d.new_business, "NB", "REN"); d["P"] = d.written_premium * d.exposure\nh = d[(d.conjunto == "train") & (d.year == 2023)].reset_index(drop=True); h["U"] = h.y / h.c_fila\nth = tabla("hold"); mu = componentes("hold", ["T2b"], th)["T2b"]\nt = th.merge(d[["row_id", "seg", "P"]], on="row_id", how="left"); t["mu"] = mu\ndef lrs(w):\n    o = {}\n    for l in LINEAS:\n        il = (h.line == l).values; lr_l = np.sum(w[il] * h.U.values[il]) / np.sum(w[il] * h.P.values[il])\n        for s in ("NB", "REN"):\n            m = il & (h.seg == f"{l}|{s}").values\n            n = np.sum(w[m] * h.n_sin.values[m]); z = n / (n + K)\n            o[f"{l}|{s}"] = z * np.sum(w[m] * h.U.values[m]) / np.sum(w[m] * h.P.values[m]) + (1 - z) * lr_l\n    return o\nc0 = lrs(np.ones(len(h))); cli, idx = np.unique(h.customer_id.values, return_inverse=True)\nrng = np.random.default_rng(20261024); sims = [lrs(rng.poisson(1.0, len(cli))[idx].astype(float)) for _ in range(500)]\nf = {}; res = {}\nfor s in c0:\n    m = (t.seg == s).values\n    g = c0[s] * t.P.values[m].sum() / t.mu.values[m].sum()\n    sig2 = np.var([np.log(x[s]) for x in sims]); z = TAU ** 2 / (TAU ** 2 + sig2)\n    f[s] = float(np.clip(np.exp(z * np.log(g)), 0.85, 1.15))\n    ae = t.y.values[m].sum() / (t.mu.values[m] * t.c_fila.values[m]).sum()\n    res[s] = {"g": g, "sigma_boot": float(np.sqrt(sig2)), "Z": z, "f": f[s], "AE_T2b_2024": float(ae)}\nmu2 = t.mu.values * t.seg.map(f).values\nout = {"segmentos": res, "D2_T2b": d2(t.y.values, t.mu.values * t.c_fila.values), "D2_con_f": d2(t.y.values, mu2 * t.c_fila.values),\n       "D2_con_g_pleno": d2(t.y.values, t.mu.values * t.seg.map({s: r["g"] for s, r in res.items()}).values * t.c_fila.values)}\nfor s, r in res.items(): print(s, {k: round(v, 3) for k, v in r.items()})\nprint("D2 2024: T2b %.4f, con f %.4f, con g sin credibilidad %.4f" % (out["D2_T2b"], out["D2_con_f"], out["D2_con_g_pleno"]))\nguardar_json(out, ENTREGA / "nivel2025" / "e27_backtest_2024.json")\n'


### Embedded source: `ml/codigo/e27_nivel.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/e27_nivel.py'] = '"""E27: derive 2025 line/segment levels from expected losses over written premium.\nFollow the frozen preregistration without leaderboard information. Save the\naggregate record and private base-times-factor predictions.\n"""\nimport json, hashlib\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom scipy.stats import spearmanr\n\nPRE = ENTREGA / "nivel2025" / "PREINSCRIPCION_NIVEL.md"\nassert sha256(PRE).startswith("13d2ef2f1596621f"), "la preinscripción cambió"\nBASE = RAIZ / "privado" / "mezcla50_designada_mlind_2025.csv"\nassert sha256(BASE).startswith("12831ef4a05c1773"), "la base no es la mezcla enviada"\nTAU, K, TOPE, B_BOOT, SEM = 0.07, 200.0, (0.85, 1.15), 1000, 20261024\n\nfilas = []\nfor l in LINEAS:\n    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet",\n                        columns=["row_id", "customer_id", "year", "conjunto", "y", "n_sin", "c_fila", "exposure",\n                                 "written_premium", "new_business"])\n    filas.append(b.assign(line=l))\nd = pd.concat(filas, ignore_index=True)\nd["seg"] = d.line + "|" + np.where(d.new_business, "NB", "REN")\nd["P"] = d.written_premium * d.exposure\nh = d[(d.conjunto == "train") & d.year.isin([2023, 2024])].reset_index(drop=True)\nh["U"] = h.y / h.c_fila\nt = d[d.conjunto == "test"].reset_index(drop=True)\nbase = pd.read_csv(BASE)\nt = t.merge(base, on="row_id", how="left"); assert t.prediction.notna().all() and len(t) == 209063\nSEGS = sorted(h.seg.unique())\n\n\ndef lr_estrella(hh, w=None, k=K):\n    w = np.ones(len(hh)) if w is None else w\n    out, raw = {}, {}\n    for l in LINEAS:\n        il = (hh.line == l).values\n        lr_l = np.sum(w[il] * hh.U.values[il]) / np.sum(w[il] * hh.P.values[il])\n        for s in ("NB", "REN"):\n            m = il & (hh.seg == f"{l}|{s}").values\n            lr_s = np.sum(w[m] * hh.U.values[m]) / np.sum(w[m] * hh.P.values[m])\n            n = np.sum(w[m] * hh.n_sin.values[m]); z = n / (n + k)\n            out[f"{l}|{s}"] = z * lr_s + (1 - z) * lr_l; raw[f"{l}|{s}"] = (lr_s, lr_l, n, z)\n    return out, raw\n\n\ndef factores(hh, tt, tau=TAU, k=K, boot=True):\n    lrs, raw = lr_estrella(hh, k=k)\n    # Customer-bootstrap variance using Poisson weights.\n    cli, idx = np.unique(hh.customer_id.values, return_inverse=True)\n    rng = np.random.default_rng(SEM)\n    sims = {s: [] for s in SEGS}\n    if boot:\n        for _ in range(B_BOOT):\n            w = rng.poisson(1.0, len(cli))[idx].astype(float)\n            o, _ = lr_estrella(hh, w, k=k)\n            for s in SEGS:\n                sims[s].append(np.log(o[s]))\n    res = {}\n    for s in SEGS:\n        l = s.split("|")[0]\n        yr = {}\n        for a in (2023, 2024):\n            m = (hh.seg == s).values & (hh.year == a).values\n            yr[a] = hh.U.values[m].sum() / hh.P.values[m].sum()\n        var_boot = float(np.var(sims[s])) if boot else np.nan\n        var_anio = 0.5 * (np.log(yr[2024]) - np.log(yr[2023])) ** 2\n        sig2 = var_boot + var_anio\n        mt = (tt.seg == s).values\n        P25 = tt.P.values[mt].sum(); Bs = tt.prediction.values[mt].sum()\n        E = lrs[s] * P25; g = E / Bs\n        z = tau ** 2 / (tau ** 2 + sig2)\n        f = float(np.clip(np.exp(z * np.log(g)), *TOPE))\n        res[s] = {"LR_segmento": raw[s][0], "LR_ramo": raw[s][1], "siniestros": raw[s][2], "Z_n": raw[s][3],\n                  "LR_estrella": lrs[s], "LR_2023": yr[2023], "LR_2024": yr[2024], "var_boot": var_boot,\n                  "var_anio": var_anio, "sigma": float(np.sqrt(sig2)), "P25": P25, "base_total": Bs, "E_tarifa": E,\n                  "g": g, "Z": z, "f": f, "filas_2025": int(mt.sum())}\n    return res\n\n\nR = factores(h, t)\nout = {"preinscripcion_sha256": sha256(PRE), "base_sha256": sha256(BASE), "tau": TAU, "K": K, "tope": TOPE, "segmentos": R}\nfor s, r in R.items():\n    print(s, "LR %.3f (2023 %.3f, 2024 %.3f) LR* %.3f | g %.3f sigma %.3f Z %.3f -> f %.4f" % (\n        r["LR_segmento"], r["LR_2023"], r["LR_2024"], r["LR_estrella"], r["g"], r["sigma"], r["Z"], r["f"]))\n\n# K sensitivities reuse central-case variance instead of running another bootstrap.\nsens = {}\nfor tau in (0.05, 0.10):\n    sens[f"tau_{tau}"] = {s: float(np.clip(np.exp(tau ** 2 / (tau ** 2 + R[s]["sigma"] ** 2) * np.log(R[s]["g"])), *TOPE)) for s in SEGS}\nfor k in (100.0, 400.0):\n    Rk = factores(h, t, k=k, boot=True)\n    sens[f"K_{int(k)}"] = {s: Rk[s]["f"] for s in SEGS}\nout["sensibilidad"] = sens\nprint("sens", json.dumps({a: {s: round(v, 4) for s, v in b.items()} for a, b in sens.items()}))\n\n# E27 output: base predictions times segment factors.\nt["f"] = t.seg.map({s: R[s]["f"] for s in SEGS})\nsub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")\no = sub[["row_id"]].merge(t[["row_id", "prediction", "f"]], on="row_id", how="left")\no["prediction"] = o.prediction * o.f\no = o[["row_id", "prediction"]]\nassert len(o) == 209063 and (o.row_id.values == sub.row_id.values).all() and o.prediction.notna().all() and (o.prediction > 0).all()\nf_out = RAIZ / "privado" / "e27_nivel_2025.csv"\no.to_csv(f_out, index=False, float_format="%.6f", lineterminator="\\n")\nout["entrega"] = {"archivo": f_out.name, "sha256": sha256(f_out), "total": float(o.prediction.sum()),\n                  "total_base": float(base.prediction.sum()),\n                  "por_ramo": {l: float(o.prediction.values[t.set_index("row_id").loc[o.row_id, "line"].values == l].sum()) for l in LINEAS}}\nprint("entrega", out["entrega"])\n\n# Compare tariff/base risk ordering for new business in 2025 and 2023-2024 PCV OOF.\npcv = pd.concat([pd.read_parquet(TRABAJO / "pred" / "pcv" / f"T2b_{l}.parquet") for l in LINEAS])\nhh = h.merge(pcv, on="row_id", how="left")\ndiag = {}\nfor l in LINEAS:\n    for s in ("NB", "REN"):\n        m25 = (t.seg == f"{l}|{s}").values; m23 = (hh.seg == f"{l}|{s}").values\n        diag[f"{l}|{s}"] = {\n            "spearman_2025": float(spearmanr(t.written_premium.values[m25], t.prediction.values[m25] / t.exposure.values[m25])[0]),\n            "spearman_2023_24": float(spearmanr(hh.written_premium.values[m23], hh.mu.values[m23] / hh.exposure.values[m23])[0]),\n            "prima_media_2025": float(t.written_premium.values[m25].mean()),\n            "prima_media_2023_24": float(hh.written_premium.values[m23].mean()),\n            "base_por_exposicion_2025": float(t.prediction.values[m25].sum() / t.exposure.values[m25].sum()),\n            "T2b_por_exposicion_2023_24": float(hh.mu.values[m23].sum() / hh.exposure.values[m23].sum())}\nout["diagnostico_orden"] = diag\nprint("orden", json.dumps({s: {k: round(v, 3) for k, v in d_.items()} for s, d_ in diag.items()}))\nguardar_json(out, ENTREGA / "nivel2025" / "e27_nivel.json")\n'


### Embedded source: `ml/codigo/evaluar.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/evaluar.py'] = '"""Evaluate saved predictions from the Week 2 transfer-model family."""\nfrom __future__ import annotations\nimport sys\nimport numpy as np, pandas as pd\nfrom comun import *\n\ndef tabla_eval(fase, anio):\n    filas = []\n    for l in LINEAS:\n        b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet",\n                            columns=["row_id", "y", "c_fila", "exposure", "customer_id", "year", "new_business", "conjunto"])\n        b = b[(b.conjunto == "train") & (b.year == anio)]\n        filas.append(b.assign(line=l))\n    return pd.concat(filas, ignore_index=True)\n\n\ndef cargar_pred(fase, cand):\n    out = []\n    for l in LINEAS:\n        f = TRABAJO / "pred" / fase / f"{cand}_{l}.parquet"\n        if not f.exists():\n            return None\n        out.append(pd.read_parquet(f))\n    return pd.concat(out, ignore_index=True)\n\n\ndef cuerpo_mask(t, q=0.995):\n    m = np.ones(len(t), bool)\n    for l in LINEAS:\n        i = (t.line == l).values & (t.y.values > 0)\n        thr = np.quantile(t.y.values[i], q)\n        m &= ~((t.line == l).values & (t.y.values > thr))\n    return m\n\n\ndef metricas(t, mu, w=None):\n    y = t.y.values; m = mu * t.c_fila.values\n    r = {"D2": d2(y, m, w)}\n    if "w2025" in t:\n        r["D2_w2025"] = d2(y, m, t.w2025.values)\n    for l in LINEAS:\n        i = (t.line == l).values\n        r[f"D2_{l}"] = d2(y[i], m[i], None if w is None else w[i])\n        r[f"AE_{l}"] = float(y[i].sum() / m[i].sum())\n    cm = cuerpo_mask(t)\n    mb = m.copy()\n    for l in LINEAS:\n        i = cm & (t.line == l).values\n        mb[i] *= y[i].sum() / m[i].sum()\n    r["D2_cuerpo"] = d2(y[cm], mb[cm])\n    for nb in (True, False):\n        i = t.new_business.values == nb\n        r[f"AE_{\'NB\' if nb else \'REN\'}"] = float(y[i].sum() / m[i].sum())\n    return r\n\n\nif __name__ == "__main__":\n    fase = sys.argv[1]; anio = 2023 if fase == "dev" else 2024\n    t = tabla_eval(fase, anio)\n    filas = []\n    for f in sorted((TRABAJO / "pred" / fase).glob("*_motor.parquet")):\n        cand = f.name.replace("_motor.parquet", "")\n        p = cargar_pred(fase, cand)\n        if p is None: continue\n        mu = t[["row_id"]].merge(p, on="row_id", how="left").mu.values\n        if np.isnan(mu).any(): continue\n        filas.append({"cand": cand, **metricas(t, mu)})\n    df = pd.DataFrame(filas).set_index("cand").sort_values("D2", ascending=False)\n    pd.set_option("display.width", 250)\n    print(df.round(4))\n\n\ndef calibrar_cruzado(t, mu, folds, por_nb=False, K=200.0):\n    """Estimate line and optional credibility-adjusted business factors on other folds; apply to the held-out fold."""\n    m = mu * t.c_fila.values; y = t.y.values\n    out = mu.copy()\n    for k in np.unique(folds):\n        te = folds == k; trm = ~te\n        for l in LINEAS:\n            il = (t.line == l).values\n            kl = factor_tweedie(y[trm & il], m[trm & il])\n            if not por_nb:\n                out[te & il] = mu[te & il] * kl\n            else:\n                for nb in (True, False):\n                    s = (t.new_business.values == nb)\n                    ks = factor_tweedie(y[trm & il & s], m[trm & il & s] * kl)\n                    nsin = np.sum((y[trm & il & s] > 0))\n                    z = nsin / (nsin + K)\n                    out[te & il & s] = mu[te & il & s] * kl * (z * ks + (1 - z))\n    return out\n\n\ndef tabla_dev(cands, por_nb=False):\n    t = tabla_eval("dev", 2023)\n    t = t.merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")\n    from comun import pliegue\n    folds = pliegue(t.customer_id.values)\n    filas = []\n    for c in cands:\n        p = cargar_pred("dev", c)\n        if p is None: continue\n        mu = t[["row_id"]].merge(p, on="row_id", how="left").mu.values\n        filas.append({"cand": c, "calib": "sin", **metricas(t, mu)})\n        filas.append({"cand": c, "calib": "ramo", **metricas(t, calibrar_cruzado(t, mu, folds))})\n        if por_nb:\n            filas.append({"cand": c, "calib": "ramo_nb", **metricas(t, calibrar_cruzado(t, mu, folds, True))})\n    return pd.DataFrame(filas)\n\n\ndef mezcla_pesos(Y, M):\n    """Fit convex weights minimizing power-1.5 deviance of M@w on observed-scale predictions."""\n    from scipy.optimize import minimize\n    k = M.shape[1]\n    if k == 1:\n        return np.ones(1)\n    def f(w):\n        mu = np.maximum(M @ w, 1e-9)\n        return np.sum(2 * (2 * Y / np.sqrt(mu) + 2 * np.sqrt(mu))), M.T @ (2 * (-Y / mu ** 1.5 + 1 / np.sqrt(mu)))\n    cons = ({"type": "eq", "fun": lambda w: w.sum() - 1, "jac": lambda w: np.ones(k)},)\n    r = minimize(f, np.ones(k) / k, jac=True, method="SLSQP", bounds=[(0, 1)] * k, constraints=cons)\n    return np.clip(r.x, 0, 1) / np.clip(r.x, 0, 1).sum()\n\n\ndef mezcla_cruzada(t, preds, folds):\n    """Estimate line weights on four folds and apply to the fifth; candidate arrays must align with t."""\n    nombres = list(preds)\n    M = np.column_stack([preds[c] * t.c_fila.values for c in nombres])\n    U = np.column_stack([preds[c] for c in nombres])\n    y = t.y.values; out = np.zeros(len(t)); pesos = []\n    for k in np.unique(folds):\n        te = folds == k\n        for l in LINEAS:\n            il = (t.line == l).values\n            w = mezcla_pesos(y[~te & il], M[~te & il])\n            out[te & il] = U[te & il] @ w\n            pesos.append({"pliegue": int(k), "ramo": l, **dict(zip(nombres, np.round(w, 4)))})\n    return out, pesos\n\n\ndef bootstrap_dif(t, mu_a, mu_b, B=1000, semilla=2026, w=None):\n    """Customer Poisson-bootstrap D2(a)-D2(b); return mean, 95% interval and P(a>b)."""\n    rng = np.random.default_rng(semilla)\n    cli, idx = np.unique(t.customer_id.values, return_inverse=True)\n    y = t.y.values; ma = mu_a * t.c_fila.values; mb = mu_b * t.c_fila.values\n    base = np.ones(len(y)) if w is None else w\n    da = desviacion(y, ma); db = desviacion(y, mb)\n    sy = np.sqrt(y)\n    res = []\n    for s in range(0, B, 100):\n        Wc = rng.poisson(1.0, size=(min(100, B - s), len(cli))).astype(float)\n        W = Wc[:, idx] * base\n        ybar = (W @ y) / W.sum(1)\n        dn = 2 * ((W @ sy) / (-0.25) - (W @ y) * ybar ** -0.5 / (-0.5) + W.sum(1) * ybar ** 0.5 / 0.5)\n        res.append((1 - (W @ da) / dn) - (1 - (W @ db) / dn))\n    r = np.concatenate(res)\n    return {"dif": float(d2(y, ma, w) - d2(y, mb, w)), "ic95": [float(np.quantile(r, 0.025)), float(np.quantile(r, 0.975))],\n            "p_mayor": float(np.mean(r > 0))}\n'


### Embedded source: `ml/codigo/figura_comparacion.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figura_comparacion.py'] = 'import json\nimport numpy as np, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom comun import *\nc1 = json.load(open(RAIZ / "comparacion" / "comparacion_lineas.json")); c2 = json.load(open(RAIZ / "comparacion" / "comparacion_lineas_2.json"))\nplt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})\nfig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6))\nseg = [("todas", "todas las filas"), ("NB", "nuevo negocio"), ("REN", "renovación"), ("pond_2025", "ponderado a 2025")]\nmods = [("R0", "Tasa por ramo", "#bbbbbb"), ("A1", "A1 (base de la designada)", "#c4673b"), ("T2b", "MLind T2b (C1)", "#2a6f97"), ("mez50", "mezcla 50/50", "#5b8c5a")]\nx = np.arange(len(seg)); w = 0.2\nfor j, (k, lab, col) in enumerate(mods):\n    axs[0].bar(x + (j - 1.5) * w, [c2["D2"][k][s] for s, _ in seg], w, label=lab, color=col)\naxs[0].set_xticks(x, [s for _, s in seg]); axs[0].set_ylabel("D² Tweedie 1,5 en 2024")\naxs[0].set_ylim(0.05, 0.15); axs[0].legend(frameon=False, fontsize=7, loc="upper center", ncol=2)\naxs[0].set_title("2024, mismas 59.202 filas (exploratorio: las dos líneas ya lo habían visto)", fontsize=8.5)\nv = c1["2025"]; ks = [("noct_R0", "R0", "#bbbbbb"), ("noct_A1fixF1b3n_designada", "designada A1fixF1b3n", "#c4673b"),\n                      ("MLind_final", "MLind final", "#2a6f97"), ("MLind_C2", "MLind con C2", "#9bb7cc")]\nx = np.arange(3)\nfor j, (k, lab, col) in enumerate(ks):\n    axs[1].bar(x + (j - 1.5) * w, [v[k][l]["tasa"] for l in LINEAS], w, label=lab, color=col)\naxs[1].set_xticks(x, LINEAS); axs[1].set_ylabel("tasa 2025 predicha (CU por año de exposición)")\naxs[1].legend(frameon=False, fontsize=7); axs[1].set_title("Producción 2025: el nivel es lo que más separa a las líneas", fontsize=8.5)\nfig.tight_layout(); fig.savefig(RAIZ / "comparacion" / "f10_comparacion_lineas.png"); plt.close(fig)\n'


### Embedded source: `ml/codigo/figura_evolucion.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figura_evolucion.py'] = '"""Figure f12: E14-E22 versus T2b in DEV 2023, HOLD 2024 and developed DES24."""\nimport json\nimport numpy as np, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom comun import *\n\nE = ENTREGA\nfuentes = ["e14_individuales.json", "e14_fs.json", "e15_cb.json", "e21_fsp.json", "e22_mezcla.json"]\nboot, des = {}, {}\nfor f in fuentes:\n    r = json.load(open(E / f))\n    for k, v in r["bootstrap_vs_T2b"].items():\n        nom, cal = k.split("|")\n        if cal == "ramo":\n            boot[nom] = v\n    des.update(r["DES24"])\nfilas = [("T2x", "T2x  +make_model y tenencia"), ("CB", "CB  CatBoost Tweedie"), ("FSp", "FSp  frec. x sev. por peligro"),\n         ("T2o", "T2o  sin occupation"), ("T2c", "T2c  sin historia"), ("T2h", "T2h  historia centrada"),\n         ("FS", "FS  frecuencia x severidad"), ("T2b+T2h+FS+T2o_e50", "Mezcla v2 (E22, e50)")]\nC = {"bg": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df",\n     "s1": "#2a78d6", "s2": "#eb6834", "s3": "#1baf7a"}\nplt.rcParams.update({"font.size": 9.5, "axes.edgecolor": C["grid"], "axes.labelcolor": C["ink2"],\n                     "xtick.color": C["ink2"], "ytick.color": C["ink"], "font.family": "DejaVu Sans"})\nfig, axs = plt.subplots(1, 2, figsize=(12.5, 5.2), facecolor=C["bg"], gridspec_kw={"width_ratios": [1.15, 1]})\ny = np.arange(len(filas))[::-1]\nax = axs[0]\nfor j, (fase, lab, col, mk) in enumerate([("dev", "2023 (DEV, OOF)", C["s1"], "o"), ("hold", "2024 (HOLD)", C["s2"], "s")]):\n    for yi, (k, _) in zip(y, filas):\n        if k not in boot:\n            continue\n        b = boot[k][fase]; off = 0.16 if j == 0 else -0.16\n        ax.plot(b["ic95"], [yi + off] * 2, color=col, lw=2, solid_capstyle="round")\n        ax.plot(b["dif"], yi + off, mk, color=col, ms=7, mec=C["bg"], mew=1.5, label=lab if yi == y[0] else None)\nax.axvline(0, color=C["ink2"], lw=1)\nax.axvspan(0.001, 0.02, color=C["grid"], alpha=0.35, lw=0)\nax.text(0.0012, y[0] + 0.55, "umbral E18: +0,001 en 2023", color=C["ink2"], fontsize=8.5, va="bottom")\nax.set_yticks(y); ax.set_yticklabels([n for _, n in filas])\nax.set_xlim(-0.026, 0.012); ax.set_ylim(-0.7, len(filas) - 0.1)\nax.set_xlabel("Diferencia de D² frente a T2b, calibración C1 (IC95 por cliente)")\nax.set_title("Emergente: 2023 y 2024", loc="left", color=C["ink"], fontsize=11)\nax.legend(loc="lower left", frameon=False)\nax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True); ax.set_facecolor(C["bg"])\nax = axs[1]\nramos = [("motor", "autos", C["s1"], "o"), ("buildings", "edificios", C["s2"], "s"), ("contents", "contenidos", C["s3"], "^")]\nfor j, (l, lab, col, mk) in enumerate(ramos):\n    for yi, (k, _) in zip(y, filas):\n        if k not in des:\n            continue\n        v = des[k][l] - des["T2b"][l]; off = (1 - j) * 0.2\n        ax.plot([0, v], [yi + off] * 2, color=col, lw=1.2, alpha=0.6)\n        ax.plot(v, yi + off, mk, color=col, ms=7, mec=C["bg"], mew=1.5, label=lab if yi == y[2] else None)\n    if l == "contents":\n        for yi, (k, _) in zip(y, filas):\n            if k in des and des[k][l] - des["T2b"][l] < -0.012:\n                ax.annotate(f"{des[k][l] - des[\'T2b\'][l]:+.3f}".replace(".", ","), (-0.0116, yi - 0.45), color=C["ink2"],\n                            fontsize=8, va="center")\nax.axvline(0, color=C["ink2"], lw=1)\nax.set_yticks(y); ax.set_yticklabels([""] * len(filas))\nax.set_xlim(-0.012, 0.009); ax.set_ylim(-0.7, len(filas) - 0.1)\nax.set_xticks([-0.01, -0.005, 0, 0.005])\nax.text(0.0, -0.55, "la mezcla no tiene DES24: es una combinación de troncos", color=C["ink2"], fontsize=8.5, ha="center")\nax.set_xlabel("Diferencia de D² frente a T2b en el desarrollado 2024 (renivelado)")\nax.set_title("DES24: desarrollado 2024, fuera de tiempo", loc="left", color=C["ink"], fontsize=11)\nax.legend(loc="upper right", frameon=False)\nax.grid(axis="x", color=C["grid"], lw=0.8); ax.set_axisbelow(True); ax.set_facecolor(C["bg"])\nfor a in axs:\n    for s in ("top", "right"):\n        a.spines[s].set_visible(False)\n    a.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:+.3f}".replace(".", ",") if v else "0"))\nfig.suptitle("Troncos nuevos frente a T2b: lo que mejora en 2024 y en el desarrollado no mejora en 2023",\n             x=0.01, ha="left", fontsize=12.5, color=C["ink"])\nfig.tight_layout(rect=(0, 0, 1, 0.95))\nout = E / "evolucion" / "f12_troncos.png"; out.parent.mkdir(exist_ok=True)\nfig.savefig(out, dpi=150, facecolor=C["bg"])\nprint(out)\n'


### Embedded source: `ml/codigo/figura_nivel.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figura_nivel.py'] = '"""Figure f13: tariff-derived E27 segment levels versus global public-probe levels."""\nimport json\nimport numpy as np, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom comun import *\n\nr = json.load(open(ENTREGA / "nivel2025" / "e27_nivel.json"))["segmentos"]\nSEG = ["motor|NB", "motor|REN", "buildings|NB", "buildings|REN", "contents|NB", "contents|REN"]\nET = {"motor|NB": "Autos\\nnuevo", "motor|REN": "Autos\\nrenov.", "buildings|NB": "Edificios\\nnuevo",\n      "buildings|REN": "Edificios\\nrenov.", "contents|NB": "Contenidos\\nnuevo", "contents|REN": "Contenidos\\nrenov."}\nC = {"bg": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e", "grid": "#e4e3df", "s1": "#2a78d6", "s2": "#eb6834", "s3": "#1baf7a"}\nplt.rcParams.update({"font.size": 9.5, "axes.edgecolor": C["grid"], "axes.labelcolor": C["ink2"], "xtick.color": C["ink"],\n                     "ytick.color": C["ink2"], "font.family": "DejaVu Sans"})\nfig, axs = plt.subplots(1, 2, figsize=(12.5, 4.9), facecolor=C["bg"], gridspec_kw={"width_ratios": [1.25, 1]})\nx = np.arange(len(SEG))\nax = axs[0]\ng = np.array([r[s]["g"] for s in SEG]); sg = np.array([r[s]["sigma"] for s in SEG]); f = np.array([r[s]["f"] for s in SEG])\nax.vlines(x - 0.12, g * np.exp(-sg), g * np.exp(sg), color=C["s2"], lw=2)\nax.plot(x - 0.12, g, "o", color=C["s2"], ms=7, mec=C["bg"], mew=1.5, label="Tarifa × siniestralidad 2023-24 (±1 σ)")\nax.plot(x + 0.12, f, "s", color=C["s1"], ms=7, mec=C["bg"], mew=1.5, label="Factor E27 con credibilidad")\nfor yv, txt, ls, va in [(1.0, "mezcla enviada (1,000)", "-", "top"), (1.1877, "tablero, nivel creíble 1,188", "--", "top"),\n                        (1.2231, "tablero, óptimo público 1,223", ":", "bottom")]:\n    ax.axhline(yv, color=C["ink2"], lw=1, ls=ls)\n    ax.text(-0.45, yv * (0.995 if va == "top" else 1.005), txt, color=C["ink2"], fontsize=8, va=va, ha="left")\nax.set_xticks(x); ax.set_xticklabels([ET[s] for s in SEG])\nax.set_ylabel("Factor sobre la mezcla enviada")\nax.set_ylim(0.45, 2.15); ax.set_yscale("log"); ax.set_yticks([0.5, 0.7, 1.0, 1.4, 2.0])\nax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); ax.set_xlim(-0.5, len(SEG) - 0.5)\nax.get_yaxis().set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))\nax.set_title("Nivel 2025 que sugiere la tarifa y factor que se aplica", loc="left", color=C["ink"], fontsize=11)\nax.legend(loc="upper right", frameon=False, fontsize=8.5)\nax = axs[1]\nl23 = [r[s]["LR_2023"] for s in SEG]; l24 = [r[s]["LR_2024"] for s in SEG]\nax.vlines(x, np.minimum(l23, l24), np.maximum(l23, l24), color=C["grid"], lw=3)\nax.plot(x, l23, "o", color=C["s1"], ms=7, mec=C["bg"], mew=1.5, label="2023")\nax.plot(x, l24, "s", color=C["s2"], ms=7, mec=C["bg"], mew=1.5, label="2024")\nax.set_xticks(x); ax.set_xticklabels([ET[s] for s in SEG])\nax.set_ylabel("Coste último / prima devengada")\nax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ",")))\nax.set_title("La siniestralidad cambia mucho de un año a otro", loc="left", color=C["ink"], fontsize=11)\nax.legend(loc="upper right", frameon=False)\nfor a in axs:\n    a.set_facecolor(C["bg"]); a.grid(axis="y", color=C["grid"], lw=0.8); a.set_axisbelow(True)\n    for s in ("top", "right"):\n        a.spines[s].set_visible(False)\nfig.suptitle("La tarifa apunta en la misma dirección que el tablero, pero es demasiado inestable para fijar el nivel",\n             x=0.01, ha="left", fontsize=12.5, color=C["ink"])\nfig.tight_layout(rect=(0, 0, 1, 0.94))\nout = ENTREGA / "nivel2025" / "f13_nivel_2025.png"\nfig.savefig(out, dpi=150, facecolor=C["bg"]); print(out)\n'


### Embedded source: `ml/codigo/figuras_datos.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figuras_datos.py'] = '"""Plot peril reporting completeness and portfolio shift from 2023 to 2025."""\nimport json\nimport numpy as np, pandas as pd, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom comun import *\n\nFIG = ENTREGA / "figuras"; FIG.mkdir(parents=True, exist_ok=True)\nplt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})\nC = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}\ninfo = json.load(open(TRABAJO / "preparacion.json"))\n\n# Peril cost-weighted completeness in developed data and emerging-market contrast.\ncdf = info["cdf_coste_desarrollado"]; con = info["contraste_reporte_emergente_2023"]\nfig, axs = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)\nfor ax, l in zip(axs, LINEAS):\n    for k, v in cdf.items():\n        ll, p = k.split("|")\n        if ll != l: continue\n        xs = [0, 30, 90, 180, 365, 730]\n        ax.plot(xs, [v[str(x)] if str(x) in v else v[x] for x in xs], marker="o", ms=3, label=p)\n    ax.set_title(l); ax.set_xlabel("días desde la ocurrencia"); ax.axvline(365, color="#999", lw=0.6, ls=":")\n    ax.legend(fontsize=7, frameon=False)\naxs[0].set_ylabel("fracción del coste ya reportada")\nfig.suptitle("Completitud de reporte ponderada por coste (desarrollado, cohorte 2021)", fontsize=10)\nfig.tight_layout(); fig.savefig(FIG / "f01_completitud_reporte.png"); plt.close(fig)\n\n# Portfolio shift: new business and policy-start month, 2023/2024 versus 2025.\nfig, axs = plt.subplots(1, 2, figsize=(9, 3.2))\nfilas = []\nfor l in LINEAS:\n    b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["year", "new_business", "start_month", "exposure"])\n    for y, g in b.groupby("year"):\n        filas.append({"ramo": l, "año": y, "nb": np.average(g.new_business, weights=g.exposure),\n                      "enero": np.average(g.start_month == 1, weights=g.exposure)})\nd = pd.DataFrame(filas)\nx = np.arange(3); w = 0.25\nfor i, l in enumerate(LINEAS):\n    s = d[d.ramo == l]\n    axs[0].bar(x + (i - 1) * w, s.nb.values, w, color=C[l], label=l)\naxs[0].set_xticks(x, ["2023", "2024", "2025 (test)"]); axs[0].set_ylabel("nuevo negocio (% de exposición)")\naxs[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1)); axs[0].legend(frameon=False)\naxs[0].set_title("La cartera de 2025 es casi toda nueva")\npes = json.load(open(TRABAJO / "pesos_2025_dev.json"))\nimp = pd.DataFrame({l: pes[l]["importancia_top"] for l in LINEAS}).fillna(0)\nimp = imp.loc[imp.mean(1).sort_values().index]\nyy = np.arange(len(imp))\nfor i, l in enumerate(LINEAS):\n    axs[1].barh(yy + (i - 1) * 0.25, imp[l].values, 0.25, color=C[l], label=f"{l} (AUC {pes[l][\'auc\']:.2f})")\naxs[1].set_yticks(yy, imp.index); axs[1].set_xlabel("ganancia relativa del clasificador 2023 frente a 2025")\naxs[1].legend(frameon=False, fontsize=7); axs[1].set_title("Qué distingue la cartera 2025")\nfig.tight_layout(); fig.savefig(FIG / "f02_desplazamiento_cartera.png"); plt.close(fig)\nprint(d.round(3))\n'


### Embedded source: `ml/codigo/figuras_extra.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figuras_extra.py'] = '"""Plot the final recipe flow (f00) and production-model feature importance (f09)."""\nimport json\nimport numpy as np, pandas as pd, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch\nfrom comun import *\n\nFIG = ENTREGA / "figuras"\nplt.rcParams.update({"font.size": 9, "figure.dpi": 130})\n\n# f00: final recipe flow.\nfig, ax = plt.subplots(figsize=(10, 3.6)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 3.6)\ncajas = [\n    (0.1, 2.2, "Desarrollado 2021-2024\\n2,4 M periodos\\n(datos, etiquetas T0,\\ncompletitud por peligro)", "#dce8f1"),\n    (2.6, 2.2, "T2b: LightGBM Tweedie 1,5\\nesquema común de variables\\ndesplazamiento log(e·c)\\nhiperparámetros por E07", "#2a6f97"),\n    (5.1, 2.2, "Transferencia al emergente\\nrelatividades del desarrollado\\n× nivel por ramo (factor\\nTweedie con 2023-2024)", "#c4673b"),\n    (7.6, 2.2, "Calibración C1\\nfactor por ramo del OOF\\ncruzado 2023-2024\\nsin tendencia (A03, A04)", "#5b8c5a"),\n    (5.1, 0.25, "Emergente 2023-2024\\n109.296 periodos, 9.258 siniestros\\nOOF por cliente (DEV, PCV) y 2024 (HOLD)", "#f3e3d6"),\n    (7.6, 0.25, "Producción 2025\\n209.063 filas, 70,8 M CU\\nprima comercial de 20 pólizas", "#e3efe2"),\n]\nfor x, y, txt, col in cajas:\n    oscuro = col in ("#2a6f97", "#c4673b", "#5b8c5a")\n    ax.add_patch(FancyBboxPatch((x, y), 2.25, 1.25, boxstyle="round,pad=0.05", fc=col, ec="#555", lw=0.6))\n    ax.text(x + 1.125, y + 0.62, txt, ha="center", va="center", fontsize=7.6, color="white" if oscuro else "#222")\nfor (x0, y0, x1, y1) in [(2.4, 2.82, 2.6, 2.82), (4.9, 2.82, 5.1, 2.82), (7.4, 2.82, 7.6, 2.82), (6.22, 1.55, 6.22, 2.15), (8.72, 2.15, 8.72, 1.55)]:\n    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", color="#444", lw=1))\nax.text(0.1, 1.2, "Alternativas probadas y no elegidas:\\nsolo emergente (R0, G1, T1), ajuste fino por boosting (T3),\\nmodelo conjunto (T4), GLM de adaptación (T5), por peligro (T6),\\nFT-Transformer desde cero (N1), preentrenado y ajustado (N2a, N2b),\\nresidual sobre T2b (N3) y mezclas (E1-E3).", fontsize=7.4, va="center", color="#333")\nfig.tight_layout(); fig.savefig(FIG / "f00_flujo_receta.png"); plt.close(fig)\n\n# f09: production T2b gain-based feature importance.\nfig, axs = plt.subplots(1, 3, figsize=(10.5, 3.6))\nimp_out = {}\nfor ax, l in zip(axs, LINEAS):\n    imp = pd.read_csv(TRABAJO / "modelos" / "prod" / f"T2b_{l}_importancia.csv", index_col=0).iloc[:, 0]\n    imp = (imp / imp.sum()).sort_values(ascending=False).head(10)[::-1]\n    imp_out[l] = imp[::-1].round(4).to_dict()\n    ax.barh(imp.index, imp.values, color={"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}[l])\n    ax.set_title(l); ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1))\n    for s in ("top", "right"): ax.spines[s].set_visible(False)\nfig.suptitle("Variables con más ganancia en el modelo del desarrollado (T2b de producción)", fontsize=10)\nfig.tight_layout(); fig.savefig(FIG / "f09_importancia_T2b.png"); plt.close(fig)\nguardar_json(imp_out, ENTREGA / "importancia_T2b_prod.json"); print(json.dumps(imp_out, indent=0))\n'


### Embedded source: `ml/codigo/figuras_resultados.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/figuras_resultados.py'] = '"""Plot DEV/HOLD results, new-business instability, 2025 sensitivity and transformer curves."""\nimport json\nimport numpy as np, pandas as pd, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom comun import *\nimport receta\n\nFIG = ENTREGA / "figuras"\nplt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})\nC = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a"}\nout = {}\n\n# f04: frozen-list DEV versus HOLD 2024, without trend.\ndev = pd.read_csv(ENTREGA / "tabla_dev.csv")\nhold = json.load(open(ENTREGA / "resultados_hold.json"))["candidatos"]\nfilas = []\nfor k, v in hold.items():\n    base, cal = k.rsplit("_", 1)\n    cc = {"C1": "ramo", "C2": "ramo_nb"}[cal]\n    dv = dev[(dev.cand == base) & (dev.calib == cc)].D2.values\n    filas.append({"cand": k, "DEV": float(dv[0]) if len(dv) else np.nan, "HOLD": v["sin_tendencia"]["D2"]})\nd = pd.DataFrame(filas).sort_values("HOLD"); out["dev_hold"] = d.round(5).to_dict("records")\nfig, ax = plt.subplots(figsize=(7, 4))\ny = np.arange(len(d))\nax.hlines(y, d.DEV, d.HOLD, color="#bbb", lw=2)\nax.scatter(d.DEV, y, color="#2a6f97", label="DEV: OOF 2023", zorder=3)\nax.scatter(d.HOLD, y, color="#c4673b", label="HOLD: 2024, abierto una vez", zorder=3)\nax.set_yticks(y, d.cand); ax.set_xlabel("D² Tweedie 1,5")\nax.legend(frameon=False, loc="lower right"); ax.set_title("Los modelos con transferencia ganan en los dos años; el orden entre ellos es ruido", fontsize=9)\nfig.tight_layout(); fig.savefig(FIG / "f04_dev_frente_hold.png"); plt.close(fig)\n\n# f06: new-business observed/expected by line/year under 2023-2024 OOF C1.\nt, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")\nm = mu * t.c_fila.values\ng = pd.DataFrame({"line": t.line, "year": t.year, "nb": t.new_business, "y": t.y, "m": m, "s": t.y > 0}).groupby(["line", "nb", "year"]).agg(\n    y=("y", "sum"), m=("m", "sum"), n=("s", "sum"))\ng["AE"] = g.y / g.m\nout["ae_nb_anio"] = {f"{a}|{b}|{c}": {"AE": round(float(r.AE), 4), "n_sin": int(r.n)} for (a, b, c), r in g.iterrows()}\nfig, ax = plt.subplots(figsize=(7, 3.4))\nxs = 0\nfor l in LINEAS:\n    for nb in (False, True):\n        for yv, mk in ((2023, "o"), (2024, "s")):\n            r = g.loc[(l, nb, yv)]\n            ax.scatter(xs, r.AE, color=C[l], marker=mk, s=20 + r.n / 8, alpha=0.85)\n        ax.text(xs, 0.3, ("NB" if nb else "REN"), ha="center", fontsize=7)\n        xs += 1\n    xs += 0.6\nax.axhline(1, color="#999", lw=0.8, ls="--")\nax.set_xticks([0.5, 3.1, 5.7], LINEAS); ax.set_ylabel("observado / esperado (C1)")\nax.set_ylim(0.25, 1.9)\nax.set_title("El exceso del nuevo negocio no se repite de un año a otro (círculo 2023, cuadrado 2024; área ∝ siniestros)", fontsize=8.5)\nfig.tight_layout(); fig.savefig(FIG / "f06_nuevo_negocio_por_anio.png"); plt.close(fig)\n\n# f07: 2025 total sensitivity and rates relative to history.\nprod = json.load(open(ENTREGA / "produccion_2025.json")); ctx = json.load(open(ENTREGA / "contexto_2025.json"))\nfig, axs = plt.subplots(1, 2, figsize=(10, 3.4))\nvar = ["final_C1", "C1_tendencia", "C2", "preinscrita_C2_tendencia"]\netq = ["final (C1)", "C1 + tendencia", "C2", "preinscrita\\n(C2 + tendencia)"]\ntot = [prod["variantes"][v]["total"] / 1e6 for v in var]\naxs[0].bar(etq, tot, color=["#2a6f97", "#9bb7cc", "#9bb7cc", "#9bb7cc"])\nfor i, v in enumerate(tot):\n    axs[0].text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=8)\naxs[0].set_ylabel("total predicho 2025 (millones de CU)"); axs[0].set_title("Sensibilidad del total a las dos reglas descartadas", fontsize=9)\nx = np.arange(3); w = 0.25\nfor j, (lab, key) in enumerate([("2023 observado último", "2023"), ("2024 observado último", "2024"), ("2025 predicho", "2025")]):\n    vals = [ctx[l][key]["tasa_ultima_obs"] if key != "2025" else ctx[l]["2025"]["tasa_predicha"] for l in LINEAS]\n    axs[1].bar(x + (j - 1) * w, vals, w, label=lab, color=["#cfcfcf", "#9a9a9a", "#2a6f97"][j])\naxs[1].set_xticks(x, LINEAS); axs[1].set_ylabel("CU por año de exposición"); axs[1].legend(frameon=False, fontsize=7)\naxs[1].set_title("Tasa por exposición: historia y predicción", fontsize=9)\nfig.tight_layout(); fig.savefig(FIG / "f07_sensibilidad_2025.png"); plt.close(fig)\n\n# f08: developed-market transformer pretraining curves.\nfig, axs = plt.subplots(1, 3, figsize=(10, 3))\nfor ax, l in zip(axs, LINEAS):\n    h = json.load(open(TRABAJO / "modelos" / "dev" / f"N_pre_{l}.json"))["hist"]\n    ep = [r[0] for r in h]; tr = [r[1] for r in h]; vl = [r[2] for r in h]\n    ax.plot(ep, tr, "o-", ms=3, color="#999", label="entrenamiento"); ax.plot(ep, vl, "o-", ms=3, color=C[l], label="validación")\n    ax.set_title(l); ax.set_xlabel("época"); ax.legend(frameon=False, fontsize=7)\naxs[0].set_ylabel("desviación Tweedie media (y con tope)")\nfig.suptitle("Preentrenamiento del FT-Transformer en el mercado desarrollado 2021-2023", fontsize=10)\nfig.tight_layout(); fig.savefig(FIG / "f08_transformer_preentrenamiento.png"); plt.close(fig)\nguardar_json(out, ENTREGA / "figuras_datos.json")\nprint(json.dumps(out, indent=0)[:3000])\n'


### Embedded source: `ml/codigo/gbm.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/gbm.py'] = '"""Fit R0, emerging-only T1, transferred T2 and boosting-fine-tuned T3 trees.\nDEV uses five-customer-fold 2023 OOF and developed 2021-2023 pretraining.\nHOLD trains emerging 2023 for 2024 with the same developed window. Production\ntrains emerging 2023-2024 for 2025 with developed 2021-2024 pretraining.\nUsage: python gbm.py <dev|hold|prod>\n"""\nfrom __future__ import annotations\nimport sys, time, hashlib, os\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\n\nFASE = sys.argv[1] if len(sys.argv) > 1 else "dev"\nSOLO = sys.argv[2].split(",") if len(sys.argv) > 2 else None\nCONF = {"dev": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=None),\n        "hold": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=2024),\n        "prod": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=2025),\n        "pcv": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=None)}[FASE]\nOUT = TRABAJO / "pred" / FASE\nOUT.mkdir(parents=True, exist_ok=True)\nMOD = TRABAJO / "modelos" / {"pcv": "prod", "hold": "dev"}.get(FASE, FASE)  # Share the developed pretraining fit.\nMOD.mkdir(parents=True, exist_ok=True)\n\nP_DEV = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.05, num_leaves=63,\n             min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,\n             max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261024)\nP_EM = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.02, num_leaves=15,\n            min_data_in_leaf=100, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,\n            max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261024)\nP_FT = dict(P_EM, learning_rate=0.01, num_leaves=7, min_data_in_leaf=200, lambda_l2=20.0)\n# E07 chooses line hyperparameters using developed-market internal validation.\nP_DEVB = {"motor": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=300),\n          "buildings": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=3000),\n          "contents": dict(P_DEV, learning_rate=0.03, lambda_l2=10.0, min_data_in_leaf=3000)}\nBACKBONE = os.environ.get("MLIND_BACKBONE", "T2")\n\n\ndef interno(cids):\n    return np.array([int(hashlib.sha256(("mlind_inner|" + c).encode()).hexdigest()[:8], 16) % 5 == 0 for c in cids])\n\n\ndef ajustar(X, y, off, params, max_rounds, es=200, val=None):\n    """Fit with an internal customer validation mask or fixed boosting rounds."""\n    if val is not None:\n        dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], free_raw_data=False)\n        dva = lgb.Dataset(X[val], y[val], init_score=off[val], reference=dtr)\n        b = lgb.train(params, dtr, max_rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(es, verbose=False)])\n        n = max(b.best_iteration, 1)\n        # Refit on the full training partition using n fixed rounds.\n        b = lgb.train(params, lgb.Dataset(X, y, init_score=off), n)\n        return b, n\n    b = lgb.train(params, lgb.Dataset(X, y, init_score=off), max_rounds)\n    return b, max_rounds\n\n\ndef preentrenar(linea):\n    """T2: fit developed-market LightGBM Tweedie using the shared feature schema."""\n    f = MOD / f"T2_{linea}.txt"\n    d = cargar("developed", linea, CONF["clima"])\n    d = d[d.year.isin(CONF["dev_years"])]\n    cols = comunes(linea)\n    return d, cols, f\n\n\ndef correr(linea):\n    t0 = time.time()\n    em = cargar("emerging", linea, CONF["clima"])\n    dv = cargar("developed", linea, CONF["clima"])\n    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)\n    colsC = comunes(linea)\n    if BACKBONE == "T2c":  # A04 excludes history features whose missingness reveals the first observed year.\n        colsC = [c for c in colsC if c not in ("hist_pol_12m", "hist_cli_12m")]\n    colsE = colsC + SOLO_EMERGENTE\n    todas = colsE + MARKETING + TARIFA + MAKE\n    cats = categorias_union([dv, em], todas)\n    # T2: developed-market pretraining.\n    Xd = matriz(dv, colsC, cats); yd = dv.y.values; offd = np.log(dv.exposure.values * dv.c_fila.values)\n    vald = dv.fold.values == 0\n    f2 = MOD / f"{BACKBONE}_{linea}.txt"\n    pdev = P_DEV if BACKBONE == "T2" else P_DEVB[linea]  # T2b and T2c share E07 settings.\n    if f2.exists():\n        b2 = lgb.Booster(model_file=str(f2)); n2 = b2.current_iteration()\n    else:\n        b2, n2 = ajustar(Xd, yd, offd, pdev, 5000, es=150, val=vald)\n        b2.save_model(str(f2))\n    # Developed releveling is diagnostic; k2 determines the emerging-market level.\n    kdev = factor_tweedie(yd, np.exp(b2.predict(Xd, raw_score=True) + offd))\n    print(linea, "T2 rondas", n2, "s", round(time.time() - t0))\n    imp = pd.Series(b2.feature_importance("gain"), index=colsC).sort_values(ascending=False)\n    imp.to_csv(MOD / f"{BACKBONE}_{linea}_importancia.csv")\n    pd.DataFrame({"row_id": em.row_id.values, "raw": b2.predict(matriz(em, colsC, cats), raw_score=True)}).to_parquet(\n        OUT / f"{BACKBONE}raw_{linea}.parquet", index=False)\n    # Emerging-market fits.\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)\n    if CONF["em_eval"] is None:\n        ev_sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)]\n    else:\n        ev = em[em.year == CONF["em_eval"]].reset_index(drop=True)\n        ev_sets = [(tr, ev)]\n    variantes = {\n        "T1": colsE, "T1mk": colsE + MARKETING, "T1wp": colsE + TARIFA, "T1mm": colsE + MAKE,\n        "T3": colsE, "T3wp": colsE + TARIFA, "T3mk": colsE + MARKETING,\n    }\n    if SOLO:\n        variantes = {k: v for k, v in variantes.items() if k in SOLO}\n    if BACKBONE != "T2":\n        variantes = {k.replace("T3", "T3b"): v for k, v in variantes.items() if k.startswith("T3")}\n    res = {k: [] for k in ["R0", BACKBONE] + list(variantes)}\n    rondas = {k: [] for k in variantes}\n    for a, b in ev_sets:\n        offa = np.log(a.exposure.values * a.c_fila.values)\n        ya = a.y.values\n        # R0: line ultimate rate, using a Tweedie factor over effective exposure.\n        k0 = factor_tweedie(ya, np.exp(offa))\n        res["R0"].append(pd.DataFrame({"row_id": b.row_id.values, "mu": k0 * b.exposure.values}))\n        # T2: developed-market relativities times the emerging-market level.\n        ra = b2.predict(matriz(a, colsC, cats), raw_score=True)\n        rb = b2.predict(matriz(b, colsC, cats), raw_score=True)\n        k2 = factor_tweedie(ya, np.exp(ra + offa))\n        res[BACKBONE].append(pd.DataFrame({"row_id": b.row_id.values, "mu": k2 * np.exp(rb) * b.exposure.values}))\n        val = interno(a.customer_id.values)\n        for nom, cols in variantes.items():\n            Xa = matriz(a, cols, cats); Xb = matriz(b, cols, cats)\n            if nom.startswith("T1"):  # Only with BACKBONE T2.\n                base_a = np.full(len(a), np.log(k0)); base_b = np.full(len(b), np.log(k0))\n                bm, n = ajustar(Xa, ya, offa + base_a, P_EM, 3000, es=200, val=val)\n            else:\n                base_a = ra + np.log(k2); base_b = rb + np.log(k2)\n                bm, n = ajustar(Xa, ya, offa + base_a, P_FT, 2000, es=200, val=val)\n            # E02: relevel within the training fold using its Tweedie factor.\n            kin = factor_tweedie(ya, np.exp(bm.predict(Xa, raw_score=True) + base_a + offa))\n            mu = kin * np.exp(bm.predict(Xb, raw_score=True) + base_b) * b.exposure.values\n            rondas[nom].append(n)\n            res[nom].append(pd.DataFrame({"row_id": b.row_id.values, "mu": mu}))\n            if CONF["em_eval"] is not None:\n                (TRABAJO / "modelos" / FASE).mkdir(parents=True, exist_ok=True)\n                bm.save_model(str(TRABAJO / "modelos" / FASE / f"{nom}_{linea}.txt"))\n    for k, v in res.items():\n        out = pd.concat(v, ignore_index=True)\n        out.to_parquet(OUT / f"{k}_{linea}.parquet", index=False)\n    guardar_json({"rondas": rondas, f"{BACKBONE}_rondas": n2, "k_desarrollado": kdev, "segundos": round(time.time() - t0)},\n                 OUT / f"gbm_{BACKBONE}_{linea}_meta.json")\n    print(linea, "listo", rondas, round(time.time() - t0), "s")\n\n\nif __name__ == "__main__":\n    lineas = [x for x in LINEAS if (len(sys.argv) <= 3 or x in sys.argv[3].split(","))]\n    for l in lineas:\n        correr(l)\n'


### Embedded source: `ml/codigo/glm.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/glm.py'] = '"""G1: emerging-only log-link power-1.5 Tweedie GLM with binned features and internally selected ridge."""\nimport sys\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom variables import *\nfrom adaptacion import ajustar, interno\n\nFASE = sys.argv[1] if len(sys.argv) > 1 else "dev"\nCONF = {"dev": ("hasta2023", [2023], None), "hold": ("hasta2023", [2023], 2024), "prod": ("hasta2024", [2023, 2024], 2025),\n        "pcv": ("hasta2024", [2023, 2024], None)}[FASE]\nOUT = TRABAJO / "pred" / FASE\nLAMBDAS = [1e2, 1e3, 1e4, 3e4, 1e5, 3e5, 1e6]\n\n\ndef diseno(df, cols, ref=None):\n    """Encode categorical levels and training-based numeric quintiles, with an explicit missing level."""\n    ref = {} if ref is None else ref\n    bloques = []; nombres = []\n    for c in cols:\n        if c in CATEGORICAS or df[c].dtype == bool or df[c].nunique() <= 6:\n            v = df[c].astype(object).where(df[c].notna(), "NA").astype(str).values\n            if c not in ref:\n                ref[c] = ("cat", sorted(set(v)))\n        else:\n            x = pd.to_numeric(df[c], errors="coerce").astype(float).values\n            if c not in ref:\n                q = np.unique(np.nanquantile(x, [0.2, 0.4, 0.6, 0.8])) if np.isfinite(x).any() else np.array([])\n                ref[c] = ("num", q)\n            q = ref[c][1]\n            v = np.where(np.isnan(x), "NA", np.searchsorted(q, x, side="right").astype(str)).astype(str)\n            if ref[c][0] == "num" and len(ref[c]) == 2:\n                ref[c] = ("num", q, sorted(set(v)))\n        niveles = ref[c][1] if ref[c][0] == "cat" else ref[c][2]\n        for n in niveles:\n            bloques.append((v == n).astype(np.float32)); nombres.append(f"{c}={n}")\n    return np.column_stack(bloques), nombres, ref\n\n\nfor l in LINEAS:\n    em = cargar("emerging", l, CONF[0])\n    cols = [c for c in comunes(l) + SOLO_EMERGENTE]\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF[1])].reset_index(drop=True)\n    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF[2] is None else [(tr, em[em.year == CONF[2]].reset_index(drop=True))]\n    preds = []; lams = []\n    for a, b in sets:\n        Za, nom, ref = diseno(a, cols); Zb, _, _ = diseno(b, cols, ref)\n        offa = np.log(a.exposure.values * a.c_fila.values); offb = np.log(b.exposure.values)\n        ya = a.y.values; v = interno(a.customer_id.values)\n        best = None\n        for lam in LAMBDAS:\n            bt = ajustar(Za[~v], ya[~v], offa[~v], lam)\n            dv_ = desviacion(ya[v], np.exp(offa[v] + bt[0] + Za[v] @ bt[1:])).sum()\n            if best is None or dv_ < best[0]:\n                best = (dv_, lam)\n        lams.append(best[1])\n        bt = ajustar(Za, ya, offa, best[1])\n        mu_a = np.exp(offa + bt[0] + Za @ bt[1:])\n        kin = factor_tweedie(ya, mu_a)\n        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": kin * np.exp(offb + bt[0] + Zb @ bt[1:])}))\n    pd.concat(preds).to_parquet(OUT / f"G1_{l}.parquet", index=False)\n    print(l, "lambdas", lams, flush=True)\n'


### Embedded source: `ml/codigo/historia.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/historia.py'] = '"""Build 365-day notification history by policy and customer across all lines."""\nimport numpy as np, pandas as pd\nfrom comun import *\nINICIO = {"developed": "2021-01-01", "emerging": "2023-01-01"}\nfor m in ["developed", "emerging"]:\n    c = pd.read_parquet(DATOS / m / "claims.parquet", columns=["claim_id", "policy_id", "line", "report_date", "loss_date"])\n    c["report_date"] = pd.to_datetime(c.report_date)\n    if m == "developed":  # Use the same duplicate handling as preparar_datos.\n        full = pd.read_parquet(DATOS / m / "claims.parquet")\n        cols = [x for x in full.columns if x != "claim_id"]\n        q = full.duplicated(cols, keep=False) & (full.claim_id.str.len() == 11)\n        c = c[~q.values]\n    bases = {l: pd.read_parquet(TRABAJO / f"base_{m}_{l}.parquet", columns=["row_id", "policy_id", "customer_id", "line", "period_start"]) for l in LINEAS}\n    mapa = pd.concat([b[["policy_id", "line", "customer_id"]] for b in bases.values()]).drop_duplicates(["policy_id", "line"])\n    cc = c.merge(mapa, on=["policy_id", "line"])\n    for l, df in bases.items():\n        df["period_start"] = pd.to_datetime(df.period_start)\n        ok = (df.period_start - pd.Timedelta(days=365)) >= pd.Timestamp(INICIO[m])\n        hp = c[c.line == l][["policy_id", "report_date"]].merge(df[["row_id", "policy_id", "period_start"]], on="policy_id")\n        hp = hp[(hp.report_date < hp.period_start) & (hp.report_date >= hp.period_start - pd.Timedelta(days=365))]\n        hc = cc[["customer_id", "report_date"]].merge(df[["row_id", "customer_id", "period_start"]], on="customer_id")\n        hc = hc[(hc.report_date < hc.period_start) & (hc.report_date >= hc.period_start - pd.Timedelta(days=365))]\n        out = pd.DataFrame({"row_id": df.row_id,\n                            "hist_pol_12m": df.row_id.map(hp.groupby("row_id").size()).fillna(0).where(ok, np.nan),\n                            "hist_cli_12m": df.row_id.map(hc.groupby("row_id").size()).fillna(0).where(ok, np.nan)})\n        out.to_parquet(TRABAJO / f"historia_{m}_{l}.parquet", index=False)\n        print(m, l, out.hist_pol_12m.notna().mean().round(3), out.hist_pol_12m.mean().round(4), out.hist_cli_12m.mean().round(4))\n'


### Embedded source: `ml/codigo/historia2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/historia2.py'] = '"""A05: end the 365-day history window at min(policy start, valuation date).\nAvoid interpreting an unobserved future interval as zero notifications. Only\npost-valuation 2025 starts change; earlier rows must remain identical.\n"""\nimport numpy as np, pandas as pd\nfrom comun import *\nINICIO = {"developed": "2021-01-01", "emerging": "2023-01-01"}\nFIN = pd.Timestamp(str(V))\nfor m in ["developed", "emerging"]:\n    c = pd.read_parquet(DATOS / m / "claims.parquet", columns=["claim_id", "policy_id", "line", "report_date", "loss_date"])\n    c["report_date"] = pd.to_datetime(c.report_date)\n    if m == "developed":\n        full = pd.read_parquet(DATOS / m / "claims.parquet")\n        cols = [x for x in full.columns if x != "claim_id"]\n        q = full.duplicated(cols, keep=False) & (full.claim_id.str.len() == 11)\n        c = c[~q.values]\n    bases = {l: pd.read_parquet(TRABAJO / f"base_{m}_{l}.parquet", columns=["row_id", "policy_id", "customer_id", "line", "period_start"]) for l in LINEAS}\n    mapa = pd.concat([b[["policy_id", "line", "customer_id"]] for b in bases.values()]).drop_duplicates(["policy_id", "line"])\n    cc = c.merge(mapa, on=["policy_id", "line"])\n    for l, df in bases.items():\n        df["period_start"] = pd.to_datetime(df.period_start)\n        df["fin"] = df.period_start.where(df.period_start <= FIN, FIN)\n        ok = (df.fin - pd.Timedelta(days=365)) >= pd.Timestamp(INICIO[m])\n        hp = c[c.line == l][["policy_id", "report_date"]].merge(df[["row_id", "policy_id", "fin"]], on="policy_id")\n        hp = hp[(hp.report_date < hp.fin) & (hp.report_date >= hp.fin - pd.Timedelta(days=365))]\n        hc = cc[["customer_id", "report_date"]].merge(df[["row_id", "customer_id", "fin"]], on="customer_id")\n        hc = hc[(hc.report_date < hc.fin) & (hc.report_date >= hc.fin - pd.Timedelta(days=365))]\n        out = pd.DataFrame({"row_id": df.row_id,\n                            "hist_pol_12m": df.row_id.map(hp.groupby("row_id").size()).fillna(0).where(ok, np.nan),\n                            "hist_cli_12m": df.row_id.map(hc.groupby("row_id").size()).fillna(0).where(ok, np.nan)})\n        old = pd.read_parquet(TRABAJO / f"historia_{m}_{l}.parquet")\n        assert (old.row_id.values == out.row_id.values).all()\n        tarde = (df.period_start > FIN).values\n        for v in ["hist_pol_12m", "hist_cli_12m"]:\n            a, b = old[v].values[~tarde], out[v].values[~tarde]\n            assert np.array_equal(np.isnan(a), np.isnan(b)) and np.allclose(np.nan_to_num(a), np.nan_to_num(b)), (m, l, v)\n        out.to_parquet(TRABAJO / f"historia2_{m}_{l}.parquet", index=False)\n        if tarde.any():\n            print(m, l, "filas tras V:", int(tarde.sum()), "media hist_pol antes/después:",\n                  round(float(np.nanmean(old.hist_pol_12m.values[tarde])), 4), round(float(np.nanmean(out.hist_pol_12m.values[tarde])), 4),\n                  "cli:", round(float(np.nanmean(old.hist_cli_12m.values[tarde])), 4), round(float(np.nanmean(out.hist_cli_12m.values[tarde])), 4), flush=True)\n'


### Embedded source: `ml/codigo/mezcla_designada.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/mezcla_designada.py'] = '"""Prepare the historical, unsent 50/50 designated-production and ML-family blend option."""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nOTROS = RAIZ / "comparacion" / "otros"\nsub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")[["row_id"]]\na = pd.read_csv(PRIVADO / "prediccion_2025_mlind.csv", dtype={"row_id": str})\nb = pd.read_csv(OTROS / "A1fixF1b3n_2025.csv", dtype={"row_id": str})\nassert (a.row_id.values == sub.row_id.values).all() and (b.row_id.values == sub.row_id.values).all()\nm = pd.DataFrame({"row_id": sub.row_id, "prediction": 0.5 * a.prediction.values + 0.5 * b.prediction.values})\nassert m.prediction.notna().all() and (m.prediction > 0).all() and len(m) == 209063 and m.row_id.is_unique\nf = PRIVADO / "mezcla50_designada_mlind_2025.csv"\nm.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")\npref = m.row_id.str[0]\nr = {"archivo": f.name, "sha256": sha256(f), "filas": len(m), "total": float(m.prediction.sum()),\n     "por_ramo": {k: float(v) for k, v in m.groupby(pref).prediction.sum().items()},\n     "fuentes": {"mlind": sha256(PRIVADO / "prediccion_2025_mlind.csv"), "designada": sha256(OTROS / "A1fixF1b3n_2025.csv")}}\nguardar_json(r, RAIZ / "comparacion" / "mezcla50_info.json"); print(json.dumps(r, indent=1))\n'


### Embedded source: `ml/codigo/mezcla_diagnostico.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/mezcla_diagnostico.py'] = '"""Diagnose the preregistered 50/50 blend without changing its prediction file."""\nimport json\nimport numpy as np, pandas as pd, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom sklearn.isotonic import IsotonicRegression\nfrom comun import *\nimport receta\n\nOTROS = RAIZ / "comparacion" / "otros"; OUT = RAIZ / "mezcla"\nres = {"fuentes": {f: sha256(p) for f, p in {"designada": OTROS / "A1fixF1b3n_2025.csv", "mlind": PRIVADO / "prediccion_2025_mlind.csv",\n       "mezcla": PRIVADO / "mezcla50_designada_mlind_2025.csv", "pred2024_noct": OTROS / "predicciones_2024.parquet"}.items()}}\n# 2024 comparison.\no = pd.read_parquet(OTROS / "predicciones_2024.parquet")\np = receta.parametros("dev", ["T2b"], "ramo"); t, mu = receta.aplicar("hold", p, None)\nd = t.merge(o[["row_id", "address_id", "A1"]], on="row_id")\nassert len(d) == 59202\nd["T2b"] = mu * t.c_fila.values\ny = d.y.values; A = d.A1.values; T = d.T2b.values\nd["ari"] = 0.5 * A + 0.5 * T; d["geo"] = np.sqrt(A * T)\npadre = {}\ndef f(x):\n    while padre.setdefault(x, x) != x:\n        padre[x] = padre[padre[x]]; x = padre[x]\n    return x\nfor c, a in zip("C" + d.customer_id, "A" + d.address_id):\n    rc, ra = f(c), f(a)\n    if rc != ra: padre[ra] = rc\n_, gi = np.unique([f("C" + c) for c in d.customer_id], return_inverse=True); G = gi.max() + 1\nws = np.round(np.arange(0, 1.0001, 0.05), 2)\nrng = np.random.default_rng(20261024)\nres["curva_2024"] = {}\nfor l in LINEAS + ["todos"]:\n    i = np.ones(len(d), bool) if l == "todos" else (d.line == l).values\n    Dm = np.column_stack([desviacion(y[i], w * T[i] + (1 - w) * A[i]) for w in ws])\n    nulo = desviacion(y[i], np.full(i.sum(), y[i].mean())).sum()\n    curva = 1 - Dm.sum(0) / nulo\n    gl = gi[i]; _, gl = np.unique(gl, return_inverse=True); Gl = gl.max() + 1\n    opt = []\n    for s in range(0, 1000, 100):\n        Wc = rng.poisson(1.0, size=(100, Gl)).astype(float)\n        S = np.zeros((100, len(ws)))\n        for j in range(len(ws)):\n            S[:, j] = np.bincount(gl, weights=Dm[:, j], minlength=Gl) @ Wc.T\n        opt.append(ws[S.argmin(1)])\n    opt = np.concatenate(opt)\n    res["curva_2024"][l] = {"pesos_MLind": ws.tolist(), "D2": curva.tolist(), "w_optimo": float(ws[curva.argmax()]),\n                            "w_optimo_ic95": [float(np.quantile(opt, .025)), float(np.quantile(opt, .975))],\n                            "D2_w0": float(curva[0]), "D2_w05": float(curva[10]), "D2_w1": float(curva[-1]), "D2_max": float(curva.max())}\n# Arithmetic versus geometric blending and calibration.\ndef murphy_mcb(yy, mm, ee):\n    r = mm / ee\n    iso = IsotonicRegression(out_of_bounds="clip").fit(r, yy / ee, sample_weight=np.sqrt(ee))\n    mc = np.maximum(iso.predict(r) * ee, 1e-9)\n    return float((desviacion(yy, mm).sum() - desviacion(yy, mc).sum()) / desviacion(yy, mm).sum())\nres["2024"] = {}\nfor m in ["A1", "T2b", "ari", "geo"]:\n    r = {"D2": d2(y, d[m].values)}\n    for l in LINEAS:\n        i = (d.line == l).values\n        r[l] = {"D2": d2(y[i], d[m].values[i]), "AE": float(y[i].sum() / d[m].values[i].sum()),\n                "AE_NB": float(y[i & d.new_business.values].sum() / d[m].values[i & d.new_business.values].sum()),\n                "AE_REN": float(y[i & ~d.new_business.values].sum() / d[m].values[i & ~d.new_business.values].sum()),\n                "MCB_sobre_S": murphy_mcb(y[i], d[m].values[i], d.exposure.values[i])}\n    res["2024"][m] = r\n# 2025 comparison.\nsub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")[["row_id"]]\nbase = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "policy_id", "exposure", "new_business", "conjunto"]).assign(line=l) for l in LINEAS])\nq = sub.merge(base[base.conjunto == "test"], on="row_id", how="left")\nfor k, fp in {"designada": OTROS / "A1fixF1b3n_2025.csv", "mlind": PRIVADO / "prediccion_2025_mlind.csv", "mezcla": PRIVADO / "mezcla50_designada_mlind_2025.csv"}.items():\n    x = pd.read_csv(fp, dtype={"row_id": str}); assert (x.row_id.values == q.row_id.values).all(); q[k] = x.prediction.values\nassert np.allclose(q.mezcla, 0.5 * q.designada + 0.5 * q.mlind, rtol=1e-6, atol=1e-5)\nres["2025"] = {"controles": {"filas": int(len(q)), "nulos": int(q.mezcla.isna().sum()), "min": float(q.mezcla.min()),\n                             "positivas": bool((q.mezcla > 0).all()), "orden_plantilla": True,\n                             "max_error_media": float(np.abs(q.mezcla - 0.5 * q.designada - 0.5 * q.mlind).max())}}\nfor k in ["designada", "mlind", "mezcla"]:\n    r = {"total": float(q[k].sum())}\n    for l in LINEAS:\n        i = (q.line == l).values; e = q.exposure.values[i]; nb = q.new_business.values[i]\n        r[l] = {"total": float(q[k].values[i].sum()), "tasa": float(q[k].values[i].sum() / e.sum()),\n                "tasa_NB": float(q[k].values[i][nb].sum() / e[nb].sum()), "tasa_REN": float(q[k].values[i][~nb].sum() / e[~nb].sum())}\n    res["2025"][k] = r\n# Commercial pricing for the 20 official references.\nex = pd.read_parquet(DATOS / "emerging" / "expenses.parquet").set_index("item").emerging\nref = pd.read_parquet(DATOS / "emerging" / "reference_policies.parquet")\ncom = {"direct": ex["commission_direct"], "broker": ex["commission_broker"], "comparison_site": ex["commission_comparison_site"]}\nexp_pol = q.groupby("policy_id").exposure.sum()\nfilas = []\nfor _, r in ref.iterrows():\n    R = float(q.loc[q.row_id == r.row_id, "mezcla"].iloc[0]); e = r.exposure\n    cat = ex["cat_reinsurance"] * R if r.line in ("buildings", "contents") else 0.0\n    den = 1 - ex["variable_expense"] - com[r.channel] - ex["premium_tax"] - ex["target_profit_margin"]\n    F = ex["fixed_expense_per_policy"] * e\n    P = (R + cat + F) / den\n    Fpol = ex["fixed_expense_per_policy"] * e / exp_pol[r.policy_id]  # Allocate 85 CU per policy across its 2025 rows.\n    filas.append({"row_id": r.row_id, "ramo": r.line, "canal": r.channel, "nuevo_negocio": bool(r.new_business), "exposicion": e,\n                  "riesgo_designada": float(q.loc[q.row_id == r.row_id, "designada"].iloc[0]),\n                  "riesgo_mlind": float(q.loc[q.row_id == r.row_id, "mlind"].iloc[0]), "riesgo_mezcla": R,\n                  "reaseguro_cat": cat, "gasto_fijo": F, "gasto_variable": P * ex["variable_expense"], "comision": P * com[r.channel],\n                  "impuesto": P * ex["premium_tax"], "margen": P * ex["target_profit_margin"], "denominador": den,\n                  "prima_comercial": P, "prima_comercial_F_por_poliza": (R + cat + Fpol) / den, "prima_escrita": r.written_premium})\npc = pd.DataFrame(filas); pc.round(2).to_csv(OUT / "prima_comercial_20_mezcla.csv", index=False)\nres["prima_comercial"] = {"media_por_ramo": pc.groupby("ramo")[["riesgo_mezcla", "prima_comercial", "prima_escrita"]].mean().round(2).to_dict(),\n                          "suma_componentes_ok": bool(np.allclose(pc.riesgo_mezcla + pc.reaseguro_cat + pc.gasto_fijo + pc.gasto_variable + pc.comision + pc.impuesto + pc.margen, pc.prima_comercial))}\nguardar_json(res, OUT / "diagnostico_mezcla.json")\n# Comparison figure.\nplt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 130})\nC = {"motor": "#2a6f97", "buildings": "#c4673b", "contents": "#5b8c5a", "todos": "#333333"}\nfig, axs = plt.subplots(1, 2, figsize=(10.5, 3.6))\nfor l in LINEAS + ["todos"]:\n    c = res["curva_2024"][l]; base = c["D2"][0]\n    axs[0].plot(ws, np.array(c["D2"]) - base, color=C[l], lw=2 if l == "todos" else 1.2, label=f"{l} (óptimo {c[\'w_optimo\']:.2f}, IC95 {c[\'w_optimo_ic95\'][0]:.2f}-{c[\'w_optimo_ic95\'][1]:.2f})")\naxs[0].axvline(0.5, color="#999", ls="--", lw=0.8); axs[0].axhline(0, color="#ccc", lw=0.6)\naxs[0].set_xlabel("peso de MLind (el resto, A1)"); axs[0].set_ylabel("D² 2024 menos D² de A1 sola")\naxs[0].legend(frameon=False, fontsize=6.8); axs[0].set_title("2024 (ya visto por las dos líneas): autos y contenidos admiten 0,5; edificios prefiere MLind", fontsize=8)\nx = np.arange(3); w = 0.26\nfor j, (k, lab, col) in enumerate([("designada", "designada A1fixF1b3n", "#c4673b"), ("mezcla", "mezcla 50/50", "#5b8c5a"), ("mlind", "MLind", "#2a6f97")]):\n    vals = [res["2025"][k][l]["tasa"] for l in LINEAS]\n    axs[1].bar(x + (j - 1) * w, vals, w, label=lab, color=col)\n    for xi, v in zip(x + (j - 1) * w, vals):\n        axs[1].text(xi, v + 5, f"{v:.0f}", ha="center", fontsize=7)\naxs[1].set_xticks(x, LINEAS); axs[1].set_ylabel("tasa 2025 (CU por año de exposición)"); axs[1].legend(frameon=False, fontsize=7)\naxs[1].set_title("Producción 2025 de la combinación", fontsize=8.5)\nfig.tight_layout(); fig.savefig(OUT / "f11_mezcla.png"); plt.close(fig)\nprint(json.dumps({k: v for k, v in res.items() if k not in ("fuentes",)}, indent=1, default=float)[:7000])\n'


### Embedded source: `ml/codigo/pcv_v2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/pcv_v2.py'] = '"""Compare v2 versus T2b using 2023-2024 PCV OOF with fold-crossed weights and levels."""\nimport json, numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import mezcla_pesos, metricas, bootstrap_dif\nimport receta\nt = receta.tabla("pcv"); comps = ["T2b", "T2h", "FS", "T2o"]; P = receta.componentes("pcv", comps, t)\nfolds = pliegue(t.customer_id.values)\ndef oof(cs, enc):\n    out = np.zeros(len(t))\n    for l in LINEAS:\n        il = (t.line == l).values\n        for f in range(5):\n            te = folds == f; tr = ~te & il\n            M = np.column_stack([P[c][tr] * t.c_fila.values[tr] for c in cs])\n            w = mezcla_pesos(t.y.values[tr], M) if len(cs) > 1 else np.ones(1)\n            w = (1 - enc) * w + enc / len(cs)\n            b = np.column_stack([P[c] for c in cs]) @ w\n            k = factor_tweedie(t.y.values[tr], b[tr] * t.c_fila.values[tr])\n            out[te & il] = b[te & il] * k\n    return out\nv1 = oof(["T2b"], 0); res = {"v1_T2b": metricas(t, v1)}\nfor enc in (0, 0.5, 1):\n    v2 = oof(comps, enc); nom = f"v2_e{int(enc*100)}"\n    res[nom] = metricas(t, v2)\n    res[nom + "_vs_v1"] = {"todo": bootstrap_dif(t, v2, v1)}\n    for a in (2023, 2024):\n        s = (t.year == a).values\n        res[nom + "_vs_v1"][str(a)] = bootstrap_dif(t[s].reset_index(drop=True), v2[s], v1[s])\n    for nb in (True, False):\n        s = (t.new_business == nb).values\n        res[nom + "_vs_v1"]["NB" if nb else "REN"] = bootstrap_dif(t[s].reset_index(drop=True), v2[s], v1[s])\n    print(nom, "D2 %.4f" % res[nom]["D2"], {k: (round(v["dif"], 4), [round(x, 4) for x in v["ic95"]]) for k, v in res[nom + "_vs_v1"].items()})\nprint("v1 D2 %.4f" % res["v1_T2b"]["D2"], {k: round(v, 4) for k, v in res["v1_T2b"].items() if k.startswith("D2")})\nprint("v2 e50", {k: round(v, 4) for k, v in res["v2_e50"].items() if k.startswith("D2") or k.startswith("AE")})\nguardar_json(res, ENTREGA / "evolucion" / "pcv_v2_frente_v1.json")\n# Reweight 2023 rows toward the 2025 portfolio using density ratios.\ns = (t.year == 2023).values\nt23 = t[s].reset_index(drop=True).merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")\nassert t23.w2025.notna().all()\nv2 = oof(comps, 0.5)\nres["w2025_2023"] = {"v1": d2(t23.y.values, v1[s] * t23.c_fila.values, t23.w2025.values),\n                     "v2_e50": d2(t23.y.values, v2[s] * t23.c_fila.values, t23.w2025.values),\n                     "dif": bootstrap_dif(t23, v2[s], v1[s], w=t23.w2025.values)}\nprint("w2025 2023", {k: (round(v, 4) if isinstance(v, float) else (round(v["dif"], 4), [round(x, 4) for x in v["ic95"]])) for k, v in res["w2025_2023"].items()})\nguardar_json(res, ENTREGA / "evolucion" / "pcv_v2_frente_v1.json")\n'


### Embedded source: `ml/codigo/peligros.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/peligros.py'] = '"""T6/E05: transfer peril-specific structure with credible emerging-market levels.\nUsage: python peligros.py <phase>\n"""\nfrom __future__ import annotations\nimport sys, time\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom completitud import por_peligro\nfrom gbm import P_DEV, ajustar, CONF, MOD, OUT\n\nPELIGROS = {"motor": ["bodily_injury", "glass_hail", "own_damage", "theft", "third_party_prop"],\n            "buildings": ["accidental_damage", "escape_of_water", "fire", "flood", "storm", "subsidence"],\n            "contents": ["accidental_damage", "escape_of_water", "flood", "theft"]}\nKCRED = 100.0\nFASE = sys.argv[1] if len(sys.argv) > 1 else "dev"\n\nfor l in LINEAS:\n    t0 = time.time()\n    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])\n    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)\n    cols = comunes(l); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)\n    Xd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats)\n    raw = {}; rondas = {}; kdevs = {}\n    for p in PELIGROS[l]:\n        f = MOD / f"T6_{l}_{p}.txt"\n        cd = por_peligro(l, p, dv.period_start, dv.period_end)\n        offd = np.log(dv.exposure.values * np.maximum(cd, 1e-4))\n        if f.exists():\n            b = lgb.Booster(model_file=str(f)); n = b.current_iteration()\n        else:\n            b, n = ajustar(Xd, dv[f"y_{p}"].values, offd, P_DEV, 4000, es=100, val=dv.fold.values == 0)\n            b.save_model(str(f))\n        rondas[p] = n\n        # Relevel each developed peril: Tweedie L2 regularization can lower its level.\n        kdev = factor_tweedie(dv[f"y_{p}"].values, np.exp(b.predict(Xd, raw_score=True) + offd))\n        kdevs[p] = kdev\n        raw[p] = b.predict(Xe, raw_score=True) + np.log(kdev)\n        em[f"c_{p}"] = np.where(em.conjunto == "test", 1.0, por_peligro(l, p, em.period_start, em.period_end))\n    for p in PELIGROS[l]:\n        em[f"raw_{p}"] = raw[p]\n    em[[c for c in em.columns if c.startswith("raw_") or c == "row_id"]].to_parquet(OUT / f"T6raw_{l}.parquet", index=False)\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)\n    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF["em_eval"] is None else \\\n        [(tr, em[em.year == CONF["em_eval"]].reset_index(drop=True))]\n    preds = []; niveles = []\n    for a, b in sets:\n        # Shared line-level calibration.\n        mu_a = sum(np.exp(a[f"raw_{p}"].values) * a.exposure.values * a[f"c_{p}"].values for p in PELIGROS[l])\n        kl = factor_tweedie(a.y.values, mu_a)\n        kp = {}\n        for p in PELIGROS[l]:\n            m = np.exp(a[f"raw_{p}"].values) * a.exposure.values * a[f"c_{p}"].values * kl\n            kem = factor_tweedie(a[f"y_{p}"].values, m)\n            n = int((a[f"y_{p}"].values > 0).sum()); z = n / (n + KCRED)\n            kp[p] = kl * (z * kem + (1 - z))\n        niveles.append({"k_ramo": kl, **{p: kp[p] for p in kp}})\n        mu = sum(kp[p] * np.exp(b[f"raw_{p}"].values) for p in PELIGROS[l]) * b.exposure.values\n        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": mu}))\n    pd.concat(preds).to_parquet(OUT / f"T6_{l}.parquet", index=False)\n    guardar_json({"rondas": rondas, "k_desarrollado": kdevs, "niveles": niveles, "s": round(time.time() - t0)}, OUT / f"T6_{l}_meta.json")\n    print(l, rondas, [{k: round(v, 3) for k, v in n.items()} for n in niveles][:2], round(time.time() - t0), "s", flush=True)\n'


### Embedded source: `ml/codigo/pesos_2025.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/pesos_2025.py'] = '"""W01: DEV-2023-to-test-2025 density weights for portfolio-weighted D2 diagnostics."""\nimport sys\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\n\nANIO = int(sys.argv[1]) if len(sys.argv) > 1 else 2023\nCLIMA = "hasta2023"\nresumen = {}\nsalida = []\nfor l in LINEAS:\n    em = cargar("emerging", l, CLIMA)\n    cols = [c for c in comunes(l) + SOLO_EMERGENTE if c not in ("hist_pol_12m", "hist_cli_12m")]\n    a = em[(em.conjunto == "train") & (em.year == ANIO)]\n    b = em[em.conjunto == "test"]\n    d = pd.concat([a.assign(z=0), b.assign(z=1)], ignore_index=True)\n    cats = categorias_union([d], cols)\n    X = matriz(d, cols, cats); z = d.z.values\n    fo = pliegue(d.customer_id.values)\n    p = np.zeros(len(d))\n    params = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8,\n                  bagging_fraction=0.8, bagging_freq=1, verbose=-1, num_threads=2, seed=7)\n    for k in range(5):\n        m = lgb.train(params, lgb.Dataset(X[fo != k], z[fo != k]), 300)\n        p[fo == k] = m.predict(X[fo == k])\n    from sklearn.metrics import roc_auc_score\n    auc = roc_auc_score(z, p)\n    imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)\n    pa = np.clip(p[z == 0], 1e-6, 1 - 1e-6)\n    w = pa / (1 - pa) * (len(a) / len(b))\n    w = np.clip(w, 0.1, 10); w = w / w.mean()\n    ess = w.sum() ** 2 / (w ** 2).sum()\n    resumen[l] = {"auc": round(float(auc), 4), "n_dev": int(len(a)), "n_test": int(len(b)), "ess": round(float(ess), 1),\n                  "nb_dev": round(float(a.new_business.mean()), 4),\n                  "nb_ponderado": round(float(np.average(a.new_business, weights=w)), 4),\n                  "nb_test": round(float(b.new_business.mean()), 4),\n                  "importancia_top": {k: round(float(v / imp.sum()), 4) for k, v in imp.head(8).items()}}\n    salida.append(pd.DataFrame({"row_id": a.row_id.values, "w2025": w}))\n    print(l, resumen[l])\nsuf = "dev" if ANIO == 2023 else "hold"\npd.concat(salida).to_parquet(TRABAJO / f"pesos_2025_{suf}.parquet", index=False)\nguardar_json(resumen, TRABAJO / f"pesos_2025_{suf}.json")\n'


### Embedded source: `ml/codigo/preparar_datos.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/preparar_datos.py'] = '"""Build labels, reporting completeness and the shared transfer-model feature schema.\nRead only fingerprint-verified original Parquet files. Write private\ntrabajo/base_<market>_<line>.parquet and trabajo/preparacion.json.\n"""\nfrom __future__ import annotations\nimport numpy as np, pandas as pd\nfrom comun import *\n\nTRABAJO.mkdir(parents=True, exist_ok=True)\ninfo = {"huellas": comprobar_huellas()}\n\nUMBRAL_CAPITAL = {"buildings": 5_000_000, "contents": 2_000_000}\nCLIMA = ["wind_exposure_index", "wildfire_index", "rainfall_mm", "mean_temperature_c"]\nCONSTR = {"brick": "brick", "brk": "brick", "stone": "stone", "stn": "stone", "t": "timber", "w": "timber",\n          "wod": "timber", "non_standard": "non_standard", "mix": "non_standard"}\n\n\ndef banda_edad(a):\n    return pd.cut(a, [-1, 24, 34, 49, 64, 200], labels=["17-24", "25-34", "35-49", "50-64", "65+"]).astype(str)\n\n\ndef leer_polizas(m):\n    out = {}\n    for l in LINEAS:\n        tr = pd.read_parquet(DATOS / m / f"policies_{l}.parquet")\n        tr["conjunto"] = "train"\n        if m == "emerging":\n            te = pd.read_parquet(DATOS / m / f"test_{l}.parquet"); te["conjunto"] = "test"\n            tr = pd.concat([tr, te], ignore_index=True)\n        tr["line"] = l\n        out[l] = tr\n    return out\n\n\ndef siniestros(m):\n    c = pd.read_parquet(DATOS / m / "claims.parquet")\n    n0 = len(c)\n    if m == "developed":\n        cols = [x for x in c.columns if x != "claim_id"]\n        dup = c.duplicated(cols, keep=False)\n        quitar = dup & (c.claim_id.str.len() == 11)\n        c = c[~quitar].copy()\n        info[f"{m}_duplicados_retirados"] = int(quitar.sum())\n    c["loss_date"] = pd.to_datetime(c.loss_date); c["report_date"] = pd.to_datetime(c.report_date)\n    c["neto"] = c.incurred_to_date - c.recoveries\n    c["aceptado"] = c.status != "repudiated"\n    info[f"{m}_siniestros"] = {"leidos": n0, "usados": len(c), "rechazados": int((~c.aceptado).sum()),\n                                "neto_negativo": int((c.neto < 0).sum())}\n    return c\n\n\n# Cost-weighted reporting completeness from mature developed-market cohorts.\ncd = siniestros("developed")\nce = siniestros("emerging")\nD = 1095\nF = {}\nresumen_cdf = {}\nfor (l, p), g in cd[cd.aceptado].groupby(["line", "peril"]):\n    g = g[(V - g.loss_date.values.astype("datetime64[D]")).astype(int) >= D]\n    dly = (g.report_date - g.loss_date).dt.days.values\n    w = g.neto.values\n    ok = dly <= D\n    tot = w[ok].sum()\n    cum = np.bincount(dly[ok], weights=w[ok], minlength=D + 1).cumsum() / tot\n    F[(l, p)] = np.clip(cum, 0, 1)\n    resumen_cdf[f"{l}|{p}"] = {k: round(float(cum[k]), 4) for k in [0, 30, 90, 180, 365, 730]}\n    resumen_cdf[f"{l}|{p}"]["n"] = int(ok.sum())\ninfo["cdf_coste_desarrollado"] = resumen_cdf\n# Emerging contrast: 2023 cohorts observed for at least 365 days, using F(d)/F(365).\ncontraste = {}\nfor (l, p), g in ce[ce.aceptado & (ce.loss_date.dt.year == 2023)].groupby(["line", "peril"]):\n    g = g[(V - g.loss_date.values.astype("datetime64[D]")).astype(int) >= 365]\n    dly = (g.report_date - g.loss_date).dt.days.values\n    w = g.neto.values; ok = dly <= 365\n    if ok.sum() < 20: continue\n    em = [float(w[ok & (dly <= d)].sum() / w[ok].sum()) for d in (30, 90, 180)]\n    dv = [float(F[(l, p)][d] / F[(l, p)][365]) for d in (30, 90, 180)]\n    contraste[f"{l}|{p}"] = {"n": int(ok.sum()), "emergente_30_90_180": [round(x, 4) for x in em],\n                              "desarrollado_30_90_180": [round(x, 4) for x in dv]}\ninfo["contraste_reporte_emergente_2023"] = contraste\nG = {k: np.concatenate([[0.0], np.cumsum(np.concatenate([v, np.ones(4000 - len(v))]))]) for k, v in F.items()}\n\n\ndef completitud_media(l, p, s, e):\n    """Average F(V-t) for t in [s,e], using cumulative G[x+1]=sum_{d<=x} F(d)."""\n    g = G[(l, p)]\n    a = (V - e).astype(int); b = (V - s).astype(int)\n    a = np.clip(a, 0, None); b = np.clip(b, 0, None)\n    return (g[b + 1] - g[a]) / (b - a + 1)\n\n\n# Build each market\'s base tables.\nfor m in ["developed", "emerging"]:\n    c = cd if m == "developed" else ce\n    pol = leer_polizas(m)\n    cu = pd.read_parquet(DATOS / m / "customers.parquet")\n    ad = pd.read_parquet(DATOS / m / "addresses.parquet")\n    we = pd.read_parquet(DATOS / m / "weather.parquet")\n    # Peril weights use accepted cost in the most mature cohort, adjusted for completeness.\n    for l in LINEAS:\n        df = pol[l].copy()\n        n_in = len(df)\n        if m == "developed" and l == "motor":\n            df["customer_id"] = df.customer_id.str.zfill(8)\n        df["period_start"] = pd.to_datetime(df.period_start); df["period_end"] = pd.to_datetime(df.period_end)\n        df["exposure_original"] = df.exposure\n        df["exposure"] = ((df.period_end - df.period_start).dt.days + 1) / 365.0\n        df["year"] = df.period_start.dt.year\n        # Insured-value processing.\n        if l in UMBRAL_CAPITAL:\n            df["sum_insured_raw"] = df.sum_insured\n            adj = df.sum_insured > UMBRAL_CAPITAL[l]\n            df["audit_capital_div1000"] = adj\n            df["sum_insured"] = np.where(adj, df.sum_insured / 1000, df.sum_insured)\n            info[f"{m}_{l}_capital_div1000"] = {"train": int((adj & (df.conjunto == "train")).sum()),\n                                                  "test": int((adj & (df.conjunto == "test")).sum())}\n        # Claims-to-period assignment.\n        cl = c[c.line == l]\n        mm = cl.merge(df[["row_id", "policy_id", "period_start", "period_end"]], on="policy_id", how="inner")\n        mm = mm[(mm.loss_date >= mm.period_start) & (mm.loss_date <= mm.period_end)]\n        dup = mm.claim_id.duplicated(keep=False).sum()\n        asign = mm.claim_id.nunique()\n        info[f"{m}_{l}_asignacion"] = {"siniestros_ramo": int(len(cl)), "asignados": int(asign),\n                                        "ambiguos": int(dup), "sin_periodo": int(len(cl) - asign)}\n        acc = mm[mm.aceptado]\n        y = acc.groupby("row_id").neto.sum()\n        df["y"] = df.row_id.map(y).fillna(0.0)\n        df.loc[df.conjunto == "test", "y"] = np.nan\n        df["n_sin"] = df.row_id.map(acc.groupby("row_id").size()).fillna(0)\n        for p_ in sorted(cl.peril.unique()):\n            df[f"y_{p_}"] = df.row_id.map(acc[acc.peril == p_].groupby("row_id").neto.sum()).fillna(0.0)\n        # Row-level reporting completeness.\n        s = df.period_start.values.astype("datetime64[D]"); e = df.period_end.values.astype("datetime64[D]")\n        perils = sorted(cl.peril.unique())\n        comp = {p_: completitud_media(l, p_, s, e) for p_ in perils}\n        # Weights use accepted training-year cost divided by average completeness.\n        tr = df.conjunto == "train"\n        cost = {p_: df.loc[tr, f"y_{p_}"].sum() for p_ in perils}\n        ult = {p_: cost[p_] / max(np.average(comp[p_][tr], weights=df.exposure[tr]), 1e-6) for p_ in perils}\n        tot = sum(ult.values())\n        wpk = {p_: ult[p_] / tot for p_ in perils}\n        df["c_fila"] = sum(wpk[p_] * comp[p_] for p_ in perils)\n        df.loc[df.conjunto == "test", "c_fila"] = 1.0\n        info[f"{m}_{l}_pesos_peligro"] = {k: round(v, 4) for k, v in wpk.items()}\n        info[f"{m}_{l}_completitud_media_por_anio"] = df[tr].groupby("year").apply(\n            lambda g: float(np.average(g.c_fila, weights=g.exposure))).round(4).to_dict()\n        # History includes notifications reported before policy start, regardless of status.\n        hp = c[c.line == l][["policy_id", "report_date"]].merge(df[["row_id", "policy_id", "period_start"]], on="policy_id")\n        hp = hp[hp.report_date < hp.period_start]\n        df["hist_pol_n"] = df.row_id.map(hp.groupby("row_id").size()).fillna(0)\n        df["prev_rows_pol"] = df.groupby("policy_id").period_start.rank(method="first") - 1\n        out_rows = df\n        pol[l] = out_rows\n        assert len(df) == n_in\n    # Customer notification history across all lines.\n    mapa = pd.concat([pol[l][["policy_id", "customer_id"]].assign(line=l) for l in LINEAS]).drop_duplicates(["policy_id", "line"])\n    cc = c[["policy_id", "line", "report_date"]].merge(mapa, on=["policy_id", "line"])\n    for l in LINEAS:\n        df = pol[l]\n        h = cc.merge(df[["row_id", "customer_id", "period_start"]], on="customer_id")\n        h = h[h.report_date < h.period_start]\n        df["hist_cli_n"] = df.row_id.map(h.groupby("row_id").size()).fillna(0)\n    # Master-data joins.\n    cu = cu.copy()\n    if m == "emerging":\n        occ = cu.occupation.str.lower().str.strip()\n        cu["self_employed"] = occ.str.startswith("self-employed")\n        cu["occupation"] = occ.str.replace("self-employed ", "", regex=False).str.strip()\n    ad = ad.copy()\n    ad["energy_rating"] = ad.energy_rating\n    for l in LINEAS:\n        df = pol[l].merge(cu, on="customer_id", how="left", validate="m:1").merge(ad, on="address_id", how="left", validate="m:1")\n        df["tenure_years"] = df.year - df.customer_since_year\n        df["property_age"] = df.year - df.year_built\n        df["start_month"] = df.period_start.dt.month\n        df["start_doy"] = df.period_start.dt.dayofyear\n        df["full_year"] = (df.exposure >= 0.99).astype(int)\n        df["log_days_quote"] = np.log1p(df.days_quote_to_inception.clip(lower=0))\n        wp = df.written_premium / df.exposure\n        med = wp.groupby([df.year]).transform("median")\n        df["log_wp_rel"] = np.log(wp / med)\n        if l == "motor":\n            if m == "developed":\n                df["driver_age_band"] = banda_edad(df.driver_age)\n            df["age_band"] = df.driver_age_band\n        else:\n            if m == "developed":\n                df["policyholder_age_band"] = banda_edad(df.policyholder_age)\n            df["age_band"] = df.policyholder_age_band\n            df["log_si"] = np.log(df.sum_insured)\n        if l == "buildings":\n            df["construction"] = df.construction_type.str.lower().map(CONSTR)\n            assert df.construction.notna().all(), df.construction_type[df.construction.isna()].unique()\n        df["market"] = m\n        df["fold"] = pliegue(df.customer_id.values)\n        df.to_parquet(TRABAJO / f"base_{m}_{l}.parquet", index=False)\n        info[f"{m}_{l}_filas"] = df.conjunto.value_counts().to_dict()\n        info[f"{m}_{l}_T0_por_anio"] = df[df.conjunto == "train"].groupby("year").y.sum().round(2).to_dict()\n    # Persistent weather: address averages over each permitted year window.\n    for nombre, anios in ({"developed": {"hasta2023": [2021, 2022, 2023], "hasta2024": [2021, 2022, 2023, 2024]},\n                           "emerging": {"hasta2023": [2023], "hasta2024": [2023, 2024]}}[m]).items():\n        wm = we[we.year.isin(anios)].groupby("address_id")[CLIMA].mean().add_prefix("clima_")\n        wm.reset_index().to_parquet(TRABAJO / f"clima_{m}_{nombre}.parquet", index=False)\nguardar_json(info, TRABAJO / "preparacion.json")\nprint(json.dumps({k: v for k, v in info.items() if k != "huellas" and "cdf" not in k}, indent=0, default=str)[:6000])\n'


### Embedded source: `ml/codigo/prima_comercial.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/prima_comercial.py'] = '"""Price the 20 reference policies from predicted 2025 risk premiums.\n\nP=[R*(1+cat)+F*e]/(1-variable expenses-channel commission-tax-margin).\ncat=0.12 for household property only; F=85 CU per policy-year, allocated by\nexposure. The sensitivity charges the entire fixed expense to each row.\n"""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\n\nex = pd.read_parquet(DATOS / "emerging" / "expenses.parquet").set_index("item").emerging\nref = pd.read_parquet(DATOS / "emerging" / "reference_policies.parquet")\npred = pd.read_csv(ENTREGA / "prediccion_2025_mlind.csv", dtype={"row_id": str})\nd = ref.merge(pred, on="row_id", how="left")\nbal = json.load(open(ENTREGA / "produccion_2025.json"))["balance_oof_pcv"]\nassert d.prediction.notna().all()\ncom = {"direct": ex["commission_direct"], "broker": ex["commission_broker"], "comparison_site": ex["commission_comparison_site"]}\nfilas = []\nfor _, r in d.iterrows():\n    R0 = r.prediction; e = r.exposure\n    R = R0 * bal[r.line]  # Use the balanced mean risk premium for commercial pricing, not the deviance-optimal factor.\n    cat = ex["cat_reinsurance"] * R if r.line in ("buildings", "contents") else 0.0\n    c = com[r.channel]\n    den = 1 - ex["variable_expense"] - c - ex["premium_tax"] - ex["target_profit_margin"]\n    F = ex["fixed_expense_per_policy"] * e\n    P = (R + cat + F) / den\n    P_alt = (R + cat + ex["fixed_expense_per_policy"]) / den\n    filas.append({"row_id": r.row_id, "ramo": r.line, "canal": r.channel, "exposicion": e, "nuevo_negocio": bool(r.new_business),\n                  "prediccion_kaggle": R0, "factor_balance": bal[r.line], "prima_riesgo": R, "reaseguro_cat": cat, "gasto_fijo": F, "carga_neta": R + cat + F, "denominador": den,\n                  "gasto_variable": P * ex["variable_expense"], "comision": P * c, "impuesto": P * ex["premium_tax"],\n                  "margen": P * ex["target_profit_margin"], "prima_comercial": P, "prima_comercial_F_entero": P_alt,\n                  "prima_escrita": r.written_premium, "razon_comercial_escrita": P / r.written_premium})\nout = pd.DataFrame(filas)\nout.round(2).to_csv(ENTREGA / "prima_comercial_20.csv", index=False)\nprint(out[["row_id", "ramo", "canal", "exposicion", "prima_riesgo", "prima_comercial", "prima_escrita", "razon_comercial_escrita"]].round(2).to_string())\n'


### Embedded source: `ml/codigo/produccion.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/produccion.py'] = '"""Produce 2025 predictions from the frozen A03 transfer recipe.\nApply developed-2021-2024 T2b structure, emerging-2023-2024 line levels and\ncross-fitted PCV C1 without trend. Retain C2, developed-trend and preregistered\nC2-plus-trend alternatives as documented sensitivities.\n"""\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nimport receta\n\nparams_c2 = receta.parametros("pcv", ["T2b"], "ramo_nb")\nparams_c1 = dict(params_c2, calib="ramo")\ntend = json.load(open(TRABAJO / "pred" / "prod" / "tendencia.json"))\np2c = receta.parametros("pcv", ["T2c"], "ramo")\nvariantes = {"final_C1": (params_c1, None), "C2": (params_c2, None), "C1_tendencia": (params_c1, tend),\n             "preinscrita_C2_tendencia": (params_c2, tend), "T2c_C1": (p2c, None)}\nsub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")\nres = {"params": params_c2, "tendencia": tend, "variantes": {}}\nfor nom, (p, tt) in variantes.items():\n    t, mu = receta.aplicar("prod", p, tt)\n    out = sub[["row_id"]].merge(pd.DataFrame({"row_id": t.row_id.values, "prediction": mu}), on="row_id", how="left")\n    ctl = {"filas": int(len(out)), "orden_igual_plantilla": bool((out.row_id.values == sub.row_id.values).all()),\n           "nulos": int(out.prediction.isna().sum()), "min": float(out.prediction.min()),\n           "positivas_estrictas": bool((out.prediction > 0).all()), "ids_unicos": bool(out.row_id.is_unique)}\n    assert ctl["filas"] == 209063 and ctl["nulos"] == 0 and ctl["positivas_estrictas"] and ctl["orden_igual_plantilla"], ctl\n    f = ENTREGA / ("prediccion_2025_mlind.csv" if nom == "final_C1" else f"prediccion_2025_mlind_{nom}.csv")\n    out.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")\n    ctl["sha256"] = sha256(f); ctl["archivo"] = f.name\n    tot = {}\n    for l in LINEAS:\n        i = (t.line == l).values; e = t.exposure.values[i]; nb = t.new_business.values[i]\n        tot[l] = {"filas": int(i.sum()), "exposicion": float(e.sum()), "total": float(mu[i].sum()), "tasa": float(mu[i].sum() / e.sum()),\n                  "tasa_NB": float(mu[i][nb].sum() / e[nb].sum()), "tasa_REN": float(mu[i][~nb].sum() / e[~nb].sum())}\n    res["variantes"][nom] = {"controles": ctl, "totales": tot, "total": float(mu.sum())}\n    print(nom, round(mu.sum()), {l: round(v["tasa"], 1) for l, v in tot.items()}, ctl["sha256"][:12])\n# Commercial balance factor by line: sum(y)/sum(mu*c) on cross-fitted PCV predictions.\nt, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")\nres["balance_oof_pcv"] = {l: float(t.y[(t.line == l).values].sum() / (mu * t.c_fila.values)[(t.line == l).values].sum()) for l in LINEAS}\nguardar_json(res, ENTREGA / "produccion_2025.json")\nprint("balance", res["balance_oof_pcv"])\n'


### Embedded source: `ml/codigo/produccion2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/produccion2.py'] = '"""Produce convex transferred-component blends with C1 and no trend.\nEstimate line weights on 2023-2024 PCV OOF and shrink toward equal weights by\nthe requested fraction. Save private 2025 predictions, the fixed 50/50 designated\nblend and an aggregate production record.\nUsage: python produccion2.py <name> <shrinkage> <component> [...]\n"""\nimport sys\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import mezcla_pesos, metricas\nimport receta\n\nNOM, ENC, COMPS = sys.argv[1], float(sys.argv[2]), sys.argv[3:]\nPRIV = RAIZ / "privado"\nDESIGNADA = RAIZ / "comparacion" / "otros" / "A1fixF1b3n_2025.csv"\nSHA_DESIGNADA = "f786deb3"\n\n\ndef pesos(t, P, il):\n    M = np.column_stack([P[c][il] * t.c_fila.values[il] for c in COMPS])\n    w = mezcla_pesos(t.y.values[il], M) if len(COMPS) > 1 else np.ones(1)\n    return (1 - ENC) * w + ENC / len(COMPS)\n\n\ndef controles(out, sub):\n    c = {"filas": int(len(out)), "orden_igual_plantilla": bool((out.row_id.values == sub.row_id.values).all()),\n         "nulos": int(out.prediction.isna().sum()), "min": float(out.prediction.min()),\n         "positivas_estrictas": bool((out.prediction > 0).all()), "ids_unicos": bool(out.row_id.is_unique)}\n    assert c["filas"] == 209063 and c["nulos"] == 0 and c["positivas_estrictas"] and c["orden_igual_plantilla"], c\n    return c\n\n\ntp = receta.tabla("pcv"); Pp = receta.componentes("pcv", COMPS, tp)\ntt = receta.tabla("prod"); Pt = receta.componentes("prod", COMPS, tt)\nmu = np.zeros(len(tt)); params = {}\n# Cross-fit weights and levels for PCV evaluation and commercial balance.\nfolds = pliegue(tp.customer_id.values); oof = np.zeros(len(tp))\nfor l in LINEAS:\n    il = (tp.line == l).values; it = (tt.line == l).values\n    w = pesos(tp, Pp, il)\n    b = np.column_stack([Pp[c][il] for c in COMPS]) @ w\n    k = factor_tweedie(tp.y.values[il], b * tp.c_fila.values[il])\n    mu[it] = (np.column_stack([Pt[c][it] for c in COMPS]) @ w) * k\n    params[l] = {"pesos": dict(zip(COMPS, np.round(w, 4).tolist())), "k_ramo": k}\n    for f in range(5):\n        te = folds == f\n        tr_ = tp[~te].reset_index(drop=True)\n        wf = pesos(tr_, {c: Pp[c][~te] for c in COMPS}, il[~te])\n        bf = np.column_stack([Pp[c] for c in COMPS]) @ wf\n        kf = factor_tweedie(tp.y.values[~te & il], bf[~te & il] * tp.c_fila.values[~te & il])\n        oof[te & il] = bf[te & il] * kf\nsub = pd.read_parquet(DATOS / "emerging" / "sample_submission.parquet")\nout = sub[["row_id"]].merge(pd.DataFrame({"row_id": tt.row_id.values, "prediction": mu}), on="row_id", how="left")\nres = {"componentes": COMPS, "encoge": ENC, "params": params, "controles": controles(out, sub)}\nf = PRIV / f"prediccion_2025_{NOM}.csv"\nout.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")\nres["controles"]["sha256"] = sha256(f); res["controles"]["archivo"] = f.name\ntot = {}\nfor l in LINEAS:\n    i = (tt.line == l).values; e = tt.exposure.values[i]; nb = tt.new_business.values[i]\n    tot[l] = {"total": float(mu[i].sum()), "tasa": float(mu[i].sum() / e.sum()),\n              "tasa_NB": float(mu[i][nb].sum() / e[nb].sum()), "tasa_REN": float(mu[i][~nb].sum() / e[~nb].sum())}\nres["totales"] = tot; res["total"] = float(mu.sum())\nres["oof_pcv"] = metricas(tp, oof)\nres["balance_oof_pcv"] = {l: float(tp.y[(tp.line == l).values].sum() / (oof * tp.c_fila.values)[(tp.line == l).values].sum()) for l in LINEAS}\n# Fixed 50/50 designated blend under the recorded blend rule.\nassert sha256(DESIGNADA).startswith(SHA_DESIGNADA)\nd = pd.read_csv(DESIGNADA)\nassert (d.row_id.values == sub.row_id.values).all()\nm50 = pd.DataFrame({"row_id": sub.row_id.values, "prediction": 0.5 * d.prediction.values + 0.5 * out.prediction.values})\nf50 = PRIV / f"mezcla50_designada_{NOM}_2025.csv"\nm50.to_csv(f50, index=False, float_format="%.6f", lineterminator="\\n")\nres["mezcla50"] = {**controles(m50, sub), "sha256": sha256(f50), "archivo": f50.name, "total": float(m50.prediction.sum())}\nlm = np.log(out.prediction.values); ld = np.log(d.prediction.values)\nres["frente_designada"] = {"corr_log": float(np.corrcoef(lm, ld)[0, 1]), "cociente_p01_p99": [\n    float(np.quantile(np.exp(lm - ld), 0.01)), float(np.quantile(np.exp(lm - ld), 0.99))]}\nv1 = pd.read_csv(PRIV / "prediccion_2025_mlind.csv")\nlv = np.log(v1.prediction.values)\nres["frente_mlind_v1"] = {"corr_log": float(np.corrcoef(lm, lv)[0, 1]), "total_v1": float(v1.prediction.sum())}\nguardar_json(res, ENTREGA / f"produccion_{NOM}.json")\nprint(NOM, round(res["total"]), {l: round(v["tasa"], 1) for l, v in tot.items()}, res["controles"]["sha256"][:12],\n      "OOF pcv D2 %.4f" % res["oof_pcv"]["D2"], "corr designada %.4f v1 %.4f" % (res["frente_designada"]["corr_log"], res["frente_mlind_v1"]["corr_log"]))\n'


### Embedded source: `ml/codigo/receta.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/receta.py'] = '"""Apply convex line blends, C1/C2 calibration and optional trend.\nparametros estimates weights/factors on DEV for HOLD or PCV for production.\naplicar generates the target phase\'s predictions from those fixed parameters.\n"""\nfrom __future__ import annotations\nimport json\nimport numpy as np, pandas as pd\nfrom comun import *\nfrom evaluar import tabla_eval, cargar_pred, mezcla_pesos, mezcla_cruzada, calibrar_cruzado, metricas, bootstrap_dif\n\nK_NB = 200.0\n\n\ndef tabla(fase):\n    if fase == "dev":\n        t = tabla_eval("dev", 2023)\n    elif fase == "hold":\n        t = tabla_eval("hold", 2024)\n    elif fase == "pcv":\n        t = pd.concat([tabla_eval("dev", 2023), tabla_eval("dev", 2024)], ignore_index=True)\n    else:  # Production uses test rows.\n        filas = []\n        for l in LINEAS:\n            b = pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet",\n                                columns=["row_id", "y", "c_fila", "exposure", "customer_id", "year", "new_business", "conjunto"])\n            filas.append(b[b.conjunto == "test"].assign(line=l))\n        t = pd.concat(filas, ignore_index=True)\n    return t.reset_index(drop=True)\n\n\ndef componentes(fase, comps, t):\n    out = {}\n    for c in comps:\n        p = cargar_pred(fase, c)\n        if p is None:\n            raise FileNotFoundError(f"{fase}/{c}")\n        mu = t[["row_id"]].merge(p, on="row_id", how="left").mu.values\n        if np.isnan(mu).any():\n            raise ValueError(f"faltan filas en {fase}/{c}")\n        out[c] = mu\n    return out\n\n\ndef parametros(fase_oof, comps, calib="ramo_nb"):\n    t = tabla(fase_oof)\n    P = componentes(fase_oof, comps, t)\n    y = t.y.values; res = {"componentes": comps, "calib": calib, "ramos": {}}\n    for l in LINEAS:\n        il = (t.line == l).values\n        M = np.column_stack([P[c][il] * t.c_fila.values[il] for c in comps])\n        w = mezcla_pesos(y[il], M)\n        m = M @ w\n        kl = factor_tweedie(y[il], m)\n        r = {"pesos": dict(zip(comps, w.round(6).tolist())), "k_ramo": kl, "k_nb": {}}\n        for nb in (True, False):\n            s = t.new_business.values[il] == nb\n            ks = factor_tweedie(y[il][s], m[s] * kl); n = int((y[il][s] > 0).sum()); z = n / (n + K_NB)\n            r["k_nb"][str(nb)] = {"k_seg": ks, "n_sin": n, "Z": z, "factor": kl * (z * ks + 1 - z)}\n        res["ramos"][l] = r\n    return res\n\n\ndef aplicar(fase, params, tend=None):\n    t = tabla(fase)\n    comps = params["componentes"]; P = componentes(fase, comps, t)\n    mu = np.zeros(len(t))\n    for l in LINEAS:\n        il = (t.line == l).values; r = params["ramos"][l]\n        U = np.column_stack([P[c][il] for c in comps]); w = np.array([r["pesos"][c] for c in comps])\n        base = U @ w\n        if params["calib"] == "ramo":\n            f = np.full(il.sum(), r["k_ramo"])\n        else:\n            nb = t.new_business.values[il]\n            f = np.where(nb, r["k_nb"]["True"]["factor"], r["k_nb"]["False"]["factor"])\n        tf = 1.0 if tend is None else tend[l]["factor"]\n        mu[il] = base * f * tf\n    return t, mu\n\n\ndef oof_cruzado(fase, comps, calib="ramo_nb"):\n    """Generate DEV/PCV OOF predictions with fold-crossed blend weights and calibration."""\n    t = tabla(fase); P = componentes(fase, comps, t)\n    folds = pliegue(t.customer_id.values)\n    if len(comps) > 1:\n        mu, pesos = mezcla_cruzada(t, P, folds)\n    else:\n        mu, pesos = P[comps[0]], []\n    mu = calibrar_cruzado(t, mu, folds, por_nb=(calib == "ramo_nb"), K=K_NB) if calib != "sin" else mu\n    return t, mu, pesos\n'


### Embedded source: `ml/codigo/red.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/red.py'] = '"""Fit an FT-Transformer with power-1.5 Tweedie loss and an exposure offset.\nPretrain on developed data and fine-tune on emerging data.\nUsage: python red.py pre <phase> <line>\n       python red.py fino <phase> <line> <N1|N2a|N2b|N3>\nDEV/HOLD share developed 2021-2023; production uses 2021-2024.\n"""\nfrom __future__ import annotations\nimport sys, time, math, hashlib\nimport numpy as np, pandas as pd, torch, torch.nn as nn\nfrom comun import *\nfrom variables import *\n\ntorch.set_num_threads(2)\nSEMILLA = 20261024\n\n\nclass Tokenizador(nn.Module):\n    def __init__(self, n_num, cards, d):\n        super().__init__()\n        self.w = nn.Parameter(torch.randn(n_num, d) * 0.1)\n        self.b = nn.Parameter(torch.zeros(n_num, d))\n        self.miss = nn.Parameter(torch.randn(n_num, d) * 0.1)\n        self.emb = nn.ModuleList([nn.Embedding(c + 1, d) for c in cards])  # Index 0 represents unknown or missing categories.\n        for e in self.emb:\n            nn.init.normal_(e.weight, std=0.1)\n\n    def forward(self, xn, mn, xc):\n        t = xn.unsqueeze(-1) * self.w + self.b\n        t = torch.where(mn.unsqueeze(-1), self.miss.expand_as(t), t)\n        if len(self.emb):\n            tc = torch.stack([e(xc[:, j]) for j, e in enumerate(self.emb)], 1)\n            t = torch.cat([t, tc], 1)\n        return t\n\n\nclass Bloque(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__()\n        self.n1 = nn.LayerNorm(d); self.att = nn.MultiheadAttention(d, h, dropout=drop, batch_first=True)\n        self.n2 = nn.LayerNorm(d)\n        self.ff = nn.Sequential(nn.Linear(d, 2 * d * 2), nn.GLU(), nn.Dropout(drop), nn.Linear(2 * d, d))\n        self.dr = nn.Dropout(drop)\n\n    def forward(self, x):\n        z = self.n1(x); x = x + self.dr(self.att(z, z, z, need_weights=False)[0])\n        return x + self.dr(self.ff(self.n2(x)))\n\n\nclass FTT(nn.Module):\n    def __init__(self, n_num, cards, d=32, h=2, L=2, drop=0.1):\n        super().__init__()\n        self.tok = Tokenizador(n_num, cards, d)\n        self.extra = None  # Additional emerging-market tokens.\n        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.1)\n        self.bloques = nn.ModuleList([Bloque(d, h, drop) for _ in range(L)])\n        self.nf = nn.LayerNorm(d); self.cab = nn.Linear(d, 1)\n        self.sesgo = nn.Parameter(torch.zeros(1))\n\n    def forward(self, xn, mn, xc, xe=None):\n        t = self.tok(xn, mn, xc)\n        if self.extra is not None and xe is not None:\n            t = torch.cat([t, torch.stack([e(xe[:, j]) for j, e in enumerate(self.extra)], 1)], 1)\n        x = torch.cat([self.cls.expand(len(t), -1, -1), t], 1)\n        for b in self.bloques:\n            x = b(x)\n        return self.cab(self.nf(x[:, 0])).squeeze(-1) + self.sesgo\n\n\ndef perdida(eta, y, off, w):\n    """Power-1.5 Tweedie deviance without constant terms, with mu=exp(eta+off)."""\n    s = eta + off\n    return torch.sum(w * 2 * (2 * y * torch.exp(-0.5 * s) + 2 * torch.exp(0.5 * s))) / torch.sum(w)\n\n\nclass Prepro:\n    """Fit developed-market quantile-normal numeric transforms and categorical codes."""\n\n    def __init__(self, df, num, cat, cats):\n        self.num, self.cat, self.cats = num, cat, cats\n        self.q = {}\n        for c in num:\n            v = pd.to_numeric(df[c], errors="coerce").astype(float).values\n            v = v[~np.isnan(v)]\n            qs = np.unique(np.quantile(v, np.linspace(0, 1, 201))) if len(v) else np.array([0.0, 1.0])\n            self.q[c] = qs\n\n    def __call__(self, df):\n        from scipy.stats import norm\n        XN = np.zeros((len(df), len(self.num)), np.float32); MN = np.zeros_like(XN, dtype=bool)\n        for j, c in enumerate(self.num):\n            v = pd.to_numeric(df[c], errors="coerce").astype(float).values if c in df else np.full(len(df), np.nan)\n            m = np.isnan(v); qs = self.q[c]\n            if len(qs) < 2:\n                z = np.zeros(len(v))\n            else:\n                r = np.interp(v, qs, np.linspace(0, 1, len(qs)))\n                z = norm.ppf(np.clip(r, 0.005, 0.995))\n            XN[:, j] = np.where(m, 0, z); MN[:, j] = m\n        XC = np.zeros((len(df), len(self.cat)), np.int64)\n        for j, c in enumerate(self.cat):\n            mapa = {v: i + 1 for i, v in enumerate(self.cats[c])}\n            s = df[c].astype(object).where(df[c].notna(), None) if c in df else pd.Series([None] * len(df))\n            XC[:, j] = [mapa.get(str(x) if x is not None else None, 0) for x in s.values]\n        return XN, MN, XC\n\n\ndef entrenar(model, datos, val, lr, epocas, lote=1024, paciencia=3, wd=1e-5, l2sp=None, params=None, log=""):\n    """Training/validation tensor tuples: (xn,mn,xc,xe,y,off,w)."""\n    params = params if params is not None else [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)\n    n = len(datos[4]); g = torch.Generator().manual_seed(SEMILLA)\n    mejor, estado, sin_mejora, hist = math.inf, None, 0, []\n    ref = None\n    if l2sp is not None:\n        ref = {k: v.detach().clone() for k, v in model.named_parameters()}\n    for ep in range(epocas):\n        model.train(); perm = torch.randperm(n, generator=g); t0 = time.time(); tot = 0.0\n        for i in range(0, n, lote):\n            idx = perm[i:i + lote]\n            xn, mn, xc, xe, y, off, w = (d[idx] if d is not None else None for d in datos)\n            loss = perdida(model(xn, mn, xc, xe), y, off, w)\n            if l2sp is not None:\n                pen = sum(((p - ref[k]) ** 2).sum() for k, p in model.named_parameters() if k in ref and p.requires_grad)\n                loss = loss + l2sp * pen\n            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()\n            tot += float(loss.detach()) * len(idx)\n        dv = evaluar(model, val)\n        hist.append((ep, tot / n, dv, round(time.time() - t0, 1)))\n        print(log, "época", ep, "entren", round(tot / n, 4), "val", round(dv, 5), "s", round(time.time() - t0, 1), flush=True)\n        if dv < mejor - 1e-6:\n            mejor, estado, sin_mejora = dv, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0\n        else:\n            sin_mejora += 1\n            if sin_mejora >= paciencia:\n                break\n    model.load_state_dict(estado)\n    return mejor, hist\n\n\n@torch.no_grad()\ndef evaluar(model, val, lote=8192):\n    model.eval(); tot = 0.0; ws = 0.0\n    n = len(val[4])\n    for i in range(0, n, lote):\n        sl = slice(i, i + lote)\n        xn, mn, xc, xe, y, off, w = (d[sl] if d is not None else None for d in val)\n        tot += float(perdida(model(xn, mn, xc, xe), y, off, w)) * float(w.sum()); ws += float(w.sum())\n    return tot / ws\n\n\n@torch.no_grad()\ndef predecir(model, xn, mn, xc, xe=None, lote=8192):\n    model.eval(); out = []\n    for i in range(0, len(xn), lote):\n        sl = slice(i, i + lote)\n        out.append(model(xn[sl], mn[sl], xc[sl], None if xe is None else xe[sl]))\n    return torch.cat(out).numpy()\n\n\ndef tens(XN, MN, XC, XE, y, off, w):\n    return (torch.tensor(XN), torch.tensor(MN), torch.tensor(XC), None if XE is None else torch.tensor(XE),\n            torch.tensor(y, dtype=torch.float32), torch.tensor(off, dtype=torch.float32), torch.tensor(w, dtype=torch.float32))\n\n\n# Command-line execution.\nFASES = {"dev": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=None, pre="dev"),\n         "hold": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], em_train=[2023], em_eval=2024, pre="dev"),\n         "prod": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=2025, pre="prod"),\n         "pcv": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], em_train=[2023, 2024], em_eval=None, pre="prod")}\nEXTRA = ["region", "gender", "self_employed"]\n\n\ndef columnas(linea):\n    cols = comunes(linea)\n    return [c for c in cols if c not in CATEGORICAS], [c for c in cols if c in CATEGORICAS]\n\n\ndef submuestra(df, frac=0.1, sal="nn"):\n    h = np.array([int(hashlib.sha256((sal + r).encode()).hexdigest()[:8], 16) / 2 ** 32 for r in df.row_id.values])\n    keep = (df.y.values > 0) | (h < frac)\n    w = np.where(df.y.values > 0, 1.0, 1.0 / frac)\n    return keep, w\n\n\ndef preentrenar(fase, linea):\n    import pickle\n    cf = FASES[fase]; dirm = TRABAJO / "modelos" / cf["pre"]; dirm.mkdir(parents=True, exist_ok=True)\n    f = dirm / f"N_pre_{linea}.pt"\n    if f.exists():\n        print("ya existe", f); return\n    torch.manual_seed(SEMILLA); np.random.seed(SEMILLA)\n    dv = cargar("developed", linea, cf["clima"]); em = cargar("emerging", linea, cf["clima"])\n    dv = dv[dv.year.isin(cf["dev_years"])].reset_index(drop=True)\n    num, cat = columnas(linea)\n    cats = categorias_union([dv, em], num + cat + EXTRA)\n    pp = Prepro(dv, num, cat, cats)\n    tope = float(np.quantile(dv.y[dv.y > 0], 0.999))\n    keep, w = submuestra(dv)\n    dv = dv[keep].reset_index(drop=True); w = w[keep]\n    XN, MN, XC = pp(dv)\n    y = np.minimum(dv.y.values, tope); off = np.log(dv.exposure.values * dv.c_fila.values)\n    base = math.log(np.sum(w * y) / np.sum(w * np.exp(off)))\n    va = dv.fold.values == 0\n    T = tens(XN, MN, XC, None, y, off, w)\n    tr = tuple(None if t is None else t[torch.tensor(~va)] for t in T)\n    vl = tuple(None if t is None else t[torch.tensor(va)] for t in T)\n    m = FTT(len(num), [len(cats[c]) for c in cat])\n    with torch.no_grad():\n        m.sesgo.fill_(base)\n    print(linea, "filas", len(dv), "val", int(va.sum()), "tope", round(tope), flush=True)\n    mejor, hist = entrenar(m, tr, vl, lr=1e-3, epocas=6, lote=1024, paciencia=2, log=f"pre {linea}")\n    torch.save(m.state_dict(), f)\n    with open(dirm / f"N_pre_{linea}.pkl", "wb") as fh:\n        pickle.dump({"prepro": pp, "num": num, "cat": cat, "cats": cats, "tope": tope, "hist": hist}, fh)\n    guardar_json({"val": mejor, "hist": hist, "tope": tope, "filas": int(len(dv))}, dirm / f"N_pre_{linea}.json")\n\n\ndef interno(cids):\n    return np.array([int(hashlib.sha256(("mlind_inner|" + c).encode()).hexdigest()[:8], 16) % 5 == 0 for c in cids])\n\n\ndef fino(fase, linea, var):\n    import pickle\n    cf = FASES[fase]; dirm = TRABAJO / "modelos" / cf["pre"]\n    meta = pickle.load(open(dirm / f"N_pre_{linea}.pkl", "rb"))\n    pp, num, cat, cats, tope = meta["prepro"], meta["num"], meta["cat"], meta["cats"], meta["tope"]\n    em = cargar("emerging", linea, cf["clima"])\n    XN, MN, XC = pp(em)\n    XE = np.zeros((len(em), len(EXTRA)), np.int64)\n    for c in EXTRA:\n        if c not in cats:\n            cats[c] = sorted(set(em[c].dropna().astype(str)))\n    for j, c in enumerate(EXTRA):\n        mapa = {v: i + 1 for i, v in enumerate(cats[c])}\n        s = em[c].astype(object).where(em[c].notna(), None)\n        XE[:, j] = [mapa.get(str(x) if x is not None else None, 0) for x in s.values]\n    raw2 = pd.read_parquet(TRABAJO / "pred" / fase / f"T2braw_{linea}.parquet")\n    em = em.reset_index(drop=True)\n    r2 = em[["row_id"]].merge(raw2, on="row_id", how="left").raw.values\n    tr_mask = ((em.conjunto == "train") & em.year.isin(cf["em_train"])).values\n    if cf["em_eval"] is None:\n        sets = [(tr_mask & (em.fold.values != k), tr_mask & (em.fold.values == k)) for k in range(5)]\n    else:\n        sets = [(tr_mask, (em.year == cf["em_eval"]).values)]\n    off_all = np.log(em.exposure.values * np.where(em.conjunto == "test", 1.0, em.c_fila.values))\n    preds = []; info = []\n    for a, b in sets:\n        torch.manual_seed(SEMILLA)\n        ya = em.y.values[a]\n        k0 = factor_tweedie(ya, np.exp(off_all[a]))\n        extra_off = np.zeros(len(em))\n        if var == "N3":\n            k2 = factor_tweedie(ya, np.exp(off_all[a] + r2[a]))\n            extra_off = r2 + math.log(k2)\n        m = FTT(len(num), [len(cats[c]) for c in cat])\n        if var in ("N2a", "N2b", "N3"):\n            m.load_state_dict(torch.load(dirm / f"N_pre_{linea}.pt"))\n        d = m.cls.shape[-1]\n        m.extra = nn.ModuleList([nn.Embedding(len(cats[c]) + 1, d) for c in EXTRA])\n        for e in m.extra:\n            nn.init.normal_(e.weight, std=0.02)\n        with torch.no_grad():\n            if var == "N3":\n                m.cab.weight.zero_(); m.cab.bias.zero_(); m.sesgo.zero_()\n            else:\n                # Preserve transferred relativities while setting the initial emerging-market level.\n                eta0 = predecir(m, *[torch.tensor(x[a]) for x in (XN, MN, XC)], torch.tensor(XE[a]))\n                m.sesgo += math.log(factor_tweedie(ya, np.exp(eta0 + off_all[a])))\n        params = None; l2sp = None; lr = 2e-4\n        if var == "N1":\n            lr = 1e-3\n        elif var == "N2b":\n            for p in m.parameters():\n                p.requires_grad = False\n            for mod in [m.cab, m.nf, m.extra]:\n                for p in mod.parameters():\n                    p.requires_grad = True\n            m.sesgo.requires_grad = True\n            lr = 1e-3\n        else:  # N2a/N3: full fine-tuning with L2-SP.\n            l2sp = 1e-2\n        idx_a = np.where(a)[0]; v = interno(em.customer_id.values[idx_a])\n        it, iv = idx_a[~v], idx_a[v]\n        yc = np.minimum(em.y.values, tope)\n        T = lambda ix: tens(XN[ix], MN[ix], XC[ix], XE[ix], yc[ix], off_all[ix] + extra_off[ix], np.ones(len(ix)))\n        mejor, hist = entrenar(m, T(it), T(iv), lr=lr, epocas=20, lote=512, paciencia=4, l2sp=l2sp, log=f"{var} {linea}")\n        # Relevel training predictions with and without the cap.\n        eta_a = predecir(m, torch.tensor(XN[a]), torch.tensor(MN[a]), torch.tensor(XC[a]), torch.tensor(XE[a]))\n        kin = factor_tweedie(ya, np.exp(eta_a + off_all[a] + extra_off[a]))\n        eta_b = predecir(m, torch.tensor(XN[b]), torch.tensor(MN[b]), torch.tensor(XC[b]), torch.tensor(XE[b]))\n        mu = kin * np.exp(eta_b + extra_off[b]) * em.exposure.values[b]\n        preds.append(pd.DataFrame({"row_id": em.row_id.values[b], "mu": mu}))\n        info.append({"val": mejor, "epocas": len(hist), "kin": kin})\n        if cf["em_eval"] is not None:\n            (TRABAJO / "modelos" / fase).mkdir(parents=True, exist_ok=True)\n            torch.save(m.state_dict(), TRABAJO / "modelos" / fase / f"{var}_{linea}.pt")\n    out = TRABAJO / "pred" / fase; out.mkdir(parents=True, exist_ok=True)\n    pd.concat(preds).to_parquet(out / f"{var}_{linea}.parquet", index=False)\n    guardar_json(info, out / f"{var}_{linea}_meta.json")\n\n\nif __name__ == "__main__":\n    modo, fase, linea = sys.argv[1], sys.argv[2], sys.argv[3]\n    if modo == "pre":\n        preentrenar(fase, linea)\n    else:\n        for var in sys.argv[4].split(","):\n            t0 = time.time(); fino(fase, linea, var); print(var, linea, "listo", round(time.time() - t0), "s", flush=True)\n'


### Embedded source: `ml/codigo/ruido_pareado.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/ruido_pareado.py'] = '"""Explore paired differences on random 33% customer subsets of 2024.\nCompare the fixed T2b/A1 blend with ML-family v2/v2.1 replacements. A1 serves\nas the designated-model proxy; this is an exploratory historical comparison.\n"""\nimport json, numpy as np, pandas as pd\nfrom comun import *\nfrom receta import tabla, componentes\nfrom evaluar import mezcla_pesos\ntd = tabla("dev"); th = tabla("hold")\ndef pred(comps):\n    Pd = componentes("dev", comps, td); Ph = componentes("hold", comps, th); mu = np.zeros(len(th))\n    for l in LINEAS:\n        il = (td.line == l).values; ih = (th.line == l).values\n        M = np.column_stack([Pd[c][il] * td.c_fila.values[il] for c in comps])\n        w = mezcla_pesos(td.y.values[il], M) if len(comps) > 1 else np.ones(1)\n        w = 0.5 * w + 0.5 / len(comps) if len(comps) > 1 else w\n        mu[ih] = np.column_stack([Ph[c][ih] for c in comps]) @ w * factor_tweedie(td.y.values[il], M @ w)\n    return mu * th.c_fila.values\no = pd.read_parquet(RAIZ / "comparacion" / "otros" / "predicciones_2024.parquet")\nd = th[["row_id", "y"]].merge(o[["row_id", "A1"]], on="row_id", how="left").merge(\n    pd.read_parquet(TRABAJO / "pesos_2025_hold.parquet"), on="row_id", how="left")\nassert d.A1.notna().all() and (d.row_id.values == th.row_id.values).all()\ny = d.y.values\nm = {"v1": 0.5 * pred(["T2b"]) + 0.5 * d.A1.values, "v2": 0.5 * pred(["T2b", "T2h", "FS", "T2o"]) + 0.5 * d.A1.values,\n     "v21": 0.5 * pred(["T2b", "T2h", "FSt", "T2o"]) + 0.5 * d.A1.values}\nrng = np.random.default_rng(20261024); res = {}\nfor k in ("v2", "v21"):\n    r = {"corr_log": float(np.corrcoef(np.log(m[k]), np.log(m["v1"]))[0, 1]), "dif_total": d2(y, m[k]) - d2(y, m["v1"]),\n         "dif_pond2025": d2(y, m[k], d.w2025.values) - d2(y, m["v1"], d.w2025.values)}\n    sims = []\n    for _ in range(1000):\n        s = rng.random(len(y)) < 0.33\n        sims.append(d2(y[s], m[k][s]) - d2(y[s], m["v1"][s]))\n    r["sd_33"] = float(np.std(sims)); r["p_negativa_33"] = float(np.mean(np.array(sims) < 0))\n    res[k] = r\n    print(k, {a: round(b, 5) for a, b in r.items()})\nguardar_json(res, ENTREGA / "evolucion" / "ruido_pareado_v2.json")\n'


### Embedded source: `ml/codigo/seg_eval.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/seg_eval.py'] = '"""Compare DEV/HOLD structure by line and new-business status after within-segment releveling."""\nimport sys, numpy as np, pandas as pd\nfrom comun import *\nfrom receta import tabla, componentes\nfrom evaluar import calibrar_cruzado\ncands = sys.argv[1:]\ntd = tabla("dev").merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left"); th = tabla("hold")\nPd = componentes("dev", cands, td); Ph = componentes("hold", cands, th)\nfolds = pliegue(td.customer_id.values)\nfilas = []\nfor c in cands:\n    for fase, t, mu in (("dev", td, calibrar_cruzado(td, Pd[c], folds)), ("hold", th, Ph[c])):\n        m = mu * t.c_fila.values; y = t.y.values\n        r = {"cand": c, "fase": fase}\n        for nb in (True, False):\n            s = t.new_business.values == nb\n            mm = m.copy()\n            for l in LINEAS:  # Relevel each line within the segment.\n                i = s & (t.line == l).values; mm[i] *= factor_tweedie(y[i], m[i])\n            r["NB" if nb else "REN"] = d2(y[s], mm[s])\n        filas.append(r)\ndf = pd.DataFrame(filas).pivot(index="cand", columns="fase", values=["NB", "REN"])\nprint(df.round(4))\n'


### Embedded source: `ml/codigo/sonda_nivel_s2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/sonda_nivel_s2.py'] = '"""Reconstruct historical Week 2 public-probe level factors with credibility.\n\nFor power 1.5, scaling a line by c changes public D2 by\n-4*[S1*(c**-0.5-1)+S2*(c**0.5-1)]/N, with S1=sum(y/sqrt(mu)) and\nS2=sum(sqrt(mu)). Two recorded probes (0.90 and 1.10) identify the normalized\nterms and the public optimum c*=S1/S2. Shrink log(c*) by\nZ=tau^2/(tau^2+sigma^2), with tau fixed before observing the probes. Estimate\nnoise at the recorded 2025 public/private sample sizes. This source preserves\nthe historical formula; the reproduction notebook does not request new probes.\nUsage: python sonda_nivel_s2.py generar\n       python sonda_nivel_s2.py resolver --base <score> --motor <P090> <P110>\n       --buildings <P090> <P110> --contents <P090> <P110> [--simular]\n"""\nimport argparse, json, datetime as dt, sys\nimport numpy as np, pandas as pd\nfrom comun import *\nimport receta\n\nBASE_SHA = "12831ef4a05c1773824abe5fa139d9503dd2989b4874e7d93cfc000406cfdb4a"\nBASE_PUB = 0.15390\nC_SONDA = {"x090": 0.90, "x110": 1.10}\nPREF = {"M": "motor", "B": "buildings", "C": "contents"}\n# Prior uncertainty of log 2025 level was frozen before observing public probes,\n# using the documented spread of defensible production-level scenarios:\n# motor: C1 1.00, trend 1.08, C2 1.28, C2+trend 1.38; prior sd 0.14.\n# buildings: C1 1.00, trend 1.09, 2023-only 1.31, 2024-only 0.81; prior sd 0.20.\n# contents: C1 1.00, trend 1.06, C2 1.05, C2+trend 1.12; prior sd 0.06.\nTAU = {"motor": 0.14, "buildings": 0.20, "contents": 0.06}\nDIR = RAIZ / "sondas"\n\n\ndef leer_base():\n    f = PRIVADO / "mezcla50_designada_mlind_2025.csv"\n    assert sha256(f) == BASE_SHA, "la base no es la mezcla enviada"\n    b = pd.read_csv(f, dtype={"row_id": str})\n    b["line"] = b.row_id.str[0].map(PREF)\n    assert b.line.notna().all()\n    return b\n\n\ndef ruido(reps=2000, semilla=20261024):\n    """Estimate line sigma as sd(log(public optimum/private optimum)) by customer-address resampling of observed-scale T2b+C1 PCV OOF to the recorded 2025 sample sizes."""\n    t, mu, _ = receta.oof_cruzado("pcv", ["T2b"], "ramo")\n    b = pd.concat([pd.read_parquet(TRABAJO / f"base_emerging_{l}.parquet", columns=["row_id", "address_id"]) for l in LINEAS])\n    t = t.merge(b, on="row_id", how="left")\n    m = mu * t.c_fila.values; y = t.y.values\n    padre = {}\n    def f(x):\n        while padre.setdefault(x, x) != x:\n            padre[x] = padre[padre[x]]; x = padre[x]\n        return x\n    for c, a in zip("C" + t.customer_id, "A" + t.address_id):\n        rc, ra = f(c), f(a)\n        if rc != ra: padre[ra] = rc\n    _, gi = np.unique([f("C" + c) for c in t.customer_id], return_inverse=True); G = gi.max() + 1\n    lam_pub = 209063 / 3 / len(t); lam_priv = 209063 * 2 / 3 / len(t)\n    rng = np.random.default_rng(semilla); out = {}\n    for l in LINEAS:\n        i = (t.line == l).values\n        a1 = np.bincount(gi[i], weights=y[i] / np.sqrt(m[i]), minlength=G); a2 = np.bincount(gi[i], weights=np.sqrt(m[i]), minlength=G)\n        Wp = rng.poisson(lam_pub, size=(reps, G)); Wq = rng.poisson(lam_priv, size=(reps, G))\n        cp = (Wp @ a1) / (Wp @ a2); cq = (Wq @ a1) / (Wq @ a2)\n        out[l] = {"sd_log_c_publico": float(np.std(np.log(cp))), "sd_log_c_privado": float(np.std(np.log(cq))),\n                  "sigma_pub_menos_priv": float(np.std(np.log(cp) - np.log(cq)))}\n    return out\n\n\ndef generar(_):\n    DIR.mkdir(parents=True, exist_ok=True)\n    b = leer_base(); sig = ruido()\n    Z = {l: TAU[l] ** 2 / (TAU[l] ** 2 + sig[l]["sigma_pub_menos_priv"] ** 2) for l in LINEAS}\n    archivos = {}\n    for l in LINEAS:\n        for k, c in C_SONDA.items():\n            x = b[["row_id", "prediction"]].copy()\n            x.loc[b.line == l, "prediction"] *= c\n            f = DIR / f"S2sonda_{l}_{k}.csv"\n            x.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")\n            archivos[f.name] = {"sha256": sha256(f), "ramo": l, "factor": c, "total": float(x.prediction.sum()), "filas": len(x)}\n    reg = {"id": "S2-SONDA-NIVEL-MLIND", "utc": dt.datetime.now(dt.UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),\n           "base": {"archivo": "mezcla50_designada_mlind_2025.csv", "sha256": BASE_SHA, "publico": BASE_PUB, "envio": 57599788},\n           "forma": "D2(c)-D2(1) = -4[S1(c^-1/2-1)+S2(c^1/2-1)]/N por ramo; c* = S1/S2",\n           "tau_log": TAU, "ruido": sig, "Z": Z, "sondas": archivos,\n           "regla": ["Enviar las seis sondas tal cual; ninguna se selecciona nunca.",\n                     "Con las seis puntuaciones públicas y la base, resolver S1/N y S2/N por ramo; si alguno sale <= 0, ese ramo no se ajusta.",\n                     "Si c* sale fuera de [0,5; 2,0], ese ramo no se ajusta (la forma supuesta no se cumple).",\n                     "Candidato: base x c_Z por ramo, log c_Z = Z log c*. Se envía una vez; su público esperado se fija antes y una desviación > 0,0001 indica que el evaluador no tiene la forma supuesta.",\n                     "Historical rule: retain the candidate only when its public result meets the recorded baseline. The participant controls submission and final selection."]}\n    guardar_json(reg, DIR / "preinscripcion_sonda_nivel_s2.json")\n    print(json.dumps({k: reg[k] for k in ["ruido", "Z", "tau_log"]}, indent=1))\n    for k, v in archivos.items(): print(k, v["sha256"][:12], round(v["total"]))\n\n\ndef resolver(a):\n    reg = json.load(open(DIR / "preinscripcion_sonda_nivel_s2.json"))\n    b = leer_base(); out = {"entradas": {"base": a.base}, "ramos": {}}\n    for l in LINEAS:\n        p90, p110 = getattr(a, l); out["entradas"][l] = [p90, p110]\n        # Solve d(c)=-4*[S1*(c**-0.5-1)+S2*(c**0.5-1)]/N for normalized s1 and s2.\n        A = np.array([[-(0.9 ** -0.5 - 1), -(0.9 ** 0.5 - 1)], [-(1.1 ** -0.5 - 1), -(1.1 ** 0.5 - 1)]])\n        s1, s2 = np.linalg.solve(A, np.array([p90 - a.base, p110 - a.base]))\n        r = {"s1": float(s1), "s2": float(s2)}\n        if s1 <= 0 or s2 <= 0:\n            r.update({"valido": False, "motivo": "S1 o S2 no positivos", "c_Z": 1.0})\n        else:\n            cs = s1 / s2; z = reg["Z"][l]\n            ok = bool(0.5 <= cs <= 2.0)\n            cz = float(np.exp(z * np.log(cs))) if ok else 1.0\n            ganancia_pub = lambda c: -(s1 * (c ** -0.5 - 1) + s2 * (c ** 0.5 - 1))\n            r.update({"valido": ok, "c_estrella": float(cs), "Z": z, "c_Z": cz, "ganancia_publica_esperada": float(ganancia_pub(cz)),\n                      "ganancia_publica_si_c_estrella": float(ganancia_pub(cs))})\n        out["ramos"][l] = r\n    out["publico_esperado_candidato"] = a.base + sum(r.get("ganancia_publica_esperada", 0.0) for r in out["ramos"].values())\n    print(json.dumps(out, indent=1, default=float))\n    if a.simular:\n        return\n    x = b[["row_id", "prediction"]].copy()\n    for l in LINEAS:\n        x.loc[b.line == l, "prediction"] *= out["ramos"][l]["c_Z"]\n    f = DIR / "S2_nivel_credibilidad.csv"\n    x.to_csv(f, index=False, float_format="%.6f", lineterminator="\\n")\n    out["candidato"] = {"archivo": f.name, "sha256": sha256(f), "total": float(x.prediction.sum()),\n                        "por_ramo": {l: float(x.prediction[b.line == l].sum()) for l in LINEAS}}\n    guardar_json(out, DIR / "resolucion_sonda_nivel_s2.json")\n    print(out["candidato"])\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); sp = ap.add_subparsers(dest="cmd", required=True)\n    sp.add_parser("generar")\n    r = sp.add_parser("resolver"); r.add_argument("--base", type=float, default=BASE_PUB); r.add_argument("--simular", action="store_true")\n    for l in LINEAS: r.add_argument(f"--{l}", type=float, nargs=2, required=True)\n    a = ap.parse_args()\n    {"generar": generar, "resolver": resolver}[a.cmd](a)\n'


### Embedded source: `ml/codigo/t2_rejilla.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/t2_rejilla.py'] = '"""E07: choose developed-market model settings using developed internal validation."""\nimport sys, itertools, time\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom gbm import P_DEV, CONF, MOD, OUT, FASE\n\nres = {}\nfor l in LINEAS:\n    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])\n    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)\n    cols = comunes(l); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)\n    X = matriz(dv, cols, cats); y = dv.y.values; off = np.log(dv.exposure.values * dv.c_fila.values)\n    va = dv.fold.values == 0\n    dtr = lgb.Dataset(X[~va], y[~va], init_score=off[~va], free_raw_data=False)\n    dva = lgb.Dataset(X[va], y[va], init_score=off[va], reference=dtr)\n    filas = []\n    for l2, md in itertools.product([0.0, 10.0], [300, 1000, 3000]):\n        t0 = time.time()\n        p = dict(P_DEV, learning_rate=0.03, lambda_l2=l2, min_data_in_leaf=md)\n        b = lgb.train(p, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False)])\n        mu = np.exp(b.predict(X[va], raw_score=True, num_iteration=b.best_iteration) + off[va])\n        k = factor_tweedie(y[~va], np.exp(b.predict(X[~va], raw_score=True, num_iteration=b.best_iteration) + off[~va]))\n        dev_v = desviacion(y[va], mu * k).sum()\n        filas.append({"lambda_l2": l2, "min_data": md, "rondas": b.best_iteration, "k_entren": k,\n                      "D2_val": d2(y[va], mu * k), "s": round(time.time() - t0)})\n        print(l, filas[-1], flush=True)\n    res[l] = filas\n    guardar_json(res, OUT / "T2_rejilla.json")\n'


### Embedded source: `ml/codigo/t2cap.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/t2cap.py'] = '"""E06: train developed T2 with line-specific positive-row p99.9 caps."""\nimport sys\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom gbm import P_DEV, ajustar, CONF, MOD, OUT, FASE\n\nfor l in LINEAS:\n    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])\n    dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)\n    cols = comunes(l); cats = categorias_union([dv, em], cols + SOLO_EMERGENTE + MARKETING + TARIFA + MAKE)\n    tope = float(np.quantile(dv.y[dv.y > 0], 0.999))\n    f = MOD / f"T2cap_{l}.txt"\n    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values)\n    if f.exists():\n        b = lgb.Booster(model_file=str(f)); n = b.current_iteration()\n    else:\n        b, n = ajustar(Xd, np.minimum(dv.y.values, tope), offd, P_DEV, 4000, es=100, val=dv.fold.values == 0)\n        b.save_model(str(f))\n    r = b.predict(matriz(em, cols, cats), raw_score=True)\n    em["raw"] = r\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)\n    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF["em_eval"] is None else \\\n        [(tr, em[em.year == CONF["em_eval"]].reset_index(drop=True))]\n    preds = []\n    for a, bb in sets:\n        k = factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)\n        preds.append(pd.DataFrame({"row_id": bb.row_id.values, "mu": k * np.exp(bb.raw.values) * bb.exposure.values}))\n    pd.concat(preds).to_parquet(OUT / f"T2cap_{l}.parquet", index=False)\n    pd.DataFrame({"row_id": em.row_id.values, "raw": r}).to_parquet(OUT / f"T2capraw_{l}.parquet", index=False)\n    print(l, "tope", round(tope), "rondas", n, flush=True)\n'


### Embedded source: `ml/codigo/t4.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/t4.py'] = '"""T4 preregistered joint developed/emerging model with a market indicator.\nSample 35% of developed customers for computation and weight emerging rows by 5.\nStop on internal emerging validation and relevel on emerging training-fold rows.\n"""\nimport sys, hashlib\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom gbm import P_DEVB, interno, CONF, FASE, OUT, MOD\n\nPESO_EM = 5.0\nfor l in LINEAS:\n    em = cargar("emerging", l, CONF["clima"]); dv = cargar("developed", l, CONF["clima"])\n    dv = dv[dv.year.isin(CONF["dev_years"])]\n    h = np.array([int(hashlib.sha256(("t4|" + c).encode()).hexdigest()[:8], 16) / 2 ** 32 for c in dv.customer_id.values])\n    dv = dv[h < 0.35].reset_index(drop=True)\n    cols = comunes(l) + SOLO_EMERGENTE + ["market"]\n    CATEGORICAS.add("market")\n    cats = categorias_union([dv, em], cols)\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])].reset_index(drop=True)\n    sets = [(tr[tr.fold != k], tr[tr.fold == k]) for k in range(5)] if CONF["em_eval"] is None else [(tr, em[em.year == CONF["em_eval"]].reset_index(drop=True))]\n    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values)\n    preds = []; rondas = []\n    for a, b in sets:\n        Xa = matriz(a, cols, cats); offa = np.log(a.exposure.values * a.c_fila.values)\n        v = interno(a.customer_id.values)\n        X = pd.concat([Xd, Xa[~v]], ignore_index=True)\n        y = np.concatenate([dv.y.values, a.y.values[~v]]); off = np.concatenate([offd, offa[~v]])\n        w = np.concatenate([np.ones(len(dv)), np.full((~v).sum(), PESO_EM)])\n        p = dict(P_DEVB[l])\n        dtr = lgb.Dataset(X, y, weight=w, init_score=off, free_raw_data=False)\n        dva = lgb.Dataset(Xa[v], a.y.values[v], init_score=offa[v], reference=dtr)\n        bst = lgb.train(p, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False)])\n        n = max(bst.best_iteration, 1); rondas.append(n)\n        X2 = pd.concat([Xd, Xa], ignore_index=True)\n        y2 = np.concatenate([dv.y.values, a.y.values]); off2 = np.concatenate([offd, offa])\n        w2 = np.concatenate([np.ones(len(dv)), np.full(len(a), PESO_EM)])\n        bst = lgb.train(p, lgb.Dataset(X2, y2, weight=w2, init_score=off2), n)\n        kin = factor_tweedie(a.y.values, np.exp(bst.predict(Xa, raw_score=True) + offa))\n        mu = kin * np.exp(bst.predict(matriz(b, cols, cats), raw_score=True)) * b.exposure.values\n        preds.append(pd.DataFrame({"row_id": b.row_id.values, "mu": mu}))\n        if CONF["em_eval"] is not None:\n            (TRABAJO / "modelos" / FASE).mkdir(parents=True, exist_ok=True)\n            bst.save_model(str(TRABAJO / "modelos" / FASE / f"T4_{l}.txt"))\n    pd.concat(preds).to_parquet(OUT / f"T4_{l}.parquet", index=False)\n    print(l, "rondas", rondas, flush=True)\n'


### Embedded source: `ml/codigo/tabla_dev_final.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/tabla_dev_final.py'] = '"""Final DEV table: candidates/blends, cross-fitted C1/C2 and intervals versus T2b."""\nimport json\nimport numpy as np, pandas as pd, matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nfrom comun import *\nfrom evaluar import metricas, bootstrap_dif\nimport receta\n\nSIMPLES = ["R0", "G1", "T1wp", "T2", "T2b", "T3bwp", "T3bmk", "T4", "T5", "T6", "N1", "N2a", "N2b", "N3"]\nMEZCLAS = {"E1": ["T2b", "T3bwp", "N3"], "E2": ["T2b", "T3bwp", "N3", "T4", "G1"],\n           "E3": ["G1", "T1wp", "T2", "T2b", "T3bwp", "T3bmk", "T4", "T5", "T6", "N1", "N2a", "N2b", "N3"]}\nt0 = receta.tabla("dev").merge(pd.read_parquet(TRABAJO / "pesos_2025_dev.parquet"), on="row_id", how="left")\nfilas = []; mus = {}; pesos_mezcla = {}\nfor nombre in SIMPLES + list(MEZCLAS):\n    comps = MEZCLAS.get(nombre, [nombre])\n    for calib in ["sin", "ramo", "ramo_nb"]:\n        if nombre in MEZCLAS and calib == "sin":\n            continue\n        t, mu, pesos = receta.oof_cruzado("dev", comps, calib)\n        assert (t.row_id.values == t0.row_id.values).all()\n        mus[(nombre, calib)] = mu\n        if pesos: pesos_mezcla[f"{nombre}|{calib}"] = pesos\n        filas.append({"cand": nombre, "calib": calib, **metricas(t0, mu)})\ndf = pd.DataFrame(filas)\nref = mus[("T2b", "ramo_nb")]\nbs = {}\nfor (n, c), mu in mus.items():\n    if c == "ramo_nb" and n != "T2b":\n        bs[n] = bootstrap_dif(t0, mu, ref, B=1000)\ndf["dif_T2b_C2"] = df.apply(lambda r: bs[r.cand]["dif"] if r.calib == "ramo_nb" and r.cand in bs else np.nan, axis=1)\ndf["ic95_lo"] = df.apply(lambda r: bs[r.cand]["ic95"][0] if r.calib == "ramo_nb" and r.cand in bs else np.nan, axis=1)\ndf["ic95_hi"] = df.apply(lambda r: bs[r.cand]["ic95"][1] if r.calib == "ramo_nb" and r.cand in bs else np.nan, axis=1)\ndf.to_csv(ENTREGA / "tabla_dev.csv", index=False)\nguardar_json({"pesos_mezcla": pesos_mezcla, "bootstrap_vs_T2b_C2": bs}, ENTREGA / "dev_mezclas_bootstrap.json")\npd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)\nprint(df[df.calib != "sin"].sort_values("D2", ascending=False).round(4).to_string())\n# Plot candidate C2 D2 with comparison intervals against T2b.\nd = df[df.calib == "ramo_nb"].set_index("cand").sort_values("D2")\nfig, ax = plt.subplots(figsize=(7.5, 4.2))\ncol = ["#7a7a7a" if c in ("R0", "G1", "T1wp", "T4") else "#2a6f97" if c.startswith("T") else "#8e5ea2" if c.startswith("N") else "#c4673b" for c in d.index]\nax.barh(d.index, d.D2, color=col)\nfor i, (c, r) in enumerate(d.iterrows()):\n    ax.text(r.D2 + 0.001, i, f"{r.D2:.4f}", va="center", fontsize=7)\nax.set_xlabel("D² Tweedie 1,5, OOF 2023, calibración por ramo y negocio cruzada")\nax.set_title("Desarrollo 2023: gris = solo emergente, azul = árboles con transferencia,\\nmorado = redes, naranja = mezclas", fontsize=9)\nfig.tight_layout(); fig.savefig(ENTREGA / "figuras" / "f03_dev_candidatos.png"); plt.close(fig)\n'


### Embedded source: `ml/codigo/tendencia.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/tendencia.py'] = '"""Estimate developed-market annual trend and apply the exposure-weighted time shift.\nExclude year and history features. Fit log-linear observed/expected trends using\ncompleteness-adjusted labels and expected-cost weights. HOLD uses developed\n2021-2023 and the 2023-to-2024 shift; production uses 2021-2024 and the shift\nfrom emerging 2023-2024 to 2025.\nUsage: python tendencia.py <phase>\n"""\nimport sys\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom gbm import P_DEVB, ajustar, CONF, FASE, MOD, OUT\n\nres = {}\nfor l in LINEAS:\n    dv = cargar("developed", l, CONF["clima"]); dv = dv[dv.year.isin(CONF["dev_years"])].reset_index(drop=True)\n    em = cargar("emerging", l, CONF["clima"])\n    cols = [c for c in comunes(l) if c not in ("hist_pol_12m", "hist_cli_12m")]\n    cats = categorias_union([dv, em], cols)\n    X = matriz(dv, cols, cats); off = np.log(dv.exposure.values * dv.c_fila.values)\n    b, n = ajustar(X, dv.y.values, off, P_DEVB[l], 5000, es=150, val=dv.fold.values == 0)\n    mu = np.exp(b.predict(X, raw_score=True) + off)\n    g = pd.DataFrame({"year": dv.year, "y": dv.y, "m": mu}).groupby("year").sum()\n    g["ae"] = g.y / g.m\n    x = g.index.values.astype(float); w = g.m.values\n    A = np.vstack([np.ones_like(x), x - x.mean()]).T\n    beta = np.linalg.lstsq(A * np.sqrt(w)[:, None], np.log(g.ae.values) * np.sqrt(w), rcond=None)[0]\n    tau = float(np.exp(beta[1]))\n    # Time shift between exposure-weighted mean coverage dates in training and target.\n    def fecha_media(d):\n        mid = d.period_start + (d.period_end - d.period_start) / 2\n        anio = mid.dt.year + (mid.dt.dayofyear - 0.5) / 365.25\n        return float(np.average(anio, weights=d.exposure))\n    tr = em[(em.conjunto == "train") & em.year.isin(CONF["em_train"])]\n    ob = em[em.year == CONF["em_eval"]]\n    dt = fecha_media(ob) - fecha_media(tr)\n    res[l] = {"ae_por_anio": g.ae.round(4).to_dict(), "tau_anual": tau, "dt": dt, "factor": tau ** dt, "rondas": n}\n    print(l, res[l], flush=True)\nguardar_json(res, OUT / "tendencia.json")\n'


### Embedded source: `ml/codigo/transfer2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/transfer2.py'] = '"""Fit alternative developed-market transfer components under the common schema.\n\nVariants include CatBoost Tweedie, frequency-severity, make/model and multiline\ntenure, occupation exclusion, and a centered history stage. E23 variants start\nboosting at the mean rate. MLIND_HISTORIA=2 ends 2025 history at min(start,V).\ndevhold trains developed 2021-2023 for DEV-2023 OOF, HOLD-2024 and DES24;\nprodpcv trains developed 2021-2024 for PCV-2023/2024 and production-2025.\nEstimate the emerging line level only on each fold\'s training rows.\nUsage: python transfer2.py <variant> <devhold|prodpcv> [<lines>]\n"""\nfrom __future__ import annotations\nimport sys, time, hashlib\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\n\nVAR = sys.argv[1]\nFASE = sys.argv[2]\nCONF = {"devhold": dict(clima="hasta2023", dev_years=[2021, 2022, 2023], mod="dev", oof=("dev", [2023]),\n                        ev=("hold", [2023], 2024), des=2024),\n        "prodpcv": dict(clima="hasta2024", dev_years=[2021, 2022, 2023, 2024], mod="prod", oof=("pcv", [2023, 2024]),\n                        ev=("prod", [2023, 2024], 2025), des=None)}[FASE]\nMOD = TRABAJO / "modelos" / CONF["mod"]\nHIST = ["hist_pol_12m", "hist_cli_12m"]\nTEN = ["n_lineas", "tiene_motor", "tiene_buildings", "tiene_contents"]\nSEMILLA = 20261024\nINICIO0 = VAR in ("T2bi", "T2ci", "T2oi", "FSi", "T2hi")  # E23 starts boosting at the mean rate.\nBASE = VAR[:-1] if INICIO0 else VAR\nimport os\nSUF = os.environ.get("MLIND_SUFIJO", "")  # E24 output suffix; saved model names remain fixed.\nP_DEV = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.05, num_leaves=63,\n             min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0,\n             max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=SEMILLA)\nP_DEVB = {"motor": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=300),\n          "buildings": dict(P_DEV, learning_rate=0.03, lambda_l2=0.0, min_data_in_leaf=3000),\n          "contents": dict(P_DEV, learning_rate=0.03, lambda_l2=10.0, min_data_in_leaf=3000)}\n\n\ndef ajustar(X, y, off, params, max_rounds, es=150, val=None, w=None):\n    """Stop on the internal validation fold, then refit on the entire training partition."""\n    wt = None if w is None else w[~val]\n    dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], weight=wt, free_raw_data=False)\n    dva = lgb.Dataset(X[val], y[val], init_score=off[val], weight=None if w is None else w[val], reference=dtr)\n    b = lgb.train(params, dtr, max_rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(es, verbose=False)])\n    n = max(b.best_iteration, 1)\n    b = lgb.train(params, lgb.Dataset(X, y, init_score=off, weight=w), n)\n    return b, n\n\n\ndef ajustar0(X, y, off, params, max_rounds, es=150, val=None, w=None):\n    """E23: include the log mean rate in init_score; return the fit, rounds and raw-score constant."""\n    ww = np.ones(len(y)) if w is None else w\n    b0 = float(np.log(np.sum(ww * y) / np.sum(ww * np.exp(off))))\n    b, n = ajustar(X, y, off + b0, params, max_rounds, es=es, val=val, w=w)\n    return b, n, b0\n\n\ndef _guardar_base(f, b0):\n    guardar_json({"base": b0}, str(f) + ".base.json")\n\n\ndef _leer_base(f):\n    import json\n    return json.load(open(str(f) + ".base.json"))["base"]\n\n\ndef tenencia(mercado):\n    b = {l: pd.read_parquet(TRABAJO / f"base_{mercado}_{l}.parquet", columns=["customer_id", "year"]) for l in LINEAS}\n    cl = pd.concat([d.assign(line=l) for l, d in b.items()]).drop_duplicates()\n    t = cl.assign(v=1).pivot_table(index=["customer_id", "year"], columns="line", values="v", fill_value=0)\n    t.columns = [f"tiene_{c}" for c in t.columns]\n    t["n_lineas"] = t.sum(axis=1)\n    return t.reset_index()\n\n\ndef columnas(linea):\n    c = comunes(linea)\n    if BASE in ("T2c", "T2h"):\n        c = [x for x in c if x not in HIST]\n    if BASE == "T2o":\n        c = [x for x in c if x != "occupation"]\n    if VAR == "T2x":\n        c = c + (["make_model"] if linea == "motor" else []) + TEN\n    return c\n\n\n# Developed-market backbone models.\ndef tronco_lgb(dv, cols, cats, linea, nombre):\n    f = MOD / f"{nombre}_{linea}.txt"\n    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values)\n    if f.exists():\n        b = lgb.Booster(model_file=str(f)); n = b.current_iteration()\n        b0 = _leer_base(f) if INICIO0 else 0.0\n    elif INICIO0:\n        b, n, b0 = ajustar0(Xd, dv.y.values, offd, P_DEVB[linea], 5000, val=dv.fold.values == 0)\n        b.save_model(str(f)); _guardar_base(f, b0)\n    else:\n        b, n = ajustar(Xd, dv.y.values, offd, P_DEVB[linea], 5000, val=dv.fold.values == 0); b0 = 0.0\n        b.save_model(str(f))\n    return (lambda df: b.predict(matriz(df, cols, cats), raw_score=True) + b0), {"rondas": n, "base": b0}\n\n\ndef tronco_cb(dv, cols, cats, linea):\n    from catboost import CatBoostRegressor, Pool\n    catc = [c for c in cols if c in CATEGORICAS]\n\n    def X(df):\n        o = pd.DataFrame(index=df.index)\n        for c in cols:\n            if c in CATEGORICAS:\n                o[c] = df[c].astype(str).where(df[c].notna(), "NA")\n            else:\n                o[c] = pd.to_numeric(df[c], errors="coerce").astype(float)\n        return o\n    f = MOD / f"CB_{linea}.cbm"\n    m = CatBoostRegressor()\n    if f.exists():\n        m.load_model(str(f)); n = m.tree_count_\n    else:\n        offd = np.log(dv.exposure.values * dv.c_fila.values); val = dv.fold.values == 0\n        Xd = X(dv)\n        prm = dict(loss_function="Tweedie:variance_power=1.5", learning_rate=0.08, depth=6, l2_leaf_reg=3.0,\n                   one_hot_max_size=8, thread_count=2, random_seed=SEMILLA, verbose=0, allow_writing_files=False)\n        m1 = CatBoostRegressor(iterations=3000, od_type="Iter", od_wait=100, **prm)\n        m1.fit(Pool(Xd[~val], dv.y.values[~val], baseline=offd[~val], cat_features=catc),\n               eval_set=Pool(Xd[val], dv.y.values[val], baseline=offd[val], cat_features=catc), use_best_model=True)\n        n = max(m1.get_best_iteration() + 1, 1)\n        m = CatBoostRegressor(iterations=n, **prm)\n        m.fit(Pool(Xd, dv.y.values, baseline=offd, cat_features=catc))\n        m.save_model(str(f))\n    return (lambda df: m.predict(Pool(X(df), cat_features=catc), prediction_type="RawFormulaVal")), {"rondas": int(n)}\n\n\ndef tronco_fs(dv, cols, cats, linea):\n    fF = MOD / f"{VAR}f_{linea}.txt"; fS = MOD / f"{VAR}s_{linea}.txt"\n    if VAR == "FSt":  # E25 keeps FS frequency unchanged and uses the internally selected severity grid.\n        fF = MOD / f"FSf_{linea}.txt"\n    Xd = matriz(dv, cols, cats); offd = np.log(dv.exposure.values * dv.c_fila.values); val = dv.fold.values == 0\n    s = (dv.n_sin.values > 0) & (dv.y.values > 0)\n    msev = float(dv.y.values[s].sum() / dv.n_sin.values[s].sum())\n    if VAR == "FSt":\n        import json\n        el = json.load(open(TRABAJO / "modelos" / "dev" / f"e25_rejilla_{linea}.json"))["elegida"]\n        bf = lgb.Booster(model_file=str(fF)); nf = bf.current_iteration(); b0 = 0.0\n        if fS.exists():\n            bs = lgb.Booster(model_file=str(fS))\n        else:\n            ps = dict(objective="gamma", learning_rate=0.03, num_leaves=el["num_leaves"],\n                      min_data_in_leaf=el["min_data_in_leaf"], lambda_l2=el["lambda_l2"], feature_fraction=0.8,\n                      bagging_fraction=0.8, bagging_freq=1, max_cat_to_onehot=8, cat_smooth=50, verbose=-1,\n                      num_threads=2, seed=SEMILLA)\n            Xs = Xd[s].reset_index(drop=True); n_ = dv.n_sin.values[s].astype(float)\n            bs, _ = ajustar(Xs, dv.y.values[s] / n_, np.full(s.sum(), np.log(msev)), ps, 3000, es=100, val=val[s], w=n_)\n            bs.save_model(str(fS))\n        ns = bs.current_iteration()\n    elif fF.exists():\n        bf = lgb.Booster(model_file=str(fF)); bs = lgb.Booster(model_file=str(fS))\n        nf, ns = bf.current_iteration(), bs.current_iteration()\n        b0 = _leer_base(fF) if INICIO0 else 0.0\n    else:\n        pf = {k: v for k, v in P_DEVB[linea].items() if k != "tweedie_variance_power"}\n        pf["objective"] = "poisson"\n        if INICIO0:\n            bf, nf, b0 = ajustar0(Xd, dv.n_sin.values.astype(float), offd, pf, 5000, val=val)\n            _guardar_base(fF, b0)\n        else:\n            bf, nf = ajustar(Xd, dv.n_sin.values, offd, pf, 5000, val=val); b0 = 0.0\n        ps = dict(objective="gamma", learning_rate=0.03, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.8,\n                  bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50,\n                  verbose=-1, num_threads=2, seed=SEMILLA)\n        Xs = Xd[s].reset_index(drop=True); n_ = dv.n_sin.values[s]\n        bs, ns = ajustar(Xs, dv.y.values[s] / n_, np.full(s.sum(), np.log(msev)), ps, 5000, val=val[s], w=n_)\n        bf.save_model(str(fF)); bs.save_model(str(fS))\n\n    def raw(df):\n        X = matriz(df, cols, cats)\n        return bf.predict(X, raw_score=True) + b0 + bs.predict(X, raw_score=True) + np.log(msev)\n    return raw, {"rondas_frec": nf, "rondas_sev": ns, "sev_media": msev, "base_frec": b0}\n\n\ndef tronco_fsp(dv, cols, cats, linea):\n    """E21 FSp: developed-market peril Poisson frequency and gamma severity, with one emerging level per line."""\n    from completitud import por_peligro\n    Xd = matriz(dv, cols, cats); val = dv.fold.values == 0\n    perils = sorted(c[2:] for c in dv.columns if c.startswith("n_") and c != "n_sin" and c != "n_lineas")\n    modelos = {}; meta = {}\n    pf = {k: v for k, v in P_DEVB[linea].items() if k != "tweedie_variance_power"}\n    pf["objective"] = "poisson"\n    ps = dict(objective="gamma", learning_rate=0.03, num_leaves=15, min_data_in_leaf=200, feature_fraction=0.8,\n              bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50,\n              verbose=-1, num_threads=2, seed=SEMILLA)\n    for p in perils:\n        fF = MOD / f"FSpf_{linea}_{p}.txt"; fS = MOD / f"FSps_{linea}_{p}.txt"\n        n = dv[f"n_{p}"].values; yp = dv[f"y_{p}"].values\n        s = (n > 0) & (yp > 0)\n        msev = float(yp[s].sum() / n[s].sum())\n        if fF.exists():\n            bf = lgb.Booster(model_file=str(fF)); bs = lgb.Booster(model_file=str(fS))\n        else:\n            cp = np.maximum(por_peligro(linea, p, dv.period_start, dv.period_end), 1e-4)\n            bf, _ = ajustar(Xd, n.astype(float), np.log(dv.exposure.values * cp), pf, 5000, val=val)\n            bs, _ = ajustar(Xd[s].reset_index(drop=True), yp[s] / n[s], np.full(s.sum(), np.log(msev)), ps, 3000,\n                            val=val[s], w=n[s].astype(float))\n            bf.save_model(str(fF)); bs.save_model(str(fS))\n        modelos[p] = (bf, bs, msev)\n        meta[p] = {"rondas_frec": bf.current_iteration(), "rondas_sev": bs.current_iteration(), "sev_media": msev,\n                   "siniestros": int(n.sum())}\n        print("  ", linea, p, meta[p], flush=True)\n\n    def raw(df):\n        X = matriz(df, cols, cats)\n        tot = np.zeros(len(df))\n        for p, (bf, bs, msev) in modelos.items():\n            tot += np.exp(bf.predict(X, raw_score=True) + bs.predict(X, raw_score=True)) * msev\n        return np.log(tot)\n    return raw, {"peligros": meta}\n\n\ndef tronco_h(dv, cats, linea):\n    """T2h: T2c plus centered history, with multiplier 1 where the history window is unobserved."""\n    colsC = [x for x in comunes(linea) if x not in HIST]\n    f1 = MOD / ("T2ci_%s.txt" % linea if INICIO0 else "T2c_%s.txt" % linea)\n    b1 = lgb.Booster(model_file=str(f1)); base1 = _leer_base(f1) if INICIO0 else 0.0\n    H = HIST + ["new_business", "renewal_count", "prior_claims_5y"]\n    f = MOD / f"{VAR}_{linea}.txt"\n    r1 = b1.predict(matriz(dv, colsC, cats), raw_score=True) + base1\n    offd = np.log(dv.exposure.values * dv.c_fila.values)\n    obs = dv.hist_pol_12m.notna().values\n    XH = matriz(dv, H, cats)\n    if f.exists():\n        b2 = lgb.Booster(model_file=str(f)); n2 = b2.current_iteration()\n    else:\n        p2 = dict(objective="tweedie", tweedie_variance_power=1.5, learning_rate=0.03, num_leaves=7,\n                  min_data_in_leaf=2000, lambda_l2=10.0, verbose=-1, num_threads=2, seed=SEMILLA)\n        b2, n2 = ajustar(XH[obs].reset_index(drop=True), dv.y.values[obs], r1[obs] + offd[obs], p2, 3000,\n                         val=(dv.fold.values == 0)[obs])\n        b2.save_model(str(f))\n    mu1 = np.exp(r1[obs] + offd[obs]); g = b2.predict(XH[obs], raw_score=True)\n    cen = float(np.log(np.sum(mu1 * np.exp(g)) / np.sum(mu1)))\n\n    def raw(df):\n        r = b1.predict(matriz(df, colsC, cats), raw_score=True) + base1\n        gg = b2.predict(matriz(df, H, cats), raw_score=True) - cen\n        return r + np.where(df.hist_pol_12m.notna().values, gg, 0.0)\n    return raw, {"rondas_etapa2": n2, "centrado": cen, "importancia": dict(zip(H, b2.feature_importance("gain").round(1).tolist()))}\n\n\n# Phase predictions and outputs.\ndef nivel(a):\n    return factor_tweedie(a.y.values, np.exp(a.raw.values) * a.exposure.values * a.c_fila.values)\n\n\ndef salidas(em, linea):\n    meta = {}\n    fo, anios = CONF["oof"]\n    tr = em[(em.conjunto == "train") & em.year.isin(anios)]\n    out, ks = [], []\n    for k in range(5):\n        a, b = tr[tr.fold != k], tr[tr.fold == k]\n        kk = nivel(a); ks.append(kk)\n        out.append(pd.DataFrame({"row_id": b.row_id.values, "mu": kk * np.exp(b.raw.values) * b.exposure.values}))\n    (TRABAJO / "pred" / fo).mkdir(parents=True, exist_ok=True)\n    pd.concat(out, ignore_index=True).to_parquet(TRABAJO / "pred" / fo / f"{VAR}{SUF}_{linea}.parquet", index=False)\n    fe, anios_tr, anio_ev = CONF["ev"]\n    a = em[(em.conjunto == "train") & em.year.isin(anios_tr)]; b = em[em.year == anio_ev]\n    kk = nivel(a)\n    (TRABAJO / "pred" / fe).mkdir(parents=True, exist_ok=True)\n    pd.DataFrame({"row_id": b.row_id.values, "mu": kk * np.exp(b.raw.values) * b.exposure.values}).to_parquet(\n        TRABAJO / "pred" / fe / f"{VAR}{SUF}_{linea}.parquet", index=False)\n    em[["row_id", "raw"]].to_parquet(TRABAJO / "pred" / fe / f"{VAR}{SUF}raw_{linea}.parquet", index=False)\n    meta.update({"k_oof": ks, "k_eval": kk})\n    return meta\n\n\ndef correr(linea):\n    t0 = time.time()\n    em = cargar("emerging", linea, CONF["clima"])\n    dva = cargar("developed", linea, CONF["clima"])\n    if VAR == "T2x":\n        em = em.merge(tenencia("emerging"), on=["customer_id", "year"], how="left")\n        dva = dva.merge(tenencia("developed"), on=["customer_id", "year"], how="left")\n    if VAR == "FSp":\n        dva = dva.merge(pd.read_parquet(TRABAJO / f"conteos_developed_{linea}.parquet"), on="row_id", how="left")\n    dv = dva[dva.year.isin(CONF["dev_years"])].reset_index(drop=True)\n    cols = columnas(linea)\n    cats = categorias_union([dv, em], cols + HIST + ["new_business", "renewal_count", "prior_claims_5y"])\n    if VAR in ("T2x", "T2o", "T2b", "T2c", "T2bi", "T2ci", "T2oi"):\n        raw, meta = tronco_lgb(dv, cols, cats, linea, VAR)\n    elif VAR == "CB":\n        raw, meta = tronco_cb(dv, cols, cats, linea)\n    elif VAR in ("FS", "FSi", "FSt"):\n        raw, meta = tronco_fs(dv, cols, cats, linea)\n    elif VAR == "FSp":\n        raw, meta = tronco_fsp(dv, cols, cats, linea)\n    elif VAR == "F0":  # Diagnostic: FS frequency times constant mean severity.\n        bf = lgb.Booster(model_file=str(MOD / f"FSf_{linea}.txt"))\n        s = (dv.n_sin.values > 0) & (dv.y.values > 0)\n        lm = float(np.log(dv.y.values[s].sum() / dv.n_sin.values[s].sum()))\n        raw, meta = (lambda df: bf.predict(matriz(df, cols, cats), raw_score=True) + lm), {"rondas_frec": bf.current_iteration()}\n    elif VAR in ("T2h", "T2hi"):\n        raw, meta = tronco_h(dv, cats, linea)\n    else:\n        raise ValueError(VAR)\n    meta["s_tronco"] = round(time.time() - t0)\n    # DES24: out-of-time developed structure, releveled within the evaluation year.\n    if CONF["des"] is not None:\n        d4 = dva[dva.year == CONF["des"]].reset_index(drop=True)\n        mu = np.exp(raw(d4)) * d4.exposure.values * d4.c_fila.values\n        k4 = factor_tweedie(d4.y.values, mu)\n        meta["DES24"] = {"D2": d2(d4.y.values, k4 * mu), "k": k4, "n": len(d4)}\n        if os.environ.get("MLIND_GUARDAR_DES24"):  # E26 saves developed-2024 raw predictions for blend-weight estimation.\n            pd.DataFrame({"row_id": d4.row_id.values, "raw": np.log(mu / (d4.exposure.values * d4.c_fila.values)),\n                          "y": d4.y.values, "exposure": d4.exposure.values, "c_fila": d4.c_fila.values}).to_parquet(\n                TRABAJO / "pred" / "hold" / f"{VAR}{SUF}des24_{linea}.parquet", index=False)\n    em = em.assign(raw=raw(em))\n    if VAR not in ("T2b", "T2c") or SUF:\n        meta.update(salidas(em, linea))\n    meta["s"] = round(time.time() - t0)\n    guardar_json(meta, TRABAJO / "pred" / CONF["ev"][0] / f"{VAR}{SUF}_{linea}_meta.json")\n    print(VAR, FASE, linea, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in meta.items() if k != "importancia"}, flush=True)\n\n\nif __name__ == "__main__":\n    lineas = sys.argv[3].split(",") if len(sys.argv) > 3 else LINEAS\n    for l in lineas:\n        correr(l)\n'


### Embedded source: `ml/codigo/variables.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/variables.py'] = '"""Shared transfer-model feature schema and phase-specific matrix loading."""\nfrom __future__ import annotations\nimport numpy as np, pandas as pd\nfrom comun import *\n\nPOLIZA = ["voluntary_excess", "channel", "payment_method", "missed_payments_12m", "new_business", "renewal_count",\n          "log_days_quote", "discount_applied", "start_month", "full_year", "hist_pol_12m", "hist_cli_12m"]\nCLIENTE = ["marital_status", "household_size", "prior_claims_5y", "licence_points", "years_at_address",\n           "tenure_years", "age_band", "occupation"]\nMARKETING = ["email_domain", "marketing_consent", "document_preference", "previous_insurer", "campaign_code",\n             "branch_id", "marketing_segment"]\nDIRECCION = ["urban_density", "flood_zone", "distance_to_river_km", "elevation_m", "soil_type", "property_age",\n             "property_type", "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms",\n             "energy_rating", "solar_panels", "clima_wind_exposure_index", "clima_wildfire_index",\n             "clima_rainfall_mm", "clima_mean_temperature_c"]\nRAMO = {\n    "motor": ["years_licensed", "ncb_years", "vehicle_value_group", "vehicle_age", "engine_size_cc", "body_type",\n              "transmission", "seats", "fuel_type", "driver_assist", "cover_type", "overnight_parking",\n              "vehicle_use", "named_young_driver"],\n    "buildings": ["log_si", "construction", "occupancy", "days_unoccupied", "smoke_alarms",\n                  "accidental_damage_cover", "mortgage"],\n    "contents": ["log_si", "tenure", "security_level", "high_value_items", "accidental_damage_cover",\n                 "works_from_home", "lodgers", "bicycle_cover"],\n}\nSOLO_EMERGENTE = ["gender", "self_employed", "region"]\nTARIFA = ["log_wp_rel"]\nMAKE = ["make_model"]\n\nCATEGORICAS = {"channel", "payment_method", "marital_status", "age_band", "occupation", "email_domain",\n               "document_preference", "previous_insurer", "campaign_code", "marketing_segment", "urban_density",\n               "soil_type", "property_type", "roof_type", "energy_rating", "body_type", "transmission", "fuel_type",\n               "cover_type", "overnight_parking", "vehicle_use", "construction", "occupancy", "tenure",\n               "security_level", "gender", "region", "make_model", "branch_id"}\n\n\ndef comunes(linea, marketing=False):\n    return POLIZA + CLIENTE + (MARKETING if marketing else []) + DIRECCION + RAMO[linea]\n\n\ndef _historia(df, claims, inicio_datos):\n    """Count all-status policy/customer notifications in the preceding 365 days; use NaN when the full window is unobserved."""\n    lo = df.period_start - pd.Timedelta(days=365)\n    ok = lo >= pd.Timestamp(inicio_datos)\n    hp = claims[["policy_id", "line", "report_date"]].merge(\n        df[["row_id", "policy_id", "line", "period_start"]], on=["policy_id", "line"])\n    hp = hp[(hp.report_date < hp.period_start) & (hp.report_date >= hp.period_start - pd.Timedelta(days=365))]\n    pol = df.row_id.map(hp.groupby("row_id").size()).fillna(0)\n    return pol.where(ok, np.nan)\n\n\n_cache = {}\n\n\ndef cargar(mercado, linea, ventana_clima):\n    """Load derived features using the hasta2023 or hasta2024 weather window."""\n    key = (mercado, linea, ventana_clima)\n    if key in _cache:\n        return _cache[key].copy()\n    df = pd.read_parquet(TRABAJO / f"base_{mercado}_{linea}.parquet")\n    cl = pd.read_parquet(TRABAJO / f"clima_{mercado}_{ventana_clima}.parquet")\n    df = df.merge(cl, on="address_id", how="left")\n    if "occupation" in df:\n        df["occupation"] = df.occupation.str.lower().str.strip()\n    import os  # A05 history mode 2 ends at min(start,V), affecting only 2025 rows.\n    pref = "historia2" if os.environ.get("MLIND_HISTORIA") == "2" else "historia"\n    hist = pd.read_parquet(TRABAJO / f"{pref}_{mercado}_{linea}.parquet")\n    df = df.merge(hist, on="row_id", how="left")\n    _cache[key] = df\n    return df.copy()\n\n\ndef matriz(df, cols, categorias):\n    X = pd.DataFrame(index=df.index)\n    for c in cols:\n        if c in CATEGORICAS:\n            v = df[c].astype(str).where(df[c].notna(), None) if c in df else pd.Series(None, index=df.index)\n            X[c] = pd.Categorical(v, categories=categorias[c])\n        else:\n            X[c] = pd.to_numeric(df[c], errors="coerce").astype(float) if c in df else np.nan\n    return X\n\n\ndef categorias_union(frames, cols):\n    out = {}\n    for c in cols:\n        if c in CATEGORICAS:\n            vals = set()\n            for f in frames:\n                if c in f:\n                    vals |= set(f[c].dropna().astype(str).unique())\n            out[c] = sorted(vals)\n    return out\n'


### Embedded source: `ml/codigo/verificar_informe.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/verificar_informe.py'] = '"""Verify report values against their source artifacts; use the standard library and pandas for CSVs."""\nimport json, re, sys, hashlib\nfrom pathlib import Path\nimport pandas as pd\n\nR = Path(__file__).resolve().parents[1]\nE = R / "entrega" if (R / "entrega").exists() else R\nT = Path(__file__).resolve().parents[1] / "trabajo"\nif not T.exists():  # Local evidence copies are packaged under entrega.\n    T = None\ntxt = (E / "INFORME.md").read_text(encoding="utf-8")\nJ = lambda f: json.load(open(E / f, encoding="utf-8"))\n\n\ndef es(x, d):\n    s = f"{x:,.{d}f}"\n    return s.replace(",", "X").replace(".", ",").replace("X", ".")\n\n\ndef pct(x, d=1):\n    return es(100 * x, d) + " %"\n\n\ndev = pd.read_csv(E / "tabla_dev.csv")\nD = lambda c, k, col="D2": float(dev[(dev.cand == c) & (dev.calib == k)][col].iloc[0])\nhold = J("resultados_hold.json")["candidatos"]\nH = lambda c: hold[c]["sin_tendencia"]["D2"]\nbs = J("bootstrap_final.json"); prod = J("produccion_2025.json"); cal = J("calibracion_pcv.json"); ctx = J("contexto_2025.json")\n_t = (lambda f, alt=None: json.load(open((T / f) if T else (E / (alt or Path(f).name)), encoding="utf-8")))\nprep = _t("preparacion.json"); pesos = _t("pesos_2025_dev.json")\nhb = J("hold_prueba_tendencia.json"); cred = J("credibilidad_entre_anios.json"); fig = J("figuras_datos.json")\ntend_h = _t("pred/hold/tendencia.json", "tendencia_hold.json")\nV = prod["variantes"]\npc = pd.read_csv(E / "prima_comercial_20.csv").groupby("ramo")[["prima_comercial", "prima_escrita"]].mean()\nic = lambda k: f"[{\'+\' if bs[k][\'ic95\'][0] >= 0 else \'−\'}{es(abs(bs[k][\'ic95\'][0]), 4)}; {\'+\' if bs[k][\'ic95\'][1] >= 0 else \'−\'}{es(abs(bs[k][\'ic95\'][1]), 4)}]"\nchecks = [\n    ("total 2025", es(V["final_C1"]["total"], 0)),\n    ("autos 2025", es(V["final_C1"]["totales"]["motor"]["total"], 0)),\n    ("edificios 2025", es(V["final_C1"]["totales"]["buildings"]["total"], 0)),\n    ("contenidos 2025", es(V["final_C1"]["totales"]["contents"]["total"], 0)),\n    ("tasa autos", es(V["final_C1"]["totales"]["motor"]["tasa"], 1)),\n    ("tasa edificios", es(V["final_C1"]["totales"]["buildings"]["tasa"], 1)),\n    ("tasa contenidos", es(V["final_C1"]["totales"]["contents"]["tasa"], 1)),\n    ("sha", V["final_C1"]["controles"]["sha256"][:16]),\n    ("DEV T2b C1", es(D("T2b", "ramo"), 4)), ("HOLD T2b C1", es(H("T2b_C1"), 4)),\n    ("DEV R0 C1", es(D("R0", "ramo"), 4)), ("HOLD R0", es(H("R0_C1"), 4)),\n    ("dif DEV T2b-R0", "+" + es(bs["DEV_T2b_C1-R0_C1"]["dif"], 4)), ("IC DEV T2b-R0", ic("DEV_T2b_C1-R0_C1")),\n    ("dif HOLD T2b-R0", "+" + es(bs["HOLD_T2b_C1-R0_C1"]["dif"], 4)), ("IC HOLD T2b-R0", ic("HOLD_T2b_C1-R0_C1")),\n    ("dif DEV T2b-T1wp", "+" + es(bs["DEV_T2b_C1-T1wp_C1"]["dif"], 4)), ("IC DEV T2b-T1wp", ic("DEV_T2b_C1-T1wp_C1")),\n    ("dif HOLD T2b-T1wp", "+" + es(bs["HOLD_T2b_C1-T1wp_C2"]["dif"], 4)), ("IC HOLD T2b-T1wp", ic("HOLD_T2b_C1-T1wp_C2")),\n    ("N2a-N1 DEV", "+" + es(bs["DEV_N2a_C2-N1_C2"]["dif"], 4)), ("IC N2a-N1 DEV", ic("DEV_N2a_C2-N1_C2")),\n    ("N2a-N1 HOLD", "+" + es(bs["HOLD_N2a_C2-N1_C2"]["dif"], 4)), ("IC N2a-N1 HOLD", ic("HOLD_N2a_C2-N1_C2")),\n    ("N3-T2b DEV", "−" + es(abs(bs["DEV_N3_C2-T2b_C2"]["dif"]), 4)), ("IC N3-T2b DEV", ic("DEV_N3_C2-T2b_C2")),\n    ("C2-C1 HOLD", es(abs(bs["HOLD_T2b_C2-T2b_C1"]["dif"]), 4)), ("IC C2-C1 HOLD", ic("HOLD_T2b_C2-T2b_C1")),\n    ("T0 emergente", es(sum(sum(prep[f"emerging_{l}_T0_por_anio"].values()) for l in ("motor", "buildings", "contents")), 2)),\n    ("compl 2024 autos", es(prep["emerging_motor_completitud_media_por_anio"]["2024"], 4)),\n    ("compl 2024 edif", es(prep["emerging_buildings_completitud_media_por_anio"]["2024"], 4)),\n    ("compl 2024 cont", es(prep["emerging_contents_completitud_media_por_anio"]["2024"], 4)),\n    ("compl 2023 autos", es(prep["emerging_motor_completitud_media_por_anio"]["2023"], 4)),\n    ("BI 90 días", pct(prep["cdf_coste_desarrollado"]["motor|bodily_injury"]["90"], 0)),\n    ("BI 365 días", pct(prep["cdf_coste_desarrollado"]["motor|bodily_injury"]["365"], 0)),\n    ("capital edif train", str(prep["emerging_buildings_capital_div1000"]["train"])),\n    ("capital cont test", str(prep["emerging_contents_capital_div1000"]["test"])),\n    ("duplicados", str(prep["developed_duplicados_retirados"])),\n    ("AUC autos", es(pesos["motor"]["auc"], 2)), ("AUC hogar", es(pesos["buildings"]["auc"], 2)),\n    ("incendio emergente", pct(prep["emerging_buildings_pesos_peligro"]["fire"], 1)),\n    ("incendio desarrollado", pct(prep["developed_buildings_pesos_peligro"]["fire"], 1)),\n    ("DEV T3bwp C2", es(D("T3bwp", "ramo_nb"), 4)), ("DEV T2b C2", es(D("T2b", "ramo_nb"), 4)),\n    ("DEV N2a C2", es(D("N2a", "ramo_nb"), 4)), ("DEV N1 C2", es(D("N1", "ramo_nb"), 4)),\n    ("DEV N3 C2", es(D("N3", "ramo_nb"), 4)), ("DEV E1 C2", es(D("E1", "ramo_nb"), 4)),\n    ("DEV T4 C2", es(D("T4", "ramo_nb"), 4)), ("DEV T6 C2", es(D("T6", "ramo_nb"), 4)), ("DEV T5 C2", es(D("T5", "ramo_nb"), 4)),\n    ("DEV T2b cuerpo", es(D("T2b", "ramo_nb", "D2_cuerpo"), 4)), ("DEV T2b w2025", es(D("T2b", "ramo_nb", "D2_w2025"), 4)),\n    ("DEV T1wp w2025", es(D("T1wp", "ramo_nb", "D2_w2025"), 4)), ("DEV G1 C1", es(D("G1", "ramo"), 4)),\n    ("HOLD N2a", es(H("N2a_C2"), 4)), ("HOLD N1", es(H("N1_C2"), 4)), ("HOLD T1wp", es(H("T1wp_C2"), 4)), ("HOLD T3bwp", es(H("T3bwp_C2"), 4)),\n    ("tendencia hold autos", pct(tend_h["motor"]["tau_anual"] - 1, 1).replace(" %", "")),\n    ("tend C2", "−" + es(abs(hb["T2b_C2"]["dif"]), 4)), ("tend C1", "−" + es(abs(hb["T2b_C1"]["dif"]), 4)),\n    ("NB autos 2023", es(fig["ae_nb_anio"]["motor|True|2023"]["AE"], 2)), ("NB autos 2024", es(fig["ae_nb_anio"]["motor|True|2024"]["AE"], 2)),\n    ("cred autos", es(cred["motor"]["z_optimo"], 1)),\n    ("MCB autos", pct(cal["murphy"]["motor"]["MCB"] / cal["murphy"]["motor"]["S"], 1)),\n    ("MCB edif", pct(cal["murphy"]["buildings"]["MCB"] / cal["murphy"]["buildings"]["S"], 1)),\n    ("MCB cont", pct(cal["murphy"]["contents"]["MCB"] / cal["murphy"]["contents"]["S"], 1)),\n    ("forma edif", es(cal["forma"]["buildings"]["ganancia_D2_en_muestra"], 4)),\n    ("balance autos", pct(prod["balance_oof_pcv"]["motor"], 1)), ("balance cont", pct(prod["balance_oof_pcv"]["contents"], 1)),\n    ("total C2", es(V["C2"]["total"], 0)), ("total preinscrita", es(V["preinscrita_C2_tendencia"]["total"], 0)),\n    ("total tendencia", es(V["C1_tendencia"]["total"], 0)), ("total T2c", es(V["T2c_C1"]["total"], 0)),\n    ("C2 sube", pct(V["C2"]["total"] / V["final_C1"]["total"] - 1, 1)),\n    ("tasa obs 2023 autos", es(ctx["motor"]["2023"]["tasa_ultima_obs"], 1)), ("tasa obs 2024 edif", es(ctx["buildings"]["2024"]["tasa_ultima_obs"], 1)),\n    ("razón edificios", es(ctx["buildings"]["2025"]["razon_riesgo_prima_escrita"], 2)),\n    ("prima comercial autos", es(pc.loc["motor", "prima_comercial"], 1)), ("prima escrita edif", es(pc.loc["buildings", "prima_escrita"], 1)),\n]\nfallos = [(n, s) for n, s in checks if s not in txt]\n# Final CSV checks.\ncands = [E / "prediccion_2025_mlind.csv", E.parent / "privado" / "prediccion_2025_mlind.csv",\n         E.parents[2] / "Repositorio_IAA2026" / "privado" / "semana2" / "transfer_ml_20261023" / "prediccion_2025_mlind.csv"]\ncsv = next((c for c in cands if c.exists()), None)\nif csv is None:\n    print("aviso: CSV de predicción no encontrado junto al informe; se omite su huella")\nelse:\n    h = hashlib.sha256(csv.read_bytes()).hexdigest()\n    if h != V["final_C1"]["controles"]["sha256"]:\n        fallos.append(("sha del CSV", h))\nfor n, s in fallos:\n    print("FALLA", n, s)\nprint(f"{len(checks) - len(fallos)} de {len(checks)} cifras del informe ligadas a su artefacto" + ("" if not fallos else " (hay fallos)"))\nsys.exit(1 if fallos else 0)\n'


### Embedded source: `noche/codigo/base_mercado.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/base_mercado.py'] = '"""Build emerging-market bases from the original Parquet files under frozen rules.\n\nEXP-N1: round inclusive covered days/365 to four decimals.\nASG-N1: match policy, line and inclusive loss-date/period boundaries; classify\nassignments as unique, missing or ambiguous.\nOBJ-N1: sum incurred minus recoveries for uniquely assigned non-declined claims.\nPaid/incurred are already net of excess, so do not subtract it again.\nCAP-N1: flag suspicious household insured-value scales using premium per mille;\ndivide by 1,000 only above the line threshold and below the normal population\'s\nminimum premium-per-mille rate. Keep raw insured values unchanged.\nWrite line bases for 2023-2025 and the claims-assignment table under salidas/s2/.\n"""\nimport json, hashlib, pathlib\nimport numpy as np, pandas as pd\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\')\nimport sys as _s\nMK = _s.argv[1] if len(_s.argv) > 1 else \'s2\'\nD = R / f\'datos/{MK}\'\nO = R / f\'salidas/{MK}\'; O.mkdir(parents=True, exist_ok=True)\nLINES = [\'motor\', \'buildings\', \'contents\']\nPERILS = {\'motor\': [\'own_damage\', \'third_party_prop\', \'bodily_injury\', \'theft\', \'glass_hail\'],\n          \'buildings\': [\'storm\', \'flood\', \'escape_of_water\', \'subsidence\', \'fire\', \'accidental_damage\'],\n          \'contents\': [\'theft\', \'escape_of_water\', \'accidental_damage\', \'flood\']}\nCUT = pd.Timestamp(\'2024-12-31\')\nlog = {}\n\ndef sha(p):\n    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()\n\nclaims = pd.read_parquet(D / \'claims.parquet\')\nif MK == \'s1\':  # DUP-N1: 604 payload-identical copies differ only in claim_id; retain the 10-character identifier.\n    claims = claims[claims.claim_id.str.len() == 10].reset_index(drop=True)\nclaims[\'ld\'] = pd.to_datetime(claims.loss_date)\nclaims[\'rd\'] = pd.to_datetime(claims.report_date)\nclaims[\'net\'] = claims.incurred_to_date - claims.recoveries\nclaims[\'accepted\'] = claims.status != \'repudiated\'\n\nasg_all = []\nfor ln in LINES:\n    tr = pd.read_parquet(D / f\'policies_{ln}.parquet\'); tr[\'split\'] = \'train\'\n    te = pd.read_parquet(D / f\'test_{ln}.parquet\'); te[\'split\'] = \'test\'\n    df = pd.concat([tr, te], ignore_index=True)\n    df[\'ps\'] = pd.to_datetime(df.period_start); df[\'pe\'] = pd.to_datetime(df.period_end)\n    df[\'year\'] = df.ps.dt.year\n    days = (df.pe - df.ps).dt.days + 1\n    df[\'days\'] = days\n    df[\'exposure_raw\'] = df.exposure\n    df[\'exposure_model\'] = (days / 365).round(4)\n    df[\'exp_changed\'] = ~np.isclose(df.exposure_raw, df.exposure_model, atol=6e-5)\n    log[f\'{ln}_exp_changed_train\'] = int(df.exp_changed[df.split == \'train\'].sum())\n    log[f\'{ln}_exp_changed_test\'] = int(df.exp_changed[df.split == \'test\'].sum())\n\n    c = claims[claims.line == ln].copy()\n    m = c[[\'claim_id\', \'policy_id\', \'ld\']].merge(df.loc[df.split == \'train\', [\'row_id\', \'policy_id\', \'ps\', \'pe\']], on=\'policy_id\', how=\'left\')\n    m[\'inside\'] = (m.ld >= m.ps) & (m.ld <= m.pe)\n    n_in = m.groupby(\'claim_id\').inside.sum()\n    c[\'n_rows\'] = c.claim_id.map(n_in).fillna(0).astype(int)\n    c[\'category\'] = np.select([c.n_rows == 1, c.n_rows == 0], [\'unica\', \'sin_fila\'], \'ambigua\')\n    first = df[df.split == \'train\'].groupby(\'policy_id\').ps.min()\n    c[\'policy_first_start\'] = c.policy_id.map(first)\n    c[\'prehistoria\'] = (c.category == \'sin_fila\') & (c.ld < c.policy_first_start)\n    rid = m[m.inside].drop_duplicates(\'claim_id\').set_index(\'claim_id\').row_id\n    c[\'row_id\'] = c.claim_id.map(rid)\n    asg_all.append(c)\n\n    ok = c[(c.category == \'unica\') & c.accepted]\n    g = ok.groupby(\'row_id\')\n    agg = pd.DataFrame({\'t0\': g.net.sum(), \'paid\': g.paid_to_date.sum(), \'incurred\': g.incurred_to_date.sum(),\n                        \'recov\': g.recoveries.sum(), \'n_claims\': g.size(),\n                        \'n_open\': g.apply(lambda x: (x.status == \'open\').sum(), include_groups=False),\n                        \'case_reserve\': g.apply(lambda x: (x.incurred_to_date - x.paid_to_date).sum(), include_groups=False)})\n    for p in PERILS[ln]:\n        sp = ok[ok.peril == p].groupby(\'row_id\')\n        agg[f\'n_{p}\'] = sp.size(); agg[f\'t0_{p}\'] = sp.net.sum()\n    df = df.merge(agg, left_on=\'row_id\', right_index=True, how=\'left\')\n    num = [x for x in agg.columns]\n    df.loc[df.split == \'train\', num] = df.loc[df.split == \'train\', num].fillna(0)\n    log[f\'{ln}_claims\'] = {k: int(v) for k, v in c.groupby([\'category\']).size().items()}\n    log[f\'{ln}_accepted_unica\'] = int(len(ok)); log[f\'{ln}_t0\'] = round(float(ok.net.sum()), 2)\n    log[f\'{ln}_prehistoria\'] = int(c.prehistoria.sum())\n    log[f\'{ln}_sin_fila_no_prehist\'] = int(((c.category == \'sin_fila\') & ~c.prehistoria).sum())\n\n    if ln in (\'buildings\', \'contents\'):\n        df[\'sum_insured_raw\'] = df.sum_insured\n        df[\'ppm\'] = df.written_premium / df.sum_insured * 1000\n        log[f\'{ln}_si_null\'] = int(df.sum_insured.isna().sum())\n    df.drop(columns=[\'ps\', \'pe\']).to_parquet(O / f\'base_{ln}.parquet\', index=False)\n\nasg = pd.concat(asg_all, ignore_index=True)\nasg.drop(columns=[\'ld\', \'rd\']).to_parquet(O / \'asignacion.parquet\', index=False)\nlog[\'t0_total\'] = round(sum(log[f\'{l}_t0\'] for l in LINES), 2)\njson.dump(log, open(O / \'log_base.json\', \'w\'), indent=1, default=str)\nprint(json.dumps(log, indent=1, default=str))\n'


### Embedded source: `noche/codigo/reporte.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/reporte.py'] = '"""Estimate peril reporting completeness with right-truncated Lynden-Bell data.\n\nREP-N1 assumes accepted-claim reporting delay depends on peril, not market or\nyear. Pool both markets and retain market contrasts as diagnostics. Normalize\nF(t) at the largest observed delay (1,460 days); the unobserved tail beyond that\nis not identifiable from one valuation. Inspect severity versus delay in mature\ndeveloped cohorts before equating cost and count completeness.\n"""\nimport json, pathlib\nimport numpy as np, pandas as pd\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\')\nCUT = pd.Timestamp(\'2024-12-31\')\nB = 7  # Days per delay bin.\n\n\ndef load(mk):\n    c = pd.read_parquet(R / f\'datos/{mk}/claims.parquet\')\n    if mk == \'s1\':  # Identical payloads except claim_id: retain the 10-character identifier.\n        c = c[c.claim_id.str.len() == 10]\n    c = c[c.status != \'repudiated\'].copy()\n    c[\'ld\'] = pd.to_datetime(c.loss_date); c[\'rd\'] = pd.to_datetime(c.report_date)\n    c[\'delay\'] = (c.rd - c.ld).dt.days; c[\'window\'] = (CUT - c.ld).dt.days\n    c[\'market\'] = mk\n    c[\'net\'] = c.incurred_to_date - c.recoveries\n    return c\n\n\ndef lynden_bell(delay, window, tmax):\n    """Estimate F(t) in B-day bins; delay is bounded by the observation window."""\n    db = (delay // B).astype(int); wb = (window // B).astype(int)\n    nb = tmax // B + 1\n    d = np.bincount(db, minlength=nb)[:nb]\n    # Reverse-time risk set: records satisfying db <= s <= wb.\n    start = np.bincount(db, minlength=nb + 1)[:nb]\n    end = np.bincount(np.minimum(wb, nb - 1) + 1, minlength=nb + 1)[:nb]\n    risk = np.cumsum(start) - np.cumsum(end)  # Records with db<=s and wb>=s.\n    h = np.where(risk > 0, d / np.maximum(risk, 1), 0.0)\n    # F(s) = prod_{u>s} (1-h_u)\n    surv = np.cumprod((1 - h)[::-1])[::-1]\n    F = np.concatenate([surv[1:], [1.0]])\n    return F  # F[s] = P(delay bucket <= s | delay <= tmax)\n\n\ndef main():\n    c = pd.concat([load(\'s1\'), load(\'s2\')], ignore_index=True)\n    tmax = int(c.window.max())\n    out = {}; tabla = {}\n    for p, g in c.groupby(\'peril\'):\n        F = lynden_bell(g.delay.values, g.window.values, tmax)\n        Fe = lynden_bell(g[g.market == \'s2\'].delay.values, g[g.market == \'s2\'].window.values, 730) if (g.market == \'s2\').sum() > 30 else None\n        Fd = lynden_bell(g[g.market == \'s1\'].delay.values, g[g.market == \'s1\'].window.values, tmax)\n        tabla[p] = F.tolist()\n        k = {lag: round(float(F[min(lag // B, len(F) - 1)]), 4) for lag in (30, 91, 182, 365, 730)}\n        kd = {lag: round(float(Fd[min(lag // B, len(Fd) - 1)]), 4) for lag in (30, 91, 182, 365, 730)}\n        ke = {lag: round(float(Fe[min(lag // B, len(Fe) - 1)] * Fd[min(730 // B, len(Fd) - 1)]), 4) for lag in (30, 91, 182, 365, 730)} if Fe is not None else None\n        out[p] = {\'n\': int(len(g)), \'n_emerging\': int((g.market == \'s2\').sum()), \'F_pooled\': k, \'F_developed\': kd,\n                  \'F_emerging_rescaled_to_dev730\': ke}\n    json.dump({\'bucket_days\': B, \'tmax_days\': tmax, \'F\': tabla}, open(R / \'salidas/reporte_F.json\', \'w\'))\n    json.dump(out, open(R / \'salidas/reporte_resumen.json\', \'w\'), indent=1)\n    # Mature developed 2021-2022 severity diagnostic: late/early mean-cost ratio.\n    m = c[(c.market == \'s1\') & (c.ld.dt.year <= 2022)]\n    sev = {}\n    for p, g in m.groupby(\'peril\'):\n        q = g.delay.quantile([0.5, 0.9])\n        sev[p] = {\'media_hasta_p50\': round(float(g[g.delay <= q[0.5]].net.mean()), 1),\n                  \'media_p50_p90\': round(float(g[(g.delay > q[0.5]) & (g.delay <= q[0.9])].net.mean()), 1),\n                  \'media_sobre_p90\': round(float(g[g.delay > q[0.9]].net.mean()), 1), \'n\': int(len(g))}\n    json.dump(sev, open(R / \'salidas/severidad_por_retraso.json\', \'w\'), indent=1)\n    print(json.dumps(out, indent=1)); print(json.dumps(sev, indent=1))\n\n\nif __name__ == \'__main__\':\n    main()\n'


### Embedded source: `noche/codigo/exposicion_efectiva.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/exposicion_efectiva.py'] = '"""Derive peril- and cost-weighted effective exposure for each training row.\ne_ef_p=sum_d F_p(cutoff-d)/365, using count- or cost-weighted reporting curves.\ne_ef_coste=sum_p(s_p*e_ef_p)/sum_p(s_p), where s_p are ultimate-cost shares.\nUse nominal exposure when predicting ultimate cost for the 2025 test period.\n"""\nimport json, pathlib\nimport numpy as np, pandas as pd\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\')\nCUT = pd.Timestamp(\'2024-12-31\')\nPERILS = {\'motor\': [\'own_damage\', \'third_party_prop\', \'bodily_injury\', \'theft\', \'glass_hail\'],\n          \'buildings\': [\'storm\', \'flood\', \'escape_of_water\', \'subsidence\', \'fire\', \'accidental_damage\'],\n          \'contents\': [\'theft\', \'escape_of_water\', \'accidental_damage\', \'flood\']}\n\n\ndef daily_G(F, B, n):\n    Fd = np.repeat(np.asarray(F), B)[:n]\n    if len(Fd) < n:\n        Fd = np.concatenate([Fd, np.ones(n - len(Fd))])\n    return np.concatenate([[0.0], np.cumsum(Fd)])  # G[k] = sum_{t<k} F(t)\n\n\ndef add_effective(df, line, Ftab, B, shares):\n    ps = pd.to_datetime(df.period_start); pe = pd.to_datetime(df.period_end)\n    w_hi = (CUT - ps).dt.days.values  # Largest observation window, at the period\'s first day.\n    w_lo = (CUT - pe).dt.days.values  # Smallest observation window, at the period\'s last day.\n    train = (w_lo >= 0)\n    tot = np.zeros(len(df)); wsum = 0.0\n    for p in PERILS[line]:\n        G = daily_G(Ftab[p], B, 4000)\n        e = np.where(train, (G[np.clip(w_hi, 0, 3999) + 1] - G[np.clip(w_lo, 0, 3999)]) / 365.0, np.nan)\n        df[f\'eef_{p}\'] = e\n        tot += shares[p] * np.nan_to_num(e); wsum += shares[p]\n    df[\'eef_coste\'] = np.where(train, tot / wsum, np.nan)\n    df[\'comp_coste\'] = df.eef_coste / df.exposure_model\n    return df\n'


### Embedded source: `noche/codigo/ultimo_mercado.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/ultimo_mercado.py'] = '"""Estimate emerging ultimate costs, effective exposure and recovery sensitivity.\nREP-N2 uses peril net-cost weights capped at p99.5 for reporting completeness.\nEstimate emerging 2023-2024 peril ultimate cost as T0_p*e/e_ef_p for cost shares.\nREC-N1 only diagnoses possible open-claim recoveries using closed-claim peril\npatterns; it does not apply that adjustment to model predictions.\n"""\nimport json, pathlib, sys\nimport numpy as np, pandas as pd\nsys.path.insert(0, str(pathlib.Path(__file__).parent))\nfrom exposicion_efectiva import add_effective, PERILS\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\')\nimport sys as _s\nMK = _s.argv[1] if len(_s.argv) > 1 else \'s2\'\nO = R / f\'salidas/{MK}\'\nrep = json.load(open(R / \'salidas/reporte_F.json\'))\nF = rep[\'F_coste\']; B = rep[\'bucket_days\']\nres = {}\nfor ln in PERILS:\n    b = pd.read_parquet(O / f\'base_{ln}.parquet\')\n    tr = b.split == \'train\'\n    tmp = add_effective(b.copy(), ln, F, B, {p: 1.0 for p in PERILS[ln]})\n    ult = {}\n    for p in PERILS[ln]:\n        ult[p] = float(b.loc[tr, f\'t0_{p}\'].sum() * tmp.loc[tr, \'exposure_model\'].sum() / tmp.loc[tr, f\'eef_{p}\'].sum())\n    tot = sum(ult.values()); shares = {p: ult[p] / tot for p in ult}\n    b = add_effective(b, ln, F, B, shares)\n    b.to_parquet(O / f\'base_{ln}.parquet\', index=False)\n    r = {\'shares\': {p: round(v, 4) for p, v in shares.items()}}\n    for y in sorted(b.loc[tr,\'year\'].unique()):\n        s = b[tr & (b.year == y)]\n        r[str(y)] = {\'exposure\': round(s.exposure_model.sum(), 2), \'eef_coste\': round(s.eef_coste.sum(), 2),\n                     \'comp\': round(s.eef_coste.sum() / s.exposure_model.sum(), 4), \'t0\': round(s.t0.sum(), 2),\n                     \'rate_t0\': round(s.t0.sum() / s.exposure_model.sum(), 2),\n                     \'rate_ult\': round(s.t0.sum() / s.eef_coste.sum(), 2),\n                     \'freq_obs\': round(s.n_claims.sum() / s.exposure_model.sum(), 4)}\n    res[ln] = r\n# Pending-recovery diagnostic.\na = pd.read_parquet(O / \'asignacion.parquet\')\na = a[(a.category == \'unica\') & a.accepted]\nrec = {}\nfor (ln, p), g in a.groupby([\'line\', \'peril\']):\n    cl = g[g.status == \'closed\']; op = g[g.status == \'open\']\n    rate = cl.recoveries.sum() / cl.incurred_to_date.sum() if cl.incurred_to_date.sum() > 0 else 0\n    if rate > 0:\n        rec[f\'{ln}/{p}\'] = {\'rate_closed\': round(rate, 4), \'open_incurred\': round(op.incurred_to_date.sum(), 2),\n                            \'implied_recov\': round(rate * op.incurred_to_date.sum(), 2), \'n_open\': int(len(op))}\nfor ln in PERILS:\n    t0 = a[a.line == ln].net.sum()\n    imp = sum(v[\'implied_recov\'] for k, v in rec.items() if k.startswith(ln + \'/\'))\n    res[ln][\'recov_open_implied_share_t0\'] = round(imp / t0, 4)\nres[\'recoveries\'] = rec\njson.dump(res, open(O / \'ultimo_resumen.json\', \'w\'), indent=1)\nprint(json.dumps(res, indent=1))\n'


### Embedded source: `noche/codigo/conf_insumos.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/conf_insumos.py'] = '"""Build 2023-to-2024 confirmation inputs without either market\'s 2024 loss labels.\nEstimate count/cost reporting curves from occurrence years <=2023, retaining the\ncommon 31 December 2024 valuation truncation. Use emerging-2023 and developed\n2021-2023 cost shares. Recompute effective exposures for emerging 2023/2024 and\ndeveloped 2021-2023; save base-plus-eef tables under salidas/conf/.\n"""\nimport json, pathlib, sys\nimport numpy as np, pandas as pd\nsys.path.insert(0, str(pathlib.Path(__file__).parent))\nimport reporte as RP\nfrom exposicion_efectiva import add_effective, PERILS\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\'); O = R / \'salidas/conf\'; O.mkdir(parents=True, exist_ok=True)\nc = pd.concat([RP.load(\'s1\'), RP.load(\'s2\')], ignore_index=True)\nc = c[c.ld.dt.year <= 2023]\ntmax = int(c.window.max())\nFc = {}\nfor p, g in c.groupby(\'peril\'):\n    cap = g.net.quantile(0.995)\n    w = np.minimum(g.net.values, cap)\n    db = (g.delay.values // RP.B).astype(int); wb = (g.window.values // RP.B).astype(int); nb = tmax // RP.B + 1\n    d = np.bincount(db, weights=w, minlength=nb)[:nb]\n    st = np.bincount(db, weights=w, minlength=nb + 1)[:nb]\n    en = np.bincount(np.minimum(wb, nb - 1) + 1, weights=w, minlength=nb + 1)[:nb]\n    risk = np.cumsum(st) - np.cumsum(en)\n    h = np.where(risk > 0, d / np.maximum(risk, 1e-9), 0)\n    surv = np.cumprod((1 - h)[::-1])[::-1]\n    Fc[p] = np.concatenate([surv[1:], [1.0]]).tolist()\njson.dump({\'bucket_days\': RP.B, \'F_coste\': Fc, \'nota\': \'ocurrencia <= 2023\'}, open(O / \'reporte_F_conf.json\', \'w\'))\nres = {}\nfor mk, years in ((\'s2\', [2023, 2024]), (\'s1\', [2021, 2022, 2023])):\n    for ln in PERILS:\n        b = pd.read_parquet(R / f\'salidas/{mk}/base_{ln}.parquet\')\n        b = b[(b.split == \'train\') & b.year.isin(years)].reset_index(drop=True)\n        fit_years = [y for y in years if y <= 2023]\n        tmp = add_effective(b.copy(), ln, Fc, RP.B, {p: 1.0 for p in PERILS[ln]})\n        f = tmp.year.isin(fit_years) if mk == \'s1\' else tmp.year == 2023\n        ult = {p: float(b.loc[f, f\'t0_{p}\'].sum() * tmp.loc[f, \'exposure_model\'].sum() / tmp.loc[f, f\'eef_{p}\'].sum()) for p in PERILS[ln]}\n        tot = sum(ult.values()); shares = {p: v / tot for p, v in ult.items()}\n        b = add_effective(b, ln, Fc, RP.B, shares)\n        b.to_parquet(O / f\'base_{mk}_{ln}.parquet\', index=False)\n        res[f\'{mk}_{ln}\'] = {\'shares\': {p: round(v, 4) for p, v in shares.items()},\n                             \'comp_by_year\': b.groupby(\'year\').apply(lambda x: x.eef_coste.sum() / x.exposure_model.sum(), include_groups=False).round(4).to_dict()}\njson.dump(res, open(O / \'conf_insumos.json\', \'w\'), indent=1, default=str)\nprint(json.dumps(res, indent=1, default=str))\n'


### Embedded source: `noche/codigo/desarrollado_gbm.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/desarrollado_gbm.py'] = '"""Transfer developed-market LightGBM Tweedie relativities (TRF-N1).\nFit each line on the requested developed years with a log-effective-exposure\noffset. Use the common market schema and exposure, excluding market-specific\nregion, credit, mileage and telematics. Emerging labels do not fit these trees.\nSave models and log-relative predictions for developed and emerging rows.\nUsage: python desarrollado_gbm.py <line> [--años 2021,2022,2023,2024]\n"""\nimport json, pathlib, sys, time\nimport numpy as np, pandas as pd, lightgbm as lgb\nsys.path.insert(0, str(pathlib.Path(__file__).parent))\nfrom variables import features\nimport marco as M\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\'); O = R / \'salidas/transfer\'; O.mkdir(parents=True, exist_ok=True)\nline = sys.argv[1]\nyears = [2021, 2022, 2023, 2024]\nif \'--años\' in sys.argv:\n    years = [int(x) for x in sys.argv[sys.argv.index(\'--años\') + 1].split(\',\')]\ntag = \'dev\' + \'\'.join(str(y)[-2:] for y in years) + (sys.argv[sys.argv.index(\'--sufijo\') + 1] if \'--sufijo\' in sys.argv else \'\')\nLR = float(sys.argv[sys.argv.index(\'--lr\') + 1]) if \'--lr\' in sys.argv else 0.05\nPAT = int(sys.argv[sys.argv.index(\'--pat\') + 1]) if \'--pat\' in sys.argv else 200\nSEED = int(sys.argv[sys.argv.index(\'--seed\') + 1]) if \'--seed\' in sys.argv else 11\nt = time.time()\nCONF = \'--conf\' in sys.argv  # Confirmation uses bases excluding 2024 labels.\nb1 = pd.read_parquet(R / (f\'salidas/conf/base_s1_{line}.parquet\' if CONF else f\'salidas/s1/base_{line}.parquet\'))\nb1 = b1[(b1.split == \'train\') & b1.year.isin(years)].reset_index(drop=True)\nX1, cols1, _ = features(b1, line, \'s1\')\nX1[\'exposure_model\'] = b1.exposure_model.values\nb2 = pd.read_parquet(R / (f\'salidas/conf/base_s2_{line}.parquet\' if CONF else f\'salidas/s2/base_{line}.parquet\'))\nX2, cols2, _ = features(b2, line, \'s2\')\nX2[\'exposure_model\'] = b2.exposure_model.values\nDROPC = sys.argv[sys.argv.index(\'--sin\') + 1].split(\',\') if \'--sin\' in sys.argv else []\nCLIMA = [\'wind_exposure_index\', \'wildfire_index\', \'rainfall_mm\', \'mean_temperature_c\']\nif \'--clima\' in sys.argv:  # CLIMA-N1: persistent address weather from other years, excluding the period year.\n    def otros(base, mk, yrs):\n        w = pd.read_parquet(R / f\'datos/{mk}/weather.parquet\'); w = w[w.year.isin(yrs)]\n        pv = w.pivot_table(index=\'address_id\', columns=\'year\', values=CLIMA)\n        out = pd.DataFrame(index=base.index)\n        for v in CLIMA:\n            vals = np.stack([base.address_id.map(pv[(v, y)]).values for y in yrs], axis=1)\n            mask = np.stack([(base.year.values != y) for y in yrs], axis=1)\n            out[f\'cl_{v}\'] = np.nansum(np.where(mask, vals, 0), axis=1) / np.maximum((mask & ~np.isnan(vals)).sum(axis=1), 1)\n        return out\n    c1x = otros(b1, \'s1\', [2021, 2022, 2023, 2024]); c2x = otros(b2, \'s2\', [2023, 2024])\n    for c in c1x: X1[c] = c1x[c].values; X2[c] = c2x[c].values\nif \'--f1\' in sys.argv:  # F1-N1: policy notifications reported before period_start.\n    def hist(base, mk):\n        a = pd.read_parquet(R / f\'salidas/{mk}/asignacion.parquet\'); a = a[(a.line == line) & a.accepted][[\'policy_id\', \'report_date\']]\n        a[\'rd\'] = pd.to_datetime(a.report_date)\n        bb = base[[\'row_id\', \'policy_id\', \'period_start\']].copy(); bb[\'ps\'] = pd.to_datetime(bb.period_start)\n        mm = bb.merge(a, on=\'policy_id\'); mm = mm[mm.rd < mm.ps]\n        n = bb.row_id.map(mm.groupby(\'row_id\').size()).fillna(0).values\n        first = bb.groupby(\'policy_id\').ps.min()\n        yrs = ((bb.ps - bb.policy_id.map(first)).dt.days / 365.25).values\n        return n, yrs\n    n1, y1 = hist(b1, \'s1\'); n2, y2 = hist(b2, \'s2\')\n    X1[\'f1_siniestros_previos\'] = n1; X1[\'f1_anios_observados\'] = y1\n    X2[\'f1_siniestros_previos\'] = n2; X2[\'f1_anios_observados\'] = y2\ncols = [c for c in X1.columns if c in X2.columns and c not in DROPC]\n# Align to developed categorical levels; unseen emerging levels become missing.\nfor c in cols:\n    if str(X1[c].dtype) == \'category\':\n        cats = list(X1[c].cat.categories)\n        v = X2[c].astype(\'string\')\n        X2[c] = pd.Categorical(v.where(v.isin(cats), \'missing\'), categories=cats)\n        unseen = int((~v.isin(cats)).sum())\n        if unseen: print(\'categoría no vista\', c, unseen)\ny = b1.t0.values; off = np.log(b1.eef_coste.values)\nif \'--cap\' in sys.argv:  # ROB-C1 caps training losses for shape; emerging balance absorbs the excess level.\n    CAP = float(sys.argv[sys.argv.index(\'--cap\') + 1])\n    asg = pd.read_parquet(R / \'salidas/s1/asignacion.parquet\'); asg = asg[(asg.category == \'unica\') & asg.accepted & (asg.line == line)]\n    full_t0 = b1.row_id.map(asg.groupby(\'row_id\').net.sum()).fillna(0.0).values\n    assert np.allclose(full_t0, y, atol=0.01), \'el T0 de la base no cuadra con la asignación\'\n    y = b1.row_id.map(asg.assign(c=np.minimum(asg.net, CAP)).groupby(\'row_id\').c.sum()).fillna(0.0).values\nbase = np.log(y.sum() / np.exp(off).sum())\np = dict(objective=\'tweedie\', tweedie_variance_power=1.5, learning_rate=LR, num_leaves=31, min_data_in_leaf=1000,\n         feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, min_data_per_group=1000,\n         cat_smooth=50, verbose=-1, seed=SEED, deterministic=True, force_row_wise=True, num_threads=2)\nif \'--reg\' in sys.argv:  # ROB-C2 uses smaller, more strongly regularized trees.\n    p.update(num_leaves=15, min_data_in_leaf=3000, lambda_l2=50.0, feature_fraction=0.7)\nif \'--params\' in sys.argv:  # ROB-C4 grid settings use developed internal validation only.\n    p.update(json.loads(sys.argv[sys.argv.index(\'--params\') + 1]))\ninner = M.fold_of(b1.customer_id.values, salt=\'noche_dev\', k=5) == 0\ndtr = lgb.Dataset(X1.loc[~inner, cols], y[~inner], init_score=off[~inner] + base, free_raw_data=False)\ndva = lgb.Dataset(X1.loc[inner, cols], y[inner], init_score=off[inner] + base, reference=dtr)\nbst = lgb.train(p, dtr, num_boost_round=4000, valid_sets=[dva], callbacks=[lgb.early_stopping(PAT, verbose=False)])\nnb = bst.best_iteration\nfull = lgb.train(p, lgb.Dataset(X1[cols], y, init_score=off + base), num_boost_round=nb)\nfull.save_model(str(O / f\'gbm_{tag}_{line}.txt\'))\ng1 = full.predict(X1[cols], raw_score=True); g2 = full.predict(X2[cols], raw_score=True)\nc1 = np.log(np.average(np.exp(g1), weights=b1.exposure_model))  # Center to mean developed-market relativity 1.\npd.DataFrame({\'row_id\': b2.row_id, \'lg_dev\': g2 - c1}).to_parquet(O / f\'lg_{tag}_{line}_em.parquet\', index=False)\npd.DataFrame({\'row_id\': b1.row_id, \'lg_dev\': g1 - c1}).to_parquet(O / f\'lg_{tag}_{line}_dev.parquet\', index=False)\nimp = pd.Series(full.feature_importance(\'gain\'), index=cols).sort_values(ascending=False)\ninfo = {\'line\': line, \'years\': years, \'rows\': int(len(b1)), \'rounds\': int(nb), \'valid_d2_inner\': None,\n        \'seconds\': round(time.time() - t, 1), \'importance_gain_share\': (imp / imp.sum()).round(4).head(25).to_dict(),\n        \'cols\': cols}\nmu_va = np.exp(bst.predict(X1.loc[inner, cols], raw_score=True) + off[inner] + base)\ninfo[\'valid_d2_inner\'] = round(float(M.d2(y[inner], mu_va)), 5)\njson.dump(info, open(O / f\'info_{tag}_{line}.json\', \'w\'), indent=1)\nprint(json.dumps(info, indent=1))\n'


### Embedded source: `noche/codigo/variables.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/variables.py'] = '"""Build comparable F0 features for both markets under the frozen availability contract.\nExclude renewal_decision, current-period service contacts, identifiers, surname,\nclaim amounts and same-year weather. Written premium and discounts are reserved\nfor separate tariff ablations. Normalize occupation text, construction codes,\nhousehold insured values and developed ages into the emerging age bands.\n"""\nimport pathlib\nimport numpy as np, pandas as pd\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\')\n# CON-N1 maps emerging construction codes and developed t/w aliases to one vocabulary.\n# Recorded mix: brick 70%, t+w 12.6%, non_standard 9.9%, stone 8.0%; emerging BRK 70%, WOD 12.0%, MIX 10.1%, STN 7.9%.\nCONS = {\'BRK\': \'brick\', \'STN\': \'stone\', \'WOD\': \'timber\', \'MIX\': \'non_standard\', \'T\': \'timber\', \'W\': \'timber\'}\nCAP_THR = {\'buildings\': 5_000_000, \'contents\': 2_000_000}  # Thresholds lie within the observed insured-value gaps (894k-60M and 283k-7.6M).\nBANDS = [(17, 24, \'17-24\'), (25, 34, \'25-34\'), (35, 49, \'35-49\'), (50, 64, \'50-64\'), (65, 200, \'65+\')]\n\nPOL_COMMON = [\'start_month\', \'voluntary_excess\', \'compulsory_excess\', \'channel\', \'payment_method\', \'missed_payments_12m\',\n              \'new_business\', \'renewal_count\', \'days_quote_to_inception\']\n# VAR-N1 excludes previous_insurer, branch_id, email_domain, marketing_consent, document_preference\n# and self_employed under the recorded developed-frequency diagnostics.\nCUST = [\'occupation_n\', \'marital_status\', \'household_size\', \'prior_claims_5y\', \'licence_points\',\n        \'years_at_address\', \'cust_tenure\', \'has_campaign\', \'marketing_segment\']\nADDR = [\'region\', \'urban_density\', \'flood_zone\', \'distance_to_river_km\', \'elevation_m\', \'soil_type\', \'prop_age\',\n        \'property_type\', \'roof_type\', \'basement\', \'trees_within_5m\', \'listed_building\', \'bedrooms\', \'energy_rating\',\n        \'solar_panels\']\nLINE = {\'motor\': [\'age_band\', \'years_licensed\', \'ncb_years\', \'vehicle_value_group\', \'vehicle_age\', \'make_model\',\n                  \'engine_size_cc\', \'body_type\', \'transmission\', \'seats\', \'vehicle_colour\', \'fuel_type\',\n                  \'driver_assist\', \'cover_type\', \'overnight_parking\', \'vehicle_use\', \'named_young_driver\'],\n        \'buildings\': [\'age_band\', \'log_si\', \'construction_type\', \'occupancy\', \'days_unoccupied\', \'smoke_alarms\',\n                      \'accidental_damage_cover\', \'mortgage\'],\n        \'contents\': [\'age_band\', \'log_si\', \'tenure\', \'security_level\', \'high_value_items\', \'accidental_damage_cover\',\n                     \'works_from_home\', \'lodgers\', \'bicycle_cover\']}\nCATEG = {\'channel\', \'payment_method\', \'occupation_n\', \'marital_status\', \'email_domain\', \'document_preference\',\n         \'previous_insurer\', \'branch_id\', \'marketing_segment\', \'region\', \'urban_density\', \'soil_type\', \'property_type\',\n         \'roof_type\', \'energy_rating\', \'age_band\', \'make_model\', \'body_type\', \'transmission\', \'vehicle_colour\',\n         \'fuel_type\', \'cover_type\', \'overnight_parking\', \'vehicle_use\', \'construction_type\', \'occupancy\',\n         \'smoke_alarms\', \'tenure\', \'security_level\'}\nWEATHER = [\'hail_days\', \'wind_exposure_index\', \'freezing_days\', \'dry_summer_index\', \'wildfire_index\', \'rainfall_mm\',\n           \'mean_temperature_c\']\n\n\ndef band(age):\n    out = pd.Series(pd.NA, index=age.index, dtype=\'object\')\n    for lo, hi, lab in BANDS:\n        out[(age >= lo) & (age <= hi)] = lab\n    return out\n\n\ndef features(base, line, market):\n    """Build (X,columns) from a policy-row base containing year and exposure_model."""\n    d = pd.read_parquet(R / f\'datos/{market}/customers.parquet\')\n    a = pd.read_parquet(R / f\'datos/{market}/addresses.parquet\')\n    occ = d.occupation.astype(\'string\').str.strip().str.lower()\n    d[\'self_employed\'] = occ.str.startswith(\'self-employed\').fillna(False)\n    d[\'occupation_n\'] = occ.str.replace(r\'^self-employed\\s+\', \'\', regex=True).str.replace(r\'\\s+\', \' \', regex=True)\n    d[\'has_campaign\'] = d.campaign_code.notna()\n    # ID01: pad developed motor customer_id to eight characters for the master-data join.\n    # Corrected on 23 October: the initial version omitted padding and left\n    # 98,110 training and 25,554 test motor rows without customer features.\n    key = base.customer_id.astype(str) if __import__(\'os\').environ.get(\'A1_PRE_ID01\') else base.customer_id.astype(str).str.zfill(8)\n    df = base.assign(_cid=key.values).merge(d.drop(columns=[\'surname\'], errors=\'ignore\').rename(columns={\'customer_id\': \'_cid\'}),\n                                            on=\'_cid\', how=\'left\', validate=\'m:1\').drop(columns=[\'_cid\'])\n    assert __import__(\'os\').environ.get(\'A1_PRE_ID01\') or df.customer_since_year.notna().all(), \'cruce de clientes incompleto\'\n    df = df.merge(a, on=\'address_id\', how=\'left\', validate=\'m:1\')\n    df[\'cust_tenure\'] = df.year - df.customer_since_year\n    df[\'start_month\'] = pd.to_datetime(df.period_start).dt.month  # NB-N2: calendar features capture part of the new-business effect.\n    df[\'prop_age\'] = df.year - df.year_built\n    if line == \'motor\':\n        df[\'age_band\'] = df[\'driver_age_band\'] if \'driver_age_band\' in df else band(df[\'driver_age\'])\n    else:\n        df[\'age_band\'] = df[\'policyholder_age_band\'] if \'policyholder_age_band\' in df else band(df[\'policyholder_age\'])\n        si = df[\'sum_insured\'].astype(float)\n        scale = si > CAP_THR[line]\n        df[\'cap_scaled\'] = scale\n        df[\'log_si\'] = np.log(np.where(scale, si / 1000.0, si))\n    if line == \'buildings\':\n        df[\'construction_type\'] = df.construction_type.astype(\'string\').str.strip()\n        df[\'construction_type\'] = df.construction_type.map(lambda v: CONS.get(v.upper(), v.lower()) if isinstance(v, str) else v)\n    if \'smoke_alarms\' in df:\n        df[\'smoke_alarms\'] = df.smoke_alarms.map({True: \'yes\', False: \'no\'}).fillna(\'missing\')\n    cols = POL_COMMON + CUST + ADDR + LINE[line]\n    if market == \'s1\':\n        cols = [c for c in cols if c != \'region\']\n    X = df[cols].copy()\n    for c in cols:\n        if c in CATEG:\n            X[c] = X[c].astype(\'string\').fillna(\'missing\').astype(\'category\')\n        elif X[c].dtype == bool or str(X[c].dtype) == \'boolean\':\n            X[c] = X[c].astype(float)\n        else:\n            X[c] = pd.to_numeric(X[c], errors=\'coerce\').astype(float)\n    return X, cols, df\n\n\ndef weather_other_year(df, market, vars_, mode):\n    """base adds no weather; otro uses the other training year and the 2023-2024 mean for 2025."""\n    w = pd.read_parquet(R / f\'datos/{market}/weather.parquet\')\n    w = w[w.year.isin([2023, 2024])].pivot_table(index=\'address_id\', columns=\'year\', values=vars_)\n    out = pd.DataFrame(index=df.index)\n    for v in vars_:\n        a23 = df.address_id.map(w[(v, 2023)]); a24 = df.address_id.map(w[(v, 2024)])\n        out[f\'w_{v}\'] = np.where(df.year == 2023, a24, np.where(df.year == 2024, a23, (a23 + a24) / 2))\n    return out\n'


### Embedded source: `noche/codigo/marco.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['noche/codigo/marco.py'] = '"""Shared Week 2 partitions, metrics, component bootstrap and standard models.\nFreeze customer folds as int(sha256(\'noche_v1|\'+customer_id)[:16],16)%5.\nCompute power-1.5 Tweedie D2 on row amounts, by line and over concatenated lines,\nusing the evaluation-set mean as the null. Development/confirmation evaluate T0\nagainst ultimate rate times effective exposure.\n"""\nimport hashlib, json, pathlib, sys, time\nimport numpy as np, pandas as pd\n\nR = pathlib.Path(\'__RUN_ROOT__/noche\')\nsys.path.insert(0, str(R / \'codigo\'))\nP = 1.5\n\n\ndef fold_of(ids, salt=\'noche_v1\', k=5):\n    return np.array([int(hashlib.sha256(f\'{salt}|{x}\'.encode()).hexdigest()[:16], 16) % k for x in ids])\n\n\ndef dev(y, mu, p=P):\n    y = np.asarray(y, float); mu = np.asarray(mu, float)\n    return 2 * (np.power(y, 2 - p) / ((1 - p) * (2 - p)) - y * np.power(mu, 1 - p) / (1 - p) + np.power(mu, 2 - p) / (2 - p))\n\n\ndef d2(y, mu):\n    y = np.asarray(y, float)\n    return 1 - dev(y, mu).sum() / dev(y, np.full_like(y, y.mean())).sum()\n\n\ndef components(cust, addr):\n    """Construct customer-address connected components with union-find."""\n    keys = pd.Index(pd.unique(np.concatenate([(\'c\' + pd.Series(cust).astype(str)).values, (\'a\' + pd.Series(addr).astype(str)).values])))\n    par = np.arange(len(keys))\n    def find(i):\n        while par[i] != i:\n            par[i] = par[par[i]]; i = par[i]\n        return i\n    ci = keys.get_indexer(\'c\' + pd.Series(cust).astype(str)); ai = keys.get_indexer(\'a\' + pd.Series(addr).astype(str))\n    for x, z in zip(ci, ai):\n        rx, rz = find(x), find(z)\n        if rx != rz: par[rx] = rz\n    roots = np.array([find(i) for i in ci])\n    return pd.factorize(roots)[0]\n\n\ndef boot_d2_diff(y, mus, comp, n=10000, seed=2026102301, batch=250):\n    """Poisson-bootstrap connected components; return replicate D2 arrays for each named prediction."""\n    y = np.asarray(y, float); K = comp.max() + 1\n    agg = lambda v: np.bincount(comp, weights=v, minlength=K)\n    Sy = agg(y); Sy2 = agg(np.power(y, 2 - P)); Sn = agg(np.ones_like(y))\n    Sd = {k: agg(dev(y, m)) for k, m in mus.items()}\n    rng = np.random.default_rng(seed); out = {k: [] for k in mus}\n    for s in range(0, n, batch):\n        W = rng.poisson(1.0, size=(min(batch, n - s), K)).astype(float)\n        ny = W @ Sy; nn = W @ Sn; c = ny / nn; ny2 = W @ Sy2\n        null = 2 * (ny2 / ((1 - P) * (2 - P)) - ny * np.power(c, 1 - P) / (1 - P) + nn * np.power(c, 2 - P) / (2 - P))\n        for k in mus:\n            out[k].append(1 - (W @ Sd[k]) / null)\n    return {k: np.concatenate(v) for k, v in out.items()}\n\n\n# Standard reference models.\n\ndef design_glm(Xtr, Xte, cols, categ, nbins=6, min_share=0.005, w=None):\n    """Bin numeric variables using training quantiles plus missing; pool categories below 0.5% into other."""\n    A = {}; B = {}\n    for c in cols:\n        if c in categ or Xtr[c].nunique(dropna=True) <= 12:  # Treat binary and low-cardinality discrete features as categorical.\n            s_tr = Xtr[c].astype(\'string\').fillna(\'missing\'); s_te = Xte[c].astype(\'string\').fillna(\'missing\')\n            share = s_tr.value_counts(normalize=True)\n            keep = set(share[share >= min_share].index)\n            A[c] = s_tr.where(s_tr.isin(keep), \'otros\'); B[c] = s_te.where(s_te.isin(keep), \'otros\')\n        else:\n            v = Xtr[c].astype(float)\n            qs = np.unique(np.nanquantile(v, np.linspace(0, 1, nbins + 1)[1:-1])) if v.notna().any() else np.array([])\n            def cut(x):\n                x = x.astype(float)\n                lab = np.searchsorted(qs, x.values, side=\'right\').astype(str)\n                lab = np.where(np.isnan(x.values), \'missing\', lab)\n                return pd.Series(lab, index=x.index)\n            A[c] = cut(Xtr[c]); B[c] = cut(Xte[c])\n    A = pd.DataFrame(A); B = pd.DataFrame(B)\n    for c in cols:\n        cats = sorted(set(A[c]) | {\'otros\'})\n        A[c] = pd.Categorical(A[c], categories=cats); B[c] = pd.Categorical(B[c].where(B[c].isin(cats), \'otros\'), categories=cats)\n    return A, B\n\n\ndef fit_glm_tweedie(Xtr, ytr, off_tr, Xte, off_te, alpha):\n    from glum import GeneralizedLinearRegressor, TweedieDistribution\n    m = GeneralizedLinearRegressor(family=TweedieDistribution(P), link=\'log\', alpha=alpha, l1_ratio=0.0,\n                                   fit_intercept=True, drop_first=False, max_iter=200)\n    m.fit(Xtr, ytr, offset=off_tr)\n    return m.predict(Xte, offset=off_te), m\n\n\ndef fit_glm_fs(Xtr, ntr, t0tr, off_tr, Xte, off_te, alpha_f, alpha_s):\n    from glum import GeneralizedLinearRegressor\n    f = GeneralizedLinearRegressor(family=\'poisson\', link=\'log\', alpha=alpha_f, l1_ratio=0.0, drop_first=False, max_iter=200)\n    f.fit(Xtr, ntr, offset=off_tr)\n    pos = (ntr > 0) & (t0tr > 0)\n    s = GeneralizedLinearRegressor(family=\'gamma\', link=\'log\', alpha=alpha_s, l1_ratio=0.0, drop_first=False, max_iter=200)\n    s.fit(Xtr[pos], (t0tr[pos] / ntr[pos]), sample_weight=ntr[pos])\n    return f.predict(Xte, offset=off_te) * s.predict(Xte), (f, s)\n\n\ndef fit_gbm_tweedie(Xtr, ytr, off_tr, Xte, off_te, ids_tr, params=None, seed=7):\n    import lightgbm as lgb\n    p = dict(objective=\'tweedie\', tweedie_variance_power=P, learning_rate=0.02, num_leaves=7, min_data_in_leaf=400,\n             feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, min_data_per_group=500,\n             cat_smooth=50, max_cat_to_onehot=8, verbose=-1, seed=seed, deterministic=True, force_row_wise=True,\n             num_threads=2)\n    if params: p.update(params)\n    base = np.log(ytr.sum() / np.exp(off_tr).sum())  # Initialize at the training mean rate.\n    off_tr = off_tr + base; off_te = off_te + base\n    inner = fold_of(ids_tr, salt=\'noche_inner\', k=5) == 0\n    dtr = lgb.Dataset(Xtr[~inner], ytr[~inner], init_score=off_tr[~inner], free_raw_data=False)\n    dva = lgb.Dataset(Xtr[inner], ytr[inner], init_score=off_tr[inner], reference=dtr)\n    b = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False)])\n    nb = b.best_iteration\n    full = lgb.train(p, lgb.Dataset(Xtr, ytr, init_score=off_tr), num_boost_round=max(nb, 1))\n    return np.exp(full.predict(Xte, raw_score=True) + off_te), (full, nb)\n'


### Embedded source: `ml/codigo/fsp_light.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/fsp_light.py'] = '"""Fit lightweight developed 2021-2023 peril Poisson frequency and gamma severity.\nUse the shared T2b schema, weather through 2023 and learning rate 0.08. Predict\nall emerging 2023-2025 rows and save ultimate expected counts f_<p> and severity\ns_<p> with row_id in trabajo/pred/fsp/<line>.parquet.\n"""\nimport sys, time, json\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom completitud import por_peligro\nlinea = sys.argv[1]\nHIST = ["hist_pol_12m", "hist_cli_12m"]\ncols = comunes(linea)\nt0 = time.time()\nem = cargar("emerging", linea, "hasta2023")\ndva = cargar("developed", linea, "hasta2023")\ndv = dva[dva.year.isin([2021, 2022, 2023])].reset_index(drop=True); del dva\ncn = pd.read_parquet(TRABAJO / f"conteos_developed_{linea}.parquet")\ndv = dv.merge(cn, on="row_id", how="left")\ncats = categorias_union([dv, em], cols + HIST + ["new_business", "renewal_count", "prior_claims_5y"])\nXd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats); val = dv.fold.values == 0\nperils = sorted(c[2:] for c in cn.columns if c.startswith("n_"))\nPF = dict(objective="poisson", learning_rate=0.08, num_leaves=31, min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8,\n          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)\nPS = dict(objective="gamma", learning_rate=0.05, num_leaves=15, min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,\n          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)\ndef fit(X, y, off, p, val, w=None, rounds=2000):\n    dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], weight=None if w is None else w[~val], free_raw_data=False)\n    dvl = lgb.Dataset(X[val], y[val], init_score=off[val], weight=None if w is None else w[val], reference=dtr)\n    b = lgb.train(p, dtr, rounds, valid_sets=[dvl], callbacks=[lgb.early_stopping(50, verbose=False)])\n    n = max(b.best_iteration, 1); b = lgb.train(p, lgb.Dataset(X, y, init_score=off, weight=w), n); return b, n\nout = pd.DataFrame({"row_id": em.row_id.values}); meta = {}\nfor p in perils:\n    n = dv[f"n_{p}"].values.astype(float); yp = dv[f"y_{p}"].values\n    cp = np.maximum(por_peligro(linea, p, dv.period_start, dv.period_end), 1e-4)\n    off = np.log(dv.exposure.values * cp); b0 = float(np.log(n.sum() / np.exp(off).sum())); off = off + b0\n    bf, nf = fit(Xd, n, off, PF, val)\n    (TRABAJO / "modelos/fsp_dev").mkdir(parents=True, exist_ok=True)\n    bf.save_model(str(TRABAJO / "modelos/fsp_dev" / f"f_{linea}_{p}.txt"))\n    s = (n > 0) & (yp > 0); msev = float(yp[s].sum() / n[s].sum())\n    bs, ns = fit(Xd[s].reset_index(drop=True), yp[s] / n[s], np.full(s.sum(), np.log(msev)), PS, val[s], w=n[s])\n    bs.save_model(str(TRABAJO / "modelos/fsp_dev" / f"s_{linea}_{p}.txt"))\n    guardar_json({"b0": b0, "msev": msev, "nf": nf, "ns": ns}, TRABAJO / "modelos/fsp_dev" / f"meta_{linea}_{p}.json")\n    out[f"f_{p}"] = np.exp(bf.predict(Xe, raw_score=True) + b0) * em.exposure.values\n    out[f"s_{p}"] = np.exp(bs.predict(Xe, raw_score=True) + np.log(msev))\n    out[f"cp_{p}"] = por_peligro(linea, p, em.period_start, em.period_end)\n    meta[p] = {"b0": b0, "rondas_f": nf, "rondas_s": ns, "n_dev": int(n.sum()), "sev_media": msev}\n    print(linea, p, meta[p], round(time.time() - t0), flush=True)\n(TRABAJO / "pred" / "fsp").mkdir(parents=True, exist_ok=True)\nout.to_parquet(TRABAJO / "pred" / "fsp" / f"{linea}.parquet", index=False)\njson.dump(meta, open(TRABAJO / "pred" / "fsp" / f"{linea}_meta.json", "w"), indent=1)\nprint("fin", linea, round(time.time() - t0))\n'


### Embedded source: `ml/codigo/fsp_light_prod.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['ml/codigo/fsp_light_prod.py'] = '"""Fit production peril Poisson frequency and gamma severity on developed 2021-2024.\nUse the common T2b schema, weather through 2024 and learning rate 0.08. Predict\nemerging target rows and save row_id, ultimate expected counts f_<p>, severity\ns_<p> and completeness cp_<p> to trabajo/pred/fsp/<line>_prod.parquet.\n"""\nimport sys, time, json\nimport numpy as np, pandas as pd, lightgbm as lgb\nfrom comun import *\nfrom variables import *\nfrom completitud import por_peligro\nlinea = sys.argv[1]\nVENT, ANIOS, SUFX = "hasta2024", [2021, 2022, 2023, 2024], "_prod"\nHIST = ["hist_pol_12m", "hist_cli_12m"]\ncols = comunes(linea)\nt0 = time.time()\nem = cargar("emerging", linea, VENT)\ndva = cargar("developed", linea, VENT)\ndv = dva[dva.year.isin(ANIOS)].reset_index(drop=True); del dva\ncn = pd.read_parquet(TRABAJO / f"conteos_developed_{linea}.parquet")\ndv = dv.merge(cn, on="row_id", how="left")\ncats = categorias_union([dv, em], cols + HIST + ["new_business", "renewal_count", "prior_claims_5y"])\nXd = matriz(dv, cols, cats); Xe = matriz(em, cols, cats); val = dv.fold.values == 0\nperils = sorted(c[2:] for c in cn.columns if c.startswith("n_"))\nPF = dict(objective="poisson", learning_rate=0.08, num_leaves=31, min_data_in_leaf=1000, feature_fraction=0.8, bagging_fraction=0.8,\n          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)\nPS = dict(objective="gamma", learning_rate=0.05, num_leaves=15, min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,\n          bagging_freq=1, lambda_l2=10.0, max_cat_to_onehot=8, cat_smooth=50, verbose=-1, num_threads=2, seed=20261026)\ndef fit(X, y, off, p, val, w=None, rounds=2000):\n    dtr = lgb.Dataset(X[~val], y[~val], init_score=off[~val], weight=None if w is None else w[~val], free_raw_data=False)\n    dvl = lgb.Dataset(X[val], y[val], init_score=off[val], weight=None if w is None else w[val], reference=dtr)\n    b = lgb.train(p, dtr, rounds, valid_sets=[dvl], callbacks=[lgb.early_stopping(50, verbose=False)])\n    n = max(b.best_iteration, 1); b = lgb.train(p, lgb.Dataset(X, y, init_score=off, weight=w), n); return b, n\nout = pd.DataFrame({"row_id": em.row_id.values}); meta = {}\nfor p in perils:\n    n = dv[f"n_{p}"].values.astype(float); yp = dv[f"y_{p}"].values\n    cp = np.maximum(por_peligro(linea, p, dv.period_start, dv.period_end), 1e-4)\n    off = np.log(dv.exposure.values * cp); b0 = float(np.log(n.sum() / np.exp(off).sum())); off = off + b0\n    bf, nf = fit(Xd, n, off, PF, val)\n    (TRABAJO / "modelos/fsp_prod").mkdir(parents=True, exist_ok=True)\n    bf.save_model(str(TRABAJO / "modelos/fsp_prod" / f"f_{linea}_{p}.txt"))\n    s = (n > 0) & (yp > 0); msev = float(yp[s].sum() / n[s].sum())\n    bs, ns = fit(Xd[s].reset_index(drop=True), yp[s] / n[s], np.full(s.sum(), np.log(msev)), PS, val[s], w=n[s])\n    bs.save_model(str(TRABAJO / "modelos/fsp_prod" / f"s_{linea}_{p}.txt"))\n    guardar_json({"b0": b0, "msev": msev, "nf": nf, "ns": ns}, TRABAJO / "modelos/fsp_prod" / f"meta_{linea}_{p}.json")\n    out[f"f_{p}"] = np.exp(bf.predict(Xe, raw_score=True) + b0) * em.exposure.values\n    out[f"s_{p}"] = np.exp(bs.predict(Xe, raw_score=True) + np.log(msev))\n    out[f"cp_{p}"] = por_peligro(linea, p, em.period_start, em.period_end)\n    meta[p] = {"b0": b0, "rondas_f": nf, "rondas_s": ns, "n_dev": int(n.sum()), "sev_media": msev}\n    print(linea, p, meta[p], round(time.time() - t0), flush=True)\n(TRABAJO / "pred" / "fsp").mkdir(parents=True, exist_ok=True)\nout.to_parquet(TRABAJO / "pred" / "fsp" / f"{linea}{SUFX}.parquet", index=False)\njson.dump(meta, open(TRABAJO / "pred" / "fsp" / f"{linea}{SUFX}_meta.json", "w"), indent=1)\nprint("fin", linea, round(time.time() - t0))\n'


### Embedded source: `m017/forma_local.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/forma_local.py'] = 'import numpy as np, pandas as pd, json\nP = 1.5\ndef dev(y, m):\n    return 2 * (np.maximum(y, 0) ** (2 - P) / ((1 - P) * (2 - P)) - y * m ** (1 - P) / (1 - P) + m ** (2 - P) / (2 - P))\ndef kopt(y, m):  # Tweedie-optimal power-1.5 scalar.\n    return np.sum(y / np.sqrt(m)) / np.sum(np.sqrt(m))\nb = pd.read_parquet("base_etiquetas.parquet").set_index("row_id")\ndef cargar(conj, mod):\n    d = pd.concat([pd.read_parquet(f"oof/pred/{conj}/{mod}_{l}.parquet") for l in ["motor", "buildings", "contents"]]).set_index("row_id")\n    t = b.loc[d.index].copy(); t["m"] = d.mu.values * t.c_fila.values; return t\np24 = pd.read_parquet("in/predicciones_2024.parquet").set_index("row_id")\nsets = {"DEV23_T2b": cargar("dev", "T2b"), "HOLD24_T2b": cargar("hold", "T2b")}\nfor c in ["A1", "R0"]:\n    t = b.loc[p24.index].copy(); t["m"] = p24[c].values; sets[f"HOLD24_{c}"] = t\nfor k, t in sets.items():\n    print(k, len(t), "y", round(t.y.sum()), "m", round(t.m.sum()), "ratio", round(t.y.sum() / t.m.sum(), 3))\nA = np.round(np.arange(0.6, 1.41, 0.05), 2)\nres = {}\nfor k, t in sets.items():\n    t = t.copy(); t["cel"] = t.line + "_" + t.new_business.astype(int).astype(str)\n    y = t.y.values; null = dev(y, np.full(len(y), y.mean())).sum(); res[k] = {}\n    base_dev = 0.0; best_dev = 0.0\n    for c, g in t.groupby("cel"):\n        yy = g.y.values; e = g.exposure.values * g.c_fila.values; r = g.m.values / e; gm = np.sum(g.m.values) / np.sum(e)\n        devs = []\n        for a in A:\n            m = e * gm * (r / gm) ** a; m = m * kopt(yy, m); devs.append(dev(yy, m).sum())\n        devs = np.array(devs); i1 = int(np.where(A == 1.0)[0][0]); ib = int(devs.argmin())\n        base_dev += devs[i1]; best_dev += devs[ib]\n        res[k][c] = {"n": len(g), "positivos": int((yy > 0).sum()), "a_opt": float(A[ib]), "ganancia_D2": round(float((devs[i1] - devs[ib]) / null), 5)}\n    res[k]["_total_ganancia_D2"] = round(float((base_dev - best_dev) / null), 5)\nprint(json.dumps(res, indent=1))\n'


### Embedded source: `m017/forma_local3.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/forma_local3.py'] = 'import numpy as np, pandas as pd, json\nexec(open("forma_local.py").read().split("A = np.round")[0].replace(\'print(k, len(t)\', \'0 and print(k, len(t)\'))\nte = b[b.conjunto == "test"].copy(); te["cel"] = te.line + "_" + te.new_business.astype(int).astype(str); n25 = te.cel.value_counts(normalize=True)\n# 2024 proxy: average A1 and T2b after matching each line\'s prediction total.\nh = sets["HOLD24_T2b"].copy(); a1 = sets["HOLD24_A1"].m.reindex(h.index)\ntot = h.groupby("line").m.transform("sum") / a1.groupby(h.line).transform("sum")\nh["m"] = 0.5 * h.m + 0.5 * a1 * tot; sets["HOLD24_mezcla"] = h\ndef aplicar(t, acel):\n    t = t.copy(); t["cel"] = t.line + "_" + t.new_business.astype(int).astype(str); out = t.m.values.copy()\n    for c, a in acel.items():\n        i = (t.cel == c).values; e = t.exposure.values[i] * t.c_fila.values[i]; r = t.m.values[i] / e; gm = t.m.values[i].sum() / e.sum()\n        mm = e * gm * (r / gm) ** a; out[i] = mm * t.m.values[i].sum() / mm.sum()\n    return out\ndef d2w(t, m, w):\n    y = t.y.values; return 1 - (dev(y, m) * w).sum() / (dev(y, np.full(len(y), y.mean())) * w).sum()\nREGLA = {"motor_1": 0.5, "motor_0": 0.85}\nout = {}\nfor k, t in sets.items():\n    if k == "HOLD24_R0": continue\n    cel = t.line + "_" + t.new_business.astype(int).astype(str); n = cel.value_counts(normalize=True); w = cel.map(n25 / n).values\n    # Match both predictions to their cell-optimal levels to inspect shape alone.\n    def nivel(m):\n        m = m.copy()\n        for c in cel.unique():\n            i = (cel == c).values; m[i] = m[i] * kopt(t.y.values[i], m[i])\n        return m\n    m0 = nivel(t.m.values); m1 = nivel(aplicar(t, REGLA))\n    out[k] = {"D2_base_comp2025": round(d2w(t, m0, w), 5), "D2_forma_comp2025": round(d2w(t, m1, w), 5),\n              "ganancia_comp2025": round(d2w(t, m1, w) - d2w(t, m0, w), 5),\n              "ganancia_comp_propia": round(d2w(t, m1, np.ones(len(t))) - d2w(t, m0, np.ones(len(t))), 5)}\nprint(json.dumps(out, indent=1))\n# Inspect observed/expected by motor NCB and business status using 2023-2024 T2b.\npm = pd.concat([pd.read_parquet(f"datos/emerging/policies_motor.parquet", columns=["row_id", "ncb_years", "years_licensed", "driver_age_band"])]).set_index("row_id")\nt = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]); t = t[t.line == "motor"].join(pm)\nt["ncb"] = pd.cut(t.ncb_years, [-1, 0, 2, 5, 9, 99], labels=["0", "1-2", "3-5", "6-9", "10+"])\nt["dec"] = t.groupby("new_business").m.transform(lambda s: pd.qcut(s / t.loc[s.index, "exposure"], 5, labels=False))\nfor nb in (False, True):\n    g = t[t.new_business == nb]; k = g.y.sum() / g.m.sum()\n    print("NB" if nb else "REN", "A/E por quintil de tasa predicha:", (g.groupby("dec").y.sum() / g.groupby("dec").m.sum() / k).round(2).tolist())\n    print("   A/E por NCB:", (g.groupby("ncb", observed=True).y.sum() / g.groupby("ncb", observed=True).m.sum() / k).round(2).to_dict(), g.groupby("ncb", observed=True).size().to_dict())\n'


### Embedded source: `m017/forma_v2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/forma_v2.py'] = '"""Inspect motor shape using the correct Tweedie scalar and 2025 cell/exposure-band composition."""\nimport numpy as np, pandas as pd, json\nexec(open("forma_local3.py").read().split("REGLA =")[0])\ndef banda(e): return np.where(e < 0.5, "c", np.where(e < 0.9, "m", "l"))\nte = b[b.conjunto == "test"]; k25 = (te.line + "_" + te.new_business.astype(int).astype(str) + "_" + banda(te.exposure.values)).value_counts(normalize=True)\ndef pesos(t):\n    k = t.line + "_" + t.new_business.astype(int).astype(str) + "_" + banda(t.exposure.values); n = k.value_counts(normalize=True)\n    return k.map(k25 / n).fillna(0).values\nA = np.round(np.arange(0.2, 1.31, 0.05), 2)\ndef nivel_celda(t, m):\n    cel = (t.line + "_" + t.new_business.astype(int).astype(str)).values; m = m.copy()\n    for c in np.unique(cel): i = cel == c; m[i] *= kopt(t.y.values[i], m[i])\n    return m\ndef d2w(t, m, w):\n    y = t.y.values; mu0 = np.average(y, weights=w); return 1 - (dev(y, m) * w).sum() / (dev(y, np.full(len(y), mu0)) * w).sum()\nout = {}\nfor k in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:\n    t = sets[k]; w = pesos(t); r = {}\n    for c in ["motor_1", "motor_0"]:\n        cel = (t.line + "_" + t.new_business.astype(int).astype(str)).values; i = cel == c; s = t[i]; ww = w[i]\n        yy = s.y.values; e = (s.exposure * s.c_fila).values; rr = s.m.values / e; gm = s.m.values.sum() / e.sum()\n        def curva(mask, wt):\n            return np.array([(dev(yy[mask], (lambda mm: mm * kopt(yy[mask], mm))(e[mask] * gm * (rr[mask] / gm) ** a)) * wt[mask]).sum() for a in A])\n        todo = curva(np.ones(len(s), bool), np.ones(len(s))); largo = curva(s.exposure.values >= 0.9, np.ones(len(s))); rep = curva(np.ones(len(s), bool), ww)\n        r[c] = {"a_opt_todas": float(A[todo.argmin()]), "a_opt_expo>=0.9": float(A[largo.argmin()]), "a_opt_reponderado2025": float(A[rep.argmin()]),\n                "positivos_expo>=0.9": int(((yy > 0) & (s.exposure.values >= 0.9)).sum())}\n    base = d2w(t, nivel_celda(t, t.m.values), w)\n    for a1, a0 in ((0.5, 0.85), (0.6, 0.9), (0.7, 0.9), (0.75, 0.9), (0.8, 0.9), (0.85, 0.9)):\n        r[f"ganancia_reponderada_a{a1}_{a0}"] = round(d2w(t, nivel_celda(t, aplicar(t, {"motor_1": a1, "motor_0": a0})), w) - base, 5)\n    out[k] = r\nprint(json.dumps(out, indent=1)); json.dump(out, open("forma_v2.json", "w"), indent=1)\n'


### Embedded source: `m017/eval_v2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/eval_v2.py'] = 'import numpy as np, pandas as pd, json\nexec(open("forma_v2.py").read().split("out = {}")[0])\nexec(open("ncb_local.py").read().split("def evalua")[0].split("exec(")[0] if False else "")\npm = pd.read_parquet("datos/emerging/policies_motor.parquet", columns=["row_id", "ncb_years"]).set_index("row_id")\ntm = pd.read_parquet("datos/emerging/test_motor.parquet", columns=["row_id", "ncb_years"]).set_index("row_id")\nncb = pd.concat([pm, tm]).ncb_years\ncu = pd.read_parquet("datos/emerging/customers.parquet").set_index("customer_id")\ndef bnd(v): return pd.cut(v, [-1, 0, 2, 5, 9, 99], labels=["0", "1-2", "3-5", "6-9", "10+"]).astype(str)\ndef celda(t): return (t.line + "_" + t.new_business.astype(int).astype(str)).values\ndef fac(t, clave, k=300):\n    c = celda(t); f = {}\n    for cc in np.unique(c):\n        i = c == cc; base = t.y.values[i].sum() / t.m.values[i].sum(); kk = clave(t)[i]\n        for v in np.unique(kk):\n            j = kk == v; n = (t.y.values[i][j] > 0).sum(); ae = t.y.values[i][j].sum() / t.m.values[i][j].sum() / base\n            f[(cc, v)] = 1 + n / (n + k) * (ae - 1)\n    return f\ndef aplica(t, f, clave, m=None):\n    m = t.m.values.copy() if m is None else m.copy(); c = celda(t); kk = clave(t); m0 = m.copy()\n    m = m * np.array([f.get((a, v), 1.0) for a, v in zip(c, kk)])\n    for cc in np.unique(c): i = c == cc; m[i] *= m0[i].sum() / m[i].sum()\n    return m\nk_ncb = lambda t: np.where(t.line.values == "motor", bnd(ncb.reindex(t.index).fillna(0).values), "x")\nk_pri = lambda t: np.minimum(cu.prior_claims_5y.reindex(t.customer_id.values).fillna(0).values, 2).astype(int).astype(str)\nres = {}\nfor tr_k, te_k in (("DEV23_T2b", "HOLD24_T2b"), ("DEV23_T2b", "HOLD24_A1"), ("DEV23_T2b", "HOLD24_mezcla"), ("HOLD24_T2b", "DEV23_T2b")):\n    tr, te = sets[tr_k], sets[te_k]; w = pesos(te)\n    f1 = fac(tr, k_ncb); f2 = fac(tr, k_pri)\n    m_n = aplica(te, f1, k_ncb); t2 = te.copy(); t2["m"] = m_n; m_np = aplica(t2, f2, k_pri)\n    base = d2w(te, nivel_celda(te, te.m.values), w); r = {"base": round(base, 5)}\n    r["ncb"] = round(d2w(te, nivel_celda(te, m_n), w) - base, 5)\n    r["ncb+prior"] = round(d2w(te, nivel_celda(te, m_np), w) - base, 5)\n    t3 = te.copy(); t3["m"] = m_np\n    for a1 in (0.6, 0.7, 0.8, 0.9, 1.0):\n        r[f"ncb+prior+forma({a1},0.85)"] = round(d2w(te, nivel_celda(te, aplicar(t3, {"motor_1": a1, "motor_0": 0.85})), w) - base, 5)\n    res[f"{tr_k}->{te_k}"] = r\nprint(json.dumps(res, indent=1)); json.dump(res, open("eval_v2.json", "w"), indent=1)\n'


### Embedded source: `m017/f3_eval.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/f3_eval.py'] = '"""F3: out-of-time residual factors over the F2 transformation.\nInspect emerging-only or T2b-excluded features, including gender, prior insurer,\noccupation and marketing segment, using 2025 cell/exposure-band composition.\n"""\nimport numpy as np, pandas as pd, json, sys\nexec(open("eval_v2.py").read().split("res = {}")[0])\nDE = "datos/emerging"\npol = pd.concat([pd.concat([pd.read_parquet(f"{DE}/policies_{l}.parquet"), pd.read_parquet(f"{DE}/test_{l}.parquet")]) for l in ["motor","buildings","contents"]], ignore_index=True).set_index("row_id")\nad = pd.read_parquet(f"{DE}/addresses.parquet").set_index("address_id")\ndef col(t, c):\n    return np.array([str(x) for x in _col(t, c)], dtype=object)\ndef _col(t, c):\n    if c in cu.columns: return cu[c].reindex(t.customer_id.values).astype(str).values\n    if c in pol.columns: return pol[c].reindex(t.index).astype(str).values\n    if c in ad.columns: return ad[c].reindex(pol.address_id.reindex(t.index).values).astype(str).values\n    if c == "occ": return cu.occupation.reindex(t.customer_id.values).astype(str).str.lower().str.replace("self-employed ", "", regex=False).str.strip().values\n    raise KeyError(c)\nFORMA = {"motor_0": 0.9}\ndef f2_fit(pool):\n    f1 = fac(pool, k_ncb); t2 = pool.copy(); t2["m"] = aplica(pool, f1, k_ncb); f2 = fac(t2, k_pri); return f1, f2\ndef f2_apply(t, f1, f2):\n    m = aplica(t, f1, k_ncb); tt = t.copy(); tt["m"] = m; m = aplica(tt, f2, k_pri); tt["m"] = m\n    return aplicar(tt, FORMA)\ndef fac_linea(t, clave, lines, k=300):\n    """Estimate a line factor across business statuses relative to each cell\'s base."""\n    c = celda(t); kk = clave(t); f = {}\n    rel = t.m.values.copy()\n    for cc in np.unique(c):\n        i = c == cc; rel[i] = rel[i] * t.y.values[i].sum() / t.m.values[i].sum()\n    for l in lines:\n        i = t.line.values == l\n        for v in np.unique(kk[i]):\n            j = i & (kk == v); n = (t.y.values[j] > 0).sum(); ae = t.y.values[j].sum() / rel[j].sum()\n            f[(l, v)] = 1 + n / (n + k) * (ae - 1)\n    return f\ndef aplica_linea(t, f, clave, m):\n    c = celda(t); kk = clave(t); m0 = m.copy()\n    m = m * np.array([f.get((l, v), 1.0) for l, v in zip(t.line.values, kk)])\n    for cc in np.unique(c): i = c == cc; m[i] *= m0[i].sum() / m[i].sum()\n    return m\nK = 300\nCANDS = {"gender": ["motor", "contents"], "gender_b": ["buildings"], "previous_insurer": ["motor"], "occ": ["motor"],\n         "marketing_segment": ["contents"], "licence_points": ["buildings"], "policyholder_age_band": ["contents"],\n         "marketing_consent": ["contents"], "payment_method": ["buildings"], "region": ["motor"], "campaign_code": ["contents", "buildings"]}\ndef clave_de(v):\n    vv = "gender" if v == "gender_b" else v\n    return lambda t: col(t, vv)\nres = {}\nfor tr_k, te_k in (("DEV23_T2b", "HOLD24_T2b"), ("DEV23_T2b", "HOLD24_A1"), ("DEV23_T2b", "HOLD24_mezcla"), ("HOLD24_T2b", "DEV23_T2b")):\n    tr, te = sets[tr_k].copy(), sets[te_k].copy(); w = pesos(te)\n    f1, f2 = f2_fit(tr); tr["m"] = f2_apply(tr, f1, f2); te["m"] = f2_apply(te, f1, f2)\n    base = d2w(te, nivel_celda(te, te.m.values), w); r = {"base_F2": round(base, 5)}\n    for v, lines in CANDS.items():\n        cl = clave_de(v); f = fac_linea(tr, cl, lines, K); m = aplica_linea(te, f, cl, te.m.values)\n        r[v] = round(d2w(te, nivel_celda(te, m), w) - base, 5)\n    # Sequential factors: gender, previous_insurer, occupation, marketing_segment.\n    seq = ["gender", "previous_insurer", "occ", "marketing_segment"]\n    trs, m_te = tr.copy(), te.m.values.copy()\n    for v in seq:\n        cl = clave_de(v); f = fac_linea(trs, cl, CANDS[v], K); trs["m"] = aplica_linea(trs, f, cl, trs.m.values); m_te = aplica_linea(te, f, cl, m_te)\n        r["seq+" + v] = round(d2w(te, nivel_celda(te, m_te), w) - base, 5)\n    res[f"{tr_k}->{te_k}"] = r; print(f"{tr_k}->{te_k}", r, flush=True)\njson.dump(res, open("exp26/f3_eval.json", "w"), indent=1)\n'


### Embedded source: `m017/construir_F2.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F2.py'] = '"""Build frozen F2 from the recorded F blend and its 1.1877 level factor.\nEstimate motor NCB and all-cell prior_claims_5y factors from pooled 2023-2024 T2b\nOOF, separately by business status, with n/(n+300) credibility and fixed cell\ntotals. Apply motor-renewal shape 0.9 and the recorded correct-scalar local\nTweedie level ratios measured across four series.\n"""\nimport numpy as np, pandas as pd, json, hashlib\nexec(open("eval_v2.py").read().split("res = {}")[0])\nFORMA = {"motor_0": 0.9}\npool = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]); f1 = fac(pool, k_ncb)\nt2 = pool.copy(); t2["m"] = aplica(pool, f1, k_ncb); f2 = fac(t2, k_pri)\ndef transformar(t):\n    m = aplica(t, f1, k_ncb); tt = t.copy(); tt["m"] = m; m = aplica(tt, f2, k_pri); tt["m"] = m\n    return aplicar(tt, FORMA)\ncels = ["motor_0", "motor_1", "buildings_0", "buildings_1", "contents_0", "contents_1"]\nraz = {}\nfor k in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:\n    t = sets[k]; m2 = transformar(t); c = celda(t)\n    raz[k] = {cc: kopt(t.y.values[c == cc], m2[c == cc]) / kopt(t.y.values[c == cc], t.m.values[c == cc]) for cc in cels}\nR = {cc: float(np.mean([raz[k][cc] for k in raz])) for cc in cels}\nF = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); pass\nt = b.loc[F.row_id].copy(); t["m"] = F.prediction.values; t["c_fila"] = 1.0\nm2 = transformar(t); c = celda(t)\nfor cc, r in R.items(): m2[c == cc] *= r\nout = F.copy(); out["prediction"] = m2\nassert len(out) == 209063 and np.isfinite(out.prediction).all() and (out.prediction > 0).all() and (out.row_id.values == F.row_id.values).all()\np = "F2_ncb_previos_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")\ninfo = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "base": "F 42659dbb…", "forma": FORMA,\n        "factores_ncb": {f"{a}|{v}": round(x, 4) for (a, v), x in f1.items() if a.startswith("motor")},\n        "factores_previos": {f"{a}|{v}": round(x, 4) for (a, v), x in f2.items()},\n        "razon_nivel": {k: round(v, 4) for k, v in R.items()}, "razon_por_serie": {k: {c_: round(v, 4) for c_, v in d.items()} for k, d in raz.items()},\n        "total": round(float(m2.sum()), 2), "total_F": round(float(F.prediction.sum()), 2),\n        "total_celda": {cc: round(float(m2[c == cc].sum()), 2) for cc in cels}, "total_F_celda": {cc: round(float(F.prediction.values[c == cc].sum()), 2) for cc in cels},\n        "evidencia_fuera_de_tiempo": "eval_v2.json: NCB+previos de 2023 aplicados en 2024 con la composición 2025 por celda y banda de exposición: +0,0029 a +0,0031 (T2b, A1, mezcla); de 2024 en 2023: +0,0093"}\njson.dump(info, open("F2_info.json", "w"), indent=1, ensure_ascii=False); print(json.dumps(info, indent=1, ensure_ascii=False))\n'


### Embedded source: `m017/construir_final.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_final.py'] = '"""Reconstruct the credibility-calibrated final base from archived public receipts.\nUse frozen cell-level probes and preregistered stopping rules. For new-business\nshape, shrink a concave quadratic vertex toward 1 using recorded Z (tau=0.25),\nonly when the vertex lies in [0.3,1.8]; otherwise keep 1. Apply F2 cell shapes\nand the credible cell factor relative to F. Save credible and unshrunk private\nCSVs plus the aggregate calculation record; do not request new probes.\nUsage: python construir_final.py <receipts.json> [--prueba]\n"""\nimport sys, json, hashlib\nimport numpy as np, pandas as pd\nexec(open("construir_F2.py").read().split("F = pd.read_csv")[0])\npre = json.load(open("in/preinscripcion.json")); B = pre["base"]; CEL = pre["celdas"]; DED = pre["celda_deducida"]\nSF = json.load(open("sondas_F2_info.json")); SIG = json.load(open("sigma_forma.json"))\nK, LO, HI = 1.2, 0.75, 1.33\nnombre_sha = {B["sha256"]: ("F", None)}; nombre_sha.update({v["sha256"]: ("celda", v["celda"]) for v in pre["sondas"].values()})\nnombre_sha[SF["base"]["sha256"]] = ("F2", None); nombre_sha.update({v["sha256"]: ("forma", (v["celda"], v["a"])) for v in SF["sondas"].values()})\ndef resolver(sc):\n    bA, bB, g, cg = B["beta_A"], B["beta_B"], B["g"], B["c_optimo_global"]; dF = sc["F"]\n    b_s = {s: bB * CEL[s]["cuota_raiz_mu"] for s in CEL}; a_s = {}\n    for s in CEL:\n        if s != DED: a_s[s] = (dF - sc[("celda", s)] - b_s[s] * ((g * K) ** 0.5 - g ** 0.5)) / ((g * K) ** -0.5 - g ** -0.5)\n    a_s[DED] = bA - sum(a_s.values())\n    assert all(v > 0 for v in a_s.values()), ("a_s no positivo", a_s)\n    cs = {s: a_s[s] / b_s[s] for s in CEL}\n    grande = [s for s in CEL if abs(np.log(cs[s] / cg)) > max(0.7, 3 * CEL[s]["sigma"])]\n    cred = {s: float(np.clip((cs[s] / cg) ** CEL[s]["Z"], LO, HI)) * g for s in CEL}\n    pleno = {s: float(np.clip(cs[s], g * LO, g * HI)) for s in CEL}\n    prev = lambda f: dF - sum(a_s[s] * (f[s] ** -0.5 - g ** -0.5) + b_s[s] * (f[s] ** 0.5 - g ** 0.5) for s in CEL)\n    return {"c_opt": cs, "rel_global": {s: cs[s] / cg for s in CEL}, "cred": cred, "pleno": pleno, "grande": grande,\n            "d2_F_celda_cred": prev(cred), "d2_F_celda_pleno": prev(pleno), "g": g}\ndef parabola(pts):\n    a = np.array([p[0] for p in pts]); d = np.array([p[1] for p in pts]); c2, c1, c0 = np.polyfit(a, d, 2)\n    if c2 >= 0: return None, (c2, c1, c0)\n    v = -c1 / (2 * c2); return (v if 0.3 <= v <= 1.8 else None), (c2, c1, c0)\ndef construir(sc, base_df, t, sets_eval=None):\n    lv = resolver(sc); out = {"nivel": lv, "forma": {}}\n    for cc in ["motor_1", "buildings_1", "contents_1"]:\n        pts = [(1.0, sc["F2"])] + [(a, sc[("forma", (cc, a))]) for (c_, a) in [k[1] for k in sc if isinstance(k, tuple) and k[0] == "forma"] if c_ == cc]\n        if len(pts) < 3: out["forma"][cc] = {"a_final": 1.0, "nota": "faltan puntos"}; continue\n        v, coef = parabola(pts); Z = SIG[cc]["Z_tau0.25"]\n        out["forma"][cc] = {"puntos": pts, "vertice": v, "coef": coef, "Z": Z,\n                            "a_cred": 1.0 if v is None else 1 + Z * (v - 1), "a_pleno": 1.0 if v is None else v,\n                            "d2_vertice": None if v is None else float(np.polyval(coef, v))}\n    res = {}\n    for modo in ("cred", "pleno"):\n        m = base_df.prediction.values.copy(); c = celda(t)\n        for cc, fz in out["forma"].items():\n            a = fz.get(f"a_{modo}", 1.0)\n            if a != 1.0:\n                tt = t.copy(); tt["m"] = m; m2 = aplicar(tt, {cc: a}); i = c == cc\n                rs = []\n                for k_, s_ in ser.items():\n                    cs_ = celda(s_); j = cs_ == cc; mm = aplicar(s_, {cc: a}); rs.append(kopt(s_.y.values[j], mm[j]) / kopt(s_.y.values[j], s_.m.values[j]))\n                m[i] = m2[i] * float(np.mean(rs)); fz[f"razon_{modo}"] = float(np.mean(rs))\n        f = lv[modo]\n        for cc in CEL: m[c == cc] *= f[cc] / lv["g"]\n        res[modo] = m\n    return out, res\nser = {}\nfor k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:\n    s_ = sets[k_].copy(); s_["m"] = transformar(s_); ser[k_] = s_\nif __name__ == "__main__" and "--prueba" not in sys.argv:\n    sc = {}\n    for r in json.load(open(sys.argv[1])):\n        if r["sha256"] not in nombre_sha: print("AVISO: SHA desconocido, se ignora:", r["sha256"][:12]); continue\n        tipo, cual = nombre_sha[r["sha256"]]; sc[tipo if cual is None else (tipo, cual)] = float(r["d2"])\n    falta = ({"F", "F2"} | {("celda", s) for s in CEL if s != DED}) - set(sc)\n    if falta: sys.exit(f"PARADA: faltan puntuaciones obligatorias {falta}")\n    falta_forma = {("forma", (v["celda"], v["a"])) for v in SF["sondas"].values()} - set(sc)\n    if falta_forma: print("AVISO: faltan sondas de forma (esa celda queda con a = 1):", falta_forma)\n    lo, hi = pre["d2_F_intervalo_redondeo"]\n    if sc["F"] + 5e-6 < lo - 1e-9 or sc["F"] - 5e-6 > hi + 1e-9: sys.exit(f"PARADA preinscrita: D2(F) = {sc[\'F\']} fuera de [{lo}, {hi}]")\n    pass\n    for v in SF["sondas"].values():\n        cc_, a_ = v["celda"], v["a"]; rs_ = [kopt(s_.y.values[celda(s_) == cc_], aplicar(s_, {cc_: a_})[celda(s_) == cc_]) / kopt(s_.y.values[celda(s_) == cc_], s_.m.values[celda(s_) == cc_]) for s_ in ser.values()]\n        assert abs(np.mean(rs_) - v["razon_nivel"]) < 6e-5, ("razón de forma distinta de la de las sondas", cc_, a_, np.mean(rs_))\n    F2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); t = b.loc[F2.row_id].copy(); t["m"] = F2.prediction.values; t["c_fila"] = 1.0\n    info, res = construir(sc, F2, t)\n    if info["nivel"]["grande"] and "--forzar" not in sys.argv:\n        print(json.dumps(info, indent=1, default=str)); sys.exit("PARADA preinscrita de nivel en " + str(info["nivel"]["grande"]))\n    for modo, m in res.items():\n        out = F2.copy(); out["prediction"] = m; assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()\n        p = f"final_{\'creible\' if modo == \'cred\' else \'pleno\'}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")\n        info[p] = {"sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "total": round(float(m.sum()), 2)}\n    json.dump(info, open("final_info.json", "w"), indent=1, default=str, ensure_ascii=False); print(json.dumps(info, indent=1, default=str, ensure_ascii=False))\n'


### Embedded source: `m017/exp26/lib26.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/exp26/lib26.py'] = 'import numpy as np, pandas as pd, json\nexec(open("__RUN_ROOT__/m017/f3_eval.py").read().split("K = 300")[0])\nW = pd.read_parquet("__RUN_ROOT__/ml/datos/emerging/weather.parquet").set_index(["address_id", "year"])\ndef wvar(t, v, year=None):\n    a = pol.address_id.reindex(t.index).values; yr = t.year.values.astype(int) if year is None else np.full(len(t), year)\n    return W[v].reindex(pd.MultiIndex.from_arrays([a, yr])).values\ndef d2u(t, m):\n    y = t.y.values; return 1 - dev(y, m).sum() / dev(y, np.full(len(y), y.mean())).sum()\ndef rel_base(t):\n    """Relevel m to its cell-specific Tweedie optimum."""\n    return nivel_celda(t, t.m.values)\ndef fac_tw(t, clave, lines, k=300, by_nb=False):\n    """Estimate a cell-relative Tweedie-optimal level factor with n/(n+k) log credibility."""\n    c = celda(t); kk = clave(t); m = rel_base(t); y = t.y.values; f = {}\n    for l in lines:\n        il = t.line.values == l\n        grupos = [("all", il)] if not by_nb else [(nb, il & (t.new_business.values == nb)) for nb in (False, True)]\n        for g, i in grupos:\n            for v in np.unique(kk[i]):\n                j = i & (kk == v); n = (y[j] > 0).sum()\n                if n == 0: continue\n                kv = np.sum(y[j] / np.sqrt(m[j])) / np.sum(np.sqrt(m[j]))\n                f[(l, g, v)] = np.exp(n / (n + k) * np.log(max(kv, 1e-3)))\n    return f\ndef aplica_tw(t, f, clave, m, by_nb=False):\n    c = celda(t); kk = clave(t); m0 = m.copy()\n    g = t.new_business.values if by_nb else np.array(["all"] * len(t), dtype=object)\n    m = m * np.array([f.get((l, gg, v), 1.0) for l, gg, v in zip(t.line.values, g, kk)])\n    for cc in np.unique(c): i = c == cc; m[i] *= m0[i].sum() / m[i].sum()\n    return m\n'


### Embedded source: `m017/exp26/fsp_blend_eval.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/exp26/fsp_blend_eval.py'] = 'exec(open("__RUN_ROOT__/m017/exp26/lib26.py").read())\nimport sys\nT = "__RUN_ROOT__/ml/trabajo"\nKP = 100.0\ndef fsp_tables():\n    out = {}\n    for l in ["motor", "buildings", "contents"]:\n        f = pd.read_parquet(f"{T}/pred/fsp/{l}.parquet").set_index("row_id")\n        per = [c[2:] for c in f.columns if c.startswith("f_")]\n        cn = pd.read_parquet(f"{T}/conteos_emerging_{l}.parquet").set_index("row_id")\n        be = pd.read_parquet(f"{T}/base_emerging_{l}.parquet", columns=["row_id", "y", "year", "conjunto"] + [f"y_{p}" for p in per]).set_index("row_id")\n        out[l] = (f, per, cn, be)\n    return out\nFT = fsp_tables()\ndef kappas(l, years):\n    f, per, cn, be = FT[l]; tr = be[(be.conjunto == "train") & be.year.astype(int).isin(years)].index\n    ef = {p: (f.loc[tr, f"f_{p}"] * f.loc[tr, f"cp_{p}"]).values for p in per}; es = {p: f.loc[tr, f"s_{p}"].values for p in per}\n    single = sum(ef[p] * es[p] for p in per); kg = be.loc[tr, "y"].sum() / single.sum(); kap = {}\n    for p in per:\n        n = cn.loc[tr, f"n_{p}"].values; kf = n.sum() / ef[p].sum(); ks = be.loc[tr, f"y_{p}"].sum() / max((n * es[p]).sum(), 1e-9)\n        z = n.sum() / (n.sum() + KP); kap[p] = float(np.exp(z * np.log(max(kf * ks / kg, 1e-3))))\n    return kap\ndef fsp_pred(idx, l, kap, ultimate=False):\n    f, per, cn, be = FT[l]\n    return sum(kap[p] * f.loc[idx, f"f_{p}"].values * (1.0 if ultimate else f.loc[idx, f"cp_{p}"].values) * f.loc[idx, f"s_{p}"].values for p in per)\ndef blend(t, m, w, kap_by_line):\n    out = m.copy()\n    for l in ["motor", "buildings", "contents"]:\n        i = (t.line == l).values\n        if w[l] == 0: continue\n        r = fsp_pred(t.index[i], l, kap_by_line[l])\n        r = r * m[i].sum() / r.sum()\n        out[i] = np.exp((1 - w[l]) * np.log(m[i]) + w[l] * np.log(r))\n    c = celda(t)\n    for cc in np.unique(c): j = c == cc; out[j] *= m[j].sum() / out[j].sum()\n    return out\nSEQ = [("gender", ["motor", "contents"]), ("works_from_home", ["contents"]), ("marketing_segment", ["contents"])]\nres = {}\nfor tr_k, te_k, tr_y in (("DEV23_T2b", "HOLD24_T2b", [2023]), ("DEV23_T2b", "HOLD24_mezcla", [2023]), ("DEV23_T2b", "HOLD24_A1", [2023]), ("HOLD24_T2b", "DEV23_T2b", [2024])):\n    tr, te = sets[tr_k].copy(), sets[te_k].copy(); w25 = pesos(te)\n    f1, f2 = f2_fit(tr); tr["m"] = f2_apply(tr, f1, f2); te["m"] = f2_apply(te, f1, f2)\n    kap = {l: kappas(l, tr_y) for l in ["motor", "buildings", "contents"]}\n    # Estimate factors on the training year\'s F2-transformed base.\n    FAC = []\n    trs = tr.copy()\n    for v, lines in SEQ:\n        cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(trs, cl, lines, 300); trs["m"] = aplica_tw(trs, f, cl, trs.m.values); FAC.append((cl, f))\n    def facs(t, m):\n        for cl, f in FAC: m = aplica_tw(t, f, cl, m)\n        return m\n    b0 = te.m.values\n    r = {}\n    for nm, W in (("w0", 0.0), ("w25", 0.25), ("w40", 0.4), ("w50", 0.5), ("w60", 0.6), ("w75", 0.75)):\n        for usefac in (False, True):\n            m = blend(te, b0, {"motor": W, "buildings": W, "contents": W}, kap)\n            if usefac: m = facs(te, m)\n            key = nm + ("+fac" if usefac else "")\n            mm = nivel_celda(te, m)\n            r[key + "|ALLu"] = round(d2u(te, mm) - d2u(te, nivel_celda(te, b0)), 5)\n            r[key + "|w2025"] = round(d2w(te, mm, w25) - d2w(te, nivel_celda(te, b0), w25), 5)\n            for l in ["motor", "buildings", "contents"]:\n                for seg, s in (("REN", ~te.new_business.values.astype(bool)), ("NB", te.new_business.values.astype(bool))):\n                    j = (te.line == l).values & s; tt = te[j]\n                    r[f"{key}|{l}_{seg}"] = round(d2u(tt, nivel_celda(tt, m[j])) - d2u(tt, nivel_celda(tt, b0[j])), 5)\n    res[f"{tr_k[:5]}->{te_k[:9]}"] = r\n    print(tr_k, te_k, flush=True)\ndf = pd.DataFrame(res); pd.set_option("display.width", 250); pd.set_option("display.max_rows", 500)\ndf.to_csv("__RUN_ROOT__/m017/exp26/fsp_blend_eval.csv")\nprint(df.to_string())\n'


### Embedded source: `m017/construir_F3b.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F3b.py'] = '"""Reconstruct the historical F3b alternative using recorded audits I and J.\nStart from F2. Apply the level-leakage-corrected motor new-business shape while\nholding the 2025 cell sum of sqrt(mu) fixed. Retain the validated motor gender\nand contents gender/home-working/marketing factors, excluding prior insurer\nand occupation under the corrected null comparison. Estimate factors from pooled\nT2b OOF after F2, with n/(n+K) credibility and fixed cell totals; calculate local\nlevel ratios using equal year weights, then apply credible cell levels.\nUsage: python construir_F3b.py [<K>] [--sin-factores] [--sin-forma] [--seq=...]\n"""\nimport sys, json, hashlib\nargv = sys.argv[:]; sys.argv = ["x", "--prueba"]\nimport numpy as np, pandas as pd\nexec(open("exp26/lib26.py").read())\nexec(open("construir_final.py").read())\nsys.argv = argv\npos = [a for a in sys.argv[1:] if not a.startswith("--")]\nKC = float(pos[0]) if pos else 300.0\nSIN_F = "--sin-factores" in sys.argv; SIN_A = "--sin-forma" in sys.argv\nSEQ_ALL = {"gender": ["motor", "contents"], "works_from_home": ["contents"], "marketing_segment": ["contents"],\n           "previous_insurer": ["motor"], "occ": ["motor"]}\nseqarg = [a for a in sys.argv if a.startswith("--seq=")]\nSEQN = seqarg[0][6:].split(",") if seqarg else ["gender", "works_from_home", "marketing_segment"]\nSEQ = [] if SIN_F else [(v, SEQ_ALL[v]) for v in SEQN]\nFUGA = json.load(open("exp26/fuga_celdas.json"))\nA_M1 = 1.0 if SIN_A else 1 + SIG["motor_1"]["Z_tau0.25"] * (FUGA["motor_1"]["sin_nivel"]["vertice"] - 1)\ncels = ["motor_0", "motor_1", "buildings_0", "buildings_1", "contents_0", "contents_1"]\n# Fit factors on the local F2-plus-shape pool with fixed sum(mu).\npool = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]).copy()\npt = pool.copy(); pt["m"] = transformar(pool); pool["m"] = aplicar(pt, {"motor_1": A_M1}) if A_M1 != 1.0 else pt.m.values\nFAC = []\nfor v, lines in SEQ:\n    cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(pool, cl, lines, KC)\n    pool["m"] = aplica_tw(pool, f, cl, pool.m.values); FAC.append((v, cl, f))\ndef factores(t, m):\n    for v, cl, f in FAC: m = aplica_tw(t, f, cl, m)\n    return m\n# Local before/after factor-level ratio over F2 plus shape; weight both years equally.\nraz = {}\nfor k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:\n    s_ = sets[k_].copy(); tt = s_.copy(); tt["m"] = transformar(s_); m0 = aplicar(tt, {"motor_1": A_M1}) if A_M1 != 1.0 else tt.m.values\n    m1 = factores(s_, m0); c_ = celda(s_)\n    raz[k_] = {cc: kopt(s_.y.values[c_ == cc], m1[c_ == cc]) / kopt(s_.y.values[c_ == cc], m0[c_ == cc]) for cc in cels}\nRf = {cc: 0.5 * raz["DEV23_T2b"][cc] + 0.5 * np.mean([raz[k][cc] for k in ["HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]]) for cc in cels}\n# Credibility-weighted cell level.\nsc = {}\nfor r in json.load(open("recibos_reales_20261026.json")):\n    if r["sha256"] in nombre_sha:\n        tipo, cual = nombre_sha[r["sha256"]]; sc[tipo if cual is None else (tipo, cual)] = float(r["d2"])\nlv = resolver(sc)\nF2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); pass\nF = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); assert (F.row_id.values == F2.row_id.values).all()\nt = b.loc[F2.row_id].copy(); t["m"] = F2.prediction.values; t["c_fila"] = 1.0; c = celda(t)\nm = F2.prediction.values.copy()\nif A_M1 != 1.0:\n    i = c == "motor_1"; ms = aplicar(t.assign(m=m), {"motor_1": A_M1})\n    m[i] = ms[i] * (np.sqrt(m[i]).sum() / np.sqrt(ms[i]).sum()) ** 2     # Hold the cell\'s sum(sqrt(mu)) fixed.\nm = factores(t, m)\nfor cc in cels: m[c == cc] *= Rf[cc] * lv["cred"][cc] / lv["g"]\nout = F2.copy(); out["prediction"] = m\nassert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()\ntag = "F3b" + ("_sinfac" if SIN_F else "") + ("_sinforma" if SIN_A else "") + (f"_K{int(KC)}" if KC != 300 else "") + ("" if not seqarg else "_" + "-".join(SEQN))\np = f"exp26/{tag}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")\ninfo = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "K": KC, "a_motor_1": A_M1, "seq": [v for v, _ in SEQ],\n        "factores": {v: {f"{a}|{vv}": round(x, 4) for (a, g_, vv), x in f.items()} for v, cl, f in FAC},\n        "razon_nivel_factores": {k: round(v, 4) for k, v in Rf.items()}, "razon_por_serie": {k: {c_: round(v, 4) for c_, v in d.items()} for k, d in raz.items()},\n        "nivel_cred_rel_F": {k: round(v / lv["g"], 4) for k, v in lv["cred"].items()},\n        "total": round(float(m.sum()), 2), "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},\n        "total_F2_celda": {cc: round(float(F2.prediction.values[c == cc].sum()), 2) for cc in cels},\n        "total_F_celda": {cc: round(float(F.prediction.values[c == cc].sum()), 2) for cc in cels}}\njson.dump(info, open(f"exp26/{tag}_info.json", "w"), indent=1, ensure_ascii=False)\nprint(json.dumps({k: v for k, v in info.items() if k not in ("razon_por_serie",)}, indent=1, ensure_ascii=False))\n'


### Embedded source: `m017/construir_F4b.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F4b.py'] = '"""Reconstruct the historical F4b frequency-severity blend alternative.\nBlend transferred FSp geometrically using recorded line weights (0.25 motor,\n0.25 buildings, 0.5 contents), credible peril calibration n/(n+100), and fixed\ncell totals. Apply F2, FSp blend, motor new-business shape, residual factors,\nlocal level ratios and credible cell levels in that order. Preserve F2\'s multiplier.\nUsage: python construir_F4b.py [--w=m,b,c] [--sin-factores]\n"""\nimport sys, json, hashlib\nargv = sys.argv[:]; sys.argv = ["x", "--prueba"]\nimport numpy as np, pandas as pd\nexec(open("exp26/lib26.py").read())\nexec(open("construir_final.py").read())\nexec(open("exp26/fsp_blend_eval.py").read().split("SEQ = [")[0].split(\'exec(open("__RUN_ROOT__/m017/exp26/lib26.py").read())\')[1])\nsys.argv = argv\nwa = [a for a in sys.argv if a.startswith("--w=")]\nWM, WB, WC = [float(x) for x in wa[0][4:].split(",")] if wa else (0.25, 0.0, 0.5)\nWL = {"motor": WM, "buildings": WB, "contents": WC}\nSIN_F = "--sin-factores" in sys.argv\nSEQ = [] if SIN_F else [("gender", ["motor", "contents"]), ("works_from_home", ["contents"]), ("marketing_segment", ["contents"])]\nFUGA = json.load(open("exp26/fuga_celdas.json"))\nA_M1 = 1 + SIG["motor_1"]["Z_tau0.25"] * (FUGA["motor_1"]["sin_nivel"]["vertice"] - 1)\ncels = ["motor_0", "motor_1", "buildings_0", "buildings_1", "contents_0", "contents_1"]\n# Fit factors on the local F2, out-of-time FSp blend and shape pool.\nKAP_DEV = {2023: {l: kappas(l, [2024]) for l in WL}, 2024: {l: kappas(l, [2023]) for l in WL}}   # Peril calibration kappa from the other year.\ndef blend_local(t, m, mbase=None):\n    """Blend FSp geometrically while preserving the F2 multiplier m/mbase."""\n    mbase = t.m.values if mbase is None else mbase\n    out = m.copy(); yr = t.year.values.astype(int)\n    for y_ in (2023, 2024):\n        for l in WL:\n            i = (yr == y_) & (t.line.values == l)\n            if not i.any() or WL[l] == 0: continue\n            r = fsp_pred(t.index[i], l, KAP_DEV[y_][l]); r = r * mbase[i].sum() / r.sum()\n            out[i] = m[i] * (r / mbase[i]) ** WL[l]\n    c_ = celda(t)\n    for cc in np.unique(c_): j = c_ == cc; out[j] *= m[j].sum() / out[j].sum()\n    return out\ndef forma(t, m):\n    tt = t.copy(); tt["m"] = m; return aplicar(tt, {"motor_1": A_M1})\npool = pd.concat([sets["DEV23_T2b"], sets["HOLD24_T2b"]]).copy()\npool["m"] = forma(pool, blend_local(pool, transformar(pool)))\nFAC = []\nfor v, lines in SEQ:\n    cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(pool, cl, lines, 300)\n    pool["m"] = aplica_tw(pool, f, cl, pool.m.values); FAC.append((v, cl, f))\ndef factores(t, m):\n    for v, cl, f in FAC: m = aplica_tw(t, f, cl, m)\n    return m\n# Compare local level before (F2 plus shape) and after blend, shape and factors.\nraz = {}\nfor k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:\n    s_ = sets[k_].copy(); m2 = transformar(s_)\n    m0 = forma(s_, m2); m1 = factores(s_, forma(s_, m2)); c_ = celda(s_)   # Factor-only ratio; the historical blend ratio was treated as noise.\n    raz[k_] = {cc: kopt(s_.y.values[c_ == cc], m1[c_ == cc]) / kopt(s_.y.values[c_ == cc], m0[c_ == cc]) for cc in cels}\nR4 = {cc: 0.5 * raz["DEV23_T2b"][cc] + 0.5 * np.mean([raz[k][cc] for k in ["HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]]) for cc in cels}\n# Credibility-weighted level.\nsc = {}\nfor r in json.load(open("recibos_reales_20261026.json")):\n    if r["sha256"] in nombre_sha:\n        tipo, cual = nombre_sha[r["sha256"]]; sc[tipo if cual is None else (tipo, cual)] = float(r["d2"])\nlv = resolver(sc)\n# Production FSp uses developed 2021-2024, kappa from 2023-2024 and ultimate completeness 1.\nFP = {}\nfor l in WL:\n    f = pd.read_parquet(f"{T}/pred/fsp/{l}_prod.parquet").set_index("row_id"); per = [c[2:] for c in f.columns if c.startswith("f_")]\n    FP[l] = (f, per, FT[l][2], FT[l][3])\nKAP_P = {}\nfor l in WL:\n    FT_save = FT[l]; FT[l] = FP[l]; KAP_P[l] = kappas(l, [2023, 2024]); FT[l] = FT_save\nF2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); pass\nt = b.loc[F2.row_id].copy(); t["m"] = F2.prediction.values; t["c_fila"] = 1.0; c = celda(t)\nFb = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); assert (Fb.row_id.values == F2.row_id.values).all()\nm = F2.prediction.values.copy()\nfor l in WL:\n    i = (t.line == l).values\n    if WL[l] == 0: continue\n    f, per, _, _ = FP[l]; idx = t.index[i]\n    r = sum(KAP_P[l][p] * f.loc[idx, f"f_{p}"].values * f.loc[idx, f"s_{p}"].values for p in per)\n    mb = Fb.prediction.values[i]; r = r * mb.sum() / r.sum(); m[i] = m[i] * (r / mb) ** WL[l]\nfor cc in cels: j = c == cc; m[j] *= F2.prediction.values[j].sum() / m[j].sum()\ni = c == "motor_1"; ms = forma(t, m); m[i] = ms[i] * (np.sqrt(m[i]).sum() / np.sqrt(ms[i]).sum()) ** 2\nm = factores(t, m)\nfor cc in cels: m[c == cc] *= R4[cc] * lv["cred"][cc] / lv["g"]\nout = F2.copy(); out["prediction"] = m\nassert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()\ntag = "F4b" + ("_sinfac" if SIN_F else "") + ("" if not wa else "_w" + wa[0][4:].replace(",", "-"))\np = f"exp26/{tag}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")\ninfo = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "pesos_fsp": WL, "a_motor_1": A_M1, "seq": [v for v, _ in SEQ],\n        "kappa_prod": {l: {p_: round(v, 3) for p_, v in d.items()} for l, d in KAP_P.items()},\n        "razon_nivel": {k: round(v, 4) for k, v in R4.items()}, "razon_por_serie": {k: {c_: round(v, 4) for c_, v in d.items()} for k, d in raz.items()},\n        "nivel_cred_rel_F": {k: round(v / lv["g"], 4) for k, v in lv["cred"].items()},\n        "total": round(float(m.sum()), 2), "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},\n        "corr_log_F2": {cc: round(float(np.corrcoef(np.log(m[c == cc] / t.exposure.values[c == cc]), np.log(F2.prediction.values[c == cc] / t.exposure.values[c == cc]))[0, 1]), 4) for cc in cels}}\njson.dump(info, open(f"exp26/{tag}_info.json", "w"), indent=1, ensure_ascii=False)\nprint(json.dumps({k: v for k, v in info.items() if k != "razon_por_serie"}, indent=1, ensure_ascii=False))\n'


### Embedded source: `m017/construir_F3c.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F3c.py'] = '"""Reconstruct selected F3c and the historical F4c alternative from the credible base.\nF3c applies validated motor/contents gender and contents home-working/marketing\nresidual factors, preserving cell totals, then their equal-year local level ratio.\nFactors and ratios follow F3b. F4c additionally blends FSp using the frozen motor\n0.25/contents 0.5 weights while preserving F2\'s multiplier, then uses F4b factors.\nSelection is recorded separately; this script does not submit predictions.\nUsage: python construir_F3c.py F3c|F4c\n"""\nimport sys, json, hashlib\nQUE = sys.argv[1]\nimport numpy as np, pandas as pd\nsrc = open("construir_F3b.py" if QUE == "F3c" else "construir_F4b.py").read()\nsrc = src.split("# nivel creíble")[0] if QUE == "F3c" else src.split("# --- nivel creíble")[0]\nsys.argv = ["x"]\nexec(src)\nR = Rf if QUE == "F3c" else R4\nFC = "revision/J_final_creible_2025.csv"; pass\nfc = pd.read_csv(FC, dtype={"row_id": str})\nF2 = pd.read_csv("F2_ncb_previos_2025.csv", dtype={"row_id": str}); assert (fc.row_id.values == F2.row_id.values).all()\nt = b.loc[fc.row_id].copy(); t["m"] = fc.prediction.values; t["c_fila"] = 1.0; c = celda(t)\nm = fc.prediction.values.copy()\nif QUE == "F4c":\n    Fb = pd.read_csv("in/final_sonda_global_2025.csv", dtype={"row_id": str}); assert (Fb.row_id.values == fc.row_id.values).all()\n    FP = {}\n    for l in WL:\n        f = pd.read_parquet(f"{T}/pred/fsp/{l}_prod.parquet").set_index("row_id"); per = [c_[2:] for c_ in f.columns if c_.startswith("f_")]\n        FP[l] = (f, per, FT[l][2], FT[l][3])\n    KAP_P = {}\n    for l in WL:\n        sv = FT[l]; FT[l] = FP[l]; KAP_P[l] = kappas(l, [2023, 2024]); FT[l] = sv\n    for l in WL:\n        i = (t.line == l).values\n        if WL[l] == 0: continue\n        f, per, _, _ = FP[l]; idx = t.index[i]\n        r = sum(KAP_P[l][p] * f.loc[idx, f"f_{p}"].values * f.loc[idx, f"s_{p}"].values for p in per)\n        mb = Fb.prediction.values[i]; r = r * mb.sum() / r.sum(); m[i] = m[i] * (r / mb) ** WL[l]\n    for cc in cels: j = c == cc; m[j] *= fc.prediction.values[j].sum() / m[j].sum()\nm = factores(t, m)\nfor cc in cels: m[c == cc] *= R[cc]\nout = fc.copy(); out["prediction"] = m\nassert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()\np = f"exp26/{QUE}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")\ninfo = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "base": "final_creible 1a003d86… (público 0,16519)",\n        "pasos": ["mezcla FSp (autos 0,25, contenidos 0,5)"] * (QUE == "F4c") + ["factores " + ",".join(v for v, _, _ in FAC)],\n        "razon_nivel_factores": {k: round(v, 4) for k, v in R.items()}, "total": round(float(m.sum()), 2),\n        "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},\n        "total_celda_base": {cc: round(float(fc.prediction.values[c == cc].sum()), 2) for cc in cels}}\njson.dump(info, open(f"exp26/{QUE}_info.json", "w"), indent=1, ensure_ascii=False); print(json.dumps(info, indent=1, ensure_ascii=False))\n'


### Embedded source: `m017/construir_F3d.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['m017/construir_F3d.py'] = '"""Reconstruct the historical F3d sensitivity with weaker contents shrinkage.\nKeep F3c\'s motor gender K=300 and use K=60 by default for contents gender,\nhome-working and marketing. Apply local factor-level ratios with equal year\nweights. This preserves the archived alternative, not a new promotion rule.\nUsage: python construir_F3d.py [<contents_K>]\n"""\nimport sys, json, hashlib\nKCONT = float(sys.argv[1]) if len(sys.argv) > 1 else 60.0\nimport numpy as np, pandas as pd\nsys.argv = ["construir_F3b.py", "--sin-factores"]\nexec(open("construir_F3b.py").read().split("# nivel creíble")[0])\nSEQK = [("gender", ["motor"], 300.0), ("gender", ["contents"], KCONT), ("works_from_home", ["contents"], KCONT), ("marketing_segment", ["contents"], KCONT)]\nFAC = []\nfor v, lines, k in SEQK:\n    cl = (lambda vv: (lambda t: col(t, vv)))(v); f = fac_tw(pool, cl, lines, k)\n    pool["m"] = aplica_tw(pool, f, cl, pool.m.values); FAC.append((v, cl, f))\ndef factores(t, m):\n    for v, cl, f in FAC: m = aplica_tw(t, f, cl, m)\n    return m\nraz = {}\nfor k_ in ["DEV23_T2b", "HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]:\n    s_ = sets[k_].copy(); tt = s_.copy(); tt["m"] = transformar(s_); m0 = aplicar(tt, {"motor_1": A_M1}); m1 = factores(s_, m0); c_ = celda(s_)\n    raz[k_] = {cc: kopt(s_.y.values[c_ == cc], m1[c_ == cc]) / kopt(s_.y.values[c_ == cc], m0[c_ == cc]) for cc in cels}\nR = {cc: 0.5 * raz["DEV23_T2b"][cc] + 0.5 * np.mean([raz[k][cc] for k in ["HOLD24_T2b", "HOLD24_A1", "HOLD24_mezcla"]]) for cc in cels}\nFC = "revision/J_final_creible_2025.csv"; pass\nfc = pd.read_csv(FC, dtype={"row_id": str}); t = b.loc[fc.row_id].copy(); t["m"] = fc.prediction.values; t["c_fila"] = 1.0; c = celda(t)\nm = factores(t, fc.prediction.values.copy())\nfor cc in cels: m[c == cc] *= R[cc]\nout = fc.copy(); out["prediction"] = m; assert len(out) == 209063 and np.isfinite(m).all() and (m > 0).all()\ntag = "F3d" + ("" if KCONT == 60 else f"_Kc{int(KCONT)}")\np = f"exp26/{tag}_2025.csv"; out.to_csv(p, index=False, float_format="%.6f", lineterminator="\\n")\nf3c = pd.read_csv("exp26/F3c_2025.csv", dtype={"row_id": str})\ninfo = {"archivo": p, "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest(), "K_contenidos": KCONT, "K_autos": 300,\n        "factores": {f"{v}|{\',\'.join(sorted(set(a for (a, g_, vv) in f)))}": {f"{a}|{vv}": round(x, 4) for (a, g_, vv), x in f.items()} for v, cl, f in FAC},\n        "razon_nivel": {k: round(v, 4) for k, v in R.items()}, "total": round(float(m.sum()), 2),\n        "total_celda": {cc: round(float(m[c == cc].sum()), 2) for cc in cels},\n        "corr_log_F3c_contenidos": round(float(np.corrcoef(np.log(m[t.line.values == "contents"]), np.log(f3c.prediction.values[t.line.values == "contents"]))[0, 1]), 4),\n        "igual_F3c_fuera_de_contenidos": bool(np.allclose(m[t.line.values != "contents"], f3c.prediction.values[t.line.values != "contents"], rtol=2e-6))}\njson.dump(info, open(f"exp26/{tag}_info.json", "w"), indent=1, ensure_ascii=False); print(json.dumps(info, indent=1, ensure_ascii=False))\n'


### Embedded source: `aux/reproduccion_cadena.py`

This cell stores the code; a later cell materializes it inside the new run.


In [ ]:
SOURCES['aux/reproduccion_cadena.py'] = '"""Link the frozen historical reconstruction stages without searches or new scoring.\nTranscribe the recorded formulas and explicitly restore the global reporting\ncurve and pre-ID01 confirmation join. Compare historical outputs outside the\ncalculation process; never load old row predictions as model inputs.\n"""\nfrom __future__ import annotations\nimport json\nimport os\nimport shutil\nimport sys\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nWORK = Path(os.environ[\'RETRAIN_WORK\'])\nN = WORK / \'noche\'\nM = WORK / \'ml\'\nF = WORK / \'m017\'\nLINES = (\'motor\', \'buildings\', \'contents\')\n\n\ndef dump(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n\n\ndef curve():\n    # Start from conf_insumos and remove only its occurrence-year restriction\n    # to reconstruct the global REP-N2 reporting curve described in the ledger.\n    import reporte as RP\n    claims = pd.concat([RP.load(\'s1\'), RP.load(\'s2\')], ignore_index=True)\n    tmax = int(claims.window.max())\n    curves = {}\n    for peril, g in claims.groupby(\'peril\'):\n        weights = np.minimum(g.net.values, g.net.quantile(.995))\n        db = (g.delay.values // RP.B).astype(int)\n        wb = (g.window.values // RP.B).astype(int)\n        nb = tmax // RP.B + 1\n        d = np.bincount(db, weights=weights, minlength=nb)[:nb]\n        start = np.bincount(db, weights=weights, minlength=nb + 1)[:nb]\n        end = np.bincount(np.minimum(wb, nb - 1) + 1, weights=weights, minlength=nb + 1)[:nb]\n        risk = np.cumsum(start) - np.cumsum(end)\n        h = np.where(risk > 0, d / np.maximum(risk, 1e-9), 0)\n        survival = np.cumprod((1 - h)[::-1])[::-1]\n        curves[peril] = np.concatenate([survival[1:], [1.0]]).tolist()\n    dump(N / \'salidas/reporte_F.json\', {\'bucket_days\': RP.B, \'F_coste\': curves,\n         \'provenance\': \'Recorded REP-N2 reconstruction from the historical completeness formula without an occurrence-year filter.\'})\n\n\ndef ml_c1():\n    import receta\n    params = receta.parametros(\'pcv\', [\'T2b\'], \'ramo\')\n    t, values = receta.aplicar(\'prod\', params)\n    sample = pd.read_parquet(M / \'datos/emerging/sample_submission.parquet\')\n    out = sample[[\'row_id\']].merge(pd.DataFrame({\'row_id\': t.row_id.values, \'prediction\': values}), on=\'row_id\', how=\'left\')\n    assert len(out) == 209063 and out.prediction.notna().all() and out.row_id.is_unique\n    out.to_csv(M / \'privado/prediccion_2025_mlind.csv\', index=False, float_format=\'%.6f\', lineterminator=\'\\n\')\n    dump(M / \'privado/params_C1.json\', params)\n\n\ndef tag(line, seed, history=False):\n    if history:\n        suffix = \'_v2fixf1\' if line == \'motor\' else \'_v2f1\'\n        return \'dev21222324\' + suffix + (f\'s{seed}\' if seed != 11 else \'\')\n    suffix = f\'_v2fixs{seed}\' if line == \'motor\' else (\'_m02\' if seed == 11 else f\'_v2s{seed}\')\n    return \'dev21222324\' + suffix\n\n\ndef a1_post():\n    sample = pd.read_parquet(N / \'datos/s2/sample_submission.parquet\').row_id\n    transfer = N / \'salidas/transfer\'\n    def lg(line, history=False):\n        return pd.concat([pd.read_parquet(transfer / f\'lg_{tag(line, seed, history)}_{line}_em.parquet\')\n                          .set_index(\'row_id\').lg_dev for seed in (11, 12, 13)], axis=1).mean(axis=1)\n    def production(history):\n        parts = []\n        for line in LINES:\n            b = pd.read_parquet(N / f\'salidas/s2/base_{line}.parquet\')\n            g = np.exp(b.row_id.map(lg(line, history)).values)\n            train = (b.split == \'train\').values\n            test = (b.split == \'test\').values\n            level = b.t0.values[train].sum() / (b.eef_coste.values[train] * g[train]).sum()\n            parts.append(pd.DataFrame({\'row_id\': b.row_id.values[test], \'prediction\': b.exposure_model.values[test] * g[test] * level}))\n        return pd.concat(parts).set_index(\'row_id\').loc[sample].reset_index()\n    # The historical procedure reads its no-history reference rounded to six decimals.\n    baseline = production(False)\n    path = N / \'salidas/correccion/A1fixb3_2025.csv\'\n    baseline.to_csv(path, index=False, float_format=\'%.6f\', lineterminator=\'\\n\')\n    baseline = pd.read_csv(path, dtype={\'row_id\': str})\n    result = production(True)\n    prefix = result.row_id.str[0].values\n    for key in \'MBC\':\n        mask = prefix == key\n        result.loc[mask, \'prediction\'] = result.prediction.values[mask] * baseline.prediction.values[mask].sum() / result.prediction.values[mask].sum()\n    result.to_csv(N / \'salidas/correccion/A1fixF1b3n_2025.csv\', index=False, float_format=\'%.6f\', lineterminator=\'\\n\')\n    # forma_local consumes only A1 and R0; do not fit unused A2/A3 models.\n    held = []\n    for line in LINES:\n        b = pd.read_parquet(N / f\'salidas/conf/base_s2_{line}.parquet\')\n        g = pd.read_parquet(transfer / f\'lg_dev212223_v2conf_{line}_em.parquet\').set_index(\'row_id\').lg_dev\n        risk = np.exp(b.row_id.map(g).values)\n        fit = (b.year == 2023).values\n        test = (b.year == 2024).values\n        level = b.t0.values[fit].sum() / (b.eef_coste.values[fit] * risk[fit]).sum()\n        held.append(pd.DataFrame({\'row_id\': b.row_id.values[test], \'A1\': b.eef_coste.values[test] * risk[test] * level,\n                     \'R0\': b.eef_coste.values[test] * b.t0.values[fit].sum() / b.eef_coste.values[fit].sum()}))\n    pd.concat(held, ignore_index=True).to_parquet(N / \'salidas/correccion/predicciones_2024.parquet\', index=False)\n\n\ndef m017_inputs():\n    # Every row-level input is generated in this run, never loaded from historical outputs.\n    for phase in (\'dev\', \'hold\'):\n        for line in LINES:\n            target = F / f\'oof/pred/{phase}/T2b_{line}.parquet\'\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(M / f\'trabajo/pred/{phase}/T2b_{line}.parquet\', target)\n    shutil.copyfile(N / \'salidas/correccion/predicciones_2024.parquet\', F / \'in/predicciones_2024.parquet\')\n    for name in (\'customers\', \'addresses\', \'weather\', *(f\'policies_{l}\' for l in LINES), *(f\'test_{l}\' for l in LINES)):\n        target = F / f\'datos/emerging/{name}.parquet\'\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(M / f\'datos/emerging/{name}.parquet\', target)\n    columns = [\'row_id\', \'line\', \'conjunto\', \'year\', \'exposure\', \'new_business\', \'start_month\', \'customer_id\', \'y\', \'c_fila\', \'written_premium\']\n    labels = pd.concat([pd.read_parquet(M / f\'trabajo/base_emerging_{line}.parquet\', columns=columns) for line in LINES], ignore_index=True)\n    labels.to_parquet(F / \'base_etiquetas.parquet\', index=False)\n    a = pd.read_csv(N / \'salidas/correccion/A1fixF1b3n_2025.csv\', dtype={\'row_id\': str})\n    b = pd.read_csv(M / \'privado/prediccion_2025_mlind.csv\', dtype={\'row_id\': str})\n    assert a.row_id.equals(b.row_id)\n    blend = a.assign(prediction=.5 * a.prediction.values + .5 * b.prediction.values)\n    p = F / \'in/mezcla50.csv\'\n    blend.to_csv(p, index=False, float_format=\'%.6f\', lineterminator=\'\\n\')\n    # Frozen public constant from the original receipt; no new public probe or learning.\n    blend = pd.read_csv(p, dtype={\'row_id\': str})\n    g = json.loads((F / \'in/constantes_publicas.json\').read_text(encoding=\'utf-8\'))[\'factor_global\']\n    blend[\'prediction\'] = blend.prediction.values * g\n    blend.to_csv(F / \'in/final_sonda_global_2025.csv\', index=False, float_format=\'%.6f\', lineterminator=\'\\n\')\n\n\ndef final_alias():\n    shutil.copyfile(F / \'final_creible_2025.csv\', F / \'revision/J_final_creible_2025.csv\')\n\n\nif __name__ == \'__main__\':\n    {\'curve\': curve, \'ml_c1\': ml_c1, \'a1_post\': a1_post, \'m017_inputs\': m017_inputs, \'final_alias\': final_alias}[sys.argv[1]]()\n'


## Build the isolated workspace

I copy the official inputs into the new run and verify their hashes. The embedded stages run in isolated child processes whose Python audit hook blocks socket connections and child process creation. They can read the run and Python environment; they cannot write to the original inputs. Historical per-row predictions are never used as inputs. Each completed stage has a receipt and log.


In [ ]:
CONFIG = {'m017/in/preinscripcion.json': {'id': 'CLA-S2-SONDA-SEGMENTO-20261025', 'autor': 'AI-assisted project workflow, sesión de validación nocturna', 'pedido': 'El participante eligió «Final + diseñar sonda por segmento» el 25-10 tras la sonda global.', 'base': {'archivo': 'Repositorio_IAA2026/privado/semana2/primary_validacion_nocturna_20261023/sonda_global/final_sonda_global_2025.csv', 'sha256': '42659dbb4101d4f9a4ab00c287d77b2bd21d42be9734d2ca4a4074eb57c9add2', 'g': 1.187658238648521, 'd2_publico_previsto': 0.15948, 'mezcla': {'sha256': '12831ef4a05c1773824abe5fa139d9503dd2989b4874e7d93cfc000406cfdb4a', 'd2_publico': 0.1539}, 'beta_A': 0.6213270716132047, 'beta_B': 0.5079950966862582, 'c_optimo_global': 1.2230965922038046}, 'celdas': {'contents_0': {'cuota_raiz_mu': 0.03863, 'filas': 15806, 'Z': 0.534, 'sigma': 0.1869, 'sigma_agrupado_simulacion': 0.1859}, 'buildings_0': {'cuota_raiz_mu': 0.067202, 'filas': 12579, 'Z': 0.36, 'sigma': 0.2666, 'sigma_agrupado_simulacion': 0.2646}, 'motor_0': {'cuota_raiz_mu': 0.112814, 'filas': 21698, 'Z': 0.81, 'sigma': 0.0969, 'sigma_agrupado_simulacion': 0.0941}, 'contents_1': {'cuota_raiz_mu': 0.128667, 'filas': 50441, 'Z': 0.736, 'sigma': 0.1198, 'sigma_agrupado_simulacion': 0.1115}, 'buildings_1': {'cuota_raiz_mu': 0.221179, 'filas': 39716, 'Z': 0.724, 'sigma': 0.1236, 'sigma_agrupado_simulacion': 0.1196}, 'motor_1': {'cuota_raiz_mu': 0.431507, 'filas': 68823, 'Z': 0.866, 'sigma': 0.0786, 'sigma_agrupado_simulacion': 0.0666}}, 'd2_F_intervalo_redondeo': [0.1594743, 0.1594849], 'celda_deducida': 'motor_1', 'sondas': {'sonda_seg_contents_0.csv': {'celda': 'contents_0', 'K': 1.2, 'filas_escaladas': 15806, 'sha256': 'fb568cefdd6cf8908b5a53ed37ed211b5a984bfaea65f6d9031eea0cc9183168', 'total': 82241464.19}, 'sonda_seg_buildings_0.csv': {'celda': 'buildings_0', 'K': 1.2, 'filas_escaladas': 12579, 'sha256': '62373a316abbb97b869c9fb315bf257b1e69fa94dcb29ee91a2515ce63efe5fb', 'total': 83110161.16}, 'sonda_seg_motor_0.csv': {'celda': 'motor_0', 'K': 1.2, 'filas_escaladas': 21698, 'sha256': 'f76c3a6c53057008b0fdf242b4cb36312822b1a6721e6100dfaee961bc966d76', 'total': 83762225.51}, 'sonda_seg_contents_1.csv': {'celda': 'contents_1', 'K': 1.2, 'filas_escaladas': 50441, 'sha256': '131716eede0d7a652063b21b86018622853f224026583450170b7d9eda3455a1', 'total': 82952117.21}, 'sonda_seg_buildings_1.csv': {'celda': 'buildings_1', 'K': 1.2, 'filas_escaladas': 39716, 'sha256': '28a61e3a5281bb56bae77836f0edc40ad625f1a104ac6b1b4678ee0d6980de7f', 'total': 85852081.51}}, 'metodo': 'b_s = βB × cuota_s (Σμ^½ de la celda en el test); a_s = [D2(F) − D2(sonda_s) − b_s((gK)^½ − g^½)] / ((gK)^−½ − g^−½) para las cinco celdas sondeadas, y a_deducida = βA − Σ a_s. c*_s = a_s / b_s. Factor final por celda: g × acotar((c*_s / c*_g)^Z_s, 0,75, 1,33). Archivo final = mezcla × factor de su celda.', 'credibilidad': {'tau': 0.2, 'tau_motivo': 'Desviación logarítmica de cada celda frente al nivel global: en 2024, con el nivel de 2023, los ramos se desviaron +0,22, −0,36 y +0,08 del global (RMS 0,25), y el cociente nuevo negocio / renovación osciló entre 0,79 y 1,70 por ramo y año.', 'sigma': 'desviación típica de log(c*_s/c*_g) público menos privado, remuestreo de 2023 y 2024 al tamaño y composición de 2025 (150 réplicas por año)'}, 'orden_de_envio': 'Primero F solo; si su D2 público es compatible con el intervalo de redondeo, después las cinco sondas. Así una parada en F no gasta las sondas.', 'entrada_de_puntuaciones': 'Por nombre de archivo (o por recibo con SHA-256), nunca por nombre de celda, para que una puntuación no se asigne a la celda equivocada.', 'paradas': ['Si el intervalo [D2(F) observado ± 0,000005] no toca el intervalo previsto por el redondeo de las puntuaciones globales, no se resuelve: la métrica no tendría la forma supuesta.', 'Si a_s o b_s ≤ 0 en alguna celda, no se resuelve: el modelo implica a_s > 0, así que un valor no positivo indica un supuesto roto (público no aleatorio u otra métrica).', 'Si |log(c*_s / c*_g)| > max(0,7; 3 σ_s) en alguna celda, el resolutor se detiene sin escribir el archivo final y se consulta al participante.'], 'nota_Z': 'Z usa el mayor σ de los dos años de la simulación (sigma_por_pool.json), más conservador que el σ agrupado con que se simularon las ganancias de diseno.json.', 'nota_previa': 'El escenario «previa» sortea seis desviaciones independientes con el mismo τ que fija Z; por construcción favorece a las segmentaciones con más celdas.', 'simulacion_privado': {'nulo_2024': {'media': -0.00048, 'p05_p50_p95': [-0.00149, -0.0004, 0.00041], 'prob_ganar': 0.22, 'techo_oraculo_media': 0.00118}, 'nulo_2023': {'media': -0.00062, 'p05_p50_p95': [-0.00203, -0.00049, 0.00037], 'prob_ganar': 0.193, 'techo_oraculo_media': 0.00128}, 'previa_2024': {'media': 0.0026, 'p05_p50_p95': [-0.0005, 0.00212, 0.00636], 'prob_ganar': 0.88, 'techo_oraculo_media': 0.00472}, 'previa_2023': {'media': 0.00222, 'p05_p50_p95': [-0.00099, 0.00182, 0.00678], 'prob_ganar': 0.813, 'techo_oraculo_media': 0.00462}, 'como_2024': {'media': 0.01125, 'p05_p50_p95': [0.00893, 0.01112, 0.01418], 'prob_ganar': 1.0, 'techo_oraculo_media': 0.01552}, 'como_2023': {'media': 0.01753, 'p05_p50_p95': [0.01271, 0.01757, 0.02366], 'prob_ganar': 1.0, 'techo_oraculo_media': 0.02114}}, 'alternativas_simuladas': {'nb': {'2024_nulo': -0.00017, '2024_como_el_año': 0.0001, '2024_previa': 0.00062, '2023_nulo': -0.00025, '2023_como_el_año': 0.00526, '2023_previa': 0.00056}, 'ramo': {'2024_nulo': -0.00038, '2024_como_el_año': 0.0112, '2024_previa': 0.0011, '2023_nulo': -0.00042, '2023_como_el_año': 0.01104, '2023_previa': 0.00095}}, 'coste': 'el archivo final global, cinco sondas y el archivo final por celda: siete envíos; quedan ocho hoy y diez por día hasta el 30 de octubre', 'limites': 'Supone un público de un tercio al azar por filas; b_s aproximado por cuotas del test; σ medido con 2023 y 2024 remuestreados (el nuevo negocio de esos años tiene pocos siniestros grandes). Solo ajusta un nivel por celda, no el orden dentro de la celda. El envío lo hace reference con autorización expresa del participante; la selección final es del participante.'}, 'm017/sigma_forma.json': {'motor_1': {'sigma': 0.07, 'a_opt_media_remuestreo': 1.092, 'bandas_2025': {'l': 0.821, 'c': 0.102, 'm': 0.077}, 'Z_tau0.25': 0.927}, 'buildings_1': {'sigma': 0.51, 'a_opt_media_remuestreo': 0.976, 'bandas_2025': {'l': 0.882, 'c': 0.067, 'm': 0.052}, 'Z_tau0.25': 0.194}, 'contents_1': {'sigma': 0.081, 'a_opt_media_remuestreo': 1.339, 'bandas_2025': {'l': 0.855, 'c': 0.082, 'm': 0.063}, 'Z_tau0.25': 0.905}}, 'm017/sondas_F2_info.json': {'base': {'archivo': 'F2_ncb_previos_2025.csv', 'sha256': '119627d93003a98933d261e773ffaf6e29a7125346367f716a0a411b97dca11a'}, 'sondas': {'sonda_F2_motor_1_a06.csv': {'celda': 'motor_1', 'a': 0.6, 'razon_nivel': 0.9466, 'sha256': '66fb61402bacd3a3f5c6bd2aba9a0979c244d553694b65515d68bb2453f78a78', 'total_celda': 37803870.18, 'total_celda_F2': 39935317.58}, 'sonda_F2_motor_1_a08.csv': {'celda': 'motor_1', 'a': 0.8, 'razon_nivel': 0.9679, 'sha256': 'b393dd709f5ef6ed208d353726cc40de8738852b74911fcc66525769f9a918eb', 'total_celda': 38654638.65, 'total_celda_F2': 39935317.58}, 'sonda_F2_buildings_1_a07.csv': {'celda': 'buildings_1', 'a': 0.7, 'razon_nivel': 0.9131, 'sha256': '451ac68bae3148da9d898fec483f45ea994a822e7aa9febbdce91a66a420a8fc', 'total_celda': 17856249.55, 'total_celda_F2': 19554638.55}, 'sonda_F2_buildings_1_a13.csv': {'celda': 'buildings_1', 'a': 1.3, 'razon_nivel': 1.1652, 'sha256': '2dff0eb76100a4116d5b4517402d0219d492adf7e708d21d08df259e92f44e01', 'total_celda': 22784970.13, 'total_celda_F2': 19554638.55}, 'sonda_F2_contents_1_a07.csv': {'celda': 'contents_1', 'a': 0.7, 'razon_nivel': 0.9525, 'sha256': 'f2401cda8e7a9c0373148995c4ee16210174011b229c2c6802c09d8b76e020b4', 'total_celda': 4785276.96, 'total_celda_F2': 5023843.53}, 'sonda_F2_contents_1_a13.csv': {'celda': 'contents_1', 'a': 1.3, 'razon_nivel': 1.0965, 'sha256': '859e2226240f682330aa7e04a8bcaaa40a101590b61beb22736496171439c4ce', 'total_celda': 5508762.58, 'total_celda_F2': 5023843.53}}}, 'm017/recibos_reales_20261026.json': [{'sha256': '42659dbb4101d4f9a4ab00c287d77b2bd21d42be9734d2ca4a4074eb57c9add2', 'd2': 0.15948}, {'sha256': 'fb568cefdd6cf8908b5a53ed37ed211b5a984bfaea65f6d9031eea0cc9183168', 'd2': 0.15888}, {'sha256': '62373a316abbb97b869c9fb315bf257b1e69fa94dcb29ee91a2515ce63efe5fb', 'd2': 0.15804}, {'sha256': 'f76c3a6c53057008b0fdf242b4cb36312822b1a6721e6100dfaee961bc966d76', 'd2': 0.15898}, {'sha256': '131716eede0d7a652063b21b86018622853f224026583450170b7d9eda3455a1', 'd2': 0.1583}, {'sha256': '28a61e3a5281bb56bae77836f0edc40ad625f1a104ac6b1b4678ee0d6980de7f', 'd2': 0.15858}, {'sha256': '119627d93003a98933d261e773ffaf6e29a7125346367f716a0a411b97dca11a', 'd2': 0.16033}, {'sha256': '66fb61402bacd3a3f5c6bd2aba9a0979c244d553694b65515d68bb2453f78a78', 'd2': 0.15246}, {'sha256': 'b393dd709f5ef6ed208d353726cc40de8738852b74911fcc66525769f9a918eb', 'd2': 0.15705}, {'sha256': '451ac68bae3148da9d898fec483f45ea994a822e7aa9febbdce91a66a420a8fc', 'd2': 0.15894}, {'sha256': '2dff0eb76100a4116d5b4517402d0219d492adf7e708d21d08df259e92f44e01', 'd2': 0.1591}, {'sha256': 'f2401cda8e7a9c0373148995c4ee16210174011b229c2c6802c09d8b76e020b4', 'd2': 0.15958}, {'sha256': '859e2226240f682330aa7e04a8bcaaa40a101590b61beb22736496171439c4ce', 'd2': 0.16001}], 'm017/exp26/fuga_celdas.json': {'motor_1': {'K_estrella_F2': 1.185914884205038, 'a0.6': {'raiz_ratio': 0.99164, 'd2_nivel': -0.000388}, 'a0.8': {'raiz_ratio': 0.99437, 'd2_nivel': -0.000257}, 'original': {'vertice': 1.4008, 'curv': -0.01637, 'ganancia': 0.00263}, 'sin_nivel': {'vertice': 1.3212, 'curv': -0.01794, 'ganancia': 0.001851}}, 'buildings_1': {'K_estrella_F2': 1.0114807998102306, 'a0.7': {'raiz_ratio': 0.98962, 'd2_nivel': -2.8e-05}, 'a1.3': {'raiz_ratio': 1.02326, 'd2_nivel': -3.3e-05}, 'original': {'vertice': 1.0092, 'curv': -0.01456, 'ganancia': 1e-06}, 'sin_nivel': {'vertice': 1.0097, 'curv': -0.01422, 'ganancia': 1e-06}}, 'contents_1': {'K_estrella_F2': 0.910477421323969, 'a0.7': {'raiz_ratio': 1.00059, 'd2_nivel': -4e-06}, 'a1.3': {'raiz_ratio': 1.00836, 'd2_nivel': -5.8e-05}, 'original': {'vertice': 1.0603, 'curv': -0.00594, 'ganancia': 2.2e-05}, 'sin_nivel': {'vertice': 1.072, 'curv': -0.0056, 'ganancia': 2.9e-05}}}, 'm017/in/constantes_publicas.json': {'factor_global': 1.1876582386485208}}
INITIAL_STEPS = [{'id': 'ml_preparar', 'argv': ['preparar_datos.py']}, {'id': 'ml_historia', 'argv': ['historia.py']}, {'id': 't2b_dev_motor', 'argv': ['gbm.py', 'dev', 'ninguno', 'motor']}, {'id': 't2b_dev_buildings', 'argv': ['gbm.py', 'dev', 'ninguno', 'buildings']}, {'id': 't2b_dev_contents', 'argv': ['gbm.py', 'dev', 'ninguno', 'contents']}, {'id': 't2b_hold_motor', 'argv': ['gbm.py', 'hold', 'ninguno', 'motor']}, {'id': 't2b_hold_buildings', 'argv': ['gbm.py', 'hold', 'ninguno', 'buildings']}, {'id': 't2b_hold_contents', 'argv': ['gbm.py', 'hold', 'ninguno', 'contents']}, {'id': 't2b_pcv_motor', 'argv': ['gbm.py', 'pcv', 'ninguno', 'motor']}, {'id': 't2b_pcv_buildings', 'argv': ['gbm.py', 'pcv', 'ninguno', 'buildings']}, {'id': 't2b_pcv_contents', 'argv': ['gbm.py', 'pcv', 'ninguno', 'contents']}, {'id': 't2b_prod_motor', 'argv': ['gbm.py', 'prod', 'ninguno', 'motor']}, {'id': 't2b_prod_buildings', 'argv': ['gbm.py', 'prod', 'ninguno', 'buildings']}, {'id': 't2b_prod_contents', 'argv': ['gbm.py', 'prod', 'ninguno', 'contents']}]
CHAIN_STEPS = [{'id': 'ml_C1', 'family': 'ml', 'argv': ['@aux', 'ml_c1']}, {'id': 'noche_base_s1', 'family': 'noche', 'argv': ['base_mercado.py', 's1']}, {'id': 'noche_base_s2', 'family': 'noche', 'argv': ['base_mercado.py', 's2']}, {'id': 'noche_curva_global', 'family': 'noche', 'argv': ['@aux', 'curve']}, {'id': 'noche_ultimo_s1', 'family': 'noche', 'argv': ['ultimo_mercado.py', 's1']}, {'id': 'noche_ultimo_s2', 'family': 'noche', 'argv': ['ultimo_mercado.py', 's2']}, {'id': 'noche_conf_insumos', 'family': 'noche', 'argv': ['conf_insumos.py']}, {'id': 'A1_full_motor_11_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixs11', '--lr', '.02', '--pat', '400', '--seed', '11', '--sin', 'make_model']}, {'id': 'A1_full_buildings_11_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_m02', '--lr', '.02', '--pat', '400', '--seed', '11']}, {'id': 'A1_full_contents_11_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_m02', '--lr', '.02', '--pat', '400', '--seed', '11']}, {'id': 'A1_full_motor_12_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixs12', '--lr', '.02', '--pat', '400', '--seed', '12', '--sin', 'make_model']}, {'id': 'A1_full_buildings_12_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2s12', '--lr', '.02', '--pat', '400', '--seed', '12']}, {'id': 'A1_full_contents_12_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2s12', '--lr', '.02', '--pat', '400', '--seed', '12']}, {'id': 'A1_full_motor_13_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixs13', '--lr', '.02', '--pat', '400', '--seed', '13', '--sin', 'make_model']}, {'id': 'A1_full_buildings_13_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2s13', '--lr', '.02', '--pat', '400', '--seed', '13']}, {'id': 'A1_full_contents_13_base', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2s13', '--lr', '.02', '--pat', '400', '--seed', '13']}, {'id': 'A1_full_motor_11_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixf1', '--lr', '.02', '--pat', '400', '--seed', '11', '--sin', 'make_model', '--f1']}, {'id': 'A1_full_buildings_11_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2f1', '--lr', '.02', '--pat', '400', '--seed', '11', '--f1']}, {'id': 'A1_full_contents_11_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2f1', '--lr', '.02', '--pat', '400', '--seed', '11', '--f1']}, {'id': 'A1_full_motor_12_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixf1s12', '--lr', '.02', '--pat', '400', '--seed', '12', '--sin', 'make_model', '--f1']}, {'id': 'A1_full_buildings_12_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2f1s12', '--lr', '.02', '--pat', '400', '--seed', '12', '--f1']}, {'id': 'A1_full_contents_12_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2f1s12', '--lr', '.02', '--pat', '400', '--seed', '12', '--f1']}, {'id': 'A1_full_motor_13_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--sufijo', '_v2fixf1s13', '--lr', '.02', '--pat', '400', '--seed', '13', '--sin', 'make_model', '--f1']}, {'id': 'A1_full_buildings_13_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--sufijo', '_v2f1s13', '--lr', '.02', '--pat', '400', '--seed', '13', '--f1']}, {'id': 'A1_full_contents_13_F1', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--sufijo', '_v2f1s13', '--lr', '.02', '--pat', '400', '--seed', '13', '--f1']}, {'id': 'A1_conf_historico_motor', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'motor', '--años', '2021,2022,2023', '--sufijo', '_v2conf', '--lr', '.02', '--pat', '400', '--conf', '--seed', '11', '--sin', 'make_model'], 'legacy_id01': True}, {'id': 'A1_conf_historico_buildings', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'buildings', '--años', '2021,2022,2023', '--sufijo', '_v2conf', '--lr', '.02', '--pat', '400', '--conf', '--seed', '11'], 'legacy_id01': False}, {'id': 'A1_conf_historico_contents', 'family': 'noche', 'argv': ['desarrollado_gbm.py', 'contents', '--años', '2021,2022,2023', '--sufijo', '_v2conf', '--lr', '.02', '--pat', '400', '--conf', '--seed', '11'], 'legacy_id01': False}, {'id': 'A1_produccion_y_hold', 'family': 'noche', 'argv': ['@aux', 'a1_post']}, {'id': 'm017_insumos_generados', 'family': 'ml', 'argv': ['@aux', 'm017_inputs']}, {'id': 'm017_F2', 'family': 'm017', 'argv': ['construir_F2.py']}, {'id': 'm017_final_creible', 'family': 'm017', 'argv': ['construir_final.py', 'recibos_reales_20261026.json']}, {'id': 'm017_alias_J', 'family': 'm017', 'argv': ['@aux', 'final_alias']}, {'id': 'm017_F3c', 'family': 'm017', 'argv': ['construir_F3c.py', 'F3c']}]
BOOTSTRAP1 = "\nimport os,sys,runpy,json\nfrom pathlib import Path\nwork=Path(sys.argv[1]).resolve()\ncode=work/'ml/codigo'\nscript=code/sys.argv[2]\nsys.path.insert(0,str(code))\nallowed=[work,Path(sys.prefix).resolve(),Path(sys.base_prefix).resolve()]\ndef inside(p,root):return p==root or root in p.parents\ndef audit(event,args):\n    if event in ('socket.connect','socket.bind','socket.sendto','subprocess.Popen'):\n        raise PermissionError('Etapa sin red ni subprocesos: '+event)\n    if event=='open' and isinstance(args[0],(str,bytes,os.PathLike)):\n        path=Path(os.fsdecode(args[0])).resolve()\n        if not any(inside(path,root) for root in allowed):\n            raise PermissionError('Lectura externa al entorno limpio: '+str(path))\n        mode=args[1] or ''\n        flags=args[2] or 0\n        writing=any(c in str(mode) for c in 'wax+') or flags & (os.O_WRONLY|os.O_RDWR|os.O_CREAT|os.O_TRUNC)\n        if writing and (not inside(path,work) or inside(path,work/'ml/datos')):\n            raise PermissionError('Escritura fuera de salidas permitidas: '+str(path))\nsys.addaudithook(audit)\nsys.argv=[str(script),*sys.argv[3:]]\nrunpy.run_path(str(script),run_name='__main__')\n"
BOOTSTRAP2 = "\nimport sys,os,runpy\nfrom pathlib import Path\nwork=Path(sys.argv[1]).resolve();family=sys.argv[2]\ncode=work/'ml/codigo' if family=='ml' else work/family/'codigo' if family=='noche' else work/'m017'\nsys.path.insert(0,str(code))\nname=sys.argv[3];script=work/'aux/reproduccion_cadena.py' if name=='@aux' else code/name\nallowed=[work,Path(sys.prefix).resolve(),Path(sys.base_prefix).resolve()]\ndef inside(p,r):return p==r or r in p.parents\ndef hook(event,args):\n    if event in ('socket.connect','socket.bind','socket.sendto','subprocess.Popen'):raise PermissionError('Etapa sin red/subprocesos: '+event)\n    if event=='open' and isinstance(args[0],(str,bytes,os.PathLike)):\n        p=Path(os.fsdecode(args[0])).resolve()\n        if not any(inside(p,r) for r in allowed):raise PermissionError('Lectura ajena a la corrida limpia: '+str(p))\n        mode=args[1] or '';flags=args[2] or 0\n        wr=any(c in str(mode) for c in 'wax+') or flags&(os.O_WRONLY|os.O_RDWR|os.O_CREAT|os.O_TRUNC)\n        if wr and (not inside(p,work) or inside(p,work/'ml/datos') or inside(p,work/'noche/datos')):raise PermissionError('Escritura no permitida: '+str(p))\nsys.addaudithook(hook)\nsys.argv=[str(script),*sys.argv[4:]]\nrunpy.run_path(str(script),run_name='__main__')\n"


### Inspect the execution plan

I show the stage identifiers and arguments before running. T2b builds the developed-market backbone first; the continuation then builds A1 and the F3c calibration. Each identifier becomes a separate log and receipt. The plan contains no historical row-level predictions.


In [ ]:
plan_rows = [{"stage": step["id"], "family": "T2b", "arguments": " ".join(step["argv"])} for step in INITIAL_STEPS]
plan_rows += [{"stage": step["id"], "family": step["family"], "arguments": " ".join(step["argv"])} for step in CHAIN_STEPS]
print(f"Stages through F3c: {len(plan_rows)}")
display(pd.DataFrame(plan_rows))


In [ ]:
def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

for rel, source in SOURCES.items():
    dest = RUN / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    source = source.replace("__RUN_ROOT__", RUN.as_posix())
    compile(source, str(dest), "exec")
    dest.write_text(source, encoding="utf-8", newline="\n")
for rel, value in CONFIG.items():
    write_json(RUN / rel, value)
for market, folder, names in (("developed", developed, BASE_FILES), ("emerging", emerging, emerging_files)):
    for name in names:
        dest = RUN / "ml/datos" / market / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(folder / name, dest)
        assert digest(dest) == digest(folder / name)
        dest2 = RUN / "noche/datos" / ("s1" if market == "developed" else "s2") / name
        dest2.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(dest, dest2)
        assert digest(dest) == digest(dest2)
for directory in ("tmp", "logs", "receipts", "chain_logs", "chain_receipts", "ml/trabajo/modelos/dev", "ml/trabajo/modelos/prod", "ml/entrega", "ml/privado", "noche/salidas/conf", "noche/salidas/transfer", "noche/salidas/correccion", "m017/in", "m017/revision"):
    (RUN / directory).mkdir(parents=True, exist_ok=True)
(RUN / "ejecutar_etapa.py").write_text(BOOTSTRAP1, encoding="utf-8")
(RUN / "ejecutar_bloque.py").write_text(BOOTSTRAP2, encoding="utf-8")
write_json(RUN / "notebook_run.json", {"python": sys.version, "packages": installed, "input_directories": {"developed": str(developed), "emerging": str(emerging)}, "source_count": len(SOURCES), "utc": datetime.now(timezone.utc).isoformat()})
print("Workspace prepared with", len(SOURCES), "embedded sources")


## Run the frozen stages

I train the six T2b source models and their internal round-selection fits, then reconstruct A1 and the calibration chain through F3c. This is the chain needed for the selected F3c reference. A failure stops the notebook and points to the stage log; it is not silently converted to success.

This notebook starts a fresh run and does not automatically resume a stage after interruption. Preserve the logs and receipt for diagnosis. Any transfer of intermediate data or checkpoints is a separate operation subject to the competition's data rules.


In [ ]:
env = {k: v for k, v in os.environ.items() if k.upper() in ("SYSTEMROOT", "WINDIR", "PATH", "COMSPEC", "PATHEXT", "NUMBER_OF_PROCESSORS")}
env.update(RETRAIN_WORK=str(RUN), MLIND_RAIZ=str(RUN / "ml"), MLIND_BACKBONE="T2b", PYTHONUTF8="1", PYTHONDONTWRITEBYTECODE="1", TEMP=str(RUN / "tmp"), TMP=str(RUN / "tmp"), OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

def run_stage(name, command, cwd, log_dir, receipt_dir, extra_env=None):
    log = RUN / log_dir / (name + ".log")
    receipt = RUN / receipt_dir / (name + ".json")
    if receipt.exists():
        raise RuntimeError(f"Stage already has a receipt: {receipt}; preserve this run and start a fresh one")
    stage_env = dict(env, **(extra_env or {}))
    start = time.monotonic()
    print("Starting", name, flush=True)
    with log.open("wb") as stream:
        result = subprocess.run(command, cwd=cwd, env=stage_env, stdout=stream, stderr=subprocess.STDOUT)
    write_json(receipt, {"stage": name, "exit_code": result.returncode, "seconds": time.monotonic() - start, "log_sha256": digest(log)})
    if result.returncode:
        print(log.read_text(encoding="utf-8", errors="replace")[-8000:])
        raise RuntimeError(f"Stage {name} failed; see {log}")
    print("Completed", name, flush=True)

for step in INITIAL_STEPS:
    run_stage(step["id"], [sys.executable, "-I", "-X", "utf8", "-u", str(RUN / "ejecutar_etapa.py"), str(RUN), *step["argv"]], RUN / "ml/codigo", "logs", "receipts")
for step in CHAIN_STEPS:
    family = step["family"]
    cwd = RUN / ("ml/codigo" if family == "ml" else "noche/codigo" if family == "noche" else "m017")
    extra = {"A1_PRE_ID01": "1"} if step.get("legacy_id01") else {}
    run_stage(step["id"], [sys.executable, "-I", "-X", "utf8", "-u", str(RUN / "ejecutar_bloque.py"), str(RUN), family, *step["argv"]], cwd, "chain_logs", "chain_receipts", extra)


## Check and save the risk-premium CSV

I compare the newly generated F3c file with the historical SHA-256 and check the submission schema and IDs. A different hash can reflect environment or numerical differences; the checks do not establish equivalence on their own.


In [ ]:
f3c = RUN / "m017/exp26/F3c_2025.csv"
assert f3c.is_file(), f"Expected output missing: {f3c}"
pred = pd.read_csv(f3c, dtype={"row_id": str})
expected_ids = pd.read_parquet(emerging / "sample_submission.parquet", columns=["row_id"]).row_id.astype(str)
assert list(pred.columns) == ["row_id", "prediction"]
assert pred.row_id.is_unique and pred.row_id.equals(expected_ids)
assert np.isfinite(pred.prediction).all() and (pred.prediction > 0).all()
historical_sha = "5ab5e4e34290cf6ed6a7a792f59e1e55a4f056400bbbdf8401933afd236b3276"
actual_sha = digest(f3c)
print("F3c SHA-256:", actual_sha, "matches historical:", actual_sha == historical_sha)
if actual_sha != historical_sha:
    raise RuntimeError(f"F3c differs from the historical CSV. Preserve and investigate {f3c} before treating this as an exact reproduction.")
output = EXPORT / "submission_week2_F3c.csv"
shutil.copyfile(f3c, output)
print("Risk premium output:", output)


## From risk premium to commercial premium

For the twenty official reference policies, I add catastrophe reinsurance for home lines and the fixed expense per policy, then divide by one minus variable expenses, commission, premium tax and target margin. The risk prediction is already for the covered period, so I do not multiply it by exposure again. I use the literal fixed expense per policy as the main interpretation and show annual proration as a sensitivity.


In [ ]:
from decimal import Decimal, localcontext, ROUND_HALF_UP
reference = pd.read_parquet(emerging / "reference_policies.parquet")
expenses = pd.read_parquet(emerging / "expenses.parquet").set_index("item").emerging
refs = reference.merge(pred, on="row_id", how="left", validate="one_to_one")
assert len(refs) == 20 and refs.prediction.notna().all()
D = lambda x: Decimal(str(x))
E = lambda k: D(expenses[k])
commercial = []
with localcontext() as context:
    context.prec = 45
    for row in refs.itertuples(index=False):
        risk, exposure = D(row.prediction), D(row.exposure)
        commission = E("commission_" + row.channel)
        cat_rate = E("cat_reinsurance") if row.line in ("buildings", "contents") else D(0)
        denominator = 1 - E("variable_expense") - commission - E("premium_tax") - E("target_profit_margin")
        assert denominator > 0 and risk >= 0 and exposure > 0
        fixed = E("fixed_expense_per_policy")
        price = (risk * (1 + cat_rate) + fixed) / denominator
        prorated = (risk * (1 + cat_rate) + fixed * exposure) / denominator
        assert abs(price - (risk * (1 + cat_rate) + fixed + price * (1 - denominator))) < D("1e-30")
        commercial.append({"row_id": row.row_id, "line": row.line, "channel": row.channel, "exposure": float(exposure), "risk_premium": float(risk), "fixed_expense": float(fixed), "commercial_premium": str(price.quantize(D("0.01"), rounding=ROUND_HALF_UP)), "commercial_premium_prorated_fixed_sensitivity": str(prorated.quantize(D("0.01"), rounding=ROUND_HALF_UP))})
commercial_path = EXPORT / "commercial_reference_premiums.csv"
pd.DataFrame(commercial).to_csv(commercial_path, index=False)
print("Commercial reference calculations:", commercial_path)


## Outputs and troubleshooting

After the final checks pass, I find `submission_week2_F3c.csv` and `commercial_reference_premiums.csv` in the `EXPORT` directory printed near the beginning. Without an override, Kaggle exports to a run-specific subdirectory of `/kaggle/working`; local execution exports beneath the new run directory. I can set `IAA_NOTEBOOK_EXPORT` before starting to choose a persistent private destination. I keep row-level outputs private.

| If execution stops | What I check |
| --- | --- |
| Missing or ambiguous input directory | The explicit market path, required filenames and attached competition data |
| Missing library or version mismatch | The optional installation cell, kernel restart and reference environment |
| Failed model stage | The stage's log and receipt under the printed run directory |
| Final fingerprint mismatch | Environment and generated artifacts; I retain the file and investigate the difference |

For Week 2, a failed stage stops execution and leaves its log and receipt. This notebook does not automatically resume the chain. I preserve the failed run before starting a fresh one or investigating a separate recovery.

I do not bypass a failed fingerprint check or interpret a resumed run as an uninterrupted execution. A schema-valid CSV and an exact historical file match answer different questions.


## License and materials

My project code and documentation are covered by the MIT License below. Competition data and third-party packages retain their own terms and are not included in that license.

MIT License

Copyright (c) 2026 Manuel Rodríguez

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
